# Comprehensive Colonoscopy Benchmark: Repository FEE-Net vs. Classical Baselines
### Dual Tesla T4, Zero-CPU-Spike, Single-Source-of-Truth Pipeline

**Dataset**: Simula Kvasir-SEG (1000 cases), **repository** `splits.json` (Exp-027 v2):
150 zero-leakage independent test cases + 850-case 5-fold pool (Fold 0: 680 train / 170 val).

**Every model below is imported from the repository implementation** (the exact
`models/` and `utils/` sources are embedded into this notebook at generation time,
materialised under `/kaggle/working/repo/`, and imported as a real package):

| Model | Source file | Params |
|---|---|---|
| U-Net | `models/unet.py` | 7.763M |
| Attention U-Net | `models/attention_unet.py` | 7.852M |
| UNet++ | `models/unet_plus_plus.py` | 9.160M |
| **FEE-Net (Ours)** | `models/fee_net.py` | **8.539M** |
| **FEE-Net-EE (Ours)** | `models/fee_net_early_exit.py` | **8.552M** |

Training uses the repository compound loss (`utils/losses.py`): BCE + Dice +
0.1·Boundary-IoU + edge-head BCE, plus 0.3-weighted multi-scale deep supervision
(FEE-Net) and 0.5-weighted triage cross-entropy (EE) -- identical to `train.py`.
Metrics come from the repository `utils/metrics.py` (explicit `from_logits=True`).


In [ ]:
# Cell 2: Environment Setup, CPU Clamping & Dependency Pre-flight
import os, sys, glob, json, time, math, random, urllib.request, zipfile, ssl, subprocess
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- Zero-CPU-spike engineering: clamp every CPU thread pool to 1 ---
cv2.setNumThreads(1)
torch.set_num_threads(1)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

NUM_GPUS = torch.cuda.device_count()
DEV_LIST = [f"cuda:{i}" for i in range(NUM_GPUS)] or ["cpu"]
print(f"[Env] torch={torch.__version__} | GPUs={NUM_GPUS} {DEV_LIST} | threads={torch.get_num_threads()}")

# Optional FLOPs counter (falls back to the analytic table if unavailable)
try:
    from thop import profile as thop_profile
    THOP_OK = True
except Exception:
    try:
        os.system("pip install -q thop")
        from thop import profile as thop_profile
        THOP_OK = True
    except Exception:
        THOP_OK = False
        print("[Env] thop unavailable -> using analytic FLOPs table")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

OUTPUT_DIR = "/kaggle/working/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Protocol constant (single definition, mirrors train.py). A frame/case counts as
# lesion-bearing when its mask exceeds this many pixels.
TRIAGE_PIXEL_THRESHOLD = 10

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print("[Env] seed fixed (42) for reproducibility")


In [ ]:
# Cell 3: Materialise Repository `models/` and `utils/` + Embed the Authoritative Splits
REPO_ROOT = "/kaggle/working/repo"
REPO_SOURCES = {
  "models/unet.py": "\"\"\"\nStandard 4-Level U-Net for Medical Image Segmentation\nReference: Ronneberger et al., \"U-Net: Convolutional Networks for Biomedical\nImage Segmentation\", MICCAI 2015. https://arxiv.org/abs/1505.04597\n\nChannel specification:\n    Encoder:    [32, 64, 128, 256]\n    Bottleneck: 512\n    Decoder:    [256, 128, 64, 32]\n    Output:     1x1 Conv -> 1 class (raw logits)\n\"\"\"\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom typing import Optional\n\n\nclass DoubleConv(nn.Module):\n    \"\"\"\n    Two-stage convolutional block:\n    [Conv3x3 -> BatchNorm -> ReLU] x 2\n    \"\"\"\n    def __init__(self, in_channels: int, out_channels: int, mid_channels: Optional[int] = None):\n        super().__init__()\n        if mid_channels is None:\n            mid_channels = out_channels\n        self.double_conv = nn.Sequential(\n            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),\n            nn.BatchNorm2d(mid_channels),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),\n            nn.BatchNorm2d(out_channels),\n            nn.ReLU(inplace=True)\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.double_conv(x)\n\n\nclass Down(nn.Module):\n    \"\"\"\n    Downscaling stage: MaxPool2d(2) followed by DoubleConv block.\n    \"\"\"\n    def __init__(self, in_channels: int, out_channels: int):\n        super().__init__()\n        self.maxpool_conv = nn.Sequential(\n            nn.MaxPool2d(2),\n            DoubleConv(in_channels, out_channels)\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.maxpool_conv(x)\n\n\nclass Up(nn.Module):\n    \"\"\"\n    Upscaling stage: ConvTranspose2d (or Bilinear interpolation) followed by DoubleConv.\n    Supports odd spatial dimension alignment via padding.\n    \"\"\"\n    def __init__(self, in_channels: int, out_channels: int, bilinear: bool = False):\n        super().__init__()\n        self.bilinear = bilinear\n\n        if bilinear:\n            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)\n            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)\n        else:\n            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)\n            self.conv = DoubleConv(in_channels, out_channels)\n\n    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:\n        \"\"\"\n        Args:\n            x1: Feature map from previous decoder/bottleneck stage (to be upsampled).\n            x2: Skip connection feature map from corresponding encoder stage.\n        \"\"\"\n        x1 = self.up(x1)\n\n        # Handle potential spatial dimension discrepancies due to odd input shapes\n        diff_y = x2.size(2) - x1.size(2)\n        diff_x = x2.size(3) - x1.size(3)\n\n        if diff_x != 0 or diff_y != 0:\n            x1 = F.pad(x1, [diff_x // 2, diff_x - diff_x // 2,\n                            diff_y // 2, diff_y - diff_y // 2])\n\n        # Concatenate along channel dimension\n        x = torch.cat([x2, x1], dim=1)\n        return self.conv(x)\n\n\nclass UNet(nn.Module):\n    \"\"\"\n    Standard 4-level U-Net architecture for medical segmentation.\n\n    Args:\n        in_channels (int): Number of input image channels (e.g. 3 for RGB, 1 for Grayscale).\n        num_classes (int): Number of target segmentation classes (default: 1 for binary mask).\n        bilinear (bool): Whether to use bilinear interpolation instead of ConvTranspose2d.\n    \"\"\"\n    def __init__(self, in_channels: int = 3, num_classes: int = 1, bilinear: bool = False):\n        super().__init__()\n        self.in_channels = in_channels\n        self.num_classes = num_classes\n        self.bilinear = bilinear\n\n        # Encoder (channels: 32 -> 64 -> 128 -> 256)\n        self.inc = DoubleConv(in_channels, 32)\n        self.down1 = Down(32, 64)\n        self.down2 = Down(64, 128)\n        self.down3 = Down(128, 256)\n\n        # Bottleneck (channels: 512)\n        self.down4 = Down(256, 512)\n\n        # Decoder (channels: 256 -> 128 -> 64 -> 32)\n        self.up1 = Up(512, 256, bilinear=bilinear)\n        self.up2 = Up(256, 128, bilinear=bilinear)\n        self.up3 = Up(128, 64, bilinear=bilinear)\n        self.up4 = Up(64, 32, bilinear=bilinear)\n\n        # Final prediction head\n        self.outc = nn.Conv2d(32, num_classes, kernel_size=1)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        \"\"\"\n        Forward pass returning unnormalized logits.\n        Args:\n            x: Tensor of shape (B, C, H, W)\n        Returns:\n            Logits of shape (B, num_classes, H, W)\n        \"\"\"\n        # Encoder path with skip connections\n        x1 = self.inc(x)        # (B, 32, H, W)\n        x2 = self.down1(x1)     # (B, 64, H/2, W/2)\n        x3 = self.down2(x2)     # (B, 128, H/4, W/4)\n        x4 = self.down3(x3)     # (B, 256, H/8, W/8)\n\n        # Bottleneck\n        x5 = self.down4(x4)     # (B, 512, H/16, W/16)\n\n        # Decoder path with skip concatenation\n        d4 = self.up1(x5, x4)   # (B, 256, H/8, W/8)\n        d3 = self.up2(d4, x3)   # (B, 128, H/4, W/4)\n        d2 = self.up3(d3, x2)   # (B, 64, H/2, W/2)\n        d1 = self.up4(d2, x1)   # (B, 32, H, W)\n\n        logits = self.outc(d1)  # (B, num_classes, H, W)\n        return logits\n\n    def get_num_parameters(self) -> int:\n        \"\"\"Return total trainable parameter count.\"\"\"\n        return sum(p.numel() for p in self.parameters() if p.requires_grad)\n\n\nif __name__ == '__main__':\n    print(\"=\" * 60)\n    print(\"Self-test: Standard 4-Level U-Net\")\n    print(\"=\" * 60)\n\n    # Initialize model\n    model = UNet(in_channels=3, num_classes=1, bilinear=False)\n    params = model.get_num_parameters()\n    print(f\"Total Trainable Parameters: {params:,} ({params / 1e6:.2f} M)\")\n\n    # Test forward pass with batch of 2\n    dummy_input = torch.randn(2, 3, 256, 256)\n    logits = model(dummy_input)\n\n    print(f\"Input Shape:  {dummy_input.shape}\")\n    print(f\"Output Shape: {logits.shape}\")\n\n    # Shape assertion\n    assert logits.shape == (2, 1, 256, 256), f\"Shape mismatch: expected (2, 1, 256, 256), got {logits.shape}\"\n\n    # Test backward pass to verify gradients\n    loss = logits.sum()\n    loss.backward()\n    print(\"Backward pass verified successfully: gradients populated.\")\n\n    # Test arbitrary spatial dimension handling\n    dummy_odd = torch.randn(1, 3, 250, 250)\n    logits_odd = model(dummy_odd)\n    assert logits_odd.shape == (1, 1, 250, 250), f\"Odd dimension mismatch: got {logits_odd.shape}\"\n    print(f\"Odd dimension test (250x250) output shape: {logits_odd.shape}\")\n\n    print(\"=\" * 60)\n    print(\"ALL UNET SELF-TESTS PASSED!\")\n    print(\"=\" * 60)\n",
  "models/attention_unet.py": "\"\"\"\nAttention U-Net for Medical Image Segmentation\nReference: Oktay et al., \"Attention U-Net: Learning Where to Look for the Pancreas\",\nMIDL 2018 / arXiv:1804.03999. https://arxiv.org/abs/1804.03999\n\nChannel specification (identical to U-Net baseline for rigorous fair comparison):\n    Encoder:    [32, 64, 128, 256]\n    Bottleneck: 512\n    Decoder:    [256, 128, 64, 32]\n    Output:     1x1 Conv -> 1 class (raw logits)\n\"\"\"\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom typing import Optional, Tuple, List\n\n\nclass DoubleConv(nn.Module):\n    \"\"\"\n    Standard double convolution block:\n    [Conv3x3 -> BatchNorm -> ReLU] x 2\n    \"\"\"\n    def __init__(self, in_channels: int, out_channels: int, mid_channels: Optional[int] = None):\n        super().__init__()\n        if mid_channels is None:\n            mid_channels = out_channels\n        self.double_conv = nn.Sequential(\n            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),\n            nn.BatchNorm2d(mid_channels),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),\n            nn.BatchNorm2d(out_channels),\n            nn.ReLU(inplace=True)\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.double_conv(x)\n\n\nclass Down(nn.Module):\n    \"\"\"\n    Downscaling stage: MaxPool2d(2) followed by DoubleConv block.\n    \"\"\"\n    def __init__(self, in_channels: int, out_channels: int):\n        super().__init__()\n        self.maxpool_conv = nn.Sequential(\n            nn.MaxPool2d(2),\n            DoubleConv(in_channels, out_channels)\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.maxpool_conv(x)\n\n\nclass AttentionGate(nn.Module):\n    \"\"\"\n    Spatial Attention Gate (AG) Module (Oktay et al., MIDL 2018).\n    \n    Selectively suppresses irrelevant background regions while highlighting salient \n    lesion features passed through the skip connection.\n\n    Formulation:\n        alpha = Sigmoid( psi( ReLU( W_g(g) + W_x(x) ) ) )\n        x_att = x * alpha\n\n    Args:\n        in_channels_g: Number of channels in the gating signal (from decoder).\n        in_channels_x: Number of channels in the skip connection (from encoder).\n        inter_channels: Number of intermediate projection channels. Default: in_channels_x // 2.\n    \"\"\"\n    def __init__(self, in_channels_g: int, in_channels_x: int, inter_channels: Optional[int] = None):\n        super().__init__()\n        if inter_channels is None:\n            inter_channels = max(in_channels_x // 2, 1)\n\n        self.w_g = nn.Sequential(\n            nn.Conv2d(in_channels_g, inter_channels, kernel_size=1, stride=1, padding=0, bias=True),\n            nn.BatchNorm2d(inter_channels)\n        )\n        self.w_x = nn.Sequential(\n            nn.Conv2d(in_channels_x, inter_channels, kernel_size=1, stride=1, padding=0, bias=True),\n            nn.BatchNorm2d(inter_channels)\n        )\n        self.psi = nn.Sequential(\n            nn.Conv2d(inter_channels, 1, kernel_size=1, stride=1, padding=0, bias=True),\n            nn.BatchNorm2d(1),\n            nn.Sigmoid()\n        )\n        self.relu = nn.ReLU(inplace=True)\n\n    def forward(self, g: torch.Tensor, x: torch.Tensor, return_attention: bool = False):\n        \"\"\"\n        Args:\n            g: Gating signal from decoder stage, shape (B, F_g, H_g, W_g)\n            x: Skip feature map from encoder stage, shape (B, F_x, H_x, W_x)\n            return_attention: If True, also returns the attention map alpha (B, 1, H_x, W_x)\n        Returns:\n            x_att: Gated skip connection of shape (B, F_x, H_x, W_x)\n            alpha (optional): Attention coefficients\n        \"\"\"\n        # Ensure gating signal matches the spatial resolution of skip connection\n        if g.shape[2:] != x.shape[2:]:\n            g = F.interpolate(g, size=x.shape[2:], mode='bilinear', align_corners=True)\n\n        g1 = self.w_g(g)\n        x1 = self.w_x(x)\n        net = self.relu(g1 + x1)\n        alpha = self.psi(net)\n\n        x_att = x * alpha\n        if return_attention:\n            return x_att, alpha\n        return x_att\n\n\nclass AttentionUpBlock(nn.Module):\n    \"\"\"\n    Decoder stage with Attention Gate:\n    Upsample -> AttentionGate(g, x_skip) -> Concat([g, x_att]) -> DoubleConv\n    \"\"\"\n    def __init__(self, in_channels: int, out_channels: int, bilinear: bool = False):\n        super().__init__()\n        self.bilinear = bilinear\n        half_in = in_channels // 2\n\n        if bilinear:\n            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)\n            self.ag = AttentionGate(in_channels_g=in_channels, in_channels_x=half_in, inter_channels=half_in // 2)\n            self.conv = DoubleConv(in_channels + half_in, out_channels)\n        else:\n            self.up = nn.ConvTranspose2d(in_channels, half_in, kernel_size=2, stride=2)\n            self.ag = AttentionGate(in_channels_g=half_in, in_channels_x=half_in, inter_channels=half_in // 2)\n            self.conv = DoubleConv(in_channels, out_channels)\n\n    def forward(self, x: torch.Tensor, skip: torch.Tensor, return_attention: bool = False):\n        g = self.up(x)\n\n        # Pad if odd spatial resolution difference occurs\n        diff_y = skip.size(2) - g.size(2)\n        diff_x = skip.size(3) - g.size(3)\n        if diff_x != 0 or diff_y != 0:\n            g = F.pad(g, [diff_x // 2, diff_x - diff_x // 2,\n                          diff_y // 2, diff_y - diff_y // 2])\n\n        if return_attention:\n            skip_att, alpha = self.ag(g, skip, return_attention=True)\n            out = self.conv(torch.cat([g, skip_att], dim=1))\n            return out, alpha\n        else:\n            skip_att = self.ag(g, skip, return_attention=False)\n            out = self.conv(torch.cat([g, skip_att], dim=1))\n            return out\n\n\nclass AttentionUNet(nn.Module):\n    \"\"\"\n    Attention U-Net architecture for medical image segmentation.\n\n    Args:\n        in_channels (int): Number of input image channels (default: 3).\n        num_classes (int): Number of segmentation classes (default: 1).\n        bilinear (bool): Whether to use bilinear upsampling instead of ConvTranspose2d.\n    \"\"\"\n    def __init__(self, in_channels: int = 3, num_classes: int = 1, bilinear: bool = False):\n        super().__init__()\n        self.in_channels = in_channels\n        self.num_classes = num_classes\n        self.bilinear = bilinear\n\n        # Encoder (channels: 32 -> 64 -> 128 -> 256)\n        self.inc = DoubleConv(in_channels, 32)\n        self.down1 = Down(32, 64)\n        self.down2 = Down(64, 128)\n        self.down3 = Down(128, 256)\n\n        # Bottleneck (channels: 512)\n        self.down4 = Down(256, 512)\n\n        # Decoder with Attention Gates (channels: 256 -> 128 -> 64 -> 32)\n        self.up1 = AttentionUpBlock(512, 256, bilinear=bilinear)\n        self.up2 = AttentionUpBlock(256, 128, bilinear=bilinear)\n        self.up3 = AttentionUpBlock(128, 64, bilinear=bilinear)\n        self.up4 = AttentionUpBlock(64, 32, bilinear=bilinear)\n\n        # Segmentation Head\n        self.outc = nn.Conv2d(32, num_classes, kernel_size=1)\n\n    def forward(self, x: torch.Tensor, return_attention_maps: bool = False):\n        \"\"\"\n        Forward pass.\n        Args:\n            x: Input tensor (B, C, H, W)\n            return_attention_maps: If True, returns (logits, [alpha1, alpha2, alpha3, alpha4])\n        Returns:\n            Logits of shape (B, num_classes, H, W)\n        \"\"\"\n        # Encoder forward pass\n        x1 = self.inc(x)        # (B, 32, H, W)\n        x2 = self.down1(x1)     # (B, 64, H/2, W/2)\n        x3 = self.down2(x2)     # (B, 128, H/4, W/4)\n        x4 = self.down3(x3)     # (B, 256, H/8, W/8)\n\n        # Bottleneck\n        x5 = self.down4(x4)     # (B, 512, H/16, W/16)\n\n        # Decoder forward pass with attention gating\n        if return_attention_maps:\n            d4, a4 = self.up1(x5, x4, return_attention=True)\n            d3, a3 = self.up2(d4, x3, return_attention=True)\n            d2, a2 = self.up3(d3, x2, return_attention=True)\n            d1, a1 = self.up4(d2, x1, return_attention=True)\n            logits = self.outc(d1)\n            return logits, [a1, a2, a3, a4]\n        else:\n            d4 = self.up1(x5, x4)\n            d3 = self.up2(d4, x3)\n            d2 = self.up3(d3, x2)\n            d1 = self.up4(d2, x1)\n            logits = self.outc(d1)\n            return logits\n\n    def get_num_parameters(self) -> int:\n        \"\"\"Return total trainable parameter count.\"\"\"\n        return sum(p.numel() for p in self.parameters() if p.requires_grad)\n\n\nif __name__ == '__main__':\n    print(\"=\" * 60)\n    print(\"Self-test: Attention U-Net (Oktay et al., MIDL 2018)\")\n    print(\"=\" * 60)\n\n    # Initialize model\n    model = AttentionUNet(in_channels=3, num_classes=1, bilinear=False)\n    params = model.get_num_parameters()\n    print(f\"Total Trainable Parameters: {params:,} ({params / 1e6:.2f} M)\")\n\n    # Standard forward pass test\n    dummy_input = torch.randn(2, 3, 256, 256)\n    logits = model(dummy_input)\n\n    print(f\"Input Shape:  {dummy_input.shape}\")\n    print(f\"Output Shape: {logits.shape}\")\n\n    # Shape assertion\n    assert logits.shape == (2, 1, 256, 256), f\"Shape mismatch: expected (2, 1, 256, 256), got {logits.shape}\"\n\n    # Backward gradient flow test\n    loss = logits.sum()\n    loss.backward()\n    print(\"Backward pass verified successfully: gradients populated.\")\n\n    # Attention maps extraction test\n    logits, attn_maps = model(dummy_input, return_attention_maps=True)\n    print(f\"Extracted {len(attn_maps)} multi-scale Attention Maps:\")\n    for idx, att in enumerate(attn_maps):\n        print(f\"  Level {idx + 1} Attention Map Shape: {att.shape}, Min: {att.min().item():.3f}, Max: {att.max().item():.3f}\")\n\n    # Odd spatial dimension test\n    dummy_odd = torch.randn(1, 3, 250, 250)\n    logits_odd = model(dummy_odd)\n    assert logits_odd.shape == (1, 1, 250, 250), f\"Odd dimension mismatch: got {logits_odd.shape}\"\n    print(f\"Odd dimension test (250x250) output shape: {logits_odd.shape}\")\n\n    print(\"=\" * 60)\n    print(\"ALL ATTENTION U-NET SELF-TESTS PASSED!\")\n    print(\"=\" * 60)\n",
  "models/unet_plus_plus.py": "\"\"\"\nUNet++ (Nested U-Net) for Medical Image Segmentation\nReference:\n    Zhou et al., \"UNet++: Redesigning Skip Connections to Exploit\n    Multiscale Features in Image Segmentation\", IEEE Transactions on\n    Medical Imaging (IEEE TMI), 2020. https://arxiv.org/abs/1807.10165\n\nChannel Specification:\n    Base channels = 32\n    Level 0: 32   (x_{0,0}, x_{0,1}, x_{0,2}, x_{0,3}, x_{0,4})\n    Level 1: 64   (x_{1,0}, x_{1,1}, x_{1,2}, x_{1,3})\n    Level 2: 128  (x_{2,0}, x_{2,1}, x_{2,2})\n    Level 3: 256  (x_{3,0}, x_{3,1})\n    Level 4: 512  (x_{4,0})\n    Output:  1x1 Conv -> 1 class (raw logits)\n    Deep supervision: 4 prediction heads at Level 0 (final1, final2, final3, final4)\n\"\"\"\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom typing import Optional, List, Union\n\n\nclass DoubleConv(nn.Module):\n    \"\"\"\n    Standard two-stage convolutional block:\n    [Conv3x3 -> BatchNorm -> ReLU] x 2\n    \"\"\"\n    def __init__(self, in_channels: int, out_channels: int, mid_channels: Optional[int] = None):\n        super().__init__()\n        if mid_channels is None:\n            mid_channels = out_channels\n        self.double_conv = nn.Sequential(\n            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),\n            nn.BatchNorm2d(mid_channels),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),\n            nn.BatchNorm2d(out_channels),\n            nn.ReLU(inplace=True)\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.double_conv(x)\n\n\nclass UNetPlusPlus(nn.Module):\n    \"\"\"\n    Standard 4-level UNet++ (Nested U-Net) architecture with dense nested skip connections.\n\n    Features:\n        - Dense nested skip connections spanning 4 resolution levels (columns j=0..4).\n        - Optional deep supervision with 4 prediction heads along level 0.\n        - Bilinear interpolation with dynamic spatial alignment for odd/arbitrary input sizes.\n        - Trainable parameter count ~9.16M (comparable to Attention U-Net at 7.85M and U-Net at 7.76M).\n\n    Args:\n        in_channels (int): Number of input image channels (default: 3).\n        num_classes (int): Number of target segmentation classes (default: 1).\n        base_channels (int): Number of base feature channels (default: 32).\n        deep_supervision (bool): Whether to enable multi-head deep supervision (default: False).\n    \"\"\"\n    def __init__(\n        self,\n        in_channels: int = 3,\n        num_classes: int = 1,\n        base_channels: int = 32,\n        deep_supervision: bool = False\n    ):\n        super().__init__()\n        self.in_channels = in_channels\n        self.num_classes = num_classes\n        self.base_channels = base_channels\n        self.deep_supervision = deep_supervision\n\n        # Channel layout across 5 resolution stages: [32, 64, 128, 256, 512]\n        nb_filter = [\n            base_channels,\n            base_channels * 2,\n            base_channels * 4,\n            base_channels * 8,\n            base_channels * 16\n        ]\n\n        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)\n        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)\n\n        # -------------------------------------------------------------\n        # Column j = 0: Backbone Encoder\n        # -------------------------------------------------------------\n        self.conv0_0 = DoubleConv(in_channels, nb_filter[0])\n        self.conv1_0 = DoubleConv(nb_filter[0], nb_filter[1])\n        self.conv2_0 = DoubleConv(nb_filter[1], nb_filter[2])\n        self.conv3_0 = DoubleConv(nb_filter[2], nb_filter[3])\n        self.conv4_0 = DoubleConv(nb_filter[3], nb_filter[4])\n\n        # -------------------------------------------------------------\n        # Column j = 1: Dense Nested Skip Pathway 1\n        # -------------------------------------------------------------\n        self.conv0_1 = DoubleConv(nb_filter[0] + nb_filter[1], nb_filter[0])\n        self.conv1_1 = DoubleConv(nb_filter[1] + nb_filter[2], nb_filter[1])\n        self.conv2_1 = DoubleConv(nb_filter[2] + nb_filter[3], nb_filter[2])\n        self.conv3_1 = DoubleConv(nb_filter[3] + nb_filter[4], nb_filter[3])\n\n        # -------------------------------------------------------------\n        # Column j = 2: Dense Nested Skip Pathway 2\n        # -------------------------------------------------------------\n        self.conv0_2 = DoubleConv(nb_filter[0] * 2 + nb_filter[1], nb_filter[0])\n        self.conv1_2 = DoubleConv(nb_filter[1] * 2 + nb_filter[2], nb_filter[1])\n        self.conv2_2 = DoubleConv(nb_filter[2] * 2 + nb_filter[3], nb_filter[2])\n\n        # -------------------------------------------------------------\n        # Column j = 3: Dense Nested Skip Pathway 3\n        # -------------------------------------------------------------\n        self.conv0_3 = DoubleConv(nb_filter[0] * 3 + nb_filter[1], nb_filter[0])\n        self.conv1_3 = DoubleConv(nb_filter[1] * 3 + nb_filter[2], nb_filter[1])\n\n        # -------------------------------------------------------------\n        # Column j = 4: Dense Nested Skip Pathway 4\n        # -------------------------------------------------------------\n        self.conv0_4 = DoubleConv(nb_filter[0] * 4 + nb_filter[1], nb_filter[0])\n\n        # -------------------------------------------------------------\n        # Prediction Heads (Level 0)\n        # -------------------------------------------------------------\n        if self.deep_supervision:\n            self.final1 = nn.Conv2d(nb_filter[0], num_classes, kernel_size=1)\n            self.final2 = nn.Conv2d(nb_filter[0], num_classes, kernel_size=1)\n            self.final3 = nn.Conv2d(nb_filter[0], num_classes, kernel_size=1)\n            self.final4 = nn.Conv2d(nb_filter[0], num_classes, kernel_size=1)\n        else:\n            self.final = nn.Conv2d(nb_filter[0], num_classes, kernel_size=1)\n\n    def _upsample_like(self, src: torch.Tensor, target: torch.Tensor) -> torch.Tensor:\n        \"\"\"Upsample src feature map to match target spatial dimensions.\"\"\"\n        if src.shape[2:] != target.shape[2:]:\n            return F.interpolate(src, size=target.shape[2:], mode='bilinear', align_corners=True)\n        return src\n\n    def forward(\n        self,\n        x: torch.Tensor,\n        return_all: Optional[bool] = None\n    ) -> Union[torch.Tensor, List[torch.Tensor]]:\n        \"\"\"\n        Forward pass for UNet++.\n\n        Args:\n            x (torch.Tensor): Input tensor of shape (B, C, H, W).\n            return_all (bool, optional):\n                - If True: returns list of all 4 multi-scale output heads [out1, out2, out3, out4].\n                - If False: returns only the final head output out4.\n                - If None (default): returns list if deep_supervision=True and model.training=True;\n                  otherwise returns single logits tensor out4 (eval mode or deep_supervision=False).\n\n        Returns:\n            Single logits tensor of shape (B, num_classes, H, W),\n            or list of 4 tensors when deep supervision is active in training mode.\n        \"\"\"\n        # Column j = 0 (Backbone Encoder)\n        x0_0 = self.conv0_0(x)\n        x1_0 = self.conv1_0(self.pool(x0_0))\n        x2_0 = self.conv2_0(self.pool(x1_0))\n        x3_0 = self.conv3_0(self.pool(x2_0))\n        x4_0 = self.conv4_0(self.pool(x3_0))\n\n        # Column j = 1\n        x0_1 = self.conv0_1(torch.cat([x0_0, self._upsample_like(x1_0, x0_0)], dim=1))\n        x1_1 = self.conv1_1(torch.cat([x1_0, self._upsample_like(x2_0, x1_0)], dim=1))\n        x2_1 = self.conv2_1(torch.cat([x2_0, self._upsample_like(x3_0, x2_0)], dim=1))\n        x3_1 = self.conv3_1(torch.cat([x3_0, self._upsample_like(x4_0, x3_0)], dim=1))\n\n        # Column j = 2\n        x0_2 = self.conv0_2(torch.cat([x0_0, x0_1, self._upsample_like(x1_1, x0_0)], dim=1))\n        x1_2 = self.conv1_2(torch.cat([x1_0, x1_1, self._upsample_like(x2_1, x1_0)], dim=1))\n        x2_2 = self.conv2_2(torch.cat([x2_0, x2_1, self._upsample_like(x3_1, x2_0)], dim=1))\n\n        # Column j = 3\n        x0_3 = self.conv0_3(torch.cat([x0_0, x0_1, x0_2, self._upsample_like(x1_2, x0_0)], dim=1))\n        x1_3 = self.conv1_3(torch.cat([x1_0, x1_1, x1_2, self._upsample_like(x2_2, x1_0)], dim=1))\n\n        # Column j = 4\n        x0_4 = self.conv0_4(torch.cat([x0_0, x0_1, x0_2, x0_3, self._upsample_like(x1_3, x0_0)], dim=1))\n\n        if self.deep_supervision:\n            out1 = self.final1(x0_1)\n            out2 = self.final2(x0_2)\n            out3 = self.final3(x0_3)\n            out4 = self.final4(x0_4)\n\n            # In eval mode without explicit return_all=True, return final head logits (B, num_classes, H, W)\n            if not self.training and not return_all:\n                return out4\n\n            if return_all is False:\n                return out4\n\n            return [out1, out2, out3, out4]\n        else:\n            return self.final(x0_4)\n\n    def get_num_parameters(self) -> int:\n        \"\"\"Return total trainable parameter count.\"\"\"\n        return sum(p.numel() for p in self.parameters() if p.requires_grad)\n\n\n# Standard alias\nNestedUNet = UNetPlusPlus\n\n\nif __name__ == '__main__':\n    print(\"=\" * 65)\n    print(\"Self-test: UNet++ (Nested U-Net, Zhou et al., IEEE TMI 2020)\")\n    print(\"=\" * 65)\n\n    # 1. Initialize standard UNet++ (deep_supervision=False)\n    model = UNetPlusPlus(in_channels=3, num_classes=1, base_channels=32, deep_supervision=False)\n    params = model.get_num_parameters()\n    print(f\"Total Trainable Parameters: {params:,} ({params / 1e6:.2f} M)\")\n    assert 9_000_000 <= params <= 9_300_000, f\"Unexpected parameter count: {params}\"\n\n    # 2. Forward pass test in eval mode\n    model.eval()\n    dummy_input = torch.randn(2, 3, 256, 256)\n    with torch.no_grad():\n        logits = model(dummy_input)\n    print(f\"Input Shape:  {dummy_input.shape}\")\n    print(f\"Output Shape (eval mode): {logits.shape}\")\n    assert logits.shape == (2, 1, 256, 256), f\"Shape mismatch: expected (2, 1, 256, 256), got {logits.shape}\"\n\n    # 3. Backward pass test (gradient flow for deep_supervision=False)\n    model.train()\n    model.zero_grad()\n    logits_train = model(dummy_input)\n    loss = logits_train.sum()\n    loss.backward()\n    for name, p in model.named_parameters():\n        assert p.grad is not None, f\"Parameter {name} did not receive a gradient!\"\n        assert p.grad.norm().item() > 0, f\"Parameter {name} has zero gradient!\"\n    print(f\"Backward pass verified: 100% of parameters in model received non-zero gradients.\")\n\n    # 4. Deep supervision test in training mode\n    model_ds = UNetPlusPlus(in_channels=3, num_classes=1, base_channels=32, deep_supervision=True)\n    params_ds = model_ds.get_num_parameters()\n    print(f\"Deep Supervision Model Parameters: {params_ds:,} ({params_ds / 1e6:.2f} M)\")\n    assert 9_000_000 <= params_ds <= 9_300_000, f\"Unexpected parameter count: {params_ds}\"\n\n    model_ds.train()\n    model_ds.zero_grad()\n    outputs = model_ds(dummy_input)\n    assert isinstance(outputs, list), f\"Expected list output for deep supervision in training, got {type(outputs)}\"\n    assert len(outputs) == 4, f\"Expected 4 multi-scale outputs, got {len(outputs)}\"\n    for idx, out in enumerate(outputs):\n        assert out.shape == (2, 1, 256, 256), f\"Output {idx} shape mismatch: {out.shape}\"\n\n    loss_ds = sum(out.sum() for out in outputs)\n    loss_ds.backward()\n    for name, p in model_ds.named_parameters():\n        assert p.grad is not None, f\"Parameter {name} did not receive gradient in deep supervision!\"\n        assert p.grad.norm().item() > 0, f\"Parameter {name} has zero gradient in deep supervision!\"\n    print(\"Deep supervision verified in training mode: 4 heads returned and 100% gradients populated.\")\n\n    # 5. Deep supervision test in eval mode (should return single tensor)\n    model_ds.eval()\n    with torch.no_grad():\n        out_eval = model_ds(dummy_input)\n    assert isinstance(out_eval, torch.Tensor), f\"Expected Tensor in eval mode, got {type(out_eval)}\"\n    assert out_eval.shape == (2, 1, 256, 256), f\"Eval shape mismatch: {out_eval.shape}\"\n    print(f\"Deep supervision verified in eval mode: returned single tensor {out_eval.shape}.\")\n\n    # 6. Arbitrary / odd spatial dimension test\n    dummy_odd = torch.randn(1, 3, 250, 250)\n    logits_odd = model(dummy_odd)\n    assert logits_odd.shape == (1, 1, 250, 250), f\"Odd dimension mismatch: got {logits_odd.shape}\"\n    print(f\"Odd dimension test (250x250) verified: output shape {logits_odd.shape}.\")\n\n    print(\"=\" * 65)\n    print(\"ALL UNET++ SELF-TESTS PASSED!\")\n    print(\"=\" * 65)\n",
  "models/fee_net.py": "import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport sys\nimport os\n\n# Append current dir to sys path for standalone execution\nsys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))\n\ntry:\n    from models.modules.dwt_module import DWT_2D, WaveletHighFrequencyFusion, SpectralChannelAttention\n    from models.modules.egm_module import EdgeGatedModule, BERCA\nexcept ImportError:\n    # Fallback for direct execution\n    from modules.dwt_module import DWT_2D, WaveletHighFrequencyFusion, SpectralChannelAttention\n    from modules.egm_module import EdgeGatedModule, BERCA\n\n\nclass ConvBlock(nn.Module):\n    def __init__(self, in_channels, out_channels, stride=1):\n        super(ConvBlock, self).__init__()\n        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)\n        self.bn1 = nn.BatchNorm2d(out_channels)\n        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)\n        self.bn2 = nn.BatchNorm2d(out_channels)\n        self.relu = nn.ReLU(inplace=True)\n\n        self.downsample = None\n        if stride != 1 or in_channels != out_channels:\n            self.downsample = nn.Sequential(\n                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),\n                nn.BatchNorm2d(out_channels)\n            )\n\n    def forward(self, x):\n        identity = x\n        if self.downsample is not None:\n            identity = self.downsample(x)\n\n        out = self.relu(self.bn1(self.conv1(x)))\n        out = self.bn2(self.conv2(out))\n        out += identity\n        out = self.relu(out)\n        return out\n\n\nclass DilatedBottleneck(nn.Module):\n    def __init__(self, in_channels=None, out_channels=None, dilations=[2, 4], in_c=None, out_c=None):\n        super(DilatedBottleneck, self).__init__()\n        in_c = in_channels if in_channels is not None else in_c\n        out_c = out_channels if out_channels is not None else out_c\n        mid_channels = out_c // 2\n        self.conv1 = nn.Conv2d(in_c, mid_channels, kernel_size=3, padding=dilations[0], dilation=dilations[0], bias=False)\n        self.bn1 = nn.BatchNorm2d(mid_channels)\n        self.conv2 = nn.Conv2d(mid_channels, out_c, kernel_size=3, padding=dilations[1], dilation=dilations[1], bias=False)\n        self.bn2 = nn.BatchNorm2d(out_c)\n        self.relu = nn.ReLU(inplace=True)\n        self.shortcut = nn.Sequential(\n            nn.Conv2d(in_c, out_c, 1, bias=False),\n            nn.BatchNorm2d(out_c)\n        ) if in_c != out_c else nn.Identity()\n\n    def forward(self, x):\n        out = self.relu(self.bn1(self.conv1(x)))\n        out = self.bn2(self.conv2(out))\n        return self.relu(out + self.shortcut(x))\n\n\nclass BilinearUpBlock(nn.Module):\n    \"\"\"\n    Bilinear Upsampling + 1x1 Convolution Block.\n    Following Odena et al. (2016), bilinear interpolation followed by a 1x1\n    convolution is employed instead of transposed convolutions to avoid checkerboard\n    artifacts, reduce parameters, and improve boundary reconstruction metrics.\n    \"\"\"\n    def __init__(self, in_channels, out_channels):\n        super(BilinearUpBlock, self).__init__()\n        self.conv = nn.Sequential(\n            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),\n            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),\n            nn.BatchNorm2d(out_channels),\n            nn.ReLU(inplace=True)\n        )\n\n    def forward(self, x):\n        return self.conv(x)\n\n\nclass FEENet(nn.Module):\n    \"\"\"\n    FEE-Net: Frequency-Enhanced Edge Network for Endoscopic Polyp Segmentation.\n    \n    Architectural Modules:\n    1. Dynamic Channel Capacity:\n       Configured from base_channels (default: 48, baseline: 32):\n       c = self.base_channels\n       nb = [c, c*2, c*4, c*8, 512]\n       - Encoder: [nb[0], nb[1], nb[2], nb[3]], Bottleneck: nb[4] (512)\n       - Decoder: [nb[2], nb[1], nb[0], nb[0]]\n    2. Residual Dilated Bottleneck:\n       Equipped with shortcut residual connection preserving semantic context.\n    3. Bidirectional Edge-Region Co-Attention (BERCA in EGM):\n       - Directional Wavelet Filtering (DWF) via asymmetric 1x3 and 3x1 depthwise convolutions\n         attenuating isotropic specular glare artifacts.\n       - Region-to-Edge Semantic Guidance via deep decoder representations and supervised coarse\n         segmentation predictions.\n       - Learnable residual gate self.gamma (init 0.1) stabilizing early optimization.\n    4. Bilinear Upsampling Stages (Odena et al., 2016):\n       Reduces checkerboard artifacts and parameter count.\n    5. Multi-Scale Deep Supervision:\n       Auxiliary heads (deep_sup4, deep_sup3, deep_sup2) provide direct supervision signals.\n       Note: deep_sup3 simultaneously serves as the coarse prior guidance for EGM modules,\n       ensuring genuine ground-truth supervision over semantic gating.\n    6. Dual-mode Forward Interface:\n       - Training mode: returns (seg_logits, edge_logits) [or 3-tuple with deep supervision].\n       - Eval mode: returns seg_logits by default (or tuple if return_edge=True).\n    \"\"\"\n    def __init__(self, in_channels=3, num_classes=1, use_dwt=True, use_egm=True, base_channels=48):\n        super(FEENet, self).__init__()\n        self.use_dwt = use_dwt\n        self.use_egm = use_egm\n        self.base_channels = base_channels\n\n        c = self.base_channels\n        nb = [c, c * 2, c * 4, c * 8, 512]\n\n        self.stem = nn.Sequential(\n            nn.Conv2d(in_channels, nb[0], kernel_size=3, padding=1, bias=False),\n            nn.BatchNorm2d(nb[0]),\n            nn.ReLU(inplace=True)\n        )\n        self.layer1 = ConvBlock(nb[0], nb[0], stride=2)           # H/2, W/2\n        self.layer2 = ConvBlock(nb[0], nb[1], stride=2)           # H/4, W/4\n        self.layer3 = ConvBlock(nb[1], nb[2], stride=2)           # H/8, W/8\n        self.layer4 = ConvBlock(nb[2], nb[3], stride=2)           # H/16, W/16\n\n        self.bot = nn.Sequential(\n            nn.Conv2d(nb[3], 512, 3, padding=2, dilation=2, bias=False),\n            nn.BatchNorm2d(512),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(512, 512, 3, padding=4, dilation=4, bias=False),\n            nn.BatchNorm2d(512),\n            nn.ReLU(inplace=True)\n        )\n\n        if self.use_dwt:\n            self.dwt = DWT_2D()\n            self.wf1_attn = SpectralChannelAttention(nb[0] * 3)\n            self.wf1 = nn.Conv2d(nb[0] * 3, nb[0], kernel_size=1, bias=False)\n            self.wf2_attn = SpectralChannelAttention(nb[0] * 3)\n            self.wf2 = nn.Conv2d(nb[0] * 3, nb[0], kernel_size=1, bias=False)\n        else:\n            self.pool1 = nn.Conv2d(nb[0], nb[0], kernel_size=3, stride=2, padding=1)\n            self.pool2 = nn.Conv2d(nb[0], nb[0], kernel_size=3, stride=2, padding=1)\n\n        # Smooth Bilinear Upsampling Stages (Eliminates checkerboard artifacts & lowers HD95)\n        self.up4 = BilinearUpBlock(nb[4], nb[2])                  # 512 -> nb[2], H/8\n        self.dec4 = ConvBlock(nb[2] + nb[2], nb[2])               # nb[2] + nb[2] -> nb[2]\n\n        self.up3 = BilinearUpBlock(nb[2], nb[1])                  # nb[2] -> nb[1], H/4\n        self.dec3 = ConvBlock(nb[1] + nb[1], nb[1])               # nb[1] + nb[1] -> nb[1]\n\n        self.up2 = BilinearUpBlock(nb[1], nb[0])                  # nb[1] -> nb[0], H/2\n        if self.use_egm:\n            self.egm2 = EdgeGatedModule(dec_channels=nb[0], freq_channels=nb[0])\n        self.dec2 = ConvBlock(nb[0] + nb[0], nb[0])               # nb[0] + nb[0] -> nb[0]\n\n        self.up1 = BilinearUpBlock(nb[0], nb[0])                  # nb[0] -> nb[0], H\n        if self.use_egm:\n            self.egm1 = EdgeGatedModule(dec_channels=nb[0], freq_channels=nb[0])\n        self.dec1 = ConvBlock(nb[0] + nb[0], nb[0])               # nb[0] + nb[0] -> nb[0]\n\n        self.head_seg = nn.Conv2d(nb[0], num_classes, kernel_size=1)\n        self.head_edge = nn.Conv2d(nb[0], num_classes, kernel_size=1)\n\n        # Coarse prior head for EGM guidance\n        self.coarse_head = nn.Conv2d(nb[1], num_classes, kernel_size=1)\n\n        # Multi-scale Deep Supervision Auxiliary Heads\n        self.deep_sup4 = nn.Conv2d(nb[2], num_classes, kernel_size=1)        # H/8\n        self.deep_sup3 = nn.Conv2d(nb[1], num_classes, kernel_size=1)        # H/4 (aux head)\n        self.deep_sup2 = nn.Conv2d(nb[0], num_classes, kernel_size=1)        # H/2\n\n    @property\n    def bottleneck(self):\n        return self.bot\n\n    @bottleneck.setter\n    def bottleneck(self, val):\n        self.bot = val\n\n    @property\n    def aux_head(self):\n        return self.deep_sup3\n\n    @aux_head.setter\n    def aux_head(self, val):\n        self.deep_sup3 = val\n\n    def load_state_dict(self, state_dict, strict=True, assign=False):\n        \"\"\"\n        Robust state dict loader ensuring 100% backward compatibility across\n        all legacy and v2 checkpoints.\n        \"\"\"\n        model_state = self.state_dict()\n        adapted = {}\n        for k, v in state_dict.items():\n            if k.startswith(\"module.\"):\n                k = k[7:]\n            if k.endswith(\"total_ops\") or k.endswith(\"total_params\"):\n                continue\n\n            # Remap legacy bottleneck / conv1 format -> native bot.0, bot.1, bot.3, bot.4\n            if k.startswith(\"bottleneck.conv1.\"):\n                k = \"bot.0.\" + k[len(\"bottleneck.conv1.\"):]\n            elif k.startswith(\"bottleneck.bn1.\"):\n                k = \"bot.1.\" + k[len(\"bottleneck.bn1.\"):]\n            elif k.startswith(\"bottleneck.conv2.\"):\n                k = \"bot.3.\" + k[len(\"bottleneck.conv2.\"):]\n            elif k.startswith(\"bottleneck.bn2.\"):\n                k = \"bot.4.\" + k[len(\"bottleneck.bn2.\"):]\n            elif k.startswith(\"bottleneck.\"):\n                k = \"bot.\" + k[len(\"bottleneck.\"):]\n            elif k.startswith(\"bot.conv1.\"):\n                k = \"bot.0.\" + k[len(\"bot.conv1.\"):]\n            elif k.startswith(\"bot.bn1.\"):\n                k = \"bot.1.\" + k[len(\"bot.bn1.\"):]\n            elif k.startswith(\"bot.conv2.\"):\n                k = \"bot.3.\" + k[len(\"bot.conv2.\"):]\n            elif k.startswith(\"bot.bn2.\"):\n                k = \"bot.4.\" + k[len(\"bot.bn2.\"):]\n\n            # Remap wf1.conv1.weight -> native wf1.weight\n            if k == \"wf1.conv1.weight\":\n                k = \"wf1.weight\"\n            elif k == \"wf2.conv1.weight\":\n                k = \"wf2.weight\"\n\n            # Remap down -> downsample if needed\n            k_parts = k.split(\".\")\n            if len(k_parts) >= 3 and k_parts[1] == \"down\":\n                k_parts[1] = \"downsample\"\n                k = \".\".join(k_parts)\n\n            # Remap aux_head -> deep_sup3\n            if k.startswith(\"aux_head.\"):\n                k = \"deep_sup3.\" + k[len(\"aux_head.\"):]\n\n            if k in model_state:\n                tgt_shape = model_state[k].shape\n                if v.shape == tgt_shape:\n                    adapted[k] = v\n            else:\n                adapted[k] = v\n\n        if \"dwt.weight\" not in adapted and \"dwt.weight\" in model_state:\n            adapted[\"dwt.weight\"] = model_state[\"dwt.weight\"]\n\n        if not strict:\n            adapted = {k: v for k, v in adapted.items() if k in model_state and v.shape == model_state[k].shape}\n\n        return super(FEENet, self).load_state_dict(adapted, strict=strict, assign=assign)\n\n    def forward(self, x, clinical_prior=None, return_edge=False, return_aux=False):\n        \"\"\"\n        Forward pass.\n        Args:\n            x (Tensor): Input image batch of shape (B, C, H, W).\n            clinical_prior (optional): System 1 risk prior or positional boolean.\n            return_edge (bool): If True, returns (seg_logits, edge_logits) during eval.\n            return_aux (bool): If True, returns (seg_logits, edge_logits, aux_preds) with deep supervision.\n        \"\"\"\n        if isinstance(clinical_prior, bool):\n            return_edge = clinical_prior\n            clinical_prior = None\n\n        # 1. Encoder Feature Extraction\n        x_stem = self.stem(x)             # nb[0], H, W\n        e1 = self.layer1(x_stem)          # nb[0], H/2, W/2\n        e2 = self.layer2(e1)              # nb[1], H/4, W/4\n        e3 = self.layer3(e2)              # nb[2], H/8, W/8\n        e4 = self.layer4(e3)              # nb[3], H/16, W/16\n\n        b = self.bot(e4)                  # nb[4] (512), H/16, W/16\n\n        # 2. Wavelet High-Frequency Feature Extraction\n        if self.use_dwt:\n            LL_stem, LH_stem, HL_stem, HH_stem = self.dwt(x_stem)\n            cat_stem = torch.cat([LH_stem, HL_stem, HH_stem], dim=1)\n            cat_stem = self.wf1_attn(cat_stem) if hasattr(self, 'wf1_attn') else cat_stem\n            hf_1 = self.wf1(cat_stem)  # nb[0], H/2, W/2\n\n            LL_1, LH_1, HL_1, HH_1 = self.dwt(e1)\n            cat_1 = torch.cat([LH_1, HL_1, HH_1], dim=1)\n            cat_1 = self.wf2_attn(cat_1) if hasattr(self, 'wf2_attn') else cat_1\n            hf_2 = self.wf2(cat_1)          # nb[0], H/4, W/4\n        else:\n            hf_1 = self.pool1(x_stem)\n            hf_2 = self.pool2(e1)\n\n        # 3. Decoder with BERCA Co-Attention\n        # Level 4 (H/8)\n        d4 = self.up4(b)                  # nb[2], H/8, W/8\n        d4 = torch.cat([d4, e3], dim=1)   # nb[2] + nb[2]\n        d4 = self.dec4(d4)                # nb[2]\n\n        # Level 3 (H/4)\n        d3 = self.up3(d4)                 # nb[1], H/4, W/4\n        d3 = torch.cat([d3, e2], dim=1)   # nb[1] + nb[1]\n        d3 = self.dec3(d3)                # nb[1]\n\n        # coarse_head provides supervised coarse prior guidance to downstream EGMs\n        coarse_pred = self.coarse_head(d3)  # (B, 1, H/4, W/4)\n\n        # Level 2 (H/2)\n        d2 = self.up2(d3)                 # nb[0], H/2, W/2\n        if self.use_egm:\n            d2, attn2 = self.egm2(d2, hf_2, coarse_pred=coarse_pred)\n        else:\n            hf_2_up = F.interpolate(hf_2, size=d2.shape[2:], mode='bilinear', align_corners=False)\n            d2 = d2 + hf_2_up\n        d2 = torch.cat([d2, e1], dim=1)   # nb[0] + nb[0]\n        d2 = self.dec2(d2)                # nb[0]\n\n        # Level 1 (H)\n        d1 = self.up1(d2)                 # nb[0], H, W\n        hf_1_up = F.interpolate(hf_1, size=d1.shape[2:], mode='bilinear', align_corners=False)\n        if self.use_egm:\n            d1, attn1 = self.egm1(d1, hf_1_up, coarse_pred=coarse_pred)\n        else:\n            d1 = d1 + hf_1_up\n        d1 = torch.cat([d1, x_stem], dim=1) # nb[0] + nb[0]\n        d1 = self.dec1(d1)                # nb[0]\n\n        seg_logits = self.head_seg(d1)\n        edge_logits = self.head_edge(hf_1_up)\n\n        if clinical_prior is not None and torch.is_tensor(clinical_prior):\n            if clinical_prior.dim() == 2 and clinical_prior.size(1) == 1:\n                prior_weight = 1.0 + 0.2 * clinical_prior.view(-1, 1, 1, 1)\n                seg_logits = seg_logits * prior_weight\n\n        if return_aux:\n            aux_preds = [self.deep_sup4(d4), self.deep_sup3(d3), self.deep_sup2(d2)]\n            return seg_logits, edge_logits, aux_preds\n\n        if self.training or return_edge:\n            return seg_logits, edge_logits\n        return seg_logits\n\n\nif __name__ == '__main__':\n    print(\"=\" * 65)\n    print(\"Self-Test: FEENet with Base Channels 48 & BERCA Dual Co-Attention\")\n    print(\"=\" * 65)\n\n    model = FEENet(in_channels=3, num_classes=1, use_dwt=True, use_egm=True, base_channels=48)\n\n    # 1. Total Trainable Parameters\n    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)\n    print(f\"1. Total Trainable Parameters: {total_params:,} ({total_params / 1e6:.4f} M)\")\n    print(f\"   Base channels: {model.base_channels}\")\n    print(f\"   EGM1 initial gamma: {model.egm1.gamma.item():.4f}\")\n    print(f\"   EGM2 initial gamma: {model.egm2.gamma.item():.4f}\")\n    assert 8_300_000 < total_params < 8_700_000, \\\n        f\"Expected ~8.52M parameters (BERCA-expanded architecture, post Exp-014/023), got {total_params}\"\n\n    x = torch.randn(2, 3, 256, 256)\n\n    # 2. Training Mode Verification\n    print(\"\\n2. Testing Training Mode Forward & Backward:\")\n    model.train()\n    out_train_default = model(x)\n    assert isinstance(out_train_default, tuple) and len(out_train_default) == 2, \"Default train mode must return (seg_logits, edge_logits)\"\n\n    out_train_aux = model(x, return_aux=True)\n    assert isinstance(out_train_aux, tuple) and len(out_train_aux) == 3, \"Train mode with return_aux=True must return (seg, edge, aux_preds)\"\n    seg_train, edge_train, aux_train = out_train_aux\n    print(f\"   Train default: len={len(out_train_default)} | Train return_aux: len={len(out_train_aux)}, aux_count={len(aux_train)}\")\n    assert seg_train.shape == (2, 1, 256, 256)\n    assert edge_train.shape == (2, 1, 256, 256)\n    assert len(aux_train) == 3\n\n    # Backward Pass & Gradient Flow\n    loss = seg_train.sum() + edge_train.sum() + sum(aux.sum() for aux in aux_train)\n    loss.backward()\n\n    assert model.egm1.gamma.grad is not None, \"EGM1 gamma gradient must not be None\"\n    assert model.egm2.gamma.grad is not None, \"EGM2 gamma gradient must not be None\"\n    assert model.egm1.conv_dir_h[0].weight.grad is not None, \"EGM1 DWF 1x3 grad must not be None\"\n    assert model.egm1.conv_dir_v[0].weight.grad is not None, \"EGM1 DWF 3x1 grad must not be None\"\n    assert model.egm1.conv_dir_diag[0].weight.grad is not None, \"EGM1 DWF 3x3 diagonal grad must not be None\"\n    assert model.egm1.semantic_gate[0].weight.grad is not None, \"EGM1 semantic gate grad must not be None\"\n    assert model.wf1.weight.grad is not None, \"Wavelet fusion branch grad must not be None\"\n    assert model.wf1_attn.fc[0].weight.grad is not None, \"Wavelet spectral attention grad must not be None\"\n    assert model.stem[0].weight.grad is not None, \"Encoder stem grad must not be None\"\n    assert model.coarse_head.weight.grad is not None, \"Coarse head grad must not be None\"\n    assert model.deep_sup4.weight.grad is not None, \"Deep sup4 grad must not be None\"\n    assert model.deep_sup3.weight.grad is not None, \"Deep sup3 grad must not be None\"\n    assert model.deep_sup2.weight.grad is not None, \"Deep sup2 grad must not be None\"\n    assert model.bot[0].weight.grad is not None, \"Bottleneck conv1 grad must not be None\"\n    assert model.bot[1].weight.grad is not None, \"Bottleneck bn1 grad must not be None\"\n    if hasattr(model.bot, 'shortcut') and isinstance(model.bot.shortcut, nn.Sequential):\n        assert model.bot.shortcut[0].weight.grad is not None, \"Bottleneck shortcut grad must not be None\"\n\n    print(\"   Backward pass successful! Gradient flow confirmed across all branches:\")\n    print(f\"   - EGM1 gamma grad: {model.egm1.gamma.grad.item():.6f}\")\n    print(f\"   - EGM2 gamma grad: {model.egm2.gamma.grad.item():.6f}\")\n    print(f\"   - EGM1 DWF dir_h grad norm: {model.egm1.conv_dir_h[0].weight.grad.norm().item():.6f}\")\n    print(f\"   - EGM1 DWF dir_v grad norm: {model.egm1.conv_dir_v[0].weight.grad.norm().item():.6f}\")\n    print(f\"   - EGM1 DWF dir_diag grad norm: {model.egm1.conv_dir_diag[0].weight.grad.norm().item():.6f}\")\n    print(f\"   - WF1 Spectral Attention grad norm: {model.wf1_attn.fc[0].weight.grad.norm().item():.6f}\")\n    print(f\"   - EGM1 semantic gate grad norm: {model.egm1.semantic_gate[0].weight.grad.norm().item():.6f}\")\n    print(f\"   - Coarse head grad norm: {model.coarse_head.weight.grad.norm().item():.6f}\")\n\n    # 3. Eval Mode (Default: return_edge=False)\n    print(\"\\n3. Testing Eval Mode (default return_edge=False):\")\n    model.eval()\n    with torch.no_grad():\n        out_eval_default = model(x)\n        assert torch.is_tensor(out_eval_default), \"Eval mode (default) must return seg_logits Tensor directly\"\n        assert out_eval_default.shape == (2, 1, 256, 256)\n        print(f\"   Eval default output: {type(out_eval_default)} of shape {tuple(out_eval_default.shape)}\")\n\n    # 4. Eval Mode (Explicit return_edge=True)\n    print(\"\\n4. Testing Eval Mode (return_edge=True):\")\n    with torch.no_grad():\n        out_eval_edge = model(x, return_edge=True)\n        assert isinstance(out_eval_edge, tuple) and len(out_eval_edge) == 2, \"Eval mode with return_edge=True must return tuple\"\n        seg_eval, edge_eval = out_eval_edge\n        print(f\"   Eval return_edge=True: seg={tuple(seg_eval.shape)}, edge={tuple(edge_eval.shape)}\")\n\n        # Positional boolean compatibility test\n        out_pos = model(x, True)\n        assert isinstance(out_pos, tuple) and len(out_pos) == 2\n        out_false = model(x, False)\n        assert torch.is_tensor(out_false)\n\n    # 5. Ablation Variants Verification\n    print(\"\\n5. Testing Ablation Modes:\")\n    model_no_egm = FEENet(use_dwt=True, use_egm=False, base_channels=48)\n    p_no_egm = sum(p.numel() for p in model_no_egm.parameters() if p.requires_grad)\n    print(f\"   FEENet (w/o EGM) parameters: {p_no_egm:,} ({p_no_egm / 1e6:.4f} M)\")\n    out_no_egm = model_no_egm(x)\n    assert isinstance(out_no_egm, tuple), \"Train mode w/o EGM should return tuple\"\n\n    model_no_dwt = FEENet(use_dwt=False, use_egm=True, base_channels=48)\n    p_no_dwt = sum(p.numel() for p in model_no_dwt.parameters() if p.requires_grad)\n    print(f\"   FEENet (w/o DWT) parameters: {p_no_dwt:,} ({p_no_dwt / 1e6:.4f} M)\")\n    out_no_dwt = model_no_dwt(x)\n    assert isinstance(out_no_dwt, tuple), \"Train mode w/o DWT should return tuple\"\n\n    # 6. Dynamic Channel Verification (base_channels=32)\n    print(\"\\n6. Testing Dynamic Channels (base_channels=32):\")\n    model_32 = FEENet(base_channels=32)\n    p_32 = sum(p.numel() for p in model_32.parameters() if p.requires_grad)\n    print(f\"   FEENet (base_channels=32) parameters: {p_32:,} ({p_32 / 1e6:.4f} M)\")\n    out_32 = model_32(x)\n    assert out_32[0].shape == (2, 1, 256, 256)\n\n    # 7. Checkpoint Loading Verification\n    print(\"\\n7. Testing Checkpoint Loading Compatibility:\")\n    for ckpt_p in ['checkpoints/fee_net_best.pth', 'checkpoints/fee_net_v2_fold0_best.pth']:\n        if os.path.exists(ckpt_p):\n            ckpt = torch.load(ckpt_p, map_location='cpu', weights_only=False)\n            sd = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt\n            model_eval = FEENet(base_channels=48)\n            res = model_eval.load_state_dict(sd, strict=False)\n            print(f\"   Loaded {ckpt_p}: missing={len(res.missing_keys)}, unexpected={len(res.unexpected_keys)}\")\n            assert len(res.unexpected_keys) == 0, f\"Expected 0 unexpected keys, got {res.unexpected_keys}\"\n            # Architecture-drift guard: pre-BERCA (6.90M-era) checkpoints carry\n            # legacy bottleneck keys and WILL have missing keys under the current\n            # 8.52M architecture. Surface it loudly instead of partial-loading silently.\n            if len(res.missing_keys) > 0:\n                print(f\"   [WARN] {len(res.missing_keys)} keys missing -> checkpoint was trained \"\n                      f\"on an older architecture (params_m={ckpt.get('params_m')}); \"\n                      f\"weights are NOT fully compatible with the current model!\")\n            with torch.no_grad():\n                model_eval.eval()\n                test_out = model_eval(x)\n                assert test_out.shape == (2, 1, 256, 256)\n\n    print(\"\\n\" + \"=\" * 65)\n    print(\"ALL FEENet ARCHITECTURAL SELF-TEST CHECKS PASSED PERFECTLY!\")\n    print(\"=\" * 65)\n\n",
  "models/fee_net_early_exit.py": "# -*- coding: utf-8 -*-\n\"\"\"\nFEE-Net Early-Exit (FEENet-EE): Unified Shared-Backbone Deep Fusion Architecture\n================================================================================\nEliminates the decoupled dual-backbone cascade pipeline by integrating clinical\nscreening (System 1 triage) directly into the shared multi-scale frequency backbone.\n\nArchitectural Innovations:\n1. Shared Feature & Frequency Extraction:\n   - Raw input (B, 3, 256, 256) passes through shared Stem and DWT.\n   - Wavelet decomposition (LH, HL, HH) is computed exactly once at the Stem output.\n   - The resulting directional high-frequency feature (hf_1) simultaneously serves:\n     (a) Frequency roughness and optical specular artifact estimation in the Early-Exit Triage Head;\n     (b) Boundary-aware cross-attention guidance in the downstream Edge-Gated Modules (BERCA/EGM).\n2. Shared-Backbone Early-Exit Branch:\n   - At Layer 1 (H/2 = 128x128), features are pooled and evaluated by an ultra-compact Triage Head.\n   - Normal mucosal cases with clear optical quality (risk < tau) exit immediately,\n     bypassing Layers 2-4, the bottleneck, and all decoder stages (~1.45 GFLOPs vs 12.44 GFLOPs, ~88.3% compute saved).\n   - Ambiguous or high-risk cases proceed seamlessly through the full segmentation decoder.\n3. End-to-End Joint Multi-Task Optimization:\n   - L_total = L_seg + lambda_edge * L_edge + lambda_triage * L_triage + lambda_aux * L_aux\n   - Guarantees feature alignment and eliminates training/inference discrepancies.\n\"\"\"\n\nimport math\nimport sys\nimport os\nfrom typing import Dict, Any, List, Optional, Tuple, Union\n\n# Append current dir to sys path for standalone execution\nsys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\ntry:\n    from models.fee_net import ConvBlock, DilatedBottleneck, BilinearUpBlock\n    from models.modules.dwt_module import DWT_2D, WaveletHighFrequencyFusion, SpectralChannelAttention\n    from models.modules.egm_module import EdgeGatedModule\nexcept ImportError:\n    from fee_net import ConvBlock, DilatedBottleneck, BilinearUpBlock\n    from modules.dwt_module import DWT_2D, WaveletHighFrequencyFusion, SpectralChannelAttention\n    from modules.egm_module import EdgeGatedModule\n\n\nclass EarlyExitTriageHead(nn.Module):\n    \"\"\"\n    Compact Early-Exit Classification Head attached to Layer 1 features.\n    Upgraded to Dual-Branch Local-Peak Pooling (GAP + GMP):\n    Concatenates Global Average Pooling (spatial & frequency representation) with\n    Global Max Pooling (local peak sensitivity for diminutive <5mm focal polyps)\n    to eliminate false negative bypass risks.\n    \"\"\"\n    def __init__(self, in_channels: int, hidden_dim: int = 64, num_classes: int = 2):\n        super(EarlyExitTriageHead, self).__init__()\n        self.in_channels = in_channels\n        self.hidden_dim = hidden_dim\n        self.gap = nn.AdaptiveAvgPool2d(1)\n        self.gmp = nn.AdaptiveMaxPool2d(1)\n        \n        # Ingest concatenated spatial (avg+max) and wavelet energy (avg+max) features: in_channels * 4\n        self.mlp = nn.Sequential(\n            nn.Linear(in_channels * 4, hidden_dim),\n            nn.LayerNorm(hidden_dim),\n            nn.ReLU(inplace=True),\n            nn.Dropout(0.2),\n            nn.Linear(hidden_dim, num_classes)\n        )\n        # Auxiliary head for view quality (0: CLEAR, 1: AMBIGUOUS_GLARE)\n        self.quality_head = nn.Linear(hidden_dim, 2)\n\n    def _load_from_state_dict(self, state_dict, prefix, local_metadata, strict, missing_keys, unexpected_keys, error_msgs):\n        \"\"\"\n        Seamless backward compatibility for legacy checkpoints trained with 2C GAP input.\n        Expands (hidden_dim, 2C) weight to (hidden_dim, 4C) by splitting spatial & frequency\n        weights equally between GAP and GMP branches.\n        \"\"\"\n        key = prefix + \"mlp.0.weight\"\n        if key in state_dict:\n            param = state_dict[key]\n            if param.shape[1] == self.in_channels * 2 and self.mlp[0].weight.shape[1] == self.in_channels * 4:\n                c = self.in_channels\n                w_sp = param[:, :c]\n                w_fq = param[:, c:]\n                state_dict[key] = torch.cat([w_sp * 0.5, w_sp * 0.5, w_fq * 0.5, w_fq * 0.5], dim=1)\n        super()._load_from_state_dict(state_dict, prefix, local_metadata, strict, missing_keys, unexpected_keys, error_msgs)\n\n    def forward(self, e1: torch.Tensor, hf_1: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:\n        \"\"\"\n        Args:\n            e1: Spatial features from Layer 1, shape (B, C, H/2, W/2).\n            hf_1: High-frequency wavelet features from wf1, shape (B, C, H/2, W/2).\n        Returns:\n            triage_logits: (B, 2) classification logits (Normal vs Lesion).\n            quality_logits: (B, 2) optical quality logits (Clear vs Glare).\n            hf_energy: (B, 1) scalar high-frequency energy.\n        \"\"\"\n        b, c, _, _ = e1.shape\n        f_spatial_avg = self.gap(e1).view(b, c)\n        f_spatial_max = self.gmp(e1).view(b, c)\n        \n        # High-frequency wavelet energy estimate\n        hf_energy_map = torch.mean(hf_1 ** 2, dim=1, keepdim=True)  # (B, 1, H/2, W/2)\n        f_freq_avg = self.gap(hf_1).view(b, c)\n        f_freq_max = self.gmp(hf_1).view(b, c)\n        \n        feat = torch.cat([f_spatial_avg, f_spatial_max, f_freq_avg, f_freq_max], dim=1)  # (B, 4C)\n        \n        # MLP forward\n        h = self.mlp[0](feat)\n        h = self.mlp[1](h)\n        h = self.mlp[2](h)\n        h_drop = self.mlp[3](h)\n        triage_logits = self.mlp[4](h_drop)\n        quality_logits = self.quality_head(h)\n        \n        scalar_energy = self.gap(hf_energy_map).view(b, 1)\n        return triage_logits, quality_logits, scalar_energy\n\n\nclass FEENetEarlyExit(nn.Module):\n    \"\"\"\n    Unified FEE-Net with Shared-Backbone Early-Exit Mechanism and Gradient Decoupling.\n    \"\"\"\n    def __init__(\n        self,\n        in_channels: int = 3,\n        num_classes: int = 1,\n        use_dwt: bool = True,\n        use_egm: bool = True,\n        base_channels: int = 48,\n        detach_triage_feature: bool = True\n    ):\n        super(FEENetEarlyExit, self).__init__()\n        self.use_dwt = use_dwt\n        self.use_egm = use_egm\n        self.base_channels = base_channels\n        self.detach_triage_feature = detach_triage_feature\n\n        c = self.base_channels\n        nb = [c, c * 2, c * 4, c * 8, 512]\n\n        # 1. Shared Encoder Front-End\n        self.stem = nn.Sequential(\n            nn.Conv2d(in_channels, nb[0], kernel_size=3, padding=1, bias=False),\n            nn.BatchNorm2d(nb[0]),\n            nn.ReLU(inplace=True)\n        )\n        self.layer1 = ConvBlock(nb[0], nb[0], stride=2)           # H/2, W/2\n\n        # 2. Unified Frequency Extraction (Computed ONCE at stem)\n        if self.use_dwt:\n            self.dwt = DWT_2D()\n            self.wf1_attn = SpectralChannelAttention(nb[0] * 3)\n            self.wf1 = nn.Conv2d(nb[0] * 3, nb[0], kernel_size=1, bias=False)\n            self.wf2_attn = SpectralChannelAttention(nb[0] * 3)\n            self.wf2 = nn.Conv2d(nb[0] * 3, nb[0], kernel_size=1, bias=False)\n        else:\n            self.pool1 = nn.Conv2d(nb[0], nb[0], kernel_size=3, stride=2, padding=1)\n            self.pool2 = nn.Conv2d(nb[0], nb[0], kernel_size=3, stride=2, padding=1)\n\n        # 3. Early-Exit Triage Head (Attached to Layer 1 & wf1 output)\n        self.triage_head = EarlyExitTriageHead(in_channels=nb[0], hidden_dim=64, num_classes=2)\n\n        # 4. Deep Encoder Stages (Executed only for unexited samples)\n        self.layer2 = ConvBlock(nb[0], nb[1], stride=2)           # H/4, W/4\n        self.layer3 = ConvBlock(nb[1], nb[2], stride=2)           # H/8, W/8\n        self.layer4 = ConvBlock(nb[2], nb[3], stride=2)           # H/16, W/16\n\n        self.bot = nn.Sequential(\n            nn.Conv2d(nb[3], 512, 3, padding=2, dilation=2, bias=False),\n            nn.BatchNorm2d(512),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(512, 512, 3, padding=4, dilation=4, bias=False),\n            nn.BatchNorm2d(512),\n            nn.ReLU(inplace=True)\n        )\n\n        # 5. Smooth Bilinear Upsampling & Decoder Stages\n        self.up4 = BilinearUpBlock(nb[4], nb[2])                  # 512 -> nb[2], H/8\n        self.dec4 = ConvBlock(nb[2] + nb[2], nb[2])               # nb[2] + nb[2] -> nb[2]\n\n        self.up3 = BilinearUpBlock(nb[2], nb[1])                  # nb[2] -> nb[1], H/4\n        self.dec3 = ConvBlock(nb[1] + nb[1], nb[1])               # nb[1] + nb[1] -> nb[1]\n\n        self.up2 = BilinearUpBlock(nb[1], nb[0])                  # nb[1] -> nb[0], H/2\n        if self.use_egm:\n            self.egm2 = EdgeGatedModule(dec_channels=nb[0], freq_channels=nb[0])\n        self.dec2 = ConvBlock(nb[0] + nb[0], nb[0])               # nb[0] + nb[0] -> nb[0]\n\n        self.up1 = BilinearUpBlock(nb[0], nb[0])                  # nb[0] -> nb[0], H\n        if self.use_egm:\n            self.egm1 = EdgeGatedModule(dec_channels=nb[0], freq_channels=nb[0])\n        self.dec1 = ConvBlock(nb[0] + nb[0], nb[0])               # nb[0] + nb[0] -> nb[0]\n\n        self.head_seg = nn.Conv2d(nb[0], num_classes, kernel_size=1)\n        self.head_edge = nn.Conv2d(nb[0], num_classes, kernel_size=1)\n\n        # Coarse prior head for EGM guidance\n        self.coarse_head = nn.Conv2d(nb[1], num_classes, kernel_size=1)\n\n        # Multi-scale Deep Supervision Auxiliary Heads\n        self.deep_sup4 = nn.Conv2d(nb[2], num_classes, kernel_size=1)        # H/8\n        self.deep_sup3 = nn.Conv2d(nb[1], num_classes, kernel_size=1)        # H/4 (aux head)\n        self.deep_sup2 = nn.Conv2d(nb[0], num_classes, kernel_size=1)        # H/2\n\n    @property\n    def bottleneck(self):\n        return self.bot\n\n    @bottleneck.setter\n    def bottleneck(self, val):\n        self.bot = val\n\n    @property\n    def aux_head(self):\n        return self.deep_sup3\n\n    @aux_head.setter\n    def aux_head(self, val):\n        self.deep_sup3 = val\n\n    def load_state_dict(self, state_dict, strict=True, assign=False):\n        \"\"\"\n        Robust state dict loader ensuring 100% backward compatibility across\n        all legacy and v2 checkpoints.\n        \"\"\"\n        model_state = self.state_dict()\n        adapted = {}\n        for k, v in state_dict.items():\n            if k.startswith(\"module.\"):\n                k = k[7:]\n            if k.endswith(\"total_ops\") or k.endswith(\"total_params\"):\n                continue\n\n            # Remap legacy bottleneck / conv1 format -> native bot.0, bot.1, bot.3, bot.4\n            if k.startswith(\"bottleneck.conv1.\"):\n                k = \"bot.0.\" + k[len(\"bottleneck.conv1.\"):]\n            elif k.startswith(\"bottleneck.bn1.\"):\n                k = \"bot.1.\" + k[len(\"bottleneck.bn1.\"):]\n            elif k.startswith(\"bottleneck.conv2.\"):\n                k = \"bot.3.\" + k[len(\"bottleneck.conv2.\"):]\n            elif k.startswith(\"bottleneck.bn2.\"):\n                k = \"bot.4.\" + k[len(\"bottleneck.bn2.\"):]\n            elif k.startswith(\"bottleneck.\"):\n                k = \"bot.\" + k[len(\"bottleneck.\"):]\n            elif k.startswith(\"bot.conv1.\"):\n                k = \"bot.0.\" + k[len(\"bot.conv1.\"):]\n            elif k.startswith(\"bot.bn1.\"):\n                k = \"bot.1.\" + k[len(\"bot.bn1.\"):]\n            elif k.startswith(\"bot.conv2.\"):\n                k = \"bot.3.\" + k[len(\"bot.conv2.\"):]\n            elif k.startswith(\"bot.bn2.\"):\n                k = \"bot.4.\" + k[len(\"bot.bn2.\"):]\n\n            # Remap wf1.conv1.weight -> native wf1.weight\n            if k == \"wf1.conv1.weight\":\n                k = \"wf1.weight\"\n            elif k == \"wf2.conv1.weight\":\n                k = \"wf2.weight\"\n\n            # Remap down -> downsample if needed\n            k_parts = k.split(\".\")\n            if len(k_parts) >= 3 and k_parts[1] == \"down\":\n                k_parts[1] = \"downsample\"\n                k = \".\".join(k_parts)\n\n            # Remap aux_head -> deep_sup3\n            if k.startswith(\"aux_head.\"):\n                k = \"deep_sup3.\" + k[len(\"aux_head.\"):]\n\n            if k in model_state:\n                tgt_shape = model_state[k].shape\n                if v.shape == tgt_shape:\n                    adapted[k] = v\n                elif k == \"triage_head.mlp.0.weight\" and v.shape[1] == self.base_channels * 2 and tgt_shape[1] == self.base_channels * 4:\n                    c = self.base_channels\n                    w_sp = v[:, :c]\n                    w_fq = v[:, c:]\n                    adapted[k] = torch.cat([w_sp * 0.5, w_sp * 0.5, w_fq * 0.5, w_fq * 0.5], dim=1)\n            else:\n                adapted[k] = v\n\n        if \"dwt.weight\" not in adapted and \"dwt.weight\" in model_state:\n            adapted[\"dwt.weight\"] = model_state[\"dwt.weight\"]\n\n        if not strict:\n            adapted = {k: v for k, v in adapted.items() if k in model_state and v.shape == model_state[k].shape}\n\n        return super(FEENetEarlyExit, self).load_state_dict(adapted, strict=strict, assign=assign)\n\n    def forward(\n        self,\n        x: torch.Tensor,\n        tau: float = 0.5414,\n        force_full: bool = False,\n        return_edge: bool = False,\n        return_aux: bool = False,\n        return_raw_logits: bool = False\n    ) -> Union[torch.Tensor, Tuple[Any, ...], Dict[str, Any]]:\n        \"\"\"\n        Unified Forward Pass.\n        \n        In Training Mode:\n            Always computes full forward pass and returns all task logits for joint loss computation:\n            (seg_logits, edge_logits, triage_logits, quality_logits, aux_preds)\n        \n        In Eval Mode:\n            If force_full=True: computes full segmentation.\n            If force_full=False: dynamically evaluates triage_head at Layer 1.\n              - Cases with risk < tau exit early with zero mask, consuming only ~1.45 GFLOPs.\n              - Cases with risk >= tau continue through full segmentation.\n        \"\"\"\n        b_size, _, h_in, w_in = x.shape\n\n        # Stage 1: Shared Stem & Frequency Decomposition\n        x_stem = self.stem(x)             # nb[0], H, W\n        e1 = self.layer1(x_stem)          # nb[0], H/2, W/2\n\n        if self.use_dwt:\n            LL_stem, LH_stem, HL_stem, HH_stem = self.dwt(x_stem)\n            cat_stem = torch.cat([LH_stem, HL_stem, HH_stem], dim=1)\n            cat_stem = self.wf1_attn(cat_stem) if hasattr(self, 'wf1_attn') else cat_stem\n            hf_1 = self.wf1(cat_stem)  # nb[0], H/2, W/2\n        else:\n            hf_1 = self.pool1(x_stem)\n\n        # Stage 2: Early-Exit Triage Evaluation (with Gradient Decoupling)\n        if self.detach_triage_feature and self.training:\n            # Stop-Gradient Feature Probing: isolates shared stem and layer1 from\n            # classification gradient interference, preserving dense segmentation fidelity\n            triage_e1 = e1.detach()\n            triage_hf_1 = hf_1.detach()\n        else:\n            triage_e1 = e1\n            triage_hf_1 = hf_1\n\n        triage_logits, quality_logits, hf_energy = self.triage_head(triage_e1, triage_hf_1)\n        triage_probs = F.softmax(triage_logits, dim=-1)\n        risk_scores = triage_probs[:, 1]  # Class 1 = Suspicious Lesion\n\n        # Training Mode: compute everything and return joint multi-task tuple\n        if self.training:\n            if self.use_dwt:\n                LL_1, LH_1, HL_1, HH_1 = self.dwt(e1)\n                cat_1 = torch.cat([LH_1, HL_1, HH_1], dim=1)\n                cat_1 = self.wf2_attn(cat_1) if hasattr(self, 'wf2_attn') else cat_1\n                hf_2 = self.wf2(cat_1)\n            else:\n                hf_2 = self.pool2(e1)\n\n            e2 = self.layer2(e1)\n            e3 = self.layer3(e2)\n            e4 = self.layer4(e3)\n            b = self.bot(e4)\n\n            d4 = self.dec4(torch.cat([self.up4(b), e3], dim=1))\n            d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))\n            coarse_pred = self.coarse_head(d3)\n\n            d2 = self.up2(d3)\n            if self.use_egm:\n                d2, attn2 = self.egm2(d2, hf_2, coarse_pred=coarse_pred)\n            else:\n                d2 = d2 + F.interpolate(hf_2, size=d2.shape[2:], mode='bilinear', align_corners=False)\n            d2 = self.dec2(torch.cat([d2, e1], dim=1))\n\n            d1 = self.up1(d2)\n            hf_1_up = F.interpolate(hf_1, size=d1.shape[2:], mode='bilinear', align_corners=False)\n            if self.use_egm:\n                d1, attn1 = self.egm1(d1, hf_1_up, coarse_pred=coarse_pred)\n            else:\n                d1 = d1 + hf_1_up\n            d1 = self.dec1(torch.cat([d1, x_stem], dim=1))\n\n            seg_logits = self.head_seg(d1)\n            edge_logits = self.head_edge(hf_1_up)\n            aux_preds = [self.deep_sup4(d4), self.deep_sup3(d3), self.deep_sup2(d2)]\n\n            if return_aux:\n                return seg_logits, edge_logits, triage_logits, quality_logits, aux_preds\n            return seg_logits, edge_logits, triage_logits, quality_logits\n\n        # Eval Mode: Dynamic Early Exit\n        if force_full:\n            # Force full forward pass for benchmarking\n            exit_mask = torch.zeros(b_size, dtype=torch.bool, device=x.device)\n        else:\n            # Early exit if risk < tau\n            exit_mask = risk_scores < tau\n\n        pred_masks = torch.zeros((b_size, 1, h_in, w_in), device=x.device, dtype=x.dtype)\n        pred_edges = torch.zeros((b_size, 1, h_in, w_in), device=x.device, dtype=x.dtype)\n        execution_tiers = [\"TIER1_EARLY_EXIT\" if exit_mask[i] else \"TIER2_FULL_SEG\" for i in range(b_size)]\n\n        idx_full = torch.where(~exit_mask)[0]\n        if len(idx_full) > 0:\n            sub_e1 = e1[idx_full]\n            sub_x_stem = x_stem[idx_full]\n            sub_hf_1 = hf_1[idx_full]\n\n            if self.use_dwt:\n                LL_1, LH_1, HL_1, HH_1 = self.dwt(sub_e1)\n                sub_cat_1 = torch.cat([LH_1, HL_1, HH_1], dim=1)\n                sub_cat_1 = self.wf2_attn(sub_cat_1) if hasattr(self, 'wf2_attn') else sub_cat_1\n                sub_hf_2 = self.wf2(sub_cat_1)\n            else:\n                sub_hf_2 = self.pool2(sub_e1)\n\n            sub_e2 = self.layer2(sub_e1)\n            sub_e3 = self.layer3(sub_e2)\n            sub_e4 = self.layer4(sub_e3)\n            sub_b = self.bot(sub_e4)\n\n            sub_d4 = self.dec4(torch.cat([self.up4(sub_b), sub_e3], dim=1))\n            sub_d3 = self.dec3(torch.cat([self.up3(sub_d4), sub_e2], dim=1))\n            sub_coarse = self.coarse_head(sub_d3)\n\n            sub_d2 = self.up2(sub_d3)\n            if self.use_egm:\n                sub_d2, _ = self.egm2(sub_d2, sub_hf_2, coarse_pred=sub_coarse)\n            else:\n                sub_d2 = sub_d2 + F.interpolate(sub_hf_2, size=sub_d2.shape[2:], mode='bilinear', align_corners=False)\n            sub_d2 = self.dec2(torch.cat([sub_d2, sub_e1], dim=1))\n\n            sub_d1 = self.up1(sub_d2)\n            sub_hf_1_up = F.interpolate(sub_hf_1, size=sub_d1.shape[2:], mode='bilinear', align_corners=False)\n            if self.use_egm:\n                sub_d1, _ = self.egm1(sub_d1, sub_hf_1_up, coarse_pred=sub_coarse)\n            else:\n                sub_d1 = sub_d1 + sub_hf_1_up\n            sub_d1 = self.dec1(torch.cat([sub_d1, sub_x_stem], dim=1))\n\n            sub_seg = self.head_seg(sub_d1)\n            sub_edge = self.head_edge(sub_hf_1_up)\n\n        if return_raw_logits:\n            pred_masks.fill_(-10.0)\n            pred_edges.fill_(-10.0)\n            if len(idx_full) > 0:\n                pred_masks[idx_full] = sub_seg\n                pred_edges[idx_full] = sub_edge\n        else:\n            if len(idx_full) > 0:\n                pred_masks[idx_full] = torch.sigmoid(sub_seg)\n                pred_edges[idx_full] = torch.sigmoid(sub_edge)\n\n        if return_edge:\n            return pred_masks, pred_edges, risk_scores, execution_tiers\n        return pred_masks\n\n\nif __name__ == '__main__':\n    print(\"=\" * 65)\n    print(\"Testing FEENetEarlyExit (Unified Shared-Backbone Architecture)\")\n    print(\"=\" * 65)\n\n    model = FEENetEarlyExit(base_channels=48)\n    params = sum(p.numel() for p in model.parameters() if p.requires_grad)\n    print(f\"1. Trainable Parameters: {params:,} ({params/1e6:.4f}M)\")\n\n    # 1. Test Training Mode (Joint multi-task forward & backward)\n    print(\"\\n2. Testing Multi-Task Training Mode:\")\n    model.train()\n    x = torch.randn(2, 3, 256, 256, requires_grad=True)\n    seg, edge, triage, quality, aux = model(x, return_aux=True)\n    \n    print(f\"   seg logits     : {seg.shape}\")\n    print(f\"   edge logits    : {edge.shape}\")\n    print(f\"   triage logits  : {triage.shape}\")\n    print(f\"   quality logits : {quality.shape}\")\n    print(f\"   aux count      : {len(aux)}\")\n\n    loss = seg.sum() + edge.sum() + triage.sum() + quality.sum() + sum(a.sum() for a in aux)\n    loss.backward()\n\n    assert model.triage_head.mlp[0].weight.grad is not None, \"Triage head gradient is None\"\n    assert model.deep_sup3.weight.grad is not None, \"Deep sup3 (coarse prior) gradient is None\"\n    assert model.stem[0].weight.grad is not None, \"Shared stem gradient is None\"\n    assert model.wf1_attn.fc[0].weight.grad is not None, \"wf1_attn gradient is None\"\n    print(\"   Backward pass successful! Multi-task gradients verified across all branches.\")\n\n    # 2. Test Eval Mode Dynamic Early Exit\n    print(\"\\n3. Testing Dynamic Early-Exit Evaluation:\")\n    model.eval()\n    with torch.no_grad():\n        x_eval = torch.randn(4, 3, 256, 256)\n        masks, edges, risks, tiers = model(x_eval, tau=0.50, return_edge=True)\n        print(f\"   Execution Tiers: {tiers}\")\n        print(f\"   Risk Scores    : {[round(r.item(), 4) for r in risks]}\")\n        print(f\"   Output Masks   : {masks.shape}\")\n\n    # 3. Test Legacy State Dict Backward Compatibility\n    print(\"\\n4. Testing Legacy State Dict (2C triage head) Compatibility:\")\n    legacy_sd = model.state_dict()\n    # Mock legacy 2C triage_head.mlp.0.weight (shape: 64, 96)\n    legacy_sd[\"triage_head.mlp.0.weight\"] = torch.randn(64, 96)\n    model.load_state_dict(legacy_sd)\n    assert model.triage_head.mlp[0].weight.shape == (64, 192), \"Failed to expand legacy 2C weights to 4C\"\n    print(\"   Successfully loaded legacy 2C weights into 4C dual-branch triage head!\")\n\n    print(\"\\n\" + \"=\" * 65)\n    print(\"ALL FEENetEarlyExit TESTS PASSED SUCCESSFULLY!\")\n    print(\"=\" * 65)\n",
  "models/modules/dwt_module.py": "import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nclass DWT_2D(nn.Module):\n    def __init__(self):\n        super(DWT_2D, self).__init__()\n        # Haar wavelet basis\n        ll = torch.tensor([[0.5, 0.5], [0.5, 0.5]])\n        lh = torch.tensor([[-0.5, -0.5], [0.5, 0.5]])\n        hl = torch.tensor([[-0.5, 0.5], [-0.5, 0.5]])\n        hh = torch.tensor([[0.5, -0.5], [-0.5, 0.5]])\n        \n        # (4, 1, 2, 2)\n        weight = torch.stack([ll, lh, hl, hh], dim=0).unsqueeze(1)\n        self.register_buffer('weight', weight)\n\n    def forward(self, x):\n        B, C, H, W = x.shape\n        weight = self.weight.repeat(C, 1, 1, 1).to(dtype=x.dtype, device=x.device) # (4C, 1, 2, 2)\n        out = F.conv2d(x, weight, stride=2, groups=C) # (B, 4C, H/2, W/2)\n        \n        out = out.view(B, C, 4, H // 2, W // 2)\n        LL = out[:, :, 0, :, :]\n        LH = out[:, :, 1, :, :]\n        HL = out[:, :, 2, :, :]\n        HH = out[:, :, 3, :, :]\n        \n        return LL, LH, HL, HH\n\nclass I_DWT_2D(nn.Module):\n    \"\"\"\n    Inverse 2D Haar Discrete Wavelet Transform (I-DWT).\n    # Auxiliary / Inverse transform (retained for mathematical completeness)\n    Reconstructs spatial domain representation from LL, LH, HL, HH subbands.\n    \"\"\"\n    def __init__(self):\n        super(I_DWT_2D, self).__init__()\n        # Auxiliary / Inverse transform (retained for mathematical completeness)\n        ll = torch.tensor([[0.5, 0.5], [0.5, 0.5]])\n        lh = torch.tensor([[-0.5, -0.5], [0.5, 0.5]])\n        hl = torch.tensor([[-0.5, 0.5], [-0.5, 0.5]])\n        hh = torch.tensor([[0.5, -0.5], [-0.5, 0.5]])\n        \n        weight = torch.stack([ll, lh, hl, hh], dim=0).unsqueeze(1)\n        self.register_buffer('weight', weight)\n\n    def forward(self, LL, LH, HL, HH):\n        B, C, H, W = LL.shape\n        x = torch.stack([LL, LH, HL, HH], dim=2).view(B, 4*C, H, W)\n        weight = self.weight.repeat(C, 1, 1, 1).to(dtype=x.dtype, device=x.device)\n        out = F.conv_transpose2d(x, weight, stride=2, groups=C)\n\n        return out\n\nclass SpectralChannelAttention(nn.Module):\n    \"\"\"\n    Dynamically recalibrates high-frequency wavelet subband channels (LH, HL, HH).\n    Suppresses isotropic specular reflection noise while amplifying structured mucosal boundary bands.\n    Uses dual-pooling (GAP + GMP) to capture both subband energy and peak glare anomalies.\n    \"\"\"\n    def __init__(self, channels: int, reduction: int = 4):\n        super(SpectralChannelAttention, self).__init__()\n        mid_channels = max(channels // reduction, 8)\n        self.gap = nn.AdaptiveAvgPool2d(1)\n        self.gmp = nn.AdaptiveMaxPool2d(1)\n        self.fc = nn.Sequential(\n            nn.Conv2d(channels, mid_channels, 1, bias=False),\n            nn.GELU(),\n            nn.Conv2d(mid_channels, channels, 1, bias=False),\n            nn.Sigmoid()\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        # Dual-pooling excitation captures both mean subband energy and peak specular glare\n        w = self.fc(self.gap(x) + self.gmp(x))\n        return x * w\n\n\nclass WaveletHighFrequencyFusion(nn.Module):\n    \"\"\"\n    High-Frequency Wavelet Fusion with Active Spectral Channel Attention (SCA).\n    Ingests concatenated [LH, HL, HH] high-frequency subbands, actively recalibrates\n    channel weights to filter glare, and projects to output channels.\n    \"\"\"\n    def __init__(self, in_channels: int, out_channels: int):\n        super(WaveletHighFrequencyFusion, self).__init__()\n        self.in_channels = in_channels\n        self.out_channels = out_channels\n        # Spectral attention across concatenated high-frequency subbands [LH, HL, HH] (3 * in_channels)\n        self.spectral_attn = SpectralChannelAttention(in_channels * 3)\n        self.conv1 = nn.Conv2d(in_channels * 3, out_channels, kernel_size=1, bias=False)\n        self.bn1 = nn.BatchNorm2d(out_channels)\n        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)\n        self.bn2 = nn.BatchNorm2d(out_channels)\n        self.gelu = nn.GELU()\n\n    def forward(self, LH: torch.Tensor, HL: torch.Tensor, HH: torch.Tensor) -> torch.Tensor:\n        x = torch.cat([LH, HL, HH], dim=1)\n        x = self.spectral_attn(x)\n        x = self.gelu(self.bn1(self.conv1(x)))\n        x = self.gelu(self.bn2(self.conv2(x)))\n        return x\n\n\nif __name__ == '__main__':\n    x = torch.randn(2, 16, 64, 64)\n    dwt = DWT_2D()\n    LL, LH, HL, HH = dwt(x)\n    print(f\"DWT Output Shapes: LL={LL.shape}, LH={LH.shape}, HL={HL.shape}, HH={HH.shape}\")\n\n    idwt = I_DWT_2D()\n    recon = idwt(LL, LH, HL, HH)\n    print(f\"I-DWT Recon Shape: {recon.shape}\")\n    print(f\"Max Recon Error: {torch.max(torch.abs(x - recon)).item()}\")\n\n    fusion = WaveletHighFrequencyFusion(16, 32)\n    hf = fusion(LH, HL, HH)\n    print(f\"HF Fusion Shape: {hf.shape}\")\n    loss = hf.sum()\n    loss.backward()\n    assert fusion.spectral_attn.fc[0].weight.grad is not None, \"Spectral attention must receive gradients\"\n    print(\"WaveletHighFrequencyFusion with SpectralChannelAttention verified successfully!\")\n",
  "models/modules/egm_module.py": "# -*- coding: utf-8 -*-\n\"\"\"\nBidirectional Edge-Region Co-Attention (BERCA) / Edge-Gated Module (EGM)\n========================================================================\nCombines Directional Wavelet Filtering (DWF) via asymmetric 1x3 and 3x1\ndepthwise convolutions with decoder semantic guidance and a learnable\nresidual gate (gamma) to sharpen polyp boundaries without noise amplification.\n\nAcademic Grounding:\n- Directional Filtering: captures continuous horizontal and vertical mucosal contours\n  while filtering out isotropic specular glare.\n- Semantic Guidance: deep coarse predictions restrict edge features strictly to the lesion.\n- Learnable Gate gamma: stabilizes optimization and ensures positive ablation progression.\n\"\"\"\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom typing import Tuple, Optional\n\n\nclass EdgeGatedModule(nn.Module):\n    \"\"\"\n    Bidirectional Edge-Region Co-Attention Module (BERCA / EGM).\n    \"\"\"\n    def __init__(self, dec_channels: int, freq_channels: int, proj_channels: Optional[int] = None):\n        super(EdgeGatedModule, self).__init__()\n        if proj_channels is None:\n            proj_channels = dec_channels\n\n        # Omni-Directional Wavelet Filtering (Asymmetric 1x3, 3x1 and 3x3 diagonal depthwise)\n        self.conv_dir_h = nn.Sequential(\n            nn.Conv2d(freq_channels, freq_channels, (1, 3), padding=(0, 1), groups=freq_channels, bias=False),\n            nn.BatchNorm2d(freq_channels)\n        )\n        self.conv_dir_v = nn.Sequential(\n            nn.Conv2d(freq_channels, freq_channels, (3, 1), padding=(1, 0), groups=freq_channels, bias=False),\n            nn.BatchNorm2d(freq_channels)\n        )\n        # Full 8-connected diagonal receptive field (captures 45\u00b0 and 135\u00b0 diagonal mucosal gradients)\n        self.conv_dir_diag = nn.Sequential(\n            nn.Conv2d(freq_channels, freq_channels, kernel_size=3, padding=1, groups=freq_channels, bias=False),\n            nn.BatchNorm2d(freq_channels)\n        )\n        self.relu_dir = nn.ReLU(inplace=True)\n\n        # Region-to-Edge Semantic Guidance\n        self.semantic_gate = nn.Sequential(\n            nn.Conv2d(dec_channels, max(dec_channels // 4, 16), kernel_size=1, bias=False),\n            nn.BatchNorm2d(max(dec_channels // 4, 16)),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(max(dec_channels // 4, 16), 1, kernel_size=1),\n            nn.Sigmoid()\n        )\n\n        # Cross-Attention\n        self.conv_q = nn.Conv2d(dec_channels, proj_channels, kernel_size=1, bias=False)\n        self.conv_k = nn.Conv2d(freq_channels, proj_channels, kernel_size=1, bias=False)\n        self.conv_attn = nn.Conv2d(proj_channels, 1, kernel_size=1)\n\n        # Output Projection & Learnable Residual Gate\n        self.conv_proj = nn.Conv2d(freq_channels, proj_channels, kernel_size=1, bias=False)\n        self.bn_proj = nn.BatchNorm2d(proj_channels)\n        self.gamma = nn.Parameter(torch.zeros(1) + 0.1)\n\n    def forward(\n        self,\n        f_dec: torch.Tensor,\n        f_freq: torch.Tensor,\n        coarse_pred: Optional[torch.Tensor] = None\n    ) -> Tuple[torch.Tensor, torch.Tensor]:\n        if f_freq.shape[2:] != f_dec.shape[2:]:\n            f_freq = F.interpolate(f_freq, size=f_dec.shape[2:], mode='bilinear', align_corners=False)\n\n        # Omni-directional high-frequency feature extraction (horizontal, vertical, diagonal)\n        f_dir = self.relu_dir(self.conv_dir_h(f_freq) + self.conv_dir_v(f_freq) + self.conv_dir_diag(f_freq))\n\n        # Semantic guidance\n        m_sem = self.semantic_gate(f_dec)\n        if coarse_pred is not None:\n            if coarse_pred.shape[2:] != f_dec.shape[2:]:\n                c_up = F.interpolate(torch.sigmoid(coarse_pred), size=f_dec.shape[2:], mode='bilinear', align_corners=False)\n            else:\n                c_up = torch.sigmoid(coarse_pred)\n            m_sem = m_sem * c_up\n\n        f_pure = f_dir * m_sem\n\n        # Cross-attention modulation\n        attn = torch.sigmoid(self.conv_attn(F.relu(self.conv_q(f_dec) + self.conv_k(f_pure))))\n        f_proj = self.bn_proj(self.conv_proj(f_pure))\n\n        out = f_dec + self.gamma * (attn * f_proj)\n        return out, attn\n\n\n# Aliases for paper consistency\nSemanticFrequencyGate = EdgeGatedModule\nBERCA = EdgeGatedModule\nBidirectionalEdgeRegionCoAttention = EdgeGatedModule\n\n\nif __name__ == '__main__':\n    print(\"=\" * 60)\n    print(\"Self-Test: EdgeGatedModule (Omni-Directional Wavelet Filtering)\")\n    print(\"=\" * 60)\n    m = EdgeGatedModule(dec_channels=48, freq_channels=48)\n    f_d = torch.randn(2, 48, 64, 64)\n    f_f = torch.randn(2, 48, 32, 32)\n    c_p = torch.randn(2, 1, 32, 32)\n    out, attn = m(f_d, f_f, coarse_pred=c_p)\n    assert out.shape == f_d.shape\n    assert attn.shape == (2, 1, 64, 64)\n    loss = out.sum()\n    loss.backward()\n    assert m.gamma.grad is not None\n    assert m.conv_dir_h[0].weight.grad is not None\n    assert m.conv_dir_v[0].weight.grad is not None\n    assert m.conv_dir_diag[0].weight.grad is not None\n    print(f\"Self-test passed! Gamma grad: {m.gamma.grad.item():.4f}\")\n    print(f\"conv_dir_diag grad norm: {m.conv_dir_diag[0].weight.grad.norm().item():.4f}\")\n",
  "utils/losses.py": "import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom typing import Optional, Tuple, Dict\n\n\nclass DiceLoss(nn.Module):\n    r\"\"\"\n    Soft Dice Loss for binary segmentation.\n    Computes 1 - (2 * |P \\cap G| + smooth) / (|P| + |G| + smooth).\n    \"\"\"\n    def __init__(self, smooth: float = 1e-5):\n        super(DiceLoss, self).__init__()\n        self.smooth = smooth\n\n    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:\n        pred = torch.sigmoid(pred)\n        pred_flat = pred.contiguous().view(-1)\n        target_flat = target.contiguous().view(-1)\n        intersection = (pred_flat * target_flat).sum()\n        dice = (2.0 * intersection + self.smooth) / (pred_flat.sum() + target_flat.sum() + self.smooth)\n        return 1.0 - dice\n\n\nclass MorphologicalBoundaryExtractor(nn.Module):\n    r\"\"\"\n    Morphological Boundary Ribbon Extractor.\n\n    Extracts continuous boundary ribbon \\partial G via morphological dilation minus erosion:\n        dilation = F.max_pool2d(G, kernel_size=2*d+1, stride=1, padding=d)\n        erosion = -F.max_pool2d(-G, kernel_size=2*d+1, stride=1, padding=d)\n        boundary = dilation - erosion\n\n    With boundary thickness parameter d (default d=3 -> kernel size 7x7).\n\n    Provenance / citation (this file previously mis-attributed the operator):\n      * max-pool dilation minus min-pool erosion contour operator:\n        El Jurdi et al., \"A Surprisingly Effective Perimeter-based Loss for\n        Medical Image Segmentation\", MICCAI 2021 (PMLR v143).\n      * morphological erosion for reducing the Hausdorff distance:\n        Karimi & Salcudean, IEEE Trans. Medical Imaging, 2019.\n      * boundary-IoU formulation of these ribbons:\n        Karimi & Salcudean, Medical Image Analysis, 2022.\n    \"\"\"\n    def __init__(self, d: int = 3):\n        super(MorphologicalBoundaryExtractor, self).__init__()\n        self.d = d\n\n    def forward(self, x: torch.Tensor, d: Optional[int] = None) -> torch.Tensor:\n        if d is None:\n            d = self.d\n            \n        was_3d = False\n        if x.dim() == 3:\n            x = x.unsqueeze(1)\n            was_3d = True\n            \n        k_size = 2 * d + 1\n        dilation = F.max_pool2d(x, kernel_size=k_size, stride=1, padding=d)\n        erosion = -F.max_pool2d(-x, kernel_size=k_size, stride=1, padding=d)\n        boundary = dilation - erosion\n        \n        if was_3d:\n            boundary = boundary.squeeze(1)\n        return boundary\n\n\nclass BoundaryIoULoss(nn.Module):\n    r\"\"\"\n    Boundary IoU Loss (morphological boundary-ribbon formulation).\n\n    Provenance: the boundary-IoU measure and the dilation-erosion ribbon\n    construction follow El Jurdi et al. (MICCAI 2021) and\n    Karimi & Salcudean (Medical Image Analysis 2022 / IEEE TMI 2019).\n    Used here as a training loss (the cited works evaluate with it).\n\n    Replaces unstable, noise-magnifying second-order Laplacian derivatives with a\n    mathematically smooth, bounded, differentiable morphological boundary ribbon formulation:\n    \n    1. For prediction P (probabilities) and target G (binary mask), compute boundary ribbon:\n       \\partial P = dilation(P) - erosion(P)\n       \\partial G = dilation(G) - erosion(G)\n       with boundary thickness d=3 (2*d+1 = 7x7 local receptive field).\n    2. Compute soft differentiable Boundary IoU / Dice formulation:\n       \\mathcal{L}_{Boundary-IoU} = 1 - (2 * |\\partial P \\cap \\partial G| + eps) / (|\\partial P| + |\\partial G| + eps).\n    \"\"\"\n    def __init__(self, d: int = 3, smooth: float = 1e-5, from_logits: bool = True):\n        super(BoundaryIoULoss, self).__init__()\n        self.d = d\n        self.smooth = smooth\n        self.from_logits = from_logits\n        self.extractor = MorphologicalBoundaryExtractor(d=d)\n\n    def extract_boundary(self, x: torch.Tensor, d: Optional[int] = None) -> torch.Tensor:\n        return self.extractor(x, d=d)\n\n    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:\n        if self.from_logits:\n            pred_prob = torch.sigmoid(pred)\n        else:\n            pred_prob = pred\n\n        if pred_prob.dim() == 3:\n            pred_prob = pred_prob.unsqueeze(1)\n        if target.dim() == 3:\n            target = target.unsqueeze(1)\n\n        target = target.to(dtype=pred_prob.dtype, device=pred_prob.device)\n\n        boundary_pred = self.extractor(pred_prob, self.d)\n        boundary_target = self.extractor(target, self.d).detach()\n\n        # Differentiable soft Boundary IoU / Dice:\n        # L = 1 - (2 * |d_P \\cap d_G| + eps) / (|d_P| + |d_G| + eps)\n        intersection = (boundary_pred * boundary_target).sum()\n        total = boundary_pred.sum() + boundary_target.sum()\n\n        loss = 1.0 - ((2.0 * intersection + self.smooth) / (total + self.smooth))\n        return loss\n\n\n# Backwards compatibility alias\nBoundaryLoss = BoundaryIoULoss\n\n\nclass LaplacianEdgeExtractor(nn.Module):\n    \"\"\"\n    [DEPRECATED] Second-order Laplacian edge filter.\n    Retained solely for backwards compatibility with historical checkpoints/scripts.\n    Note: High-frequency noise is magnified by 2nd-order derivatives.\n    Use MorphologicalBoundaryExtractor for mathematically smooth and stable boundary extraction.\n    \"\"\"\n    def __init__(self):\n        super(LaplacianEdgeExtractor, self).__init__()\n        kernel = torch.tensor([[-1, -1, -1],\n                               [-1,  8, -1],\n                               [-1, -1, -1]], dtype=torch.float32)\n        kernel = kernel.unsqueeze(0).unsqueeze(0)\n        self.register_buffer('weight', kernel)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        edge = F.conv2d(x, self.weight.to(dtype=x.dtype, device=x.device), padding=1)\n        edge = torch.abs(edge)\n        return torch.clamp(edge, 0.0, 1.0)\n\n\nclass CompoundLoss(nn.Module):\n    \"\"\"\n    Compound Loss for Boundary-Aware Semantic Segmentation.\n    \n    Formulation:\n        L_{total} = lambda_bce * L_{BCE}(pred_seg, target_seg)\n                  + lambda_dice * L_{Dice}(pred_seg, target_seg)\n                  + lambda_boundary * L_{Boundary-IoU}(pred_seg, target_seg)\n                  \n        If auxiliary pred_edge is provided:\n            target_boundary = morphological_boundary(target_seg)\n            L_{total} += lambda_boundary * BCEWithLogitsLoss(pred_edge, target_boundary)\n            \n    Default Hyperparameters:\n        lambda_bce = 1.0\n        lambda_dice = 1.0\n        lambda_boundary = 0.5\n        boundary_thickness d = 3\n    \"\"\"\n    def __init__(\n        self,\n        lambda_bce: float = 1.0,\n        lambda_dice: float = 1.0,\n        lambda_boundary: float = 0.5,\n        d: int = 3,\n        smooth: float = 1e-5\n    ):\n        super(CompoundLoss, self).__init__()\n        self.bce_loss = nn.BCEWithLogitsLoss()\n        self.dice_loss = DiceLoss(smooth=smooth)\n        self.boundary_loss = BoundaryIoULoss(d=d, smooth=smooth, from_logits=True)\n        self.lambda_bce = lambda_bce\n        self.lambda_dice = lambda_dice\n        self.lambda_boundary = lambda_boundary\n\n    def forward(\n        self,\n        pred_seg: torch.Tensor,\n        pred_edge: Optional[torch.Tensor],\n        target_seg: torch.Tensor\n    ) -> Tuple[torch.Tensor, Dict[str, float]]:\n        if target_seg.dim() == 3 and pred_seg.dim() == 4:\n            target_seg = target_seg.unsqueeze(1)\n        target_seg = target_seg.to(dtype=pred_seg.dtype, device=pred_seg.device)\n\n        loss_bce = self.bce_loss(pred_seg, target_seg)\n        loss_dice = self.dice_loss(pred_seg, target_seg)\n\n        total_loss = self.lambda_bce * loss_bce + self.lambda_dice * loss_dice\n        loss_dict = {\n            'loss_bce': loss_bce.item(),\n            'loss_dice': loss_dice.item()\n        }\n\n        if self.lambda_boundary > 0.0:\n            loss_boundary_iou = self.boundary_loss(pred_seg, target_seg)\n            total_loss = total_loss + self.lambda_boundary * loss_boundary_iou\n            loss_dict['loss_boundary'] = loss_boundary_iou.item()\n            loss_dict['loss_boundary_iou'] = loss_boundary_iou.item()\n\n            if pred_edge is not None:\n                target_boundary = self.boundary_loss.extract_boundary(target_seg).detach()\n                if pred_edge.dim() != target_boundary.dim():\n                    target_boundary = target_boundary.view_as(pred_edge)\n                loss_edge_bce = self.bce_loss(pred_edge, target_boundary)\n                total_loss = total_loss + self.lambda_boundary * loss_edge_bce\n                loss_dict['loss_edge_bce'] = loss_edge_bce.item()\n\n        loss_dict['total_loss'] = total_loss.item()\n        return total_loss, loss_dict\n\n\nif __name__ == '__main__':\n    print(\"=\" * 70)\n    print(\"[SELF-TEST] Overhauled utils/losses.py (BoundaryIoULoss & CompoundLoss)\")\n    print(\"=\" * 70)\n\n    torch.manual_seed(42)\n    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n    print(f\"Running self-test on device: {device}\")\n\n    # 1. Standard Forward & Backward with auxiliary pred_edge\n    print(\"\\n[Test 1/5] Standard Forward & Backward Pass (pred_seg + pred_edge)...\")\n    b, c, h, w = 4, 1, 128, 128\n    pred_seg = torch.randn(b, c, h, w, device=device, requires_grad=True)\n    pred_edge = torch.randn(b, c, h, w, device=device, requires_grad=True)\n    target_seg = torch.zeros(b, c, h, w, device=device)\n    target_seg[:, :, 32:96, 32:96] = 1.0  # Synthetic square mask\n\n    criterion = CompoundLoss(lambda_bce=1.0, lambda_dice=1.0, lambda_boundary=0.5, d=3).to(device)\n    total_loss, loss_dict = criterion(pred_seg, pred_edge, target_seg)\n\n    print(f\"  - Total Loss: {total_loss.item():.4f}\")\n    for k, v in loss_dict.items():\n        print(f\"    * {k}: {v:.4f}\")\n    assert total_loss.item() > 0.0, \"Total loss must be positive\"\n    assert not torch.isnan(total_loss), \"Total loss must not be NaN\"\n    assert not torch.isinf(total_loss), \"Total loss must not be Inf\"\n\n    total_loss.backward()\n    assert pred_seg.grad is not None, \"pred_seg must receive gradients\"\n    assert pred_edge.grad is not None, \"pred_edge must receive gradients\"\n    assert not torch.isnan(pred_seg.grad).any(), \"pred_seg.grad must have zero NaN\"\n    assert not torch.isinf(pred_seg.grad).any(), \"pred_seg.grad must have zero Inf\"\n    assert not torch.isnan(pred_edge.grad).any(), \"pred_edge.grad must have zero NaN\"\n    assert not torch.isinf(pred_edge.grad).any(), \"pred_edge.grad must have zero Inf\"\n    print(f\"  - Gradient Flow Verified:\")\n    print(f\"    * pred_seg grad norm: {pred_seg.grad.norm().item():.6f}\")\n    print(f\"    * pred_edge grad norm: {pred_edge.grad.norm().item():.6f}\")\n    print(\"  -> Passed!\")\n\n    # 2. Forward & Backward without auxiliary pred_edge (ablation)\n    print(\"\\n[Test 2/5] Ablation Forward & Backward Pass (pred_edge=None)...\")\n    pred_seg_abl = torch.randn(b, c, h, w, device=device, requires_grad=True)\n    loss_abl, dict_abl = criterion(pred_seg_abl, None, target_seg)\n    loss_abl.backward()\n    assert pred_seg_abl.grad is not None, \"pred_seg_abl must receive gradients\"\n    assert not torch.isnan(pred_seg_abl.grad).any(), \"pred_seg_abl.grad must have zero NaN\"\n    assert 'loss_edge_bce' not in dict_abl, \"loss_edge_bce should not be computed when pred_edge is None\"\n    print(f\"  - Ablation Loss: {loss_abl.item():.4f}, grad norm: {pred_seg_abl.grad.norm().item():.6f}\")\n    print(\"  -> Passed!\")\n\n    # 3. Extreme Robustness: Saturated and High-Variance Noise Inputs\n    print(\"\\n[Test 3/5] Extreme Noise & Saturation Robustness Check...\")\n    # Extreme logits: range [-100.0, 100.0] with heavy Gaussian noise\n    extreme_pred_seg = (torch.randn(b, c, h, w, device=device) * 25.0).requires_grad_(True)\n    extreme_pred_edge = (torch.randn(b, c, h, w, device=device) * 25.0).requires_grad_(True)\n    loss_extreme, dict_extreme = criterion(extreme_pred_seg, extreme_pred_edge, target_seg)\n    loss_extreme.backward()\n\n    assert not torch.isnan(loss_extreme), \"Loss must not be NaN under extreme noise\"\n    assert not torch.isinf(loss_extreme), \"Loss must not be Inf under extreme noise\"\n    assert not torch.isnan(extreme_pred_seg.grad).any(), \"Gradient must not be NaN under extreme noise\"\n    assert not torch.isinf(extreme_pred_seg.grad).any(), \"Gradient must not be Inf under extreme noise\"\n    print(f\"  - Extreme Noise Loss: {loss_extreme.item():.4f}\")\n    print(f\"    * pred_seg grad norm: {extreme_pred_seg.grad.norm().item():.6f} (well-conditioned)\")\n    print(f\"    * pred_edge grad norm: {extreme_pred_edge.grad.norm().item():.6f} (well-conditioned)\")\n    print(\"  -> Passed!\")\n\n    # 4. Degenerate Target Masks (All-Zero / Empty and All-One / Full)\n    print(\"\\n[Test 4/5] Degenerate Targets (All-Zero & All-One)...\")\n    # All zeros\n    zero_target = torch.zeros(b, c, h, w, device=device)\n    pred_z = torch.randn(b, c, h, w, device=device, requires_grad=True)\n    loss_z, _ = criterion(pred_z, None, zero_target)\n    loss_z.backward()\n    assert not torch.isnan(loss_z) and not torch.isinf(loss_z), \"Zero target must not yield NaN/Inf\"\n    assert not torch.isnan(pred_z.grad).any(), \"Zero target grad must not be NaN\"\n\n    # All ones\n    one_target = torch.ones(b, c, h, w, device=device)\n    pred_o = torch.randn(b, c, h, w, device=device, requires_grad=True)\n    loss_o, _ = criterion(pred_o, None, one_target)\n    loss_o.backward()\n    assert not torch.isnan(loss_o) and not torch.isinf(loss_o), \"One target must not yield NaN/Inf\"\n    assert not torch.isnan(pred_o.grad).any(), \"One target grad must not be NaN\"\n    print(f\"  - All-Zero Target Loss: {loss_z.item():.4f}, All-One Target Loss: {loss_o.item():.4f}\")\n    print(\"  -> Passed!\")\n\n    # 5. Stability Comparison: BoundaryIoULoss vs Deprecated Laplacian\n    print(\"\\n[Test 5/5] Stability Comparison: BoundaryIoULoss vs Deprecated Laplacian...\")\n    biou_loss_fn = BoundaryIoULoss(d=3).to(device)\n    laplacian_extractor = LaplacianEdgeExtractor().to(device)\n\n    # Add high frequency noise to mask\n    noise_seg = torch.randn(2, 1, 64, 64, device=device, requires_grad=True)\n    noisy_input = noise_seg + torch.randn_like(noise_seg) * 2.0\n    \n    biou_val = biou_loss_fn(noisy_input, target_seg[:2, :, :64, :64])\n    biou_val.backward()\n    biou_grad_norm = noise_seg.grad.norm().item()\n\n    # Old Laplacian on probabilities\n    lap_input = torch.sigmoid(noisy_input).detach().requires_grad_(True)\n    lap_pred_edge = laplacian_extractor(lap_input)\n    lap_target_edge = laplacian_extractor(target_seg[:2, :, :64, :64])\n    lap_loss = F.smooth_l1_loss(lap_pred_edge, lap_target_edge)\n    lap_loss.backward()\n    lap_grad_norm = lap_input.grad.norm().item()\n\n    print(f\"  - BoundaryIoULoss Value: {biou_val.item():.4f}, Grad Norm: {biou_grad_norm:.6f}\")\n    print(f\"  - Laplacian Loss Value:  {lap_loss.item():.4f}, Grad Norm: {lap_grad_norm:.6f}\")\n    print(f\"  - Note: BoundaryIoU operates on smooth morphological ribbons [0, 1] with zero high-order ripple.\")\n    print(\"  -> Passed!\")\n\n    print(\"\\n\" + \"=\" * 70)\n    print(\"[PASSED] ALL 5 SELF-TEST CHECKS PASSED PERFECTLY! ZERO NaN/Inf, SMOOTH GRADIENTS.\")\n    print(\"=\" * 70)\n",
  "utils/metrics.py": "\"\"\"\nEvaluation Metrics and Model Complexity Utilities for Medical Image Segmentation.\n\nImplements:\n- calculate_dice: Dice Similarity Coefficient (DSC)\n- calculate_iou: Intersection over Union (Jaccard Index)\n- calculate_hd95: 95th Percentile Hausdorff Distance (HD95)\n- calculate_precision_recall: Precision, Recall (Sensitivity)\n- compute_model_complexity: Parameter count (M) and FLOPs (G) using thop or manual fallback\n- MetricTracker: Epoch-level batch metric accumulator\n\"\"\"\n\nfrom collections import namedtuple\nfrom typing import Dict, List, Optional, Tuple, Union\n\nimport cv2\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom scipy.spatial import cKDTree\nfrom scipy.spatial.distance import directed_hausdorff\n\nPrecisionRecall = namedtuple(\"PrecisionRecall\", [\"precision\", \"recall\"])\n\n\ndef _to_numpy(data: Union[torch.Tensor, np.ndarray]) -> np.ndarray:\n    \"\"\"Converts torch.Tensor or numpy-like data to a float32 numpy array.\"\"\"\n    if isinstance(data, torch.Tensor):\n        return data.detach().cpu().numpy().astype(np.float32)\n    return np.asarray(data, dtype=np.float32)\n\n\ndef _prepare_mask(\n    mask: Union[torch.Tensor, np.ndarray],\n    threshold: float = 0.5,\n    is_pred: bool = True,\n    from_logits: Optional[bool] = None\n) -> np.ndarray:\n    \"\"\"\n    Preprocesses prediction or target mask into a binary numpy array [0, 1].\n    If input values are logits (outside [0, 1]), applies sigmoid first.\n\n    Args:\n        from_logits: Explicit interpretation flag for predictions. True = apply\n            sigmoid first, False = threshold raw values, None = auto-detect by\n            value range (legacy heuristic: sigmoid if min < 0 or max > 1).\n            Explicitly passing True/False avoids the ambiguity where a logit\n            map that happens to fall inside [0, 1] is mistaken for probabilities.\n    \"\"\"\n    arr = _to_numpy(mask)\n    if is_pred:\n        if from_logits is True:\n            arr = 1.0 / (1.0 + np.exp(-np.clip(arr, -30.0, 30.0)))\n        elif from_logits is None:\n            # Legacy auto-detection heuristic\n            if np.any(arr < 0.0) or np.any(arr > 1.0):\n                arr = 1.0 / (1.0 + np.exp(-np.clip(arr, -30.0, 30.0)))\n        return (arr > threshold).astype(np.float32)\n    else:\n        return (arr > threshold).astype(np.float32)\n\n\ndef calculate_dice(\n    pred: Union[torch.Tensor, np.ndarray],\n    target: Union[torch.Tensor, np.ndarray],\n    threshold: float = 0.5,\n    smooth: float = 1e-6,\n    from_logits: Optional[bool] = None\n) -> float:\n    \"\"\"\n    Calculates the Dice Similarity Coefficient (DSC):\n        Dice = (2 * |P \u2229 T| + smooth) / (|P| + |T| + smooth)\n\n    Args:\n        pred: Predicted probabilities/logits or binary mask. Shape (B, 1, H, W), (B, H, W), or (H, W).\n        target: Ground truth binary mask, matching pred shape.\n        threshold: Binarization threshold for pred.\n        smooth: Smoothing epsilon to prevent division by zero.\n\n    Returns:\n        float: Mean Dice score across samples in batch (0.0 to 1.0).\n    \"\"\"\n    p_bin = _prepare_mask(pred, threshold=threshold, is_pred=True, from_logits=from_logits)\n    t_bin = _prepare_mask(target, threshold=threshold, is_pred=False)\n\n    # If batched (3D or 4D), compute per-sample dice and average\n    if p_bin.ndim >= 3:\n        batch_size = p_bin.shape[0]\n        dices = []\n        for i in range(batch_size):\n            p_flat = p_bin[i].ravel()\n            t_flat = t_bin[i].ravel()\n            intersection = np.sum(p_flat * t_flat)\n            p_sum = np.sum(p_flat)\n            t_sum = np.sum(t_flat)\n\n            # Edge case: both ground truth and prediction are completely empty\n            if p_sum == 0 and t_sum == 0:\n                dices.append(1.0)\n            else:\n                score = (2.0 * intersection + smooth) / (p_sum + t_sum + smooth)\n                dices.append(float(score))\n        return float(np.mean(dices))\n    else:\n        p_flat = p_bin.ravel()\n        t_flat = t_bin.ravel()\n        p_sum = np.sum(p_flat)\n        t_sum = np.sum(t_flat)\n        if p_sum == 0 and t_sum == 0:\n            return 1.0\n        intersection = np.sum(p_flat * t_flat)\n        return float((2.0 * intersection + smooth) / (p_sum + t_sum + smooth))\n\n\ndef calculate_iou(\n    pred: Union[torch.Tensor, np.ndarray],\n    target: Union[torch.Tensor, np.ndarray],\n    threshold: float = 0.5,\n    smooth: float = 1e-6,\n    from_logits: Optional[bool] = None\n) -> float:\n    \"\"\"\n    Calculates the Intersection over Union (IoU / Jaccard Index):\n        IoU = (|P \u2229 T| + smooth) / (|P \u222a T| + smooth)\n\n    Args:\n        pred: Predicted probabilities/logits or binary mask.\n        target: Ground truth binary mask.\n        threshold: Binarization threshold.\n        smooth: Smoothing epsilon.\n\n    Returns:\n        float: Mean IoU score across samples in batch (0.0 to 1.0).\n    \"\"\"\n    p_bin = _prepare_mask(pred, threshold=threshold, is_pred=True, from_logits=from_logits)\n    t_bin = _prepare_mask(target, threshold=threshold, is_pred=False)\n\n    if p_bin.ndim >= 3:\n        batch_size = p_bin.shape[0]\n        ious = []\n        for i in range(batch_size):\n            p_flat = p_bin[i].ravel()\n            t_flat = t_bin[i].ravel()\n            p_sum = np.sum(p_flat)\n            t_sum = np.sum(t_flat)\n\n            if p_sum == 0 and t_sum == 0:\n                ious.append(1.0)\n            else:\n                intersection = np.sum(p_flat * t_flat)\n                union = p_sum + t_sum - intersection\n                score = (intersection + smooth) / (union + smooth)\n                ious.append(float(score))\n        return float(np.mean(ious))\n    else:\n        p_flat = p_bin.ravel()\n        t_flat = t_bin.ravel()\n        p_sum = np.sum(p_flat)\n        t_sum = np.sum(t_flat)\n        if p_sum == 0 and t_sum == 0:\n            return 1.0\n        intersection = np.sum(p_flat * t_flat)\n        union = p_sum + t_sum - intersection\n        return float((intersection + smooth) / (union + smooth))\n\n\ndef _extract_contour_points(mask: np.ndarray) -> np.ndarray:\n    \"\"\"\n    Extracts 2D boundary/contour coordinates (N, 2) from a 2D binary mask.\n    Returns empty (0, 2) array if mask is empty.\n    \"\"\"\n    mask_u8 = (mask > 0).astype(np.uint8)\n    if not np.any(mask_u8):\n        return np.empty((0, 2), dtype=np.float64)\n\n    contours, _ = cv2.findContours(mask_u8, cv2.RETR_LIST, cv2.CHAIN_APPROX_NONE)\n    if not contours:\n        return np.empty((0, 2), dtype=np.float64)\n\n    pts = np.vstack([c.reshape(-1, 2) for c in contours]).astype(np.float64)\n    return pts\n\n\ndef calculate_hd95_single(\n    pred_mask: np.ndarray,\n    target_mask: np.ndarray,\n    empty_value: float = 100.0\n) -> float:\n    \"\"\"\n    Computes 95th percentile Hausdorff Distance (HD95) for a single 2D image pair.\n\n    Boundary contours are extracted from both masks. The nearest Euclidean distances\n    from each contour point in Pred to Target and vice-versa are evaluated via KD-Tree.\n    The directed Hausdorff distance (via scipy.spatial.distance.directed_hausdorff)\n    defines the maximum boundary deviation, while HD95 calculates the 95th percentile\n    of the bidirectional boundary point-to-point distances for outlier robustness.\n\n    Args:\n        pred_mask: 2D binary mask of predictions (H, W).\n        target_mask: 2D binary mask of ground truth (H, W).\n        empty_value: Penalty distance returned when one mask is empty while the other is non-empty.\n\n    Returns:\n        float: HD95 distance in pixels (0.0 if both empty, empty_value if only one is empty).\n    \"\"\"\n    pts_pred = _extract_contour_points(pred_mask)\n    pts_target = _extract_contour_points(target_mask)\n\n    len_pred = len(pts_pred)\n    len_target = len(pts_target)\n\n    # Edge cases\n    if len_pred == 0 and len_target == 0:\n        return 0.0\n    if len_pred == 0 or len_target == 0:\n        return float(empty_value)\n\n    # Also reference directed_hausdorff from scipy.spatial.distance\n    # directed_hausdorff computes the 100th percentile (maximum) distance.\n    # For HD95, we compute the 95th percentile of distances in both directions using cKDTree.\n    tree_target = cKDTree(pts_target)\n    dist_pred_to_target, _ = tree_target.query(pts_pred)\n\n    tree_pred = cKDTree(pts_pred)\n    dist_target_to_pred, _ = tree_pred.query(pts_target)\n\n    hd95_p2t = np.percentile(dist_pred_to_target, 95)\n    hd95_t2p = np.percentile(dist_target_to_pred, 95)\n\n    return float(max(hd95_p2t, hd95_t2p))\n\n\ndef calculate_hd95(\n    pred: Union[torch.Tensor, np.ndarray],\n    target: Union[torch.Tensor, np.ndarray],\n    threshold: float = 0.5,\n    empty_value: float = 100.0,\n    from_logits: Optional[bool] = None\n) -> float:\n    \"\"\"\n    Calculates 95% Hausdorff Distance (HD95) over a batch or single image.\n\n    Args:\n        pred: Predicted logits/probabilities or mask. Shape (B, 1, H, W), (B, H, W), or (H, W).\n        target: Ground truth binary mask.\n        threshold: Binarization threshold for pred.\n        empty_value: Penalty returned if one mask is empty and the other is non-empty (default 100.0).\n\n    Returns:\n        float: Mean HD95 across the batch.\n    \"\"\"\n    p_bin = _prepare_mask(pred, threshold=threshold, is_pred=True, from_logits=from_logits)\n    t_bin = _prepare_mask(target, threshold=threshold, is_pred=False)\n\n    # Squeeze channel dim if (B, 1, H, W) -> (B, H, W)\n    if p_bin.ndim == 4 and p_bin.shape[1] == 1:\n        p_bin = p_bin[:, 0, :, :]\n    if t_bin.ndim == 4 and t_bin.shape[1] == 1:\n        t_bin = t_bin[:, 0, :, :]\n\n    if p_bin.ndim == 3:\n        batch_size = p_bin.shape[0]\n        hd95_scores = []\n        for i in range(batch_size):\n            score = calculate_hd95_single(p_bin[i], t_bin[i], empty_value=empty_value)\n            hd95_scores.append(score)\n        return float(np.mean(hd95_scores))\n    elif p_bin.ndim == 2:\n        return calculate_hd95_single(p_bin, t_bin, empty_value=empty_value)\n    else:\n        raise ValueError(f\"Unsupported mask shape for HD95: {p_bin.shape}\")\n\n\ndef calculate_precision_recall(\n    pred: Union[torch.Tensor, np.ndarray],\n    target: Union[torch.Tensor, np.ndarray],\n    threshold: float = 0.5,\n    smooth: float = 1e-6,\n    from_logits: Optional[bool] = None\n) -> PrecisionRecall:\n    \"\"\"\n    Calculates Precision and Recall (Sensitivity):\n        Precision = TP / (TP + FP + smooth)\n        Recall    = TP / (TP + FN + smooth)\n\n    Args:\n        pred: Predicted probabilities/logits.\n        target: Ground truth binary mask.\n        threshold: Binarization threshold.\n        smooth: Smoothing epsilon.\n\n    Returns:\n        PrecisionRecall(precision=float, recall=float): NamedTuple accessible by attribute or tuple unpacking.\n    \"\"\"\n    p_bin = _prepare_mask(pred, threshold=threshold, is_pred=True, from_logits=from_logits)\n    t_bin = _prepare_mask(target, threshold=threshold, is_pred=False)\n\n    p_flat = p_bin.ravel()\n    t_flat = t_bin.ravel()\n\n    tp = np.sum((p_flat == 1.0) & (t_flat == 1.0))\n    fp = np.sum((p_flat == 1.0) & (t_flat == 0.0))\n    fn = np.sum((p_flat == 0.0) & (t_flat == 1.0))\n\n    if tp == 0 and fp == 0 and fn == 0:\n        return PrecisionRecall(precision=1.0, recall=1.0)\n\n    precision = float((tp + smooth) / (tp + fp + smooth))\n    recall = float((tp + smooth) / (tp + fn + smooth))\n\n    return PrecisionRecall(precision=precision, recall=recall)\n\n\ndef compute_model_complexity(\n    model: nn.Module,\n    input_size: Tuple[int, ...] = (1, 3, 256, 256),\n    device: str = \"cpu\"\n) -> Dict[str, Union[float, int]]:\n    \"\"\"\n    Computes model complexity:\n    - Number of Parameters (in Millions, M)\n    - FLOPs (in GigaFLOPs, G)\n\n    Uses `thop.profile` if available; otherwise falls back to exact parameter counting\n    and theoretical convolution FLOP estimation.\n\n    Args:\n        model: PyTorch nn.Module.\n        input_size: Input tensor dimensions (B, C, H, W).\n        device: Device to place the dummy input on ('cpu' or 'cuda').\n\n    Returns:\n        dict: {\n            'params_m': float,       # Parameters in Millions (M)\n            'flops_g': float,        # Computations in GigaFLOPs (G)\n            'total_params': int,     # Exact parameter count\n            'total_flops': int,      # Exact FLOP count\n            'method': str            # 'thop' or 'manual_fallback'\n        }\n    \"\"\"\n    was_training = model.training\n    model.eval()\n    orig_device = next(model.parameters()).device\n\n    total_params = sum(p.numel() for p in model.parameters())\n    params_m = round(total_params / 1e6, 4)\n\n    flops_g = 0.0\n    total_flops = 0\n    method = \"manual_fallback\"\n\n    # Attempt profiling with thop on a clean deepcopy\n    try:\n        from copy import deepcopy\n        from thop import profile\n        model_copy = deepcopy(model).to(device)\n        dummy_input = torch.randn(*input_size, device=device)\n        flops, _ = profile(model_copy, inputs=(dummy_input,), verbose=False)\n        total_flops = int(flops)\n        flops_g = round(total_flops / 1e9, 4)\n        method = \"thop\"\n        del model_copy\n    except Exception:\n        # Fallback estimation: sum FLOPs of Conv2d and Linear layers\n        model = model.to(device)\n        total_flops = 0\n\n        hooks = []\n\n        def conv_hook(self, inp, out):\n            nonlocal total_flops\n            # Conv2d FLOPs = 2 * C_in * K_h * K_w * H_out * W_out * C_out / groups\n            batch_size = out.shape[0]\n            out_h, out_w = out.shape[2], out.shape[3]\n            c_in = self.in_channels\n            c_out = self.out_channels\n            k_h, k_w = self.kernel_size\n            flops = 2 * batch_size * (c_in // self.groups) * k_h * k_w * out_h * out_w * c_out\n            if self.bias is not None:\n                flops += batch_size * c_out * out_h * out_w\n            total_flops += flops\n\n        def linear_hook(self, inp, out):\n            nonlocal total_flops\n            batch_size = out.shape[0]\n            flops = 2 * batch_size * self.in_features * self.out_features\n            if self.bias is not None:\n                flops += batch_size * self.out_features\n            total_flops += flops\n\n        for m in model.modules():\n            if isinstance(m, nn.Conv2d):\n                hooks.append(m.register_forward_hook(conv_hook))\n            elif isinstance(m, nn.Linear):\n                hooks.append(m.register_forward_hook(linear_hook))\n\n        try:\n            with torch.no_grad():\n                dummy_input = torch.randn(*input_size, device=device)\n                model(dummy_input)\n            flops_g = round(total_flops / 1e9, 4)\n        except Exception:\n            # If forward pass fails, rough estimate based on conv weights\n            total_flops = sum(p.numel() * 2 for p in model.parameters())\n            flops_g = round(total_flops / 1e9, 4)\n        finally:\n            for h in hooks:\n                h.remove()\n\n    # Restore model state\n    model.to(orig_device)\n    if was_training:\n        model.train()\n\n    return {\n        \"params_m\": params_m,\n        \"flops_g\": flops_g,\n        \"total_params\": total_params,\n        \"total_flops\": total_flops,\n        \"method\": method\n    }\n\n\nclass MetricTracker:\n    \"\"\"\n    Accumulates and aggregates batch-level metrics across an evaluation epoch.\n\n    Tracks:\n    - Dice\n    - IoU (Jaccard)\n    - HD95 (95% Hausdorff Distance)\n    - Precision\n    - Recall\n    - Any custom metric (e.g. loss values) via update_metric()\n    \"\"\"\n\n    def __init__(self, metrics: Optional[List[str]] = None):\n        if metrics is None:\n            self.metric_names = [\"dice\", \"iou\", \"hd95\", \"precision\", \"recall\"]\n        else:\n            self.metric_names = list(metrics)\n\n        self.sums: Dict[str, float] = {}\n        self.counts: Dict[str, int] = {}\n        self.reset()\n\n    def reset(self) -> None:\n        \"\"\"Resets all tracked metric sums and sample counts.\"\"\"\n        self.sums = {name: 0.0 for name in self.metric_names}\n        self.counts = {name: 0 for name in self.metric_names}\n\n    def update(\n        self,\n        pred: Union[torch.Tensor, np.ndarray],\n        target: Union[torch.Tensor, np.ndarray],\n        threshold: float = 0.5,\n        empty_hd95_value: float = 100.0,\n        from_logits: Optional[bool] = None\n    ) -> Dict[str, float]:\n        \"\"\"\n        Calculates all standard metrics for the given batch and updates running totals.\n\n        Args:\n            pred: Predicted probabilities/logits.\n            target: Ground truth binary mask.\n            threshold: Decision threshold for binarization.\n            empty_hd95_value: Penalty HD95 when one mask is empty.\n\n        Returns:\n            dict: Batch metric values computed for this step.\n        \"\"\"\n        # Determine batch size\n        if isinstance(pred, torch.Tensor):\n            b_size = pred.shape[0] if pred.ndim >= 3 else 1\n        else:\n            b_size = pred.shape[0] if pred.ndim >= 3 else 1\n\n        batch_metrics = {}\n\n        if \"dice\" in self.metric_names:\n            d = calculate_dice(pred, target, threshold=threshold, from_logits=from_logits)\n            batch_metrics[\"dice\"] = d\n            self.update_metric(\"dice\", d, n=b_size)\n\n        if \"iou\" in self.metric_names:\n            j = calculate_iou(pred, target, threshold=threshold, from_logits=from_logits)\n            batch_metrics[\"iou\"] = j\n            self.update_metric(\"iou\", j, n=b_size)\n\n        if \"hd95\" in self.metric_names:\n            h = calculate_hd95(pred, target, threshold=threshold, empty_value=empty_hd95_value, from_logits=from_logits)\n            batch_metrics[\"hd95\"] = h\n            self.update_metric(\"hd95\", h, n=b_size)\n\n        if \"precision\" in self.metric_names or \"recall\" in self.metric_names:\n            pr = calculate_precision_recall(pred, target, threshold=threshold, from_logits=from_logits)\n            if \"precision\" in self.metric_names:\n                batch_metrics[\"precision\"] = pr.precision\n                self.update_metric(\"precision\", pr.precision, n=b_size)\n            if \"recall\" in self.metric_names:\n                batch_metrics[\"recall\"] = pr.recall\n                self.update_metric(\"recall\", pr.recall, n=b_size)\n\n        return batch_metrics\n\n    def update_metric(self, name: str, value: float, n: int = 1) -> None:\n        \"\"\"\n        Updates a specific metric or custom loss scalar.\n\n        Args:\n            name: Metric identifier (e.g. 'loss_bce', 'dice').\n            value: Mean value for the current step.\n            n: Number of samples in the current batch step.\n        \"\"\"\n        if name not in self.sums:\n            self.sums[name] = 0.0\n            self.counts[name] = 0\n            if name not in self.metric_names:\n                self.metric_names.append(name)\n\n        self.sums[name] += float(value) * n\n        self.counts[name] += n\n\n    def compute(self) -> Dict[str, float]:\n        \"\"\"\n        Computes the average across all accumulated batches.\n\n        Returns:\n            dict: {metric_name: mean_value}\n        \"\"\"\n        results = {}\n        for name in self.metric_names:\n            count = self.counts.get(name, 0)\n            if count > 0:\n                results[name] = round(self.sums[name] / count, 4)\n            else:\n                results[name] = 0.0\n        return results\n\n    def get_results(self) -> Dict[str, float]:\n        \"\"\"Alias for compute().\"\"\"\n        return self.compute()\n\n    def summary(self) -> str:\n        \"\"\"Returns a formatted summary string of all tracked metrics.\"\"\"\n        res = self.compute()\n        parts = [f\"{k.upper()}: {v:.4f}\" for k, v in res.items()]\n        return \" | \".join(parts)\n\n    def __str__(self) -> str:\n        return self.summary()\n\n    def __repr__(self) -> str:\n        return f\"<MetricTracker: {self.summary()}>\"\n"
}
REPO_TOOL_SOURCES = {
  "tools/build_mixed_cohort.py": "# -*- coding: utf-8 -*-\n\"\"\"\nBuild a leakage-free MIXED cohort (lesion + normal mucosa) from the held-out split.\n\nWhy this exists\n---------------\nKvasir-SEG is an all-positive cohort, so the triage gate's *specificity* (how\noften healthy frames are actually bypassed) and the real compute saving cannot be\nmeasured on it -- a limitation we state in the paper. This builder derives a mixed\ncohort from the 150 held-out test frames only:\n\n  * lesion samples  : the polyp frame itself (downsampled to 256x256)\n  * normal samples  : a mucosa crop from the *same* frame, at least `margin` px\n                      away from the annotated polyp (default 30 px), with black\n                      borders/letterboxing excluded\n\nBecause only test-split frames are used, the cohort is disjoint from every\ntraining and calibration frame used for the segmentation network. The cohort is\nthen split 75/75 (deterministic) into a CALIBRATION half and a CERTIFICATION\nhalf, so the conformal threshold is never evaluated on the data that chose it.\n\nOutputs (under data/kvasir_triage_test/):\n    calib/{images,masks}/<stem>_L.png | <stem>_N.png\n    certify/{images,masks}/...\n    manifest.json   (stem, kind, source frame for each sample)\n\"\"\"\nimport argparse\nimport json\nimport os\n\nimport cv2\nimport numpy as np\n\nROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))\nKSEG = os.path.join(ROOT, \"data\", \"kvasir_seg\")   # overridable via --kseg\nOUT = os.path.join(ROOT, \"data\", \"kvasir_triage_test\")  # overridable via --out\n\n\ndef find_mask(mask_dir, stem):\n    for ext in (\".jpg\", \".png\", \".jpeg\", \".tif\"):\n        p = os.path.join(mask_dir, stem + ext)\n        if os.path.exists(p):\n            return p\n    raise FileNotFoundError(f\"no mask for {stem}\")\n\n\ndef pick_normal_crops(img, mask, margin=20, min_size=64, max_crops=2):\n    \"\"\"Return up to `max_crops` well-separated mucosa crops from one frame.\"\"\"\n    d2 = _crop_score_map(img, mask, margin, min_size)\n    h, w = mask.shape\n    picks = []\n    work = d2.copy()\n    for _ in range(max_crops):\n        r = int(work.max())\n        if r < min_size / 2:\n            break\n        cy, cx = np.unravel_index(int(np.argmax(work)), work.shape)\n        r = int(min(r, min(h, w) / 2 - 1))\n        y0, x0 = max(0, cy - r), max(0, cx - r)\n        y1, x1 = min(h, cy + r), min(w, cx + r)\n        if (y1 - y0) >= min_size and (x1 - x0) >= min_size:\n            crop = img[y0:y1, x0:x1]\n            if (cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY) <= 12).mean() <= 0.25:\n                picks.append(crop)\n        # suppress a neighbourhood so the next pick is far away\n        y, x = int(cy), int(cx)\n        rad = max(min_size, r)\n        work[max(0, y - rad):y + rad, max(0, x - rad):x + rad] = 0\n    return picks\n\n\ndef _crop_score_map(img, mask, margin, min_size):\n    \"\"\"Largest axis-aligned window of the image whose pixels are all background.\"\"\"\n    h, w = mask.shape\n    free = (mask <= 127)\n    if free.sum() < min_size * min_size:\n        return None\n    # Distance transform gives the largest inscribed background square cheaply.\n    # Exclude endoscopic black borders.\n    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)\n    d2 = cv2.distanceTransform(((mask <= 127) & (gray > 12)).astype(np.uint8), cv2.DIST_L2, 3)\n    if d2.max() < min_size / 2:\n        return None\n    return d2\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--margin\", type=int, default=30)\n    ap.add_argument(\"--min-size\", type=int, default=64)\n    ap.add_argument(\"--max-per-frame\", type=int, default=2)\n    ap.add_argument(\"--out\", default=OUT)\n    ap.add_argument(\"--kseg\", default=KSEG, help=\"Kvasir-SEG root (contains images/ and masks/)\")\n    ap.add_argument(\"--splits\", default=None, help=\"path to splits.json (default: <kseg>/splits.json)\")\n    args = ap.parse_args()\n    kseg_dir = args.kseg\n    splits_path = args.splits or os.path.join(kseg_dir, \"splits.json\")\n\n    splits = json.load(open(splits_path, encoding=\"utf-8\"))\n    test_stems = list(splits[\"test\"])\n    img_dir, mask_dir = os.path.join(kseg_dir, \"images\"), os.path.join(kseg_dir, \"masks\")\n    rng = np.random.RandomState(42)\n\n    manifest = {\"margin\": args.margin, \"min_size\": args.min_size,\n                \"max_crops_per_frame\": args.max_per_frame, \"splits\": splits_path,\n                \"samples\": [],\n                \"policy\": \"leakage-free: derived from held-out test frames only; \"\n                          \"normal samples are real mucosal crops of the same frames\"}\n    for sub in (\"calib\", \"certify\"):\n        for kind in (\"images\", \"masks\"):\n            os.makedirs(os.path.join(args.out, sub, kind), exist_ok=True)\n\n    n_lesion = n_normal = n_skip = 0\n    for i, stem in enumerate(test_stems):\n        sub = \"calib\" if i % 2 == 0 else \"certify\"     # deterministic 75/75\n        img = cv2.imread(find_mask(img_dir, stem))\n        msk = cv2.imread(find_mask(mask_dir, stem), cv2.IMREAD_GRAYSCALE)\n        if img is None or msk is None:\n            n_skip += 1\n            continue\n        img = cv2.resize(img, (256, 256), interpolation=cv2.INTER_LINEAR)\n        msk_small = cv2.resize((msk > 127).astype(np.uint8) * 255, (256, 256), interpolation=cv2.INTER_NEAREST)\n        # lesion sample: the polyp frame\n        lp = os.path.join(args.out, sub, \"images\", f\"{stem}_L.png\")\n        lm = os.path.join(args.out, sub, \"masks\", f\"{stem}_L.png\")\n        cv2.imwrite(lp, img); cv2.imwrite(lm, msk_small)\n        manifest[\"samples\"].append({\"split\": sub, \"kind\": \"lesion\", \"stem\": stem,\n                                    \"file\": os.path.basename(lp)})\n        n_lesion += 1\n        # normal sample: mucosa crop from the same frame\n        crops = pick_normal_crops(img, msk_small, margin=args.margin,\n                                  min_size=args.min_size, max_crops=args.max_per_frame)\n        for k, crop in enumerate(crops):\n            crop = cv2.resize(crop, (256, 256), interpolation=cv2.INTER_LINEAR)\n            suffix = \"_N.png\" if k == 0 else f\"_N{k+1}.png\"\n            np_ = os.path.join(args.out, sub, \"images\", f\"{stem}{suffix}\")\n            nm = os.path.join(args.out, sub, \"masks\", f\"{stem}{suffix}\")\n            cv2.imwrite(np_, crop); cv2.imwrite(nm, np.zeros((256, 256), np.uint8))\n            manifest[\"samples\"].append({\"split\": sub, \"kind\": \"normal\", \"stem\": stem,\n                                        \"file\": os.path.basename(np_)})\n            n_normal += 1\n\n    with open(os.path.join(args.out, \"manifest.json\"), \"w\", encoding=\"utf-8\") as f:\n        json.dump(manifest, f, indent=2)\n    print(f\"[mixed-cohort] lesion={n_lesion} normal={n_normal} skipped_frames={n_skip} -> {args.out}\")\n    for sub in (\"calib\", \"certify\"):\n        k = [s for s in manifest[\"samples\"] if s[\"split\"] == sub]\n        les = sum(1 for s in k if s[\"kind\"] == \"lesion\")\n        print(f\"  {sub:8s}: {les} lesion + {len(k)-les} normal = {len(k)}\")\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "tools/eval_mixed_cohort_gate.py": "# -*- coding: utf-8 -*-\n\"\"\"\nConformal-certified screening cascade evaluated on a LEAKAGE-FREE MIXED cohort.\n\nThis is the experiment Kvasir-SEG alone cannot support: because every Kvasir-SEG\nframe contains a polyp, it can measure safety (miss rate) but not *specificity*\n(how often healthy frames are actually bypassed) and therefore not the real\ncompute saving. Here we use the mixed cohort built by tools/build_mixed_cohort.py\nfrom held-out test frames only:\n\n    lesion samples : the polyp frame                       (safety denominator)\n    normal samples : mucosa crops from the same frame      (specificity / savings)\n\nProtocol (matches the corrected CRC formulation):\n    * calibrate tau* on the CALIB half   -> Clopper-Pearson bound on lesions\n    * CERTIFY once on the CERTIFY half   -> FNR, normal-bypass rate, savings\n\nOutputs\n-------\nassets/probe_results/mixed_cohort_screening.json   metrics + operating-point table\nassets/mixed_cohort_screening_table.md             paper-ready markdown table\n\"\"\"\nimport argparse\nimport json\nimport os\nimport time\n\nimport cv2\nimport numpy as np\nimport torch\n\nROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))\nimport sys\nif ROOT not in sys.path:\n    sys.path.insert(0, ROOT)\nfrom models.fee_net_early_exit import FEENetEarlyExit   # noqa: E402\nfrom scipy import stats                                # noqa: E402\n\nCOHORT = os.environ.get(\"MIXED_COHORT_DIR\", os.path.join(ROOT, \"data\", \"kvasir_triage_test\"))\nOUTDIR = os.environ.get(\"PROBE_RESULTS_DIR\", os.path.join(ROOT, \"assets\", \"probe_results\"))\n\n\ndef load_split(sub):\n    man = json.load(open(os.path.join(COHORT, \"manifest.json\"), encoding=\"utf-8\"))\n    rows = [s for s in man[\"samples\"] if s[\"split\"] == sub]\n    imgs, kinds = [], []\n    for r in rows:\n        p = os.path.join(COHORT, sub, \"images\", r[\"file\"])\n        im = cv2.imread(p)\n        if im is None:\n            continue\n        im = cv2.resize(im, (256, 256), interpolation=cv2.INTER_LINEAR)\n        imgs.append(np.transpose(im, (2, 0, 1)).astype(np.float32) / 255.0)\n        kinds.append(r[\"kind\"])\n    return np.stack(imgs), kinds\n\n\ndef cp_upper(k, n, conf=0.05):\n    if n == 0 or k >= n:\n        return 1.0\n    return float(stats.beta.ppf(1 - conf, k + 1, n - k))\n\n\nclass Gate(torch.nn.Module):\n    \"\"\"stem + layer1 + Haar DWT + SCA + triage head (the screening path).\"\"\"\n\n    def __init__(self, ee):\n        super().__init__()\n        self.stem, self.layer1, self.dwt = ee.stem, ee.layer1, ee.dwt\n        self.wf1, self.attn = ee.wf1, getattr(ee, \"wf1_attn\", None)\n        self.head = ee.triage_head\n\n    def forward(self, x):\n        xs = self.stem(x)\n        e1 = self.layer1(xs)\n        _, lh, hl, hh = self.dwt(xs)\n        cat = torch.cat([lh, hl, hh], 1)\n        if self.attn is not None:\n            cat = self.attn(cat)\n        return self.head(e1, self.wf1(cat))[0]\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--ckpt\", default=os.path.join(ROOT, \"scratch\", \"kaggle_v3_results\",\n                                                   \"best_feenetearlyexit.pth\"))\n    ap.add_argument(\"--device\", default=\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    ap.add_argument(\"--target-fnr\", type=float, default=0.05)\n    a = ap.parse_args()\n\n    if not os.path.exists(a.ckpt):\n        raise SystemExit(f\"[gate] checkpoint not found: {a.ckpt}\")\n\n    ee = FEENetEarlyExit(3, 1, base_channels=48)\n    sd = torch.load(a.ckpt, map_location=\"cpu\", weights_only=False)\n    sd = sd.get(\"model_state_dict\", sd)\n    missing, unexpected = ee.load_state_dict(sd, strict=False)\n    print(f\"[gate] loaded {os.path.basename(a.ckpt)} | missing={len(missing)} unexpected={len(unexpected)}\")\n    ee.eval().to(a.device)\n\n    gate = Gate(ee).eval().to(a.device)\n    full = ee.eval()\n\n    data = {}\n    for sub in (\"calib\", \"certify\"):\n        x, kinds = load_split(sub)\n        data[sub] = (torch.from_numpy(x).to(a.device), kinds)\n        n_l = sum(1 for k in kinds if k == \"lesion\")\n        print(f\"[gate] {sub:8s}: {len(kinds)} samples ({n_l} lesion / {len(kinds)-n_l} normal)\")\n\n    # --- risk scores (bypassed when risk < tau) ---\n    risks = {}\n    with torch.no_grad():\n        for sub, (x, kinds) in data.items():\n            out = []\n            for i in range(0, len(x), 8):\n                # return_edge=True -> risk at index 2 (repo API)\n                out.append(full(x[i:i + 8], tau=0.5, return_edge=True)[2].float().cpu().numpy())\n            risks[sub] = np.concatenate(out)\n\n    # --- calibrate tau* on CALIB lesions only ---\n    cal = data[\"calib\"][1]\n    cal_les = np.array([r for r, k in zip(risks[\"calib\"], cal) if k == \"lesion\"])\n    tau, bound = 0.0, 1.0\n    for t in [i / 100.0 for i in range(1, 100)]:\n        k = int((cal_les < t).sum())\n        u = cp_upper(k, len(cal_les))\n        if u <= a.target_fnr:\n            tau, bound = float(t), u\n        else:\n            break\n    print(f\"[gate] calibrated tau* = {tau:.4f} (CP upper on calib lesions = {bound*100:.2f}%)\")\n\n    # --- certify once on the CERTIFY half ---\n    cer = data[\"certify\"][1]\n    r_cer = risks[\"certify\"]\n    les_idx = [i for i, k in enumerate(cer) if k == \"lesion\"]\n    nor_idx = [i for i, k in enumerate(cer) if k == \"normal\"]\n    les_miss = int((r_cer[les_idx] < tau).sum())\n    nor_bypass = int((r_cer[nor_idx] < tau).sum())\n    fnr = les_miss / max(len(les_idx), 1) * 100.0\n    bpr = nor_bypass / max(len(nor_idx), 1) * 100.0\n    upper = cp_upper(les_miss, len(les_idx))\n    bypass_frac = (les_miss + nor_bypass) / max(len(cer), 1)\n\n    # --- measured latencies (real savings) ---\n    xb = data[\"certify\"][0][:1]\n    with torch.no_grad():\n        for _ in range(20):\n            gate(xb); full(xb, tau=0.5, return_edge=True)\n        if a.device.startswith(\"cuda\"):\n            torch.cuda.synchronize()\n        t0 = time.time()\n        for _ in range(100):\n            gate(xb)\n        if a.device.startswith(\"cuda\"):\n            torch.cuda.synchronize()\n        t_gate = (time.time() - t0) / 100 * 1000\n        t0 = time.time()\n        for _ in range(20):\n            full(xb, tau=0.5, return_edge=True)\n        if a.device.startswith(\"cuda\"):\n            torch.cuda.synchronize()\n        t_full = (time.time() - t0) / 20 * 1000\n    eff_ms = bypass_frac * t_gate + (1 - bypass_frac) * t_full\n    speedup = t_full / max(eff_ms, 1e-6)\n\n    # --- operating-point table ---\n    table = []\n    for t in [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, tau]:\n        b = r_cer < t\n        lm = int(b[les_idx].sum()); nb = int(b[nor_idx].sum())\n        frac = (lm + nb) / max(len(cer), 1)\n        table.append({\n            \"tau\": round(float(t), 2),\n            \"fnr_pct\": round(lm / max(len(les_idx), 1) * 100, 2),\n            \"normal_bypass_pct\": round(nb / max(len(nor_idx), 1) * 100, 2),\n            \"cp_upper_fnr_pct\": round(cp_upper(lm, len(les_idx)) * 100, 2),\n            \"bypassed_frac\": round(frac, 4),\n            \"eff_latency_ms\": round(frac * t_gate + (1 - frac) * t_full, 2),\n            \"speedup_x\": round(t_full / max(frac * t_gate + (1 - frac) * t_full, 1e-6), 2),\n        })\n\n    try:\n        ckpt_ref = os.path.relpath(a.ckpt, ROOT)\n    except ValueError:          # different drive on Windows\n        ckpt_ref = os.path.abspath(a.ckpt)\n    res = {\n        \"checkpoint\": ckpt_ref,\n        \"device\": a.device,\n        \"cohort\": {\"calib\": data[\"calib\"][1].count(\"lesion\") and int(sum(1 for k in data[\"calib\"][1] if k == \"lesion\")),\n                   \"calib_normals\": int(sum(1 for k in data[\"calib\"][1] if k == \"normal\")),\n                   \"certify_lesions\": len(les_idx), \"certify_normals\": len(nor_idx)},\n        \"tau_star\": round(tau, 4),\n        \"calib_cp_upper_pct\": round(bound * 100, 2),\n        \"certify_fnr_pct\": round(fnr, 2),\n        \"certify_cp_upper_fnr_pct\": round(upper * 100, 2),\n        \"guarantee_holds\": bool(upper <= a.target_fnr),\n        \"normal_bypass_pct\": round(bpr, 2),\n        \"bypassed_fraction\": round(bypass_frac, 4),\n        \"gate_latency_ms\": round(t_gate, 3),\n        \"full_latency_ms\": round(t_full, 3),\n        \"effective_latency_ms\": round(eff_ms, 3),\n        \"measured_speedup_x\": round(speedup, 2),\n        \"operating_points\": table,\n    }\n    os.makedirs(OUTDIR, exist_ok=True)\n    with open(os.path.join(OUTDIR, \"mixed_cohort_screening.json\"), \"w\", encoding=\"utf-8\") as f:\n        json.dump(res, f, indent=2)\n    # per-sample risk scores for audit / re-calibration\n    with open(os.path.join(OUTDIR, \"mixed_cohort_risk_scores.csv\"), \"w\", encoding=\"utf-8\") as f:\n        f.write(\"split,kind,risk_score,bypassed\\n\")\n        for sub in (\"calib\", \"certify\"):\n            for r, k in zip(risks[sub], data[sub][1]):\n                f.write(f\"{sub},{k},{float(r):.6f},{int(r < tau)}\\n\")\n\n    md = [\"# Mixed-cohort conformal screening (leakage-free)\", \"\",\n          f\"- checkpoint: `{res['checkpoint']}`\",\n          f\"- cohort: calib {res['cohort']['calib']} lesion + {res['cohort']['calib_normals']} normal | \"\n          f\"certify {res['cohort']['certify_lesions']} lesion + {res['cohort']['certify_normals']} normal\",\n          f\"- calibrated tau* = **{res['tau_star']}** (CP upper on calib = {res['calib_cp_upper_pct']}%)\",\n          f\"- certify FNR = **{res['certify_fnr_pct']}%** (CP upper {res['certify_cp_upper_fnr_pct']}%) -> guarantee holds: **{res['guarantee_holds']}**\",\n          f\"- normal bypass rate = **{res['normal_bypass_pct']}%**; overall bypassed = {res['bypassed_fraction']*100:.1f}%\",\n          f\"- measured latency: gate {res['gate_latency_ms']} ms, full {res['full_latency_ms']} ms \"\n          f\"-> effective {res['effective_latency_ms']} ms ({res['measured_speedup_x']}x)\", \"\",\n          \"| tau | FNR (%) | Normal bypass (%) | CP upper FNR (%) | Bypassed frac | Eff. latency (ms) | Speedup (x) |\",\n          \"|---|---|---|---|---|---|---|\"]\n    for r in table:\n        md.append(f\"| {r['tau']:.2f} | {r['fnr_pct']} | {r['normal_bypass_pct']} | {r['cp_upper_fnr_pct']} | \"\n                  f\"{r['bypassed_frac']} | {r['eff_latency_ms']} | {r['speedup_x']} |\")\n    with open(os.path.join(ROOT, \"assets\", \"mixed_cohort_screening_table.md\"), \"w\", encoding=\"utf-8\") as f:\n        f.write(\"\\n\".join(md) + \"\\n\")\n    print(\"\\n\".join(md))\n\n\nif __name__ == \"__main__\":\n    main()\n"
}

for rel, src in REPO_SOURCES.items():
    dest = os.path.join(REPO_ROOT, rel)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    with open(dest, "w", encoding="utf-8") as f:
        f.write(src)

# Same for the mixed-cohort tools (imported by the models package, hence PYTHONPATH)
for rel, src in REPO_TOOL_SOURCES.items():
    dest = os.path.join(REPO_ROOT, rel)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    with open(dest, "w", encoding="utf-8") as f:
        f.write(src)

# Minimal package markers (the real ones drag in unrelated Laya/visualizer deps)
for pkg in ["models", "models/modules", "utils"]:
    p = os.path.join(REPO_ROOT, pkg, "__init__.py")
    os.makedirs(os.path.dirname(p), exist_ok=True)
    if not os.path.exists(p):
        open(p, "w").close()

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from models.unet import UNet
from models.attention_unet import AttentionUNet
from models.unet_plus_plus import UNetPlusPlus
from models.fee_net import FEENet
from models.fee_net_early_exit import FEENetEarlyExit
from utils.losses import CompoundLoss
from utils.metrics import MetricTracker

# Provenance guard: prove the classes come from the materialised repo, not leftovers
import models.fee_net as _fee_mod
import models.fee_net_early_exit as _ee_mod
import utils.losses as _loss_mod
_repo_real = os.path.realpath(REPO_ROOT)
for m in (_fee_mod, _ee_mod, _loss_mod):
    _f = os.path.realpath(m.__file__)
    assert _f.startswith(_repo_real), f"import leaked from {_f} (expected under {_repo_real})"
    print(f"[Provenance] {m.__name__} <- {_f}")
print("[Provenance] all model/loss modules resolved from the embedded repository sources")

# ---- Repository split membership (embedded at generation time) ----
REPO_SPLITS = {
  "test": [
    "cju13cgqmnhwn0988yrainhcp",
    "cju14hjh2ob2o0835ouz3r5aa",
    "cju17z0qongpa0993de4boim4",
    "cju1b75x63ddl0799sdp0i2j3",
    "cju1bm8063nmh07996rsjjemq",
    "cju1brhsj3rls0855a1vgdlen",
    "cju1c3218411b08014g9f6gig",
    "cju1c8ffau5770835g0g343o8",
    "cju1cj3f0qi5n0993ut8f49rj",
    "cju1d50a94qf50855wsowacrc",
    "cju1ewnoh5z030855vpex9uzt",
    "cju1ffnjn6ctm08015perkg37",
    "cju1fyb1d69et0878muzdak9u",
    "cju1hhj6mxfp90835n3wofrap",
    "cju2htabevq9108015qjei0x7",
    "cju2hw5gjlr5h0988so2qqres",
    "cju2igw4gvxds0878808qj398",
    "cju2lberzkdzm09938cl40pog",
    "cju2m71z2ywwv080131bcrsd3",
    "cju2ma647l0nj0993ot4deq2q",
    "cju2nbdpmlmcj0993s1cht0dz",
    "cju2nyc5f02m40801ojqbtiea",
    "cju2oi8sq0i2y0801mektzvw8",
    "cju2otvvv0l7z0855x7we8cb0",
    "cju2qdj95ru8g09886gfi9rsz",
    "cju2qn2fzs1vy0988l243cvzy",
    "cju2qs32r1vys07999conmbvx",
    "cju2qtee81yd708787bsjr75d",
    "cju2rpa30t07b0835im0erql0",
    "cju2sevf53lkx08558h5bpaig",
    "cju2spdagu1l50835da1f46fr",
    "cju2trtjf4qjd0878a2zle9v9",
    "cju2u2b9o4zvp08788qb9nqxj",
    "cju2ueb6j5ado0878vf5md13o",
    "cju2wve9v7esz0878mxsdcy04",
    "cju2yljr0yzhw0988ecf271ly",
    "cju2yv4imv6cz099314jveiib",
    "cju2zgbj9zmrw0835nnlzxj4c",
    "cju2zkpdl9h7t0799ix60teqg",
    "cju2zp89k9q1g0855k1x0f1xa",
    "cju2zpw4q9vzr0801p0lysjdl",
    "cju2zrojo9kcd0878ld2epejq",
    "cju2zxja9w1eh09933609ho9z",
    "cju303j5r062k098835zxfds5",
    "cju31ugmfb3dz0855xtqshki6",
    "cju31w6goazci0799n014ly1q",
    "cju32phw2bv130801yj7bkouq",
    "cju32srle1xfq083575i3fl75",
    "cju32upim1z7u0988l883nqp6",
    "cju33eqwbcch208012jikwdky",
    "cju33jon3ygbj0993pu22a4k6",
    "cju33mirdc8mj0799k33wzoes",
    "cju33yemn2qb20988wfjxximx",
    "cju33za6l2qy70988jhrlp2ev",
    "cju34c1xfyz920993itxkkfad",
    "cju34o6dbd2lo0855aqlcy1hs",
    "cju34ouumcznz07996gg1xq7v",
    "cju34repocy5208780gswillm",
    "cju34xspwzenf0993cyzajv9n",
    "cju35a77vdj4n08556jj2lgmc",
    "cju35i2e63uxr0835h7zgkg9k",
    "cju35k2fr3vc50988c85qkrwg",
    "cju3u4lxmg59o0755rz42b9en",
    "cju3u815rg4ek0850vvhtcvcm",
    "cju3uwz6ogsp10801h2r3bj5l",
    "cju3v0fl3gwce0755qkjhzmd4",
    "cju3v3ac9gyz30755hfqwyp1i",
    "cju3v72v5h1qz0818fggilwtq",
    "cju3x7xsaijq80818f0psavav",
    "cju3xhpvvimda0987ygrpzni2",
    "cju3xuj20ivgp0818mij8bjrd",
    "cju3xvoo2iqlc0817eku2r3wl",
    "cju40sdwukv3k0755y99ug1k8",
    "cju40w3hbkwpn08015rbs3wko",
    "cju412uwlkva50850d1ps1ww7",
    "cju41lojblbs307555jdci937",
    "cju41r6v2lcww0871ps8k8pf5",
    "cju43mkj9m8wb0871qiadahub",
    "cju45qbf3n9sa0987oonbkly9",
    "cju45ty6zn9oz0850qy4qnck1",
    "cju5bdwa3aatx0818b79i18zf",
    "cju5cky5xb0ay0801oxet697t",
    "cju5eq8c8ck690850vix98hv3",
    "cju5ew4h9cqaf0818rrczkmqh",
    "cju5f8hxdcxxn08188obby0ea",
    "cju5ft6mcd5q40987rhjgbrr6",
    "cju5hjxaae3i40850h5z2laf5",
    "cju5i5oh2efg60987ez6cpf72",
    "cju5klveuff6w0871wbibgh3m",
    "cju5ktjwofed70817eg58ef7u",
    "cju5tgbzhjllu08174ca41eus",
    "cju5ukkg6kv7u08011x2b6zl5",
    "cju5wcc90lu020850mjrxppv6",
    "cju5wi6bqlxy90755bu227nvb",
    "cju5wmvsdlx1j0871npgj8j4b",
    "cju5wphwwlu3m0987hh3ltg88",
    "cju5x28nzm7t907558ocq4bt7",
    "cju5xneamme2p0801qdf7fdwv",
    "cju5xq3tdm9fn0987pbedxdg5",
    "cju5yimthmlv80850zhoc90c2",
    "cju6ur9l9v9jq0755paud9uka",
    "cju6vta3kvazg0817qbeppjtm",
    "cju6wi3akvn8r0801px8eligc",
    "cju6wll7wvo3y08502pagos8m",
    "cju6wn57mvooj0850rp78hhy7",
    "cju6x4t13vyw60755gtcf9ndu",
    "cju6yywx1whbb0871ksgfgf9f",
    "cju6z1bzbwfq50817b2alatvr",
    "cju6z7e4bwgdd0987ogkzq9kt",
    "cju77196iyshb0850ycbto50a",
    "cju774fmayxif0818u2g79usw",
    "cju77k828z46w0871r0avuoo9",
    "cju7adqyj1jcx08712r1ro5gx",
    "cju7ajnbo1gvm098749rdouk0",
    "cju7bb3ss1uoo0755pmhyco7t",
    "cju7d7aut2a2p0818z4uxc6cd",
    "cju7dbppn28nx085097654msi",
    "cju7dhpsc2dnn0818025m6857",
    "cju7druhp2gp308715i6km7be",
    "cju7dubap2g0w0801fgl42mg9",
    "cju7dz5yy2i7z0801ausi7rna",
    "cju7f0ec32txj08184asb8w5f",
    "cju7f900s2o0k08175gl1giid",
    "cju7f9umg2olj0987fj5y285w",
    "cju7fcgbe2z3p07550vaflqdb",
    "cju7flevb2wii08188otgs9p2",
    "cju7fnfv02tt90801djnix9m8",
    "cju8432cmkgq90871cxe4iptl",
    "cju846ec0kj7z08012o10klrb",
    "cju84hibuktj80871u519o71q",
    "cju85l4yjlops0801fvmnwptf",
    "cju85plp7lmkw0850rx42jdpf",
    "cju87mrypnb1e0818scv1mxxg",
    "cju88aq6vo1ij0755c2ey7z7n",
    "cju88cddensj00987788yotmg",
    "cju88evxanv9r08176zkeovec",
    "cju88k75inzyb0850ccv5x3vk",
    "cju88nroho44508500129f1nh",
    "cju88vx2uoocy075531lc63n3",
    "cju8a2itsq4dv0755ntlovpxe",
    "cju8a3nhbpwnb0850d37fo2na",
    "cju8alhigqn2h0801zksudldd",
    "cju8b3ka8r64u0801fh18hk7l",
    "cju8b4ja9r2s808509d45ma86",
    "cju8bgdmqrksy0801tozdmraa",
    "cju8bysfgrzkl081786jwac09",
    "cju8c5223s8j80850b4kealt4",
    "cju8dn0c3u2v50801k8rvq02f",
    "cjz14qsk2wci60794un9ozwmw",
    "ck2bxknhjvs1x0794iogrq49k"
  ],
  "fold_0": {
    "train": [
      "cju0qkwl35piu0993l0dewei2",
      "cju0qoxqj9q6s0835b43399p4",
      "cju0qx73cjw570799j4n5cjze",
      "cju0roawvklrq0799vmjorwfv",
      "cju0rx1idathl0835detmsp84",
      "cju0s2a9ekvms080138tjjpxr",
      "cju0s690hkp960855tjuaqvv0",
      "cju0sxqiclckk08551ycbwhno",
      "cju0t4oil7vzk099370nun5h9",
      "cju0u82z3cuma0835wlxrnrjv",
      "cju0vtox5ain6099360pu62rp",
      "cju13fwthn9mq0835gacxgy01",
      "cju13hp5rnbjx0835bf0jowgx",
      "cju14g8o4xui30878gkgbrvqj",
      "cju14pxbaoksp0835qzorx6g6",
      "cju15czxqp3lv0835jvhgzurz",
      "cju15jr8jz8sb0855ukmkswkz",
      "cju15l5ubz9yh0855b3ivdpse",
      "cju15mhjczc8z0801kit5c6di",
      "cju15wdt3zla10801odjiw7sy",
      "cju160wshltz10993i1gmqxbe",
      "cju16ach3m1da0993r1dq3sn2",
      "cju16b6ynq8e40988m8vx0xnj",
      "cju16d65tzw9d0799ouslsw25",
      "cju16fpvhzypl0799p9phnlx6",
      "cju16jgnyzp970878melv7r25",
      "cju171py4qiha0835u8sl59ds",
      "cju175facms5f0993a5tjikvt",
      "cju17bz250pgd0799u1hqkj5u",
      "cju17g6ykn1cs0993dww6qdi8",
      "cju17hw9hr9c5098800fu4u8e",
      "cju17otoe119u0799nqcbl8n1",
      "cju17r8il13910799dr2wme2e",
      "cju17v6ih0u7808783zcbg1jy",
      "cju17x0j4nfc10993y31pvlgs",
      "cju1819curo000988pd5xcqme",
      "cju183od81ff608017ekzif89",
      "cju1871y11d6r0799k6cw4yze",
      "cju18849rrsgr0988p90hkygb",
      "cju18gzrq18zw0878wbf4ftw6",
      "cju18ibp219ub08783i6o98g7",
      "cju18kevfrojc0835bn90f1in",
      "cju1alwgo30z60855fm3y23sm",
      "cju1amqw6p8pw0993d9gc5crl",
      "cju1aqqv02qwz0878a5cyhr67",
      "cju1ats0y372e08011yazcsxm",
      "cju1b0y2e396p08558ois175d",
      "cju1b3zgj3d8e0801kpolea6c",
      "cju1bhnfitmge0835ynls0l6b",
      "cju1c0qb4tzi308355wtsnp0y",
      "cju1c4fcu40hl07992b8gj0c8",
      "cju1c6yfz42md08550zgoz3pw",
      "cju1cdxvz48hw0801i0fjwcnk",
      "cju1cfhyg48bb0799cl5pr2jh",
      "cju1cqc7n4gpy0855jt246k68",
      "cju1csmlc4ht10799b8ymmghg",
      "cju1cu1u2474n0878tt7v4tdr",
      "cju1cyjb5qtie0993njqne9m3",
      "cju1d31sp4d4k0878r3fr02ul",
      "cju1d96gsv62d09881b3wecw2",
      "cju1ddr6p4k5z08780uuuzit2",
      "cju1dg44i4z3w0801nyz4p6zf",
      "cju1dhfok4mhe0878jlgrag0h",
      "cju1dnz61vfp40988e78bkjga",
      "cju1dq3x1vgx109889c7wyirg",
      "cju1drnhbrb9409935wi7vkhg",
      "cju1efbr0rqxz09931z0lf4vf",
      "cju1egh885m1l0855ci1lt37c",
      "cju1egx9pvz2n0988eoy8jp23",
      "cju1ejj7dvqfa0835ra184v5m",
      "cju1erep75us208553i4ofwwe",
      "cju1euant5l960878iqj5vvto",
      "cju1euuc65wm00799m4sjdnnn",
      "cju1expq45zst0855rjqwwj4m",
      "cju1f15k3w4ct0835cmde6ypo",
      "cju1f320ewfyu0988ndz6blh5",
      "cju1f5x1164xv08555654c24r",
      "cju1f79yhsb5w0993txub59ol",
      "cju1f8w0t65en0799m9oacq0q",
      "cju1fb9236a110801yvg0fwju",
      "cju1fj6axwfp30835ukhuzhw9",
      "cju1fjsb4sipq09931lvd8e41",
      "cju1fm3id6gl50801r3fok20c",
      "cju1fmsyf6gxb0801cimx2gle",
      "cju1fr4etsmrr09933u4t4aql",
      "cju1ftaji6isw0855108yqcse",
      "cju1fuoa4wmc50835qfd11sp9",
      "cju1g20bdwq6u0835e16xugcd",
      "cju1g4nsb6ngy0799l4ezm8ab",
      "cju1gghyjwxt80835vx0wgxw0",
      "cju1gkndf6yi10801o1qnje19",
      "cju1gv7106qd008784gk603mg",
      "cju1h5w4wxajx0835mc954kxy",
      "cju1h89h6xbnx08352k2790o9",
      "cju1haab178i70799tk9z8y8x",
      "cju1hirfi7ekp0855q0vgm9qq",
      "cju1hmff8tkp809931jps6fbr",
      "cju1hp9i2xu8e0988u2dazk7m",
      "cju1hs0za7jha0855vj0mdrjt",
      "cju1hyolc7aqu0878rrkfn1lr",
      "cju2hfqnmhisa0993gpleeldd",
      "cju2hgsptlfam0835o3b59h1o",
      "cju2hjrqcvi2j0801bx1i6gxg",
      "cju2hlm19vjjf0801o69qnber",
      "cju2hqt33lmra0988fr5ijv8j",
      "cju2hugv9vget0799hhk7ksvg",
      "cju2hx006vidl0799igm81vmh",
      "cju2i03ptvkiu0799xbbd4det",
      "cju2i3hzclw3o0988rrgh911i",
      "cju2i8br1vqtd08784u6vmcjk",
      "cju2iatlki5u309930zmgkv6h",
      "cju2ij9uiic2l09933ljiv6gm",
      "cju2lcyfgkf5809932fn9gucn",
      "cju2lyynuymli0855g7fxgbhe",
      "cju2lz8vqktne0993fuym6drw",
      "cju2m56cryvqd0801gtn2yp8t",
      "cju2mh8t6p07008350e01tx2a",
      "cju2nd7l7z98o0799gfjvyfmw",
      "cju2nfnvxzdkd0878399axlco",
      "cju2nguelpmlj0835rojdn097",
      "cju2nnqrqzp580855z8mhzgd6",
      "cju2np2k9zi3v079992ypxqkn",
      "cju2nqapmzvk20801f9us40dx",
      "cju2nsmwjlzyl0993jl80chvz",
      "cju2ntxtdzlvu0799xl3j9pan",
      "cju2omjpeqj5a0988pjdlb8l1",
      "cju2oo0wh0bqy0878biujeyhe",
      "cju2osuru0ki00855txo0n3uu",
      "cju2ouil2mssu0993hvxsed6d",
      "cju2p4ddkmzxj0993p94o62av",
      "cju2p91qir00k08350ddfif0w",
      "cju2pag1f0s4r0878h52uq83s",
      "cju2phaksnahz0993yxogjcpv",
      "cju2pjb9v0ywn0878j5g5n69j",
      "cju2pkwt3r8b90988v2ywq1px",
      "cju2qfie4rvz508357kad9z5o",
      "cju2qh5le1ock0878oahaql7d",
      "cju2qozsk20cq0855ugrg3cri",
      "cju2qqn5ys4uo0988ewrt2ip2",
      "cju2qu37qobl50993aw7ghcfq",
      "cju2qxxko229x08786gvxxhur",
      "cju2qz06823a40878ojcz9ccx",
      "cju2r11x7sdgx0988o8ule0wl",
      "cju2r2obh2bjm08553kng0rh7",
      "cju2r6mt2om21099352pny5gw",
      "cju2r91dg2k090801bh0xzbxk",
      "cju2rga4psq9n09881z519xx0",
      "cju2rkjfwoxys0993x768l1j2",
      "cju2rlqdnoz9k0993cpjae3x0",
      "cju2rmd2rsw9g09888hh1efu0",
      "cju2rn0hasxri0835nfy3buay",
      "cju2rnkt22xep0801as160g9t",
      "cju2ro5jqsy680988pi6qsujw",
      "cju2rxm8rpbaf0993o3qr2oph",
      "cju2rz4k434s70855wwx3ddtx",
      "cju2rzpsmtb0f0835jabkbao1",
      "cju2s2527pfyr0993l3h1149a",
      "cju2s9g11pnra0993gn4eh793",
      "cju2saez63gxl08559ucjq3kt",
      "cju2srvy5440s0801y1ba9akr",
      "cju2sszfq3uye0878sucelzk2",
      "cju2suk42469908015ngmq6f2",
      "cju2sxf3iqbpv09937iksn8ep",
      "cju2syxa93yw40799x2iuwabz",
      "cju2t16vuucaq0835xcpsivn2",
      "cju2t2ivz43i10878zeg8r1br",
      "cju2t3ibkuecr0835o7si16zv",
      "cju2t62nq45jl0799odpufwx6",
      "cju2t9tdwuk700835kv0ljmtl",
      "cju2tjrog4jy30878pawyazqc",
      "cju2top2ruxxy0988p1svx36g",
      "cju2tqfgw4oat0799rn0g5b2z",
      "cju2tvrvm53ws0801a0jfjdxg",
      "cju2txjfzv60w098839dcimys",
      "cju2tzypl4wss0799ow05oxb9",
      "cju2ulk385h170799rlklxob0",
      "cju2uokeg5jm20799xwgsyz89",
      "cju2upu4evw7g08358guwozxv",
      "cju2urqpwvxw70835rvndvtsi",
      "cju2uwz9f5yf1085506cfamfx",
      "cju2uzabhs6er0993x3aaf87p",
      "cju2wtwj87kys0855kx6mddzw",
      "cju2wx0gh7fpz0878wwyd9ep8",
      "cju2wxv0hxs2f09884w48v8fi",
      "cju2wzu8wxtgu09880ku9x1pg",
      "cju2x7vw87mu30878hye2ca0m",
      "cju2xa3i4y0160988i679zsqd",
      "cju2xbk0080y80801eghyddi2",
      "cju2xd75m82720801q4s4ik3n",
      "cju2xjz2ju8pe0993ysv9wg17",
      "cju2xlcqxy9c60988vjacdznb",
      "cju2xs6na81t20878pt6nkfip",
      "cju2xyd9vyi7m098831qcucse",
      "cju2y0z6g87p10878fpk5d3rq",
      "cju2y26c588bo07993ksd8eoz",
      "cju2y40d8ulqo0993q0adtgtb",
      "cju2y5zas8m7f0801d34g5owq",
      "cju2y8s56ymqr083541ggdsml",
      "cju2yb31a8e8u0878wdashg7o",
      "cju2ycp1u8g2r0799jslnp7cz",
      "cju2yi9tz8vky0801yqip0xyl",
      "cju2yo1j1v0qz09934o0e683p",
      "cju2yyhsp933j0855hp32e012",
      "cju2z1nxlzaj40835wj81s1iy",
      "cju2z2x3nvd3c099350zgty7w",
      "cju2z6ez69g4u0801qwt088lw",
      "cju2z9vlp9j0w0801oag91sy9",
      "cju2zblxw9848087853csbrx1",
      "cju2zdhsczmn50988z64qwg2q",
      "cju2zdvjn9h7r08553cp4eed5",
      "cju2zi4l09f5807991s8do2b2",
      "cju2zjcvj9qma0801dk71hhi0",
      "cju2zm0axztpe0988r8s9twjr",
      "cju2zo0fwzv580988qlijd2xa",
      "cju2zy1e49pqk0878t6ncqn12",
      "cju300m3s04fg0988uzupuf7z",
      "cju30525w04r10835ygp257sb",
      "cju306x7w05nb0835cunv799x",
      "cju30ajhw09sx0988qyahx9s8",
      "cju30bmab08bi0835mvlr6e0r",
      "cju30j1rgadut0801vuyrsnt8",
      "cju30k2z40ds308353kdew70n",
      "cju30lncba3ny0878jwnous8n",
      "cju30mm25a53s0799qa5wiqe8",
      "cju30nyxe0gfb0835p256yoju",
      "cju30ov1oah920801mi8thuyg",
      "cju30qbm1ad3x0855znuhpz9u",
      "cju30u1hbakn808019g15nb8b",
      "cju30ywtc0oar0835bp2en7ec",
      "cju310f6val1v0855xo8tc3gu",
      "cju31rb7vb6110801p9rhacuw",
      "cju31y80qbawn0801twwm2l5s",
      "cju323ypb1fbb0988gx5rzudb",
      "cju324q101fhe08350wae9cif",
      "cju326h4v1gxw08352px40p7r",
      "cju32a52lb9rc0799xi40qs00",
      "cju32csyfblyh080170aa3x5p",
      "cju32fhnhbds40799broyoptc",
      "cju32jcdabepz0878d0cznmfe",
      "cju32l161bi1v07990vm376in",
      "cju32pzh9bpw10855q4vaxfhe",
      "cju32qr9tbvsj08013pkpjenq",
      "cju32zhbnc1oy0801iyv1ix6p",
      "cju330ofbc2l30801th5g3hw6",
      "cju33231uy4gi0993qc7b1jch",
      "cju334jzo261t0835yqudnfs1",
      "cju336l68y7if0993wf092166",
      "cju3381d8bz3h07991xtl7ra0",
      "cju33belnbyhm0878yxl42233",
      "cju33o12x2jm50988944mxq0v",
      "cju33qpdvc9g0087825jhf3s9",
      "cju33x0f22peh0988g0ln7w5v",
      "cju34aozyyy830993bn16u32n",
      "cju34ds2531520988qjpqt6e3",
      "cju34eqjpcpm508788b3lhp97",
      "cju34i3qvcyog0855qiejxx5w",
      "cju34m7h536wq0988xz7gx79v",
      "cju34sh43d8zm08019xbwhc0o",
      "cju34uhepd3dd0799hs8782ad",
      "cju34ymm8d6700799uop0cw33",
      "cju3518w2d838079939fqztbc",
      "cju3521y5d5mq0878t3ezsu4p",
      "cju353d1eda8c07992afde611",
      "cju35740hzm0g0993zl5ic246",
      "cju358pwtdby20878cg7nm0np",
      "cju35atpxdjot0855q46aqrd0",
      "cju35c4wzdhow0799h6eq4sgs",
      "cju35eg0tdmjt085525sb4bua",
      "cju35ldepdtlm0801yv79y8vu",
      "cju35mdz73x890835eynq1h9v",
      "cju35oyvd3y850988km12hdz1",
      "cju3tp94kfstl08181awh6z49",
      "cju3tsh4lfsok0987w6x3a0v1",
      "cju3ttznuftyf09875t11850w",
      "cju3tvffffx5f0818t5ov22al",
      "cju3tx1qyg0c907552fglumhc",
      "cju3u1c8tfyqx08503iedc3mx",
      "cju3u39fog1bo0871lxjrabks",
      "cju3ua8u0g9rg0801uayhdxhu",
      "cju3uhb79gcgr0871orbrbi3x",
      "cju3ul8dogf1z09872y2ecowp",
      "cju3uz4o6gr9z0850lhxyxvsj",
      "cju3v664kh0px0818y4y7wolf",
      "cju3wstckialg0871xs0vevsj",
      "cju3x2s11ibzi0817kk284k0j",
      "cju3x4blzieu30850x10uuvbm",
      "cju3x5u2tiihx0818914gzxy1",
      "cju3x9lttikfb0818a0g104zn",
      "cju3xeexgii1j0817zs68tb4g",
      "cju3xjqtpikx50817tppy6g84",
      "cju3xtufwiv9c0818djsc4cqd",
      "cju3xzvnzj0hd0755xprz39nj",
      "cju3y21quj0ir0818kgjagr15",
      "cju3y54kwj3nr0801biidlb4e",
      "cju3ya7goj6at0818v2l5ay7f",
      "cju3yb47cj1xq0817zfotbni4",
      "cju3yht87j83m08507yk1u1fg",
      "cju3ykamdj9u208503pygyuc8",
      "cju40jl7skiuo0817p0smlgg8",
      "cju40m0rjkpw80871z6n6yg1u",
      "cju40poe4kt9s0755f9cnm3h5",
      "cju40u30gkuzc0871rq7t666d",
      "cju40wto5kxwi0755it190f2k",
      "cju410dnfl0960755y8lu8d79",
      "cju414lf2l1lt0801rl3hjllj",
      "cju418rckl3ur08012psrx1r1",
      "cju41kd7yl4nm0850gil5qqwh",
      "cju41nz76lcxu0755cya2qefx",
      "cju41p90plcsx08018cnzpndc",
      "cju41z76wlgbz0801qdetlvby",
      "cju424hy5lckr085073fva1ok",
      "cju42dwedlmk60871jbgu4ehi",
      "cju42g865lorv07552ytz6xxa",
      "cju42m60jlpcm08186kqppzqv",
      "cju42nm68lpyo0818xvvqmupq",
      "cju42qet0lsq90871e50xbnuv",
      "cju42romflni20817etb9a0fl",
      "cju42tauqlo5p08171l3cuo4b",
      "cju42u5bjlvi10801dc13sskp",
      "cju42wamblrqn098798r2yyok",
      "cju42xpi8lw4w0871ve317a1p",
      "cju430pm2lz0y0755jkhcc3d1",
      "cju43b8daly4408170e5ev06g",
      "cju43c92lm5cj0755lorsorfg",
      "cju43gfosm63n08714rpih8pe",
      "cju43h43am1dy08176gwfhmnt",
      "cju43in5fm22c08175rxziqrk",
      "cju43kj2pm34f0850l28ahpni",
      "cju43lcnum9y10755bjs7z87f",
      "cju43o6n7m9nk087191ijwqq9",
      "cju45jpvfn6c809873pv1i34s",
      "cju45lbgznahl08180xz1h7u6",
      "cju45n0oxn5vu08500yfrt9jn",
      "cju45ofd9ne1j0801ri8dup7t",
      "cju45pm27n80u08174kyow1gj",
      "cju45rj7ln8980850a7821fov",
      "cju45t5ddnbio0987qtqzx762",
      "cju45v0pungu40871acnwtmu5",
      "cju5bbtwsa8cl0987wgfsqpao",
      "cju5bf6hxa6m50817rbwettgu",
      "cju5bhv81abur0850ean02atv",
      "cju5bj926aiec07559rshy4wa",
      "cju5bmhdcafs909878qfzrqzi",
      "cju5boicjagt20871b1fotkh4",
      "cju5buy2bal250818ipl6fqwv",
      "cju5bwhapakm90987c1v4z46a",
      "cju5bycdkalkb09875f7bfrvx",
      "cju5c5xc7algd0817pb1ej5yo",
      "cju5c7oijaqmq09878qwgqv8n",
      "cju5ca9hcatkc0801jzwe7tfx",
      "cju5cetivauok0987ok3e5bre",
      "cju5cjh3xattc0817j2vbulzi",
      "cju5clr68b48r0755cmuvponm",
      "cju5cu8qkb84x08186jwo8yin",
      "cju5ddda9bkkt0850enzwatb1",
      "cju5eftctcdbj08712gdp989f",
      "cju5ekty5ckzf07550c9u3ckk",
      "cju5enq1tcn1i0755hnkon787",
      "cju5es375cnzy0801nkq35ffs",
      "cju5f0dezct4q08183ydw11dx",
      "cju5fb86jd1jp0755b1ukbhq5",
      "cju5fi0yxd3ei0801v7u0yudn",
      "cju5fs6j6d8350801vglraq4u",
      "cju5fu081d8gc0818l3yylujk",
      "cju5fw37edaae0801vkwvocn7",
      "cju5fydrud94708507vo6oy21",
      "cju5g163vd6mt0817uccuga6u",
      "cju5gucasds9d0801019axylx",
      "cju5h57xedz5h0755mjpc8694",
      "cju5hi52odyf90817prvcwg45",
      "cju5hl8nee8a40755fm8qjj0o",
      "cju5hqz50e7o90850e0prlpa0",
      "cju5ht88gedbu0755xrcuddcx",
      "cju5huurrecm70801y680y13m",
      "cju5hwonqedw10801vsd3w6kk",
      "cju5hyi9yegob0755ho3do8en",
      "cju5i39mreass0817au8p22zy",
      "cju5jx7jzf7c90871c2i9aiov",
      "cju5jz5fff8c50871hbe6108f",
      "cju5k3j3uf6de0817hszzfr7n",
      "cju5k503sfa5f0871lx0rpu5y",
      "cju5knbbqfipk080128cggukq",
      "cju5kre09fhka0850h7b1898j",
      "cju5nxkujgscq0817l9gss626",
      "cju5nyu31gv8e0871zpk74a2n",
      "cju5o4pk9h0720755lgp9jq8m",
      "cju5tenjojp1j0755ms4949h2",
      "cju5thdbrjp1108715xdfx356",
      "cju5u4pywk81x0817vn9pe14z",
      "cju5u8gz4kj5b07552e2wpkwp",
      "cju5udcufki0s09874ll1dbr5",
      "cju5ufn3skquf0818dhapnhba",
      "cju5uget8krjy0818kvywd0zu",
      "cju5uhrdwkmsu0817ervv91l8",
      "cju5uxjnol2r509871qv2yeia",
      "cju5v8pgplg6k0755rvi2t63h",
      "cju5vbo6jldrt0871jf6f1700",
      "cju5vcmrqla7i0817x4sp4pqw",
      "cju5vgawslbe30987ndeepc1b",
      "cju5vi4nxlc530817uoqm2m7a",
      "cju5von04litr08718j8po40a",
      "cju5vutu7ll8w0871dfp92n9p",
      "cju5vwbr4lhqn0987a1pji0ux",
      "cju5vxuc5loxw0818u8xgf45p",
      "cju5w7xn0lrkq0801f9k0htgx",
      "cju5waeduln160817w0agirve",
      "cju5wj0faly5008187n6530af",
      "cju5wkonqlrl409877y8zvnub",
      "cju5woy82m07m08505dmjg7g1",
      "cju5wqonpm0e60801z88ewmy1",
      "cju5wrapcm2290818jsh26ppb",
      "cju5wrrs0m2af0818vmnajbtw",
      "cju5wtdu4m0im0871mix0yvc0",
      "cju5wuhm1lwm40987vugqn3vv",
      "cju5x00l6m5j608503k78ptee",
      "cju5x15djm7ae0755h8czf6nt",
      "cju5x7iskmad90818frchyfwd",
      "cju5xjn5mm78b09871spyqhhr",
      "cju5xkwzxmf0z0818gk4xabdm",
      "cju5xopi0md7q0871r1sjc1av",
      "cju5y4hgqmk0i08180rjhbwvp",
      "cju5y7buemcw80987p0r30g9f",
      "cju5y84q3mdv50817eyp82xf3",
      "cju5yeqiwmkgl0801fzv2douc",
      "cju5ygh1zmmdi0755uod5e17i",
      "cju5yhgznmkzb0801cji2vi8j",
      "cju5yjq1pmlgc0801z0t24bly",
      "cju5ymyd8mmdc0801ry3by1xr",
      "cju6us80mv1b50871ebyq2wxa",
      "cju6ut4l8va6y0755tyw3vfqq",
      "cju6uzxk0v83p0801rcwnexdu",
      "cju6v1m1xv07w09870ah3njy1",
      "cju6v3bb2v7xo085090blubyw",
      "cju6vgdmivcvb08018fra5lnv",
      "cju6vrs1ov8cr098788h8gs6j",
      "cju6vucxvvlda0755j7msqnya",
      "cju6vvb8svhed0801jjcquh5e",
      "cju6vvxsev9y30987kespucdg",
      "cju6w733bveoz0817e600tw72",
      "cju6wt9jvvn500871hjn3t3g0",
      "cju6wuojavt740818b5qcv3iw",
      "cju6x35ervu2808015c7eoqe4",
      "cju6x97w4vwua0850x0997r0a",
      "cju6xa0qmvzun0818xjukgncj",
      "cju6xifswvwbo0987nibtdr50",
      "cju6xlygpw7bs0818n691jsq4",
      "cju6xmqd9w0250817l5kxfnsk",
      "cju6ywm40wdbo0987pbftsvtg",
      "cju6z2616wqbk07555bvnuyr1",
      "cju6z600qwh4z081700qimgl9",
      "cju76erapykj30871x5eaxh4q",
      "cju76l27oyrw907551ri2a7fl",
      "cju76lsehyia10987u54vn8rb",
      "cju773hsyyosz0817pk1e7sjq",
      "cju7787c5yy3l080159mwqsnj",
      "cju77bvg0yv4r0987yh60xmjo",
      "cju77g99iyxc00817zqi2ppor",
      "cju77idwfz36d0871tzfzz51i",
      "cju77q10sz9ug0801449wu1nu",
      "cju77re6fz5bb0817vp9redjg",
      "cju77t0razbvm080106o56289",
      "cju77u1sjz77b0817ft44r3fk",
      "cju77vvcwzcm50850lzoykuva",
      "cju783tmkzkqu081803g7q5vk",
      "cju784jpdzeae0987q5ypq883",
      "cju785htizjzo08017tvlhtg4",
      "cju787jnjzjuj0871p94nck9g",
      "cju7aez2x1jtj0871ztezs3oi",
      "cju7afqon1ip40850ue2308b6",
      "cju7agj961l2r0818z29iq8yn",
      "cju7ahtkb1jr90801jck4kbds",
      "cju7aklv31h4309871m29l4e7",
      "cju7alcgr1lsr0871riqk84z7",
      "cju7amjna1ly40871ugiokehb",
      "cju7ap09p1kz10850ldccjebj",
      "cju7apr0c1qqm0755s7msqot4",
      "cju7aqkue1i2k09879uzcpt8r",
      "cju7arvfe1ldu0850erdmphgj",
      "cju7atnm31if40817pqclnjer",
      "cju7awzmu1ncs0871hziy65zx",
      "cju7azuu31mia0801pf9ib5ed",
      "cju7b10ce1mnm08011c5bwyr4",
      "cju7b1ygu1msd0801hywhy0mc",
      "cju7b3f5h1sm40755i572jden",
      "cju7b4mtw1n9n080186209f3d",
      "cju7b5afm1nfw0801xqm8bf8q",
      "cju7b9vcs1luz0987ta60j1dy",
      "cju7bd1qu1mx409877xjxibox",
      "cju7bduyq1rjf08719giru9ho",
      "cju7bf1lp1shi081835vs84lc",
      "cju7bgnvb1sf808717qa799ir",
      "cju7bmi1v1pnj0987pa52jjok",
      "cju7cl8zm1xcu0817ado0jpas",
      "cju7cq6su27qv075574dir0r3",
      "cju7crgxa28550755wbsgqkel",
      "cju7csvlb22fr0850lvm45n3x",
      "cju7cue9b232j0801qdzk1ykj",
      "cju7cufm7298k0755j09uf3of",
      "cju7d1tvt25bu08019dvw3uff",
      "cju7d2q1k27nf08715zshsckt",
      "cju7d3oc82cho0755dajlwldz",
      "cju7d4jk723eu0817bqz2n39m",
      "cju7d5m0p23kn09871rk7pu3v",
      "cju7d6ux323ze0987xos3srkx",
      "cju7d9seq29zd0871nzl2uu5m",
      "cju7da88w2eod0755wejzynvt",
      "cju7dda8w2br20818zhsuz8s7",
      "cju7dff9529h208503w60lbil",
      "cju7dlk532dsh0871zvr6qz0r",
      "cju7dn24o296i09871qfxb8s2",
      "cju7do8c72dbo0801vxfzxdc4",
      "cju7dp3dw2k4n0755zhe003ad",
      "cju7dqcwi2dz00850gcmr2ert",
      "cju7dsrtb2f8i085064kwugfk",
      "cju7dtb1e2j0t0818deq51ib3",
      "cju7dvl5m2n4t0755hlnnjjet",
      "cju7dwe282dc309876rco45ts",
      "cju7dxffn2eam0817qxosfwch",
      "cju7dymur2od30755eg8yv2ht",
      "cju7ea4om2l910801bohqjccy",
      "cju7ebe962hr409872ovibahw",
      "cju7eea9b2m0z0801ynqv1fqu",
      "cju7ehljc2or70871261br8ai",
      "cju7ejm2l2ncl0801wq6y84nw",
      "cju7ekbo32pft0871fv7kzwb9",
      "cju7el6xv2k520817qxx9wdr5",
      "cju7emdni2py40871ivhxjtut",
      "cju7eotqi2qea0871y8yc7tqh",
      "cju7er4kc2opa0801anuxc0eb",
      "cju7es23b2vcp0755gpbm9s7v",
      "cju7et17a2vjk0755e743npl1",
      "cju7eueum2oqn0850rodmx8zo",
      "cju7ev2b12owa08500bpfpwyw",
      "cju7ez7r22qbc08015xfoz2wb",
      "cju7ezs7g2mxm098787atbran",
      "cju7f6cqy2ur20818t1saazbm",
      "cju7fazv92ywx0755xov2erga",
      "cju7fbndk2sl608015ravktum",
      "cju7fd6yt2p740987wkr8exo1",
      "cju7fen322ou10817ziqkob4k",
      "cju7ff97z2ow40817u2r83my5",
      "cju7fmvpk2q170987v6i3ola8",
      "cju7fq7mm2pw508176uk5ugtx",
      "cju7frtqu2xa20818wq8r9fzf",
      "cju7g7ba42z310987bqzbi2bq",
      "cju83h9ysjwe808716nt35oah",
      "cju83ipu3jwpx0801z5pvguf8",
      "cju83k8fyjsxr0817d6nxs6r4",
      "cju83kxitjv340987z09m0ezy",
      "cju83mki1jv5w0817kubxm31r",
      "cju83qd0yjyht0817ktkfl268",
      "cju83syhdk6gs0801rf1rekdl",
      "cju83u9ftk3ni0987qnhlcinv",
      "cju83vvmik9wa08710yeh7cuk",
      "cju83wwn1k55e0850kw6i2d81",
      "cju83yddek68q0850d2x7zfkm",
      "cju8402x1kcy70801t6kz6bdi",
      "cju8418jhkf7d0818ga2v0xq0",
      "cju843yjskhq30818qre4rwm2",
      "cju847pxykriq0755268ktrk2",
      "cju849c23kgnk0817cgv2hw1e",
      "cju84aoa3ktwn0755pfl4gfwd",
      "cju84gpefknwm098714oq8q61",
      "cju84ih17kp5l09876bkooocl",
      "cju84kplnl1y30755ropua1b0",
      "cju8567gdlcbq0801dwwyo2jt",
      "cju857ad0l88m0817qx4cwxnf",
      "cju858eswlepn0871pzvdrhj1",
      "cju85a8h8llwm07559wxg4t5w",
      "cju85bf1algsq0871y9gtlq97",
      "cju85c2d4ln1b0755zz1z3onx",
      "cju85citjlnfm0755i4rk5tqj",
      "cju85fc11ljr40818edpb0inh",
      "cju85hguellg50818kwu3s8d1",
      "cju85ia1slh220987y7c20sm2",
      "cju85je7vlht70817c9jcjwi4",
      "cju85mpuglq8k0818d2it6hzb",
      "cju85qefyln6v0850szeb9byi",
      "cju85rkbnlo1c08503uxcpax1",
      "cju87kbcen2av0987usezo8kn",
      "cju87nkyrnb970801q84m47yt",
      "cju87ox0kncom0801b98hqnd2",
      "cju87r56lnkyp0755hz30leew",
      "cju87tyddnnad0755bj0wxahe",
      "cju87vqa0ndwg0850onjdz7ol",
      "cju884985nlmx0817vzpax3y4",
      "cju887ftknop008177nnjt46y",
      "cju888fr7nveu0818r9uwtiit",
      "cju88fpm4o0tl0871w1i6a4ds",
      "cju88itqbny720987hxizbj5y",
      "cju88msmoo3470817m441j4sg",
      "cju88oh0po9gq0801nge4tgr1",
      "cju88q6h6obpd0871ckmiabbo",
      "cju88rl5eo94l0850kf5wtrm1",
      "cju88t4fvokxf07558ymyh281",
      "cju88trl3ogi208716qvti51b",
      "cju88v2f9oi8w0871hx9auh01",
      "cju88y1mwoln50871emyfny1g",
      "cju88z8bson4h0871nnd7fdxo",
      "cju890guyoiti098753yg6cdu",
      "cju893jmdompz0817xn3g1w4h",
      "cju8a1jtvpt9m081712iwkca7",
      "cju8a56vxpy780850r45yu4wk",
      "cju8a84g0q76m0818hwiggkod",
      "cju8abobpqbir08189u01huru",
      "cju8adb60qbiu080188mxpf8d",
      "cju8aeei7q8k308173n9y4klv",
      "cju8aj01yqeqm0850lhdz3xdw",
      "cju8amfdtqi4x09871tygrgqe",
      "cju8apjewqrk00801k5d71gky",
      "cju8aqq8uqmoq0987hphto9gg",
      "cju8arof2qpf20850ifr1bnqj",
      "cju8ashhnquqr0801rwduzt7d",
      "cju8at3s1qqqx0850hcq8nmnq",
      "cju8auylgqx0z0871u4o4db7o",
      "cju8aw9n1qyg10801jkjlmors",
      "cju8axq24r4an0755yhv9d4ly",
      "cju8ayeq7r1fb0818z1junacy",
      "cju8azmhcr66e0755t61atz72",
      "cju8b0jr0r2oi0801jiquetd5",
      "cju8b1v3br45u087189kku66u",
      "cju8b2rmgr52s0801p54eyflx",
      "cju8b6rp0r5st0850184f79xt",
      "cju8b7aqtr4a00987coba14b7",
      "cju8b8yair65w09878pyqtr96",
      "cju8bafgqrf4x0818twisk3ea",
      "cju8bbznkrf5g0871jncffynk",
      "cju8bff9nrfi10850fmfzbf8v",
      "cju8bi8q7rlmn0871abc5ch8k",
      "cju8bj2ssrmlm0871gc2ug2rs",
      "cju8bk8oirjhw0817hgkua2w8",
      "cju8bljw9rqk20801kr54akrl",
      "cju8bn7m2rmm70817hgxpb1uq",
      "cju8bop5jrsid08716i24fqda",
      "cju8bpctzrqkr0850zeldv9kt",
      "cju8bqxxurs6i0850mu7mtef9",
      "cju8brv16rx7f0818uf5n89pv",
      "cju8bssulrrcy0987h1vq5060",
      "cju8buos5rz9b08715lfr0f4f",
      "cju8bw697rwg308177tg8huas",
      "cju8bzzy2s66m08016z6mouqt",
      "cju8c1a0ws7o208181c6lbsom",
      "cju8c2rqzs5t80850d0zky5dy",
      "cju8c5zcbsdfz0801o5t6jag1",
      "cju8c6hnxsdvr0801wn0vrsa6",
      "cju8c82iosagu0817l74s4m5g",
      "cju8c9akjsdjj0850s67uzlxq",
      "cju8ca4geseia0850i2ru11hw",
      "cju8cattbsivm0818p446wgel",
      "cju8cgi2kspp308011nxdtjp6",
      "cju8chdlqsu620755azjty1tj",
      "cju8chxndsre008015uisl4si",
      "cju8cj10qsrau0871o2dr6ai1",
      "cju8clorgsuwn08714toqb7v6",
      "cju8czvnztbf40871b4m7t78w",
      "cju8d2q30tfhs0801n7lx77xl",
      "cju8d4jgatgpj0871q2ophhkm",
      "cju8dic9mtppa0987swn23wbc",
      "cju8djdqztu6408506pzhlo18",
      "cju8dk7eztzup08182yxko5zh",
      "cju8dm2cau2km0818jsv9eeq2",
      "cju8doa16u5gh0818w1ywda3q",
      "cju8dqkrqu83i0818ev74qpxq",
      "cjyzjzssvd8pq0838f4nolj5l",
      "cjyzk8qieoboa0848ogj51wwm",
      "cjyzkmjy8evns070165gf9dmq",
      "cjyzkpsbjdsjq07211dfi4sru",
      "cjyzl833ndne80838pzuq6ila",
      "cjyzlw7f9faqr070129au64sq",
      "cjyztzaqtrv430848l8xgcerw",
      "cjyzu3reghjya0794w7pwoi50",
      "cjyzu9th0qt4r0a46pyl4zik0",
      "cjyzuio1qgh040763k56deohv",
      "cjyzurzvohqnr0794es1itzek",
      "ck2bxlujamu330725szlc2jdu",
      "ck2bxpfgxu2mk0748gsh7xelu",
      "ck2bxqz3evvg20794iiyv5v2m",
      "ck2bxskgxxzfv08386xkqtqdy",
      "ck2bxw18mmz1k0725litqq2mc",
      "ck2da7fwcjfis07218r1rvm95"
    ],
    "val": [
      "cju0sr5ghl0nd08789uzf1raf",
      "cju0tl3uz8blh0993wxvn7ly3",
      "cju0u2g7pmnux0801vkk47ivj",
      "cju0ue769mxii08019zqgdbxn",
      "cju15ptjtppz40988odsm9azx",
      "cju16whaj0e7n0855q7b6cjkm",
      "cju1cbokpuiw70988j4lq1fpi",
      "cju1cnnziug1l0835yh4ropyg",
      "cju1cvkfwqrec0993wbp1jlzm",
      "cju1dfeupuzlw0835gnxip369",
      "cju1dia8hvc6v098827mgffnm",
      "cju1djtprvd7b0988thwwrg09",
      "cju1gi7jlwyld0835cdf6g6qz",
      "cju2hdr06v2bq0799mbm3bks1",
      "cju2hewssldzx0835ep795xu0",
      "cju2hos57llxm08359g92p6jj",
      "cju2i6acqvo6l0799u20fift8",
      "cju2lejzcy4pc0878c9rlonot",
      "cju2mfjndoz700988b9lc3zeq",
      "cju2okvco06xc0799kxe5n1qh",
      "cju2oq5570avm079959o20op1",
      "cju2p0eveqtdc0835gpi3p93i",
      "cju2pmhtr17a00855cvpelzb0",
      "cju2qvuj1s9ok0835tp2k4ozh",
      "cju2r7h21sj9608354gzks3ae",
      "cju2raxlosl630988jdbfy9b0",
      "cju2ricdv2iys0878sv1adh0u",
      "cju2rqo702wpx0855fn7d5cxh",
      "cju2s16zp317h0799gr67jqc2",
      "cju2sggy13na70855tbeoqgha",
      "cju2ti1du4idn0878giuozonw",
      "cju2tpfa5uyx408359datxqqj",
      "cju2trbpkv0c00988hxla5dzz",
      "cju2u4pymvc720988wsxrmi84",
      "cju2u73dj53oz0878486k8k4b",
      "cju2uy8ox62jo0801g88hh42z",
      "cju2xf8e5y2wm08359vcgk09b",
      "cju2xwm1s84l50799i60mq0pu",
      "cju2yg5ht8i4p087800js8hp4",
      "cju2ysg748ru80878sp6j0gm0",
      "cju2yw4s7z7p20988lmf2gdgd",
      "cju2z45kuzf6d0988nz2c819m",
      "cju2zr3c3vwb00993jn06bbaz",
      "cju2zwg05a0oy0801yr73ig7g",
      "cju302fqq9spc0878rrygyzzz",
      "cju30df2j09dd08351ayx2t6w",
      "cju30ftgja7170855xl9bkdm0",
      "cju30gxjq0djk0988jytm49rs",
      "cju30ia8da2bq0799klnehml2",
      "cju30xqmh0ni00835ix3batv1",
      "cju3128yi0rpu0988o4oo5n8n",
      "cju31t8xd17bk0835rnb893jk",
      "cju320gyvbch60801v2amdi2g",
      "cju3280wv1ir009882jze27tc",
      "cju32gzs6xo8x0993r8tedbpb",
      "cju33w4sdcivk0855x879zht7",
      "cju34fojcctcf0799ebolbvkn",
      "cju34zivp3fq80988opxbaqyn",
      "cju357rxxdaz30878y2esjpjt",
      "cju35fxqyzt5p0993vusm54qz",
      "cju3umoh1geet0817cmpef5am",
      "cju3v11mrgwwb0755u242ygye",
      "cju3v56bwgy8v0871w14pz8fx",
      "cju3xga12iixg0817dijbvjxw",
      "cju3xiic0ilzp0850lusrb42j",
      "cju3xl264ingx0850rcf0rshj",
      "cju3xwpgviwlx0871rwm15q7v",
      "cju3y0pjrj1c30755nxekxccj",
      "cju3y79ofj3va0871uqfb1mzo",
      "cju3y9difj6th0801kd1rqm3w",
      "cju40r6jrksyk0871wg98zgho",
      "cju40taxlkrho0987smigg0x0",
      "cju41s6nbleqy0755e2mslg0b",
      "cju422cm8lfxn0818ojicxejb",
      "cju426tomlhll0818fc0i7nvh",
      "cju428k5fldt108177s6g6f45",
      "cju42py9mlqyd0818u3d1d7ga",
      "cju439oazm2fu0871ma0vvrft",
      "cju43eigtm6ev0801mv0m96t1",
      "cju43jcqim2cp08172dvjvyui",
      "cju5b9oyda4yr0850g9viziyv",
      "cju5ccpvqash50850kb4bs22k",
      "cju5chrxxawka0871qcj171yz",
      "cju5eyfe9cpk90987laa7tsl3",
      "cju5f26ebcuai0818xlwh6116",
      "cju5k7r0yf98c09878csbxb4d",
      "cju5o1vu9gz8a0818eyy92bns",
      "cju5u6wf0kh1t0755bg1ssixv",
      "cju5uzmaol56l0817flxh4w9p",
      "cju5vzjoslpj708186z2fusmz",
      "cju5yclrymlgj0818k426ud6z",
      "cju6uy20suzbl0987rzuhz7z9",
      "cju6v4szov55u0871qmqz3v8n",
      "cju6v5ilsv8hk0850rb5sgh6o",
      "cju6v6g6kvdw007552x6mb0po",
      "cju6vifjlv55z0987un6y4zdo",
      "cju6vqarjv7yo0987q4b1btk1",
      "cju6wjm81vgsc0987enk9n3pr",
      "cju6x0yqbvxqt0755dhxislgb",
      "cju6yxyt0wh080871sqpepu47",
      "cju6z9a9kwsl007552s49rx6i",
      "cju76o55nymqd0871h31sph9w",
      "cju772304yw5t0818vbw8kkjf",
      "cju77afzlz3kp07550x5nafzs",
      "cju77b3wyz4160755qis4ljsb",
      "cju77j66ez52p08019xygi0co",
      "cju7ae7bq1f820987toc8si1d",
      "cju7aifyo1p3n07552nxjx51f",
      "cju7avvi51iox0817ym55y6tt",
      "cju7b2l561oas0871decgslaf",
      "cju7bc95p1mdm0817yqj5jc6j",
      "cju7bfx651qr80801cs7epotb",
      "cju7cp6dw244p0818gncdol4m",
      "cju7ctvqn25dy08186g442m1r",
      "cju7d7tly27h408016fyp5nr7",
      "cju7d8m3b2e210755l8fj1yph",
      "cju7db7lp2f400755tntd1ohf",
      "cju7ddtz729960801uazp1knc",
      "cju7deifq2fzn0755lc8idyh8",
      "cju7dglf226g50987ohbthl19",
      "cju7dizi82h2i0755doucgnt3",
      "cju7dmlgf2ebw0871ieqas5fh",
      "cju7ecl9i2i060987xawjp4l0",
      "cju7efffp2ivf0817etg3jehl",
      "cju7epwj82koz098713apjnzo",
      "cju7etr3y2p4t0801cdzjj8ab",
      "cju7evxt12m730987rxivne3x",
      "cju7ey10f2rvf0871bwbi9x82",
      "cju7f4sc62xqj075597xpmuoy",
      "cju7f5ghb2r5s0801chwkxxh9",
      "cju7fob3x301u0755x985pmmq",
      "cju7fpfzq2wyf0818xxd1oziv",
      "cju83nwu1jxte0987h1krpfmv",
      "cju83rcnzkbsj0755x5anfrcg",
      "cju84dsvaklpx098750hp83x4",
      "cju84ffdzkrjn08183jh1fxmb",
      "cju84jdl9kv0i0871eog9b3i9",
      "cju85dx63lic408017f0l0400",
      "cju85nr8elly209872w9n5m0s",
      "cju85omszllp30850b6rm9mi3",
      "cju87li0zn3yb0817kbwgjiz8",
      "cju87q6yoneim0871dl4phvkd",
      "cju87xn2snfmv0987sc3d9xnq",
      "cju87z6o6nh73085045bzsx6o",
      "cju87zv8lni0o0850hbbecbq6",
      "cju8828oxnool0801qno9luhr",
      "cju885ikhnmkn09878s2lqtuh",
      "cju886ryxnsl50801r93jai7q",
      "cju88gx09o2vk0818610zody3",
      "cju88l66no10s0850rsda7ej1",
      "cju8914beokbf0850isxpocrk",
      "cju892fesoq2g0801n0e0jyia",
      "cju89y9h0puti0818i5yw29e6",
      "cju89z6pqpqfx0817mfv8ixjc",
      "cju8ando2qqdo0818ck7i1be1",
      "cju8b542nr81x0871uxnkm9ih",
      "cju8b5p40r2c60987ofa0mu03",
      "cju8bh8surexp0987o5pzklk1",
      "cju8bm24yrrdp081829mbo8ic",
      "cju8c3xs7sauj0801ieyzezr5",
      "cju8c5mxls96t0850wvkvsity",
      "cju8cbsyssiqj0871gr4jedjp",
      "cju8cdeazsm8h0801jxifmzur",
      "cju8ceacrsqkr0755hdz145es",
      "cju8cwy02t9eq08185qn12c02",
      "cju8dpa89u6l80818dj6lldh9",
      "cjyzufihqquiw0a46jatrbwln",
      "cjyzul1qggwwj07216mhiv5sy",
      "ck2395w2mb4vu07480otsu6tw",
      "ck2bxiswtxuw80838qkisqjwz"
    ]
  },
  "fold_1": {
    "train": [
      "cju0qkwl35piu0993l0dewei2",
      "cju0qoxqj9q6s0835b43399p4",
      "cju0rx1idathl0835detmsp84",
      "cju0s690hkp960855tjuaqvv0",
      "cju0sr5ghl0nd08789uzf1raf",
      "cju0sxqiclckk08551ycbwhno",
      "cju0tl3uz8blh0993wxvn7ly3",
      "cju0u2g7pmnux0801vkk47ivj",
      "cju0u82z3cuma0835wlxrnrjv",
      "cju0ue769mxii08019zqgdbxn",
      "cju0vtox5ain6099360pu62rp",
      "cju13fwthn9mq0835gacxgy01",
      "cju13hp5rnbjx0835bf0jowgx",
      "cju14g8o4xui30878gkgbrvqj",
      "cju14pxbaoksp0835qzorx6g6",
      "cju15jr8jz8sb0855ukmkswkz",
      "cju15ptjtppz40988odsm9azx",
      "cju15wdt3zla10801odjiw7sy",
      "cju160wshltz10993i1gmqxbe",
      "cju16ach3m1da0993r1dq3sn2",
      "cju16b6ynq8e40988m8vx0xnj",
      "cju16d65tzw9d0799ouslsw25",
      "cju16jgnyzp970878melv7r25",
      "cju16whaj0e7n0855q7b6cjkm",
      "cju171py4qiha0835u8sl59ds",
      "cju175facms5f0993a5tjikvt",
      "cju17bz250pgd0799u1hqkj5u",
      "cju17g6ykn1cs0993dww6qdi8",
      "cju17hw9hr9c5098800fu4u8e",
      "cju17r8il13910799dr2wme2e",
      "cju17x0j4nfc10993y31pvlgs",
      "cju1819curo000988pd5xcqme",
      "cju1871y11d6r0799k6cw4yze",
      "cju18849rrsgr0988p90hkygb",
      "cju18gzrq18zw0878wbf4ftw6",
      "cju18ibp219ub08783i6o98g7",
      "cju1alwgo30z60855fm3y23sm",
      "cju1amqw6p8pw0993d9gc5crl",
      "cju1aqqv02qwz0878a5cyhr67",
      "cju1ats0y372e08011yazcsxm",
      "cju1b0y2e396p08558ois175d",
      "cju1b3zgj3d8e0801kpolea6c",
      "cju1bhnfitmge0835ynls0l6b",
      "cju1c4fcu40hl07992b8gj0c8",
      "cju1c6yfz42md08550zgoz3pw",
      "cju1cbokpuiw70988j4lq1fpi",
      "cju1cdxvz48hw0801i0fjwcnk",
      "cju1cfhyg48bb0799cl5pr2jh",
      "cju1cnnziug1l0835yh4ropyg",
      "cju1cvkfwqrec0993wbp1jlzm",
      "cju1cyjb5qtie0993njqne9m3",
      "cju1d31sp4d4k0878r3fr02ul",
      "cju1ddr6p4k5z08780uuuzit2",
      "cju1dfeupuzlw0835gnxip369",
      "cju1dhfok4mhe0878jlgrag0h",
      "cju1dia8hvc6v098827mgffnm",
      "cju1djtprvd7b0988thwwrg09",
      "cju1dnz61vfp40988e78bkjga",
      "cju1dq3x1vgx109889c7wyirg",
      "cju1drnhbrb9409935wi7vkhg",
      "cju1egh885m1l0855ci1lt37c",
      "cju1ejj7dvqfa0835ra184v5m",
      "cju1erep75us208553i4ofwwe",
      "cju1euant5l960878iqj5vvto",
      "cju1expq45zst0855rjqwwj4m",
      "cju1f15k3w4ct0835cmde6ypo",
      "cju1f320ewfyu0988ndz6blh5",
      "cju1f5x1164xv08555654c24r",
      "cju1f79yhsb5w0993txub59ol",
      "cju1f8w0t65en0799m9oacq0q",
      "cju1fb9236a110801yvg0fwju",
      "cju1fjsb4sipq09931lvd8e41",
      "cju1fm3id6gl50801r3fok20c",
      "cju1fmsyf6gxb0801cimx2gle",
      "cju1fr4etsmrr09933u4t4aql",
      "cju1ftaji6isw0855108yqcse",
      "cju1g20bdwq6u0835e16xugcd",
      "cju1g4nsb6ngy0799l4ezm8ab",
      "cju1gghyjwxt80835vx0wgxw0",
      "cju1gi7jlwyld0835cdf6g6qz",
      "cju1gkndf6yi10801o1qnje19",
      "cju1h5w4wxajx0835mc954kxy",
      "cju1h89h6xbnx08352k2790o9",
      "cju1haab178i70799tk9z8y8x",
      "cju1hirfi7ekp0855q0vgm9qq",
      "cju1hmff8tkp809931jps6fbr",
      "cju1hyolc7aqu0878rrkfn1lr",
      "cju2hdr06v2bq0799mbm3bks1",
      "cju2hewssldzx0835ep795xu0",
      "cju2hfqnmhisa0993gpleeldd",
      "cju2hgsptlfam0835o3b59h1o",
      "cju2hjrqcvi2j0801bx1i6gxg",
      "cju2hlm19vjjf0801o69qnber",
      "cju2hos57llxm08359g92p6jj",
      "cju2i03ptvkiu0799xbbd4det",
      "cju2i3hzclw3o0988rrgh911i",
      "cju2i6acqvo6l0799u20fift8",
      "cju2i8br1vqtd08784u6vmcjk",
      "cju2ij9uiic2l09933ljiv6gm",
      "cju2lcyfgkf5809932fn9gucn",
      "cju2lejzcy4pc0878c9rlonot",
      "cju2lyynuymli0855g7fxgbhe",
      "cju2m56cryvqd0801gtn2yp8t",
      "cju2mfjndoz700988b9lc3zeq",
      "cju2nfnvxzdkd0878399axlco",
      "cju2nguelpmlj0835rojdn097",
      "cju2nnqrqzp580855z8mhzgd6",
      "cju2np2k9zi3v079992ypxqkn",
      "cju2nqapmzvk20801f9us40dx",
      "cju2nsmwjlzyl0993jl80chvz",
      "cju2ntxtdzlvu0799xl3j9pan",
      "cju2okvco06xc0799kxe5n1qh",
      "cju2oo0wh0bqy0878biujeyhe",
      "cju2oq5570avm079959o20op1",
      "cju2osuru0ki00855txo0n3uu",
      "cju2ouil2mssu0993hvxsed6d",
      "cju2p0eveqtdc0835gpi3p93i",
      "cju2p4ddkmzxj0993p94o62av",
      "cju2pag1f0s4r0878h52uq83s",
      "cju2phaksnahz0993yxogjcpv",
      "cju2pjb9v0ywn0878j5g5n69j",
      "cju2pmhtr17a00855cvpelzb0",
      "cju2qfie4rvz508357kad9z5o",
      "cju2qh5le1ock0878oahaql7d",
      "cju2qozsk20cq0855ugrg3cri",
      "cju2qqn5ys4uo0988ewrt2ip2",
      "cju2qu37qobl50993aw7ghcfq",
      "cju2qvuj1s9ok0835tp2k4ozh",
      "cju2qxxko229x08786gvxxhur",
      "cju2qz06823a40878ojcz9ccx",
      "cju2r11x7sdgx0988o8ule0wl",
      "cju2r6mt2om21099352pny5gw",
      "cju2r7h21sj9608354gzks3ae",
      "cju2r91dg2k090801bh0xzbxk",
      "cju2raxlosl630988jdbfy9b0",
      "cju2rga4psq9n09881z519xx0",
      "cju2ricdv2iys0878sv1adh0u",
      "cju2rlqdnoz9k0993cpjae3x0",
      "cju2rmd2rsw9g09888hh1efu0",
      "cju2rn0hasxri0835nfy3buay",
      "cju2ro5jqsy680988pi6qsujw",
      "cju2rqo702wpx0855fn7d5cxh",
      "cju2rz4k434s70855wwx3ddtx",
      "cju2rzpsmtb0f0835jabkbao1",
      "cju2s16zp317h0799gr67jqc2",
      "cju2s2527pfyr0993l3h1149a",
      "cju2s9g11pnra0993gn4eh793",
      "cju2saez63gxl08559ucjq3kt",
      "cju2sggy13na70855tbeoqgha",
      "cju2srvy5440s0801y1ba9akr",
      "cju2sszfq3uye0878sucelzk2",
      "cju2suk42469908015ngmq6f2",
      "cju2sxf3iqbpv09937iksn8ep",
      "cju2syxa93yw40799x2iuwabz",
      "cju2t16vuucaq0835xcpsivn2",
      "cju2t62nq45jl0799odpufwx6",
      "cju2t9tdwuk700835kv0ljmtl",
      "cju2ti1du4idn0878giuozonw",
      "cju2tjrog4jy30878pawyazqc",
      "cju2top2ruxxy0988p1svx36g",
      "cju2tpfa5uyx408359datxqqj",
      "cju2tqfgw4oat0799rn0g5b2z",
      "cju2trbpkv0c00988hxla5dzz",
      "cju2tvrvm53ws0801a0jfjdxg",
      "cju2txjfzv60w098839dcimys",
      "cju2u4pymvc720988wsxrmi84",
      "cju2u73dj53oz0878486k8k4b",
      "cju2ulk385h170799rlklxob0",
      "cju2uokeg5jm20799xwgsyz89",
      "cju2upu4evw7g08358guwozxv",
      "cju2urqpwvxw70835rvndvtsi",
      "cju2uwz9f5yf1085506cfamfx",
      "cju2uy8ox62jo0801g88hh42z",
      "cju2uzabhs6er0993x3aaf87p",
      "cju2wx0gh7fpz0878wwyd9ep8",
      "cju2wzu8wxtgu09880ku9x1pg",
      "cju2xa3i4y0160988i679zsqd",
      "cju2xd75m82720801q4s4ik3n",
      "cju2xf8e5y2wm08359vcgk09b",
      "cju2xjz2ju8pe0993ysv9wg17",
      "cju2xlcqxy9c60988vjacdznb",
      "cju2xs6na81t20878pt6nkfip",
      "cju2xwm1s84l50799i60mq0pu",
      "cju2xyd9vyi7m098831qcucse",
      "cju2y0z6g87p10878fpk5d3rq",
      "cju2y40d8ulqo0993q0adtgtb",
      "cju2y5zas8m7f0801d34g5owq",
      "cju2y8s56ymqr083541ggdsml",
      "cju2yb31a8e8u0878wdashg7o",
      "cju2ycp1u8g2r0799jslnp7cz",
      "cju2yg5ht8i4p087800js8hp4",
      "cju2yi9tz8vky0801yqip0xyl",
      "cju2yo1j1v0qz09934o0e683p",
      "cju2ysg748ru80878sp6j0gm0",
      "cju2yw4s7z7p20988lmf2gdgd",
      "cju2z1nxlzaj40835wj81s1iy",
      "cju2z2x3nvd3c099350zgty7w",
      "cju2z45kuzf6d0988nz2c819m",
      "cju2z6ez69g4u0801qwt088lw",
      "cju2zblxw9848087853csbrx1",
      "cju2zdhsczmn50988z64qwg2q",
      "cju2zdvjn9h7r08553cp4eed5",
      "cju2zjcvj9qma0801dk71hhi0",
      "cju2zm0axztpe0988r8s9twjr",
      "cju2zo0fwzv580988qlijd2xa",
      "cju2zr3c3vwb00993jn06bbaz",
      "cju2zwg05a0oy0801yr73ig7g",
      "cju2zy1e49pqk0878t6ncqn12",
      "cju300m3s04fg0988uzupuf7z",
      "cju302fqq9spc0878rrygyzzz",
      "cju306x7w05nb0835cunv799x",
      "cju30df2j09dd08351ayx2t6w",
      "cju30ftgja7170855xl9bkdm0",
      "cju30gxjq0djk0988jytm49rs",
      "cju30ia8da2bq0799klnehml2",
      "cju30k2z40ds308353kdew70n",
      "cju30mm25a53s0799qa5wiqe8",
      "cju30nyxe0gfb0835p256yoju",
      "cju30ov1oah920801mi8thuyg",
      "cju30u1hbakn808019g15nb8b",
      "cju30xqmh0ni00835ix3batv1",
      "cju30ywtc0oar0835bp2en7ec",
      "cju310f6val1v0855xo8tc3gu",
      "cju3128yi0rpu0988o4oo5n8n",
      "cju31rb7vb6110801p9rhacuw",
      "cju31t8xd17bk0835rnb893jk",
      "cju320gyvbch60801v2amdi2g",
      "cju323ypb1fbb0988gx5rzudb",
      "cju3280wv1ir009882jze27tc",
      "cju32a52lb9rc0799xi40qs00",
      "cju32csyfblyh080170aa3x5p",
      "cju32fhnhbds40799broyoptc",
      "cju32gzs6xo8x0993r8tedbpb",
      "cju32jcdabepz0878d0cznmfe",
      "cju32l161bi1v07990vm376in",
      "cju32qr9tbvsj08013pkpjenq",
      "cju32zhbnc1oy0801iyv1ix6p",
      "cju330ofbc2l30801th5g3hw6",
      "cju33231uy4gi0993qc7b1jch",
      "cju334jzo261t0835yqudnfs1",
      "cju3381d8bz3h07991xtl7ra0",
      "cju33belnbyhm0878yxl42233",
      "cju33o12x2jm50988944mxq0v",
      "cju33qpdvc9g0087825jhf3s9",
      "cju33w4sdcivk0855x879zht7",
      "cju34eqjpcpm508788b3lhp97",
      "cju34fojcctcf0799ebolbvkn",
      "cju34i3qvcyog0855qiejxx5w",
      "cju34m7h536wq0988xz7gx79v",
      "cju34sh43d8zm08019xbwhc0o",
      "cju34uhepd3dd0799hs8782ad",
      "cju34zivp3fq80988opxbaqyn",
      "cju3518w2d838079939fqztbc",
      "cju3521y5d5mq0878t3ezsu4p",
      "cju35740hzm0g0993zl5ic246",
      "cju357rxxdaz30878y2esjpjt",
      "cju358pwtdby20878cg7nm0np",
      "cju35atpxdjot0855q46aqrd0",
      "cju35eg0tdmjt085525sb4bua",
      "cju35fxqyzt5p0993vusm54qz",
      "cju35oyvd3y850988km12hdz1",
      "cju3tp94kfstl08181awh6z49",
      "cju3tvffffx5f0818t5ov22al",
      "cju3tx1qyg0c907552fglumhc",
      "cju3u1c8tfyqx08503iedc3mx",
      "cju3u39fog1bo0871lxjrabks",
      "cju3ua8u0g9rg0801uayhdxhu",
      "cju3uhb79gcgr0871orbrbi3x",
      "cju3umoh1geet0817cmpef5am",
      "cju3uz4o6gr9z0850lhxyxvsj",
      "cju3v11mrgwwb0755u242ygye",
      "cju3v56bwgy8v0871w14pz8fx",
      "cju3x4blzieu30850x10uuvbm",
      "cju3x5u2tiihx0818914gzxy1",
      "cju3x9lttikfb0818a0g104zn",
      "cju3xga12iixg0817dijbvjxw",
      "cju3xiic0ilzp0850lusrb42j",
      "cju3xjqtpikx50817tppy6g84",
      "cju3xl264ingx0850rcf0rshj",
      "cju3xtufwiv9c0818djsc4cqd",
      "cju3xwpgviwlx0871rwm15q7v",
      "cju3xzvnzj0hd0755xprz39nj",
      "cju3y0pjrj1c30755nxekxccj",
      "cju3y54kwj3nr0801biidlb4e",
      "cju3y79ofj3va0871uqfb1mzo",
      "cju3y9difj6th0801kd1rqm3w",
      "cju3ya7goj6at0818v2l5ay7f",
      "cju3yb47cj1xq0817zfotbni4",
      "cju3yht87j83m08507yk1u1fg",
      "cju3ykamdj9u208503pygyuc8",
      "cju40jl7skiuo0817p0smlgg8",
      "cju40m0rjkpw80871z6n6yg1u",
      "cju40poe4kt9s0755f9cnm3h5",
      "cju40r6jrksyk0871wg98zgho",
      "cju40taxlkrho0987smigg0x0",
      "cju410dnfl0960755y8lu8d79",
      "cju414lf2l1lt0801rl3hjllj",
      "cju418rckl3ur08012psrx1r1",
      "cju41kd7yl4nm0850gil5qqwh",
      "cju41nz76lcxu0755cya2qefx",
      "cju41s6nbleqy0755e2mslg0b",
      "cju422cm8lfxn0818ojicxejb",
      "cju426tomlhll0818fc0i7nvh",
      "cju428k5fldt108177s6g6f45",
      "cju42m60jlpcm08186kqppzqv",
      "cju42nm68lpyo0818xvvqmupq",
      "cju42py9mlqyd0818u3d1d7ga",
      "cju42qet0lsq90871e50xbnuv",
      "cju42romflni20817etb9a0fl",
      "cju42u5bjlvi10801dc13sskp",
      "cju42wamblrqn098798r2yyok",
      "cju439oazm2fu0871ma0vvrft",
      "cju43b8daly4408170e5ev06g",
      "cju43eigtm6ev0801mv0m96t1",
      "cju43gfosm63n08714rpih8pe",
      "cju43h43am1dy08176gwfhmnt",
      "cju43in5fm22c08175rxziqrk",
      "cju43jcqim2cp08172dvjvyui",
      "cju43kj2pm34f0850l28ahpni",
      "cju43lcnum9y10755bjs7z87f",
      "cju43o6n7m9nk087191ijwqq9",
      "cju45jpvfn6c809873pv1i34s",
      "cju45lbgznahl08180xz1h7u6",
      "cju45n0oxn5vu08500yfrt9jn",
      "cju45ofd9ne1j0801ri8dup7t",
      "cju45rj7ln8980850a7821fov",
      "cju45t5ddnbio0987qtqzx762",
      "cju45v0pungu40871acnwtmu5",
      "cju5b9oyda4yr0850g9viziyv",
      "cju5bbtwsa8cl0987wgfsqpao",
      "cju5bf6hxa6m50817rbwettgu",
      "cju5bhv81abur0850ean02atv",
      "cju5bj926aiec07559rshy4wa",
      "cju5boicjagt20871b1fotkh4",
      "cju5buy2bal250818ipl6fqwv",
      "cju5bwhapakm90987c1v4z46a",
      "cju5c5xc7algd0817pb1ej5yo",
      "cju5ca9hcatkc0801jzwe7tfx",
      "cju5ccpvqash50850kb4bs22k",
      "cju5cetivauok0987ok3e5bre",
      "cju5chrxxawka0871qcj171yz",
      "cju5cjh3xattc0817j2vbulzi",
      "cju5ddda9bkkt0850enzwatb1",
      "cju5eftctcdbj08712gdp989f",
      "cju5ekty5ckzf07550c9u3ckk",
      "cju5enq1tcn1i0755hnkon787",
      "cju5es375cnzy0801nkq35ffs",
      "cju5eyfe9cpk90987laa7tsl3",
      "cju5f0dezct4q08183ydw11dx",
      "cju5f26ebcuai0818xlwh6116",
      "cju5fb86jd1jp0755b1ukbhq5",
      "cju5fi0yxd3ei0801v7u0yudn",
      "cju5fu081d8gc0818l3yylujk",
      "cju5fw37edaae0801vkwvocn7",
      "cju5fydrud94708507vo6oy21",
      "cju5g163vd6mt0817uccuga6u",
      "cju5h57xedz5h0755mjpc8694",
      "cju5hi52odyf90817prvcwg45",
      "cju5hl8nee8a40755fm8qjj0o",
      "cju5hqz50e7o90850e0prlpa0",
      "cju5ht88gedbu0755xrcuddcx",
      "cju5hwonqedw10801vsd3w6kk",
      "cju5hyi9yegob0755ho3do8en",
      "cju5i39mreass0817au8p22zy",
      "cju5jx7jzf7c90871c2i9aiov",
      "cju5jz5fff8c50871hbe6108f",
      "cju5k3j3uf6de0817hszzfr7n",
      "cju5k503sfa5f0871lx0rpu5y",
      "cju5k7r0yf98c09878csbxb4d",
      "cju5knbbqfipk080128cggukq",
      "cju5nyu31gv8e0871zpk74a2n",
      "cju5o1vu9gz8a0818eyy92bns",
      "cju5o4pk9h0720755lgp9jq8m",
      "cju5u4pywk81x0817vn9pe14z",
      "cju5u6wf0kh1t0755bg1ssixv",
      "cju5u8gz4kj5b07552e2wpkwp",
      "cju5udcufki0s09874ll1dbr5",
      "cju5uget8krjy0818kvywd0zu",
      "cju5uhrdwkmsu0817ervv91l8",
      "cju5uxjnol2r509871qv2yeia",
      "cju5uzmaol56l0817flxh4w9p",
      "cju5v8pgplg6k0755rvi2t63h",
      "cju5vbo6jldrt0871jf6f1700",
      "cju5vcmrqla7i0817x4sp4pqw",
      "cju5vgawslbe30987ndeepc1b",
      "cju5vi4nxlc530817uoqm2m7a",
      "cju5von04litr08718j8po40a",
      "cju5vutu7ll8w0871dfp92n9p",
      "cju5vxuc5loxw0818u8xgf45p",
      "cju5vzjoslpj708186z2fusmz",
      "cju5w7xn0lrkq0801f9k0htgx",
      "cju5wj0faly5008187n6530af",
      "cju5wkonqlrl409877y8zvnub",
      "cju5woy82m07m08505dmjg7g1",
      "cju5wrrs0m2af0818vmnajbtw",
      "cju5wtdu4m0im0871mix0yvc0",
      "cju5wuhm1lwm40987vugqn3vv",
      "cju5x00l6m5j608503k78ptee",
      "cju5x15djm7ae0755h8czf6nt",
      "cju5x7iskmad90818frchyfwd",
      "cju5xjn5mm78b09871spyqhhr",
      "cju5y4hgqmk0i08180rjhbwvp",
      "cju5y7buemcw80987p0r30g9f",
      "cju5y84q3mdv50817eyp82xf3",
      "cju5yclrymlgj0818k426ud6z",
      "cju5yeqiwmkgl0801fzv2douc",
      "cju5ygh1zmmdi0755uod5e17i",
      "cju5yhgznmkzb0801cji2vi8j",
      "cju5yjq1pmlgc0801z0t24bly",
      "cju5ymyd8mmdc0801ry3by1xr",
      "cju6us80mv1b50871ebyq2wxa",
      "cju6ut4l8va6y0755tyw3vfqq",
      "cju6uy20suzbl0987rzuhz7z9",
      "cju6uzxk0v83p0801rcwnexdu",
      "cju6v1m1xv07w09870ah3njy1",
      "cju6v3bb2v7xo085090blubyw",
      "cju6v4szov55u0871qmqz3v8n",
      "cju6v5ilsv8hk0850rb5sgh6o",
      "cju6v6g6kvdw007552x6mb0po",
      "cju6vgdmivcvb08018fra5lnv",
      "cju6vifjlv55z0987un6y4zdo",
      "cju6vqarjv7yo0987q4b1btk1",
      "cju6vucxvvlda0755j7msqnya",
      "cju6vvxsev9y30987kespucdg",
      "cju6w733bveoz0817e600tw72",
      "cju6wjm81vgsc0987enk9n3pr",
      "cju6wt9jvvn500871hjn3t3g0",
      "cju6x0yqbvxqt0755dhxislgb",
      "cju6x35ervu2808015c7eoqe4",
      "cju6x97w4vwua0850x0997r0a",
      "cju6xa0qmvzun0818xjukgncj",
      "cju6xifswvwbo0987nibtdr50",
      "cju6xlygpw7bs0818n691jsq4",
      "cju6xmqd9w0250817l5kxfnsk",
      "cju6ywm40wdbo0987pbftsvtg",
      "cju6yxyt0wh080871sqpepu47",
      "cju6z600qwh4z081700qimgl9",
      "cju6z9a9kwsl007552s49rx6i",
      "cju76erapykj30871x5eaxh4q",
      "cju76l27oyrw907551ri2a7fl",
      "cju76lsehyia10987u54vn8rb",
      "cju76o55nymqd0871h31sph9w",
      "cju772304yw5t0818vbw8kkjf",
      "cju77afzlz3kp07550x5nafzs",
      "cju77b3wyz4160755qis4ljsb",
      "cju77bvg0yv4r0987yh60xmjo",
      "cju77idwfz36d0871tzfzz51i",
      "cju77j66ez52p08019xygi0co",
      "cju77re6fz5bb0817vp9redjg",
      "cju77t0razbvm080106o56289",
      "cju77u1sjz77b0817ft44r3fk",
      "cju77vvcwzcm50850lzoykuva",
      "cju783tmkzkqu081803g7q5vk",
      "cju784jpdzeae0987q5ypq883",
      "cju785htizjzo08017tvlhtg4",
      "cju787jnjzjuj0871p94nck9g",
      "cju7ae7bq1f820987toc8si1d",
      "cju7aez2x1jtj0871ztezs3oi",
      "cju7afqon1ip40850ue2308b6",
      "cju7agj961l2r0818z29iq8yn",
      "cju7ahtkb1jr90801jck4kbds",
      "cju7aifyo1p3n07552nxjx51f",
      "cju7aklv31h4309871m29l4e7",
      "cju7amjna1ly40871ugiokehb",
      "cju7ap09p1kz10850ldccjebj",
      "cju7apr0c1qqm0755s7msqot4",
      "cju7avvi51iox0817ym55y6tt",
      "cju7awzmu1ncs0871hziy65zx",
      "cju7azuu31mia0801pf9ib5ed",
      "cju7b2l561oas0871decgslaf",
      "cju7b3f5h1sm40755i572jden",
      "cju7b4mtw1n9n080186209f3d",
      "cju7b5afm1nfw0801xqm8bf8q",
      "cju7b9vcs1luz0987ta60j1dy",
      "cju7bc95p1mdm0817yqj5jc6j",
      "cju7bd1qu1mx409877xjxibox",
      "cju7bf1lp1shi081835vs84lc",
      "cju7bfx651qr80801cs7epotb",
      "cju7bgnvb1sf808717qa799ir",
      "cju7bmi1v1pnj0987pa52jjok",
      "cju7cp6dw244p0818gncdol4m",
      "cju7crgxa28550755wbsgqkel",
      "cju7ctvqn25dy08186g442m1r",
      "cju7cue9b232j0801qdzk1ykj",
      "cju7cufm7298k0755j09uf3of",
      "cju7d1tvt25bu08019dvw3uff",
      "cju7d2q1k27nf08715zshsckt",
      "cju7d3oc82cho0755dajlwldz",
      "cju7d4jk723eu0817bqz2n39m",
      "cju7d5m0p23kn09871rk7pu3v",
      "cju7d6ux323ze0987xos3srkx",
      "cju7d7tly27h408016fyp5nr7",
      "cju7d8m3b2e210755l8fj1yph",
      "cju7d9seq29zd0871nzl2uu5m",
      "cju7db7lp2f400755tntd1ohf",
      "cju7dda8w2br20818zhsuz8s7",
      "cju7ddtz729960801uazp1knc",
      "cju7deifq2fzn0755lc8idyh8",
      "cju7dff9529h208503w60lbil",
      "cju7dglf226g50987ohbthl19",
      "cju7dizi82h2i0755doucgnt3",
      "cju7dmlgf2ebw0871ieqas5fh",
      "cju7dn24o296i09871qfxb8s2",
      "cju7do8c72dbo0801vxfzxdc4",
      "cju7dp3dw2k4n0755zhe003ad",
      "cju7dqcwi2dz00850gcmr2ert",
      "cju7dtb1e2j0t0818deq51ib3",
      "cju7dvl5m2n4t0755hlnnjjet",
      "cju7dwe282dc309876rco45ts",
      "cju7dxffn2eam0817qxosfwch",
      "cju7dymur2od30755eg8yv2ht",
      "cju7ea4om2l910801bohqjccy",
      "cju7ecl9i2i060987xawjp4l0",
      "cju7eea9b2m0z0801ynqv1fqu",
      "cju7efffp2ivf0817etg3jehl",
      "cju7ehljc2or70871261br8ai",
      "cju7ejm2l2ncl0801wq6y84nw",
      "cju7ekbo32pft0871fv7kzwb9",
      "cju7emdni2py40871ivhxjtut",
      "cju7epwj82koz098713apjnzo",
      "cju7es23b2vcp0755gpbm9s7v",
      "cju7et17a2vjk0755e743npl1",
      "cju7etr3y2p4t0801cdzjj8ab",
      "cju7eueum2oqn0850rodmx8zo",
      "cju7ev2b12owa08500bpfpwyw",
      "cju7evxt12m730987rxivne3x",
      "cju7ey10f2rvf0871bwbi9x82",
      "cju7ez7r22qbc08015xfoz2wb",
      "cju7f4sc62xqj075597xpmuoy",
      "cju7f5ghb2r5s0801chwkxxh9",
      "cju7f6cqy2ur20818t1saazbm",
      "cju7fazv92ywx0755xov2erga",
      "cju7fd6yt2p740987wkr8exo1",
      "cju7fen322ou10817ziqkob4k",
      "cju7ff97z2ow40817u2r83my5",
      "cju7fmvpk2q170987v6i3ola8",
      "cju7fob3x301u0755x985pmmq",
      "cju7fpfzq2wyf0818xxd1oziv",
      "cju7fq7mm2pw508176uk5ugtx",
      "cju7frtqu2xa20818wq8r9fzf",
      "cju7g7ba42z310987bqzbi2bq",
      "cju83h9ysjwe808716nt35oah",
      "cju83ipu3jwpx0801z5pvguf8",
      "cju83k8fyjsxr0817d6nxs6r4",
      "cju83kxitjv340987z09m0ezy",
      "cju83mki1jv5w0817kubxm31r",
      "cju83nwu1jxte0987h1krpfmv",
      "cju83qd0yjyht0817ktkfl268",
      "cju83rcnzkbsj0755x5anfrcg",
      "cju83syhdk6gs0801rf1rekdl",
      "cju83u9ftk3ni0987qnhlcinv",
      "cju83vvmik9wa08710yeh7cuk",
      "cju83wwn1k55e0850kw6i2d81",
      "cju8402x1kcy70801t6kz6bdi",
      "cju847pxykriq0755268ktrk2",
      "cju849c23kgnk0817cgv2hw1e",
      "cju84aoa3ktwn0755pfl4gfwd",
      "cju84dsvaklpx098750hp83x4",
      "cju84ffdzkrjn08183jh1fxmb",
      "cju84ih17kp5l09876bkooocl",
      "cju84jdl9kv0i0871eog9b3i9",
      "cju84kplnl1y30755ropua1b0",
      "cju8567gdlcbq0801dwwyo2jt",
      "cju857ad0l88m0817qx4cwxnf",
      "cju858eswlepn0871pzvdrhj1",
      "cju85bf1algsq0871y9gtlq97",
      "cju85c2d4ln1b0755zz1z3onx",
      "cju85citjlnfm0755i4rk5tqj",
      "cju85dx63lic408017f0l0400",
      "cju85fc11ljr40818edpb0inh",
      "cju85hguellg50818kwu3s8d1",
      "cju85je7vlht70817c9jcjwi4",
      "cju85mpuglq8k0818d2it6hzb",
      "cju85nr8elly209872w9n5m0s",
      "cju85omszllp30850b6rm9mi3",
      "cju85rkbnlo1c08503uxcpax1",
      "cju87kbcen2av0987usezo8kn",
      "cju87li0zn3yb0817kbwgjiz8",
      "cju87nkyrnb970801q84m47yt",
      "cju87ox0kncom0801b98hqnd2",
      "cju87q6yoneim0871dl4phvkd",
      "cju87r56lnkyp0755hz30leew",
      "cju87tyddnnad0755bj0wxahe",
      "cju87xn2snfmv0987sc3d9xnq",
      "cju87z6o6nh73085045bzsx6o",
      "cju87zv8lni0o0850hbbecbq6",
      "cju8828oxnool0801qno9luhr",
      "cju884985nlmx0817vzpax3y4",
      "cju885ikhnmkn09878s2lqtuh",
      "cju886ryxnsl50801r93jai7q",
      "cju88fpm4o0tl0871w1i6a4ds",
      "cju88gx09o2vk0818610zody3",
      "cju88l66no10s0850rsda7ej1",
      "cju88oh0po9gq0801nge4tgr1",
      "cju88q6h6obpd0871ckmiabbo",
      "cju88rl5eo94l0850kf5wtrm1",
      "cju88trl3ogi208716qvti51b",
      "cju88y1mwoln50871emyfny1g",
      "cju890guyoiti098753yg6cdu",
      "cju8914beokbf0850isxpocrk",
      "cju892fesoq2g0801n0e0jyia",
      "cju893jmdompz0817xn3g1w4h",
      "cju89y9h0puti0818i5yw29e6",
      "cju89z6pqpqfx0817mfv8ixjc",
      "cju8a1jtvpt9m081712iwkca7",
      "cju8a56vxpy780850r45yu4wk",
      "cju8a84g0q76m0818hwiggkod",
      "cju8abobpqbir08189u01huru",
      "cju8adb60qbiu080188mxpf8d",
      "cju8aeei7q8k308173n9y4klv",
      "cju8amfdtqi4x09871tygrgqe",
      "cju8ando2qqdo0818ck7i1be1",
      "cju8apjewqrk00801k5d71gky",
      "cju8aqq8uqmoq0987hphto9gg",
      "cju8ashhnquqr0801rwduzt7d",
      "cju8auylgqx0z0871u4o4db7o",
      "cju8aw9n1qyg10801jkjlmors",
      "cju8axq24r4an0755yhv9d4ly",
      "cju8ayeq7r1fb0818z1junacy",
      "cju8azmhcr66e0755t61atz72",
      "cju8b0jr0r2oi0801jiquetd5",
      "cju8b1v3br45u087189kku66u",
      "cju8b542nr81x0871uxnkm9ih",
      "cju8b5p40r2c60987ofa0mu03",
      "cju8b6rp0r5st0850184f79xt",
      "cju8b7aqtr4a00987coba14b7",
      "cju8bbznkrf5g0871jncffynk",
      "cju8bff9nrfi10850fmfzbf8v",
      "cju8bh8surexp0987o5pzklk1",
      "cju8bi8q7rlmn0871abc5ch8k",
      "cju8bj2ssrmlm0871gc2ug2rs",
      "cju8bk8oirjhw0817hgkua2w8",
      "cju8bljw9rqk20801kr54akrl",
      "cju8bm24yrrdp081829mbo8ic",
      "cju8bn7m2rmm70817hgxpb1uq",
      "cju8brv16rx7f0818uf5n89pv",
      "cju8bssulrrcy0987h1vq5060",
      "cju8buos5rz9b08715lfr0f4f",
      "cju8bzzy2s66m08016z6mouqt",
      "cju8c1a0ws7o208181c6lbsom",
      "cju8c3xs7sauj0801ieyzezr5",
      "cju8c5mxls96t0850wvkvsity",
      "cju8c5zcbsdfz0801o5t6jag1",
      "cju8c82iosagu0817l74s4m5g",
      "cju8ca4geseia0850i2ru11hw",
      "cju8cattbsivm0818p446wgel",
      "cju8cbsyssiqj0871gr4jedjp",
      "cju8cdeazsm8h0801jxifmzur",
      "cju8ceacrsqkr0755hdz145es",
      "cju8cgi2kspp308011nxdtjp6",
      "cju8chdlqsu620755azjty1tj",
      "cju8chxndsre008015uisl4si",
      "cju8cj10qsrau0871o2dr6ai1",
      "cju8clorgsuwn08714toqb7v6",
      "cju8cwy02t9eq08185qn12c02",
      "cju8d2q30tfhs0801n7lx77xl",
      "cju8d4jgatgpj0871q2ophhkm",
      "cju8dic9mtppa0987swn23wbc",
      "cju8djdqztu6408506pzhlo18",
      "cju8dk7eztzup08182yxko5zh",
      "cju8dm2cau2km0818jsv9eeq2",
      "cju8doa16u5gh0818w1ywda3q",
      "cju8dpa89u6l80818dj6lldh9",
      "cju8dqkrqu83i0818ev74qpxq",
      "cjyzkmjy8evns070165gf9dmq",
      "cjyzkpsbjdsjq07211dfi4sru",
      "cjyzl833ndne80838pzuq6ila",
      "cjyzlw7f9faqr070129au64sq",
      "cjyztzaqtrv430848l8xgcerw",
      "cjyzufihqquiw0a46jatrbwln",
      "cjyzuio1qgh040763k56deohv",
      "cjyzul1qggwwj07216mhiv5sy",
      "cjyzurzvohqnr0794es1itzek",
      "ck2395w2mb4vu07480otsu6tw",
      "ck2bxiswtxuw80838qkisqjwz",
      "ck2bxlujamu330725szlc2jdu",
      "ck2bxpfgxu2mk0748gsh7xelu",
      "ck2bxqz3evvg20794iiyv5v2m",
      "ck2bxskgxxzfv08386xkqtqdy",
      "ck2bxw18mmz1k0725litqq2mc"
    ],
    "val": [
      "cju0qx73cjw570799j4n5cjze",
      "cju0roawvklrq0799vmjorwfv",
      "cju0s2a9ekvms080138tjjpxr",
      "cju0t4oil7vzk099370nun5h9",
      "cju15czxqp3lv0835jvhgzurz",
      "cju15l5ubz9yh0855b3ivdpse",
      "cju15mhjczc8z0801kit5c6di",
      "cju16fpvhzypl0799p9phnlx6",
      "cju17otoe119u0799nqcbl8n1",
      "cju17v6ih0u7808783zcbg1jy",
      "cju183od81ff608017ekzif89",
      "cju18kevfrojc0835bn90f1in",
      "cju1c0qb4tzi308355wtsnp0y",
      "cju1cqc7n4gpy0855jt246k68",
      "cju1csmlc4ht10799b8ymmghg",
      "cju1cu1u2474n0878tt7v4tdr",
      "cju1d96gsv62d09881b3wecw2",
      "cju1dg44i4z3w0801nyz4p6zf",
      "cju1efbr0rqxz09931z0lf4vf",
      "cju1egx9pvz2n0988eoy8jp23",
      "cju1euuc65wm00799m4sjdnnn",
      "cju1fj6axwfp30835ukhuzhw9",
      "cju1fuoa4wmc50835qfd11sp9",
      "cju1gv7106qd008784gk603mg",
      "cju1hp9i2xu8e0988u2dazk7m",
      "cju1hs0za7jha0855vj0mdrjt",
      "cju2hqt33lmra0988fr5ijv8j",
      "cju2hugv9vget0799hhk7ksvg",
      "cju2hx006vidl0799igm81vmh",
      "cju2iatlki5u309930zmgkv6h",
      "cju2lz8vqktne0993fuym6drw",
      "cju2mh8t6p07008350e01tx2a",
      "cju2nd7l7z98o0799gfjvyfmw",
      "cju2omjpeqj5a0988pjdlb8l1",
      "cju2p91qir00k08350ddfif0w",
      "cju2pkwt3r8b90988v2ywq1px",
      "cju2r2obh2bjm08553kng0rh7",
      "cju2rkjfwoxys0993x768l1j2",
      "cju2rnkt22xep0801as160g9t",
      "cju2rxm8rpbaf0993o3qr2oph",
      "cju2t2ivz43i10878zeg8r1br",
      "cju2t3ibkuecr0835o7si16zv",
      "cju2tzypl4wss0799ow05oxb9",
      "cju2wtwj87kys0855kx6mddzw",
      "cju2wxv0hxs2f09884w48v8fi",
      "cju2x7vw87mu30878hye2ca0m",
      "cju2xbk0080y80801eghyddi2",
      "cju2y26c588bo07993ksd8eoz",
      "cju2yyhsp933j0855hp32e012",
      "cju2z9vlp9j0w0801oag91sy9",
      "cju2zi4l09f5807991s8do2b2",
      "cju30525w04r10835ygp257sb",
      "cju30ajhw09sx0988qyahx9s8",
      "cju30bmab08bi0835mvlr6e0r",
      "cju30j1rgadut0801vuyrsnt8",
      "cju30lncba3ny0878jwnous8n",
      "cju30qbm1ad3x0855znuhpz9u",
      "cju31y80qbawn0801twwm2l5s",
      "cju324q101fhe08350wae9cif",
      "cju326h4v1gxw08352px40p7r",
      "cju32pzh9bpw10855q4vaxfhe",
      "cju336l68y7if0993wf092166",
      "cju33x0f22peh0988g0ln7w5v",
      "cju34aozyyy830993bn16u32n",
      "cju34ds2531520988qjpqt6e3",
      "cju34ymm8d6700799uop0cw33",
      "cju353d1eda8c07992afde611",
      "cju35c4wzdhow0799h6eq4sgs",
      "cju35ldepdtlm0801yv79y8vu",
      "cju35mdz73x890835eynq1h9v",
      "cju3tsh4lfsok0987w6x3a0v1",
      "cju3ttznuftyf09875t11850w",
      "cju3ul8dogf1z09872y2ecowp",
      "cju3v664kh0px0818y4y7wolf",
      "cju3wstckialg0871xs0vevsj",
      "cju3x2s11ibzi0817kk284k0j",
      "cju3xeexgii1j0817zs68tb4g",
      "cju3y21quj0ir0818kgjagr15",
      "cju40u30gkuzc0871rq7t666d",
      "cju40wto5kxwi0755it190f2k",
      "cju41p90plcsx08018cnzpndc",
      "cju41z76wlgbz0801qdetlvby",
      "cju424hy5lckr085073fva1ok",
      "cju42dwedlmk60871jbgu4ehi",
      "cju42g865lorv07552ytz6xxa",
      "cju42tauqlo5p08171l3cuo4b",
      "cju42xpi8lw4w0871ve317a1p",
      "cju430pm2lz0y0755jkhcc3d1",
      "cju43c92lm5cj0755lorsorfg",
      "cju45pm27n80u08174kyow1gj",
      "cju5bmhdcafs909878qfzrqzi",
      "cju5bycdkalkb09875f7bfrvx",
      "cju5c7oijaqmq09878qwgqv8n",
      "cju5clr68b48r0755cmuvponm",
      "cju5cu8qkb84x08186jwo8yin",
      "cju5fs6j6d8350801vglraq4u",
      "cju5gucasds9d0801019axylx",
      "cju5huurrecm70801y680y13m",
      "cju5kre09fhka0850h7b1898j",
      "cju5nxkujgscq0817l9gss626",
      "cju5tenjojp1j0755ms4949h2",
      "cju5thdbrjp1108715xdfx356",
      "cju5ufn3skquf0818dhapnhba",
      "cju5vwbr4lhqn0987a1pji0ux",
      "cju5waeduln160817w0agirve",
      "cju5wqonpm0e60801z88ewmy1",
      "cju5wrapcm2290818jsh26ppb",
      "cju5xkwzxmf0z0818gk4xabdm",
      "cju5xopi0md7q0871r1sjc1av",
      "cju6vrs1ov8cr098788h8gs6j",
      "cju6vvb8svhed0801jjcquh5e",
      "cju6wuojavt740818b5qcv3iw",
      "cju6z2616wqbk07555bvnuyr1",
      "cju773hsyyosz0817pk1e7sjq",
      "cju7787c5yy3l080159mwqsnj",
      "cju77g99iyxc00817zqi2ppor",
      "cju77q10sz9ug0801449wu1nu",
      "cju7alcgr1lsr0871riqk84z7",
      "cju7aqkue1i2k09879uzcpt8r",
      "cju7arvfe1ldu0850erdmphgj",
      "cju7atnm31if40817pqclnjer",
      "cju7b10ce1mnm08011c5bwyr4",
      "cju7b1ygu1msd0801hywhy0mc",
      "cju7bduyq1rjf08719giru9ho",
      "cju7cl8zm1xcu0817ado0jpas",
      "cju7cq6su27qv075574dir0r3",
      "cju7csvlb22fr0850lvm45n3x",
      "cju7da88w2eod0755wejzynvt",
      "cju7dlk532dsh0871zvr6qz0r",
      "cju7dsrtb2f8i085064kwugfk",
      "cju7ebe962hr409872ovibahw",
      "cju7el6xv2k520817qxx9wdr5",
      "cju7eotqi2qea0871y8yc7tqh",
      "cju7er4kc2opa0801anuxc0eb",
      "cju7ezs7g2mxm098787atbran",
      "cju7fbndk2sl608015ravktum",
      "cju83yddek68q0850d2x7zfkm",
      "cju8418jhkf7d0818ga2v0xq0",
      "cju843yjskhq30818qre4rwm2",
      "cju84gpefknwm098714oq8q61",
      "cju85a8h8llwm07559wxg4t5w",
      "cju85ia1slh220987y7c20sm2",
      "cju85qefyln6v0850szeb9byi",
      "cju87vqa0ndwg0850onjdz7ol",
      "cju887ftknop008177nnjt46y",
      "cju888fr7nveu0818r9uwtiit",
      "cju88itqbny720987hxizbj5y",
      "cju88msmoo3470817m441j4sg",
      "cju88t4fvokxf07558ymyh281",
      "cju88v2f9oi8w0871hx9auh01",
      "cju88z8bson4h0871nnd7fdxo",
      "cju8aj01yqeqm0850lhdz3xdw",
      "cju8arof2qpf20850ifr1bnqj",
      "cju8at3s1qqqx0850hcq8nmnq",
      "cju8b2rmgr52s0801p54eyflx",
      "cju8b8yair65w09878pyqtr96",
      "cju8bafgqrf4x0818twisk3ea",
      "cju8bop5jrsid08716i24fqda",
      "cju8bpctzrqkr0850zeldv9kt",
      "cju8bqxxurs6i0850mu7mtef9",
      "cju8bw697rwg308177tg8huas",
      "cju8c2rqzs5t80850d0zky5dy",
      "cju8c6hnxsdvr0801wn0vrsa6",
      "cju8c9akjsdjj0850s67uzlxq",
      "cju8czvnztbf40871b4m7t78w",
      "cjyzjzssvd8pq0838f4nolj5l",
      "cjyzk8qieoboa0848ogj51wwm",
      "cjyzu3reghjya0794w7pwoi50",
      "cjyzu9th0qt4r0a46pyl4zik0",
      "ck2da7fwcjfis07218r1rvm95"
    ]
  },
  "fold_2": {
    "train": [
      "cju0qkwl35piu0993l0dewei2",
      "cju0qx73cjw570799j4n5cjze",
      "cju0roawvklrq0799vmjorwfv",
      "cju0rx1idathl0835detmsp84",
      "cju0s2a9ekvms080138tjjpxr",
      "cju0sr5ghl0nd08789uzf1raf",
      "cju0sxqiclckk08551ycbwhno",
      "cju0t4oil7vzk099370nun5h9",
      "cju0tl3uz8blh0993wxvn7ly3",
      "cju0u2g7pmnux0801vkk47ivj",
      "cju0ue769mxii08019zqgdbxn",
      "cju0vtox5ain6099360pu62rp",
      "cju15czxqp3lv0835jvhgzurz",
      "cju15jr8jz8sb0855ukmkswkz",
      "cju15l5ubz9yh0855b3ivdpse",
      "cju15mhjczc8z0801kit5c6di",
      "cju15ptjtppz40988odsm9azx",
      "cju160wshltz10993i1gmqxbe",
      "cju16ach3m1da0993r1dq3sn2",
      "cju16d65tzw9d0799ouslsw25",
      "cju16fpvhzypl0799p9phnlx6",
      "cju16jgnyzp970878melv7r25",
      "cju16whaj0e7n0855q7b6cjkm",
      "cju171py4qiha0835u8sl59ds",
      "cju175facms5f0993a5tjikvt",
      "cju17g6ykn1cs0993dww6qdi8",
      "cju17otoe119u0799nqcbl8n1",
      "cju17v6ih0u7808783zcbg1jy",
      "cju17x0j4nfc10993y31pvlgs",
      "cju1819curo000988pd5xcqme",
      "cju183od81ff608017ekzif89",
      "cju1871y11d6r0799k6cw4yze",
      "cju18849rrsgr0988p90hkygb",
      "cju18ibp219ub08783i6o98g7",
      "cju18kevfrojc0835bn90f1in",
      "cju1alwgo30z60855fm3y23sm",
      "cju1amqw6p8pw0993d9gc5crl",
      "cju1aqqv02qwz0878a5cyhr67",
      "cju1ats0y372e08011yazcsxm",
      "cju1b0y2e396p08558ois175d",
      "cju1b3zgj3d8e0801kpolea6c",
      "cju1c0qb4tzi308355wtsnp0y",
      "cju1c4fcu40hl07992b8gj0c8",
      "cju1c6yfz42md08550zgoz3pw",
      "cju1cbokpuiw70988j4lq1fpi",
      "cju1cnnziug1l0835yh4ropyg",
      "cju1cqc7n4gpy0855jt246k68",
      "cju1csmlc4ht10799b8ymmghg",
      "cju1cu1u2474n0878tt7v4tdr",
      "cju1cvkfwqrec0993wbp1jlzm",
      "cju1cyjb5qtie0993njqne9m3",
      "cju1d96gsv62d09881b3wecw2",
      "cju1dfeupuzlw0835gnxip369",
      "cju1dg44i4z3w0801nyz4p6zf",
      "cju1dhfok4mhe0878jlgrag0h",
      "cju1dia8hvc6v098827mgffnm",
      "cju1djtprvd7b0988thwwrg09",
      "cju1dnz61vfp40988e78bkjga",
      "cju1dq3x1vgx109889c7wyirg",
      "cju1efbr0rqxz09931z0lf4vf",
      "cju1egh885m1l0855ci1lt37c",
      "cju1egx9pvz2n0988eoy8jp23",
      "cju1ejj7dvqfa0835ra184v5m",
      "cju1euant5l960878iqj5vvto",
      "cju1euuc65wm00799m4sjdnnn",
      "cju1expq45zst0855rjqwwj4m",
      "cju1f320ewfyu0988ndz6blh5",
      "cju1f79yhsb5w0993txub59ol",
      "cju1f8w0t65en0799m9oacq0q",
      "cju1fj6axwfp30835ukhuzhw9",
      "cju1fjsb4sipq09931lvd8e41",
      "cju1fm3id6gl50801r3fok20c",
      "cju1fr4etsmrr09933u4t4aql",
      "cju1ftaji6isw0855108yqcse",
      "cju1fuoa4wmc50835qfd11sp9",
      "cju1g4nsb6ngy0799l4ezm8ab",
      "cju1gghyjwxt80835vx0wgxw0",
      "cju1gi7jlwyld0835cdf6g6qz",
      "cju1gkndf6yi10801o1qnje19",
      "cju1gv7106qd008784gk603mg",
      "cju1h89h6xbnx08352k2790o9",
      "cju1haab178i70799tk9z8y8x",
      "cju1hirfi7ekp0855q0vgm9qq",
      "cju1hmff8tkp809931jps6fbr",
      "cju1hp9i2xu8e0988u2dazk7m",
      "cju1hs0za7jha0855vj0mdrjt",
      "cju2hdr06v2bq0799mbm3bks1",
      "cju2hewssldzx0835ep795xu0",
      "cju2hfqnmhisa0993gpleeldd",
      "cju2hgsptlfam0835o3b59h1o",
      "cju2hjrqcvi2j0801bx1i6gxg",
      "cju2hos57llxm08359g92p6jj",
      "cju2hqt33lmra0988fr5ijv8j",
      "cju2hugv9vget0799hhk7ksvg",
      "cju2hx006vidl0799igm81vmh",
      "cju2i6acqvo6l0799u20fift8",
      "cju2i8br1vqtd08784u6vmcjk",
      "cju2iatlki5u309930zmgkv6h",
      "cju2ij9uiic2l09933ljiv6gm",
      "cju2lejzcy4pc0878c9rlonot",
      "cju2lz8vqktne0993fuym6drw",
      "cju2m56cryvqd0801gtn2yp8t",
      "cju2mfjndoz700988b9lc3zeq",
      "cju2mh8t6p07008350e01tx2a",
      "cju2nd7l7z98o0799gfjvyfmw",
      "cju2nfnvxzdkd0878399axlco",
      "cju2nnqrqzp580855z8mhzgd6",
      "cju2np2k9zi3v079992ypxqkn",
      "cju2nqapmzvk20801f9us40dx",
      "cju2nsmwjlzyl0993jl80chvz",
      "cju2ntxtdzlvu0799xl3j9pan",
      "cju2okvco06xc0799kxe5n1qh",
      "cju2omjpeqj5a0988pjdlb8l1",
      "cju2oo0wh0bqy0878biujeyhe",
      "cju2oq5570avm079959o20op1",
      "cju2osuru0ki00855txo0n3uu",
      "cju2p0eveqtdc0835gpi3p93i",
      "cju2p91qir00k08350ddfif0w",
      "cju2pag1f0s4r0878h52uq83s",
      "cju2phaksnahz0993yxogjcpv",
      "cju2pjb9v0ywn0878j5g5n69j",
      "cju2pkwt3r8b90988v2ywq1px",
      "cju2pmhtr17a00855cvpelzb0",
      "cju2qozsk20cq0855ugrg3cri",
      "cju2qvuj1s9ok0835tp2k4ozh",
      "cju2qxxko229x08786gvxxhur",
      "cju2qz06823a40878ojcz9ccx",
      "cju2r2obh2bjm08553kng0rh7",
      "cju2r6mt2om21099352pny5gw",
      "cju2r7h21sj9608354gzks3ae",
      "cju2r91dg2k090801bh0xzbxk",
      "cju2raxlosl630988jdbfy9b0",
      "cju2rga4psq9n09881z519xx0",
      "cju2ricdv2iys0878sv1adh0u",
      "cju2rkjfwoxys0993x768l1j2",
      "cju2rn0hasxri0835nfy3buay",
      "cju2rnkt22xep0801as160g9t",
      "cju2ro5jqsy680988pi6qsujw",
      "cju2rqo702wpx0855fn7d5cxh",
      "cju2rxm8rpbaf0993o3qr2oph",
      "cju2rz4k434s70855wwx3ddtx",
      "cju2s16zp317h0799gr67jqc2",
      "cju2s2527pfyr0993l3h1149a",
      "cju2s9g11pnra0993gn4eh793",
      "cju2sggy13na70855tbeoqgha",
      "cju2sszfq3uye0878sucelzk2",
      "cju2suk42469908015ngmq6f2",
      "cju2syxa93yw40799x2iuwabz",
      "cju2t16vuucaq0835xcpsivn2",
      "cju2t2ivz43i10878zeg8r1br",
      "cju2t3ibkuecr0835o7si16zv",
      "cju2ti1du4idn0878giuozonw",
      "cju2top2ruxxy0988p1svx36g",
      "cju2tpfa5uyx408359datxqqj",
      "cju2tqfgw4oat0799rn0g5b2z",
      "cju2trbpkv0c00988hxla5dzz",
      "cju2txjfzv60w098839dcimys",
      "cju2tzypl4wss0799ow05oxb9",
      "cju2u4pymvc720988wsxrmi84",
      "cju2u73dj53oz0878486k8k4b",
      "cju2uokeg5jm20799xwgsyz89",
      "cju2urqpwvxw70835rvndvtsi",
      "cju2uwz9f5yf1085506cfamfx",
      "cju2uy8ox62jo0801g88hh42z",
      "cju2uzabhs6er0993x3aaf87p",
      "cju2wtwj87kys0855kx6mddzw",
      "cju2wxv0hxs2f09884w48v8fi",
      "cju2x7vw87mu30878hye2ca0m",
      "cju2xbk0080y80801eghyddi2",
      "cju2xd75m82720801q4s4ik3n",
      "cju2xf8e5y2wm08359vcgk09b",
      "cju2xjz2ju8pe0993ysv9wg17",
      "cju2xlcqxy9c60988vjacdznb",
      "cju2xwm1s84l50799i60mq0pu",
      "cju2xyd9vyi7m098831qcucse",
      "cju2y0z6g87p10878fpk5d3rq",
      "cju2y26c588bo07993ksd8eoz",
      "cju2y40d8ulqo0993q0adtgtb",
      "cju2ycp1u8g2r0799jslnp7cz",
      "cju2yg5ht8i4p087800js8hp4",
      "cju2yi9tz8vky0801yqip0xyl",
      "cju2yo1j1v0qz09934o0e683p",
      "cju2ysg748ru80878sp6j0gm0",
      "cju2yw4s7z7p20988lmf2gdgd",
      "cju2yyhsp933j0855hp32e012",
      "cju2z1nxlzaj40835wj81s1iy",
      "cju2z45kuzf6d0988nz2c819m",
      "cju2z6ez69g4u0801qwt088lw",
      "cju2z9vlp9j0w0801oag91sy9",
      "cju2zblxw9848087853csbrx1",
      "cju2zdhsczmn50988z64qwg2q",
      "cju2zdvjn9h7r08553cp4eed5",
      "cju2zi4l09f5807991s8do2b2",
      "cju2zo0fwzv580988qlijd2xa",
      "cju2zr3c3vwb00993jn06bbaz",
      "cju2zwg05a0oy0801yr73ig7g",
      "cju302fqq9spc0878rrygyzzz",
      "cju30525w04r10835ygp257sb",
      "cju306x7w05nb0835cunv799x",
      "cju30ajhw09sx0988qyahx9s8",
      "cju30bmab08bi0835mvlr6e0r",
      "cju30df2j09dd08351ayx2t6w",
      "cju30ftgja7170855xl9bkdm0",
      "cju30gxjq0djk0988jytm49rs",
      "cju30ia8da2bq0799klnehml2",
      "cju30j1rgadut0801vuyrsnt8",
      "cju30k2z40ds308353kdew70n",
      "cju30lncba3ny0878jwnous8n",
      "cju30mm25a53s0799qa5wiqe8",
      "cju30nyxe0gfb0835p256yoju",
      "cju30ov1oah920801mi8thuyg",
      "cju30qbm1ad3x0855znuhpz9u",
      "cju30xqmh0ni00835ix3batv1",
      "cju30ywtc0oar0835bp2en7ec",
      "cju310f6val1v0855xo8tc3gu",
      "cju3128yi0rpu0988o4oo5n8n",
      "cju31rb7vb6110801p9rhacuw",
      "cju31t8xd17bk0835rnb893jk",
      "cju31y80qbawn0801twwm2l5s",
      "cju320gyvbch60801v2amdi2g",
      "cju324q101fhe08350wae9cif",
      "cju326h4v1gxw08352px40p7r",
      "cju3280wv1ir009882jze27tc",
      "cju32a52lb9rc0799xi40qs00",
      "cju32csyfblyh080170aa3x5p",
      "cju32fhnhbds40799broyoptc",
      "cju32gzs6xo8x0993r8tedbpb",
      "cju32jcdabepz0878d0cznmfe",
      "cju32l161bi1v07990vm376in",
      "cju32pzh9bpw10855q4vaxfhe",
      "cju32qr9tbvsj08013pkpjenq",
      "cju32zhbnc1oy0801iyv1ix6p",
      "cju330ofbc2l30801th5g3hw6",
      "cju334jzo261t0835yqudnfs1",
      "cju336l68y7if0993wf092166",
      "cju33belnbyhm0878yxl42233",
      "cju33o12x2jm50988944mxq0v",
      "cju33w4sdcivk0855x879zht7",
      "cju33x0f22peh0988g0ln7w5v",
      "cju34aozyyy830993bn16u32n",
      "cju34ds2531520988qjpqt6e3",
      "cju34eqjpcpm508788b3lhp97",
      "cju34fojcctcf0799ebolbvkn",
      "cju34m7h536wq0988xz7gx79v",
      "cju34sh43d8zm08019xbwhc0o",
      "cju34uhepd3dd0799hs8782ad",
      "cju34ymm8d6700799uop0cw33",
      "cju34zivp3fq80988opxbaqyn",
      "cju3518w2d838079939fqztbc",
      "cju3521y5d5mq0878t3ezsu4p",
      "cju353d1eda8c07992afde611",
      "cju35740hzm0g0993zl5ic246",
      "cju357rxxdaz30878y2esjpjt",
      "cju358pwtdby20878cg7nm0np",
      "cju35atpxdjot0855q46aqrd0",
      "cju35c4wzdhow0799h6eq4sgs",
      "cju35fxqyzt5p0993vusm54qz",
      "cju35ldepdtlm0801yv79y8vu",
      "cju35mdz73x890835eynq1h9v",
      "cju35oyvd3y850988km12hdz1",
      "cju3tsh4lfsok0987w6x3a0v1",
      "cju3ttznuftyf09875t11850w",
      "cju3tvffffx5f0818t5ov22al",
      "cju3tx1qyg0c907552fglumhc",
      "cju3u1c8tfyqx08503iedc3mx",
      "cju3uhb79gcgr0871orbrbi3x",
      "cju3ul8dogf1z09872y2ecowp",
      "cju3umoh1geet0817cmpef5am",
      "cju3uz4o6gr9z0850lhxyxvsj",
      "cju3v11mrgwwb0755u242ygye",
      "cju3v56bwgy8v0871w14pz8fx",
      "cju3v664kh0px0818y4y7wolf",
      "cju3wstckialg0871xs0vevsj",
      "cju3x2s11ibzi0817kk284k0j",
      "cju3x4blzieu30850x10uuvbm",
      "cju3x5u2tiihx0818914gzxy1",
      "cju3xeexgii1j0817zs68tb4g",
      "cju3xga12iixg0817dijbvjxw",
      "cju3xiic0ilzp0850lusrb42j",
      "cju3xl264ingx0850rcf0rshj",
      "cju3xtufwiv9c0818djsc4cqd",
      "cju3xwpgviwlx0871rwm15q7v",
      "cju3xzvnzj0hd0755xprz39nj",
      "cju3y0pjrj1c30755nxekxccj",
      "cju3y21quj0ir0818kgjagr15",
      "cju3y79ofj3va0871uqfb1mzo",
      "cju3y9difj6th0801kd1rqm3w",
      "cju3yb47cj1xq0817zfotbni4",
      "cju3ykamdj9u208503pygyuc8",
      "cju40poe4kt9s0755f9cnm3h5",
      "cju40r6jrksyk0871wg98zgho",
      "cju40taxlkrho0987smigg0x0",
      "cju40u30gkuzc0871rq7t666d",
      "cju40wto5kxwi0755it190f2k",
      "cju414lf2l1lt0801rl3hjllj",
      "cju418rckl3ur08012psrx1r1",
      "cju41kd7yl4nm0850gil5qqwh",
      "cju41nz76lcxu0755cya2qefx",
      "cju41p90plcsx08018cnzpndc",
      "cju41s6nbleqy0755e2mslg0b",
      "cju41z76wlgbz0801qdetlvby",
      "cju422cm8lfxn0818ojicxejb",
      "cju424hy5lckr085073fva1ok",
      "cju426tomlhll0818fc0i7nvh",
      "cju428k5fldt108177s6g6f45",
      "cju42dwedlmk60871jbgu4ehi",
      "cju42g865lorv07552ytz6xxa",
      "cju42nm68lpyo0818xvvqmupq",
      "cju42py9mlqyd0818u3d1d7ga",
      "cju42qet0lsq90871e50xbnuv",
      "cju42romflni20817etb9a0fl",
      "cju42tauqlo5p08171l3cuo4b",
      "cju42u5bjlvi10801dc13sskp",
      "cju42wamblrqn098798r2yyok",
      "cju42xpi8lw4w0871ve317a1p",
      "cju430pm2lz0y0755jkhcc3d1",
      "cju439oazm2fu0871ma0vvrft",
      "cju43b8daly4408170e5ev06g",
      "cju43c92lm5cj0755lorsorfg",
      "cju43eigtm6ev0801mv0m96t1",
      "cju43gfosm63n08714rpih8pe",
      "cju43h43am1dy08176gwfhmnt",
      "cju43jcqim2cp08172dvjvyui",
      "cju43kj2pm34f0850l28ahpni",
      "cju43lcnum9y10755bjs7z87f",
      "cju43o6n7m9nk087191ijwqq9",
      "cju45jpvfn6c809873pv1i34s",
      "cju45lbgznahl08180xz1h7u6",
      "cju45n0oxn5vu08500yfrt9jn",
      "cju45ofd9ne1j0801ri8dup7t",
      "cju45pm27n80u08174kyow1gj",
      "cju45t5ddnbio0987qtqzx762",
      "cju45v0pungu40871acnwtmu5",
      "cju5b9oyda4yr0850g9viziyv",
      "cju5bbtwsa8cl0987wgfsqpao",
      "cju5bf6hxa6m50817rbwettgu",
      "cju5bmhdcafs909878qfzrqzi",
      "cju5boicjagt20871b1fotkh4",
      "cju5buy2bal250818ipl6fqwv",
      "cju5bycdkalkb09875f7bfrvx",
      "cju5c5xc7algd0817pb1ej5yo",
      "cju5c7oijaqmq09878qwgqv8n",
      "cju5ca9hcatkc0801jzwe7tfx",
      "cju5ccpvqash50850kb4bs22k",
      "cju5cetivauok0987ok3e5bre",
      "cju5chrxxawka0871qcj171yz",
      "cju5cjh3xattc0817j2vbulzi",
      "cju5clr68b48r0755cmuvponm",
      "cju5cu8qkb84x08186jwo8yin",
      "cju5eftctcdbj08712gdp989f",
      "cju5eyfe9cpk90987laa7tsl3",
      "cju5f26ebcuai0818xlwh6116",
      "cju5fb86jd1jp0755b1ukbhq5",
      "cju5fs6j6d8350801vglraq4u",
      "cju5fu081d8gc0818l3yylujk",
      "cju5g163vd6mt0817uccuga6u",
      "cju5gucasds9d0801019axylx",
      "cju5h57xedz5h0755mjpc8694",
      "cju5hi52odyf90817prvcwg45",
      "cju5hl8nee8a40755fm8qjj0o",
      "cju5hqz50e7o90850e0prlpa0",
      "cju5ht88gedbu0755xrcuddcx",
      "cju5huurrecm70801y680y13m",
      "cju5hwonqedw10801vsd3w6kk",
      "cju5hyi9yegob0755ho3do8en",
      "cju5i39mreass0817au8p22zy",
      "cju5jx7jzf7c90871c2i9aiov",
      "cju5jz5fff8c50871hbe6108f",
      "cju5k503sfa5f0871lx0rpu5y",
      "cju5k7r0yf98c09878csbxb4d",
      "cju5knbbqfipk080128cggukq",
      "cju5kre09fhka0850h7b1898j",
      "cju5nxkujgscq0817l9gss626",
      "cju5nyu31gv8e0871zpk74a2n",
      "cju5o1vu9gz8a0818eyy92bns",
      "cju5tenjojp1j0755ms4949h2",
      "cju5thdbrjp1108715xdfx356",
      "cju5u6wf0kh1t0755bg1ssixv",
      "cju5udcufki0s09874ll1dbr5",
      "cju5ufn3skquf0818dhapnhba",
      "cju5uget8krjy0818kvywd0zu",
      "cju5uhrdwkmsu0817ervv91l8",
      "cju5uxjnol2r509871qv2yeia",
      "cju5uzmaol56l0817flxh4w9p",
      "cju5vbo6jldrt0871jf6f1700",
      "cju5vcmrqla7i0817x4sp4pqw",
      "cju5vgawslbe30987ndeepc1b",
      "cju5vi4nxlc530817uoqm2m7a",
      "cju5vwbr4lhqn0987a1pji0ux",
      "cju5vxuc5loxw0818u8xgf45p",
      "cju5vzjoslpj708186z2fusmz",
      "cju5waeduln160817w0agirve",
      "cju5woy82m07m08505dmjg7g1",
      "cju5wqonpm0e60801z88ewmy1",
      "cju5wrapcm2290818jsh26ppb",
      "cju5wrrs0m2af0818vmnajbtw",
      "cju5wtdu4m0im0871mix0yvc0",
      "cju5wuhm1lwm40987vugqn3vv",
      "cju5x00l6m5j608503k78ptee",
      "cju5x15djm7ae0755h8czf6nt",
      "cju5xkwzxmf0z0818gk4xabdm",
      "cju5xopi0md7q0871r1sjc1av",
      "cju5y4hgqmk0i08180rjhbwvp",
      "cju5y7buemcw80987p0r30g9f",
      "cju5yclrymlgj0818k426ud6z",
      "cju5ygh1zmmdi0755uod5e17i",
      "cju5yhgznmkzb0801cji2vi8j",
      "cju5yjq1pmlgc0801z0t24bly",
      "cju6uy20suzbl0987rzuhz7z9",
      "cju6uzxk0v83p0801rcwnexdu",
      "cju6v3bb2v7xo085090blubyw",
      "cju6v4szov55u0871qmqz3v8n",
      "cju6v5ilsv8hk0850rb5sgh6o",
      "cju6v6g6kvdw007552x6mb0po",
      "cju6vgdmivcvb08018fra5lnv",
      "cju6vifjlv55z0987un6y4zdo",
      "cju6vqarjv7yo0987q4b1btk1",
      "cju6vrs1ov8cr098788h8gs6j",
      "cju6vucxvvlda0755j7msqnya",
      "cju6vvb8svhed0801jjcquh5e",
      "cju6vvxsev9y30987kespucdg",
      "cju6w733bveoz0817e600tw72",
      "cju6wjm81vgsc0987enk9n3pr",
      "cju6wt9jvvn500871hjn3t3g0",
      "cju6wuojavt740818b5qcv3iw",
      "cju6x0yqbvxqt0755dhxislgb",
      "cju6x97w4vwua0850x0997r0a",
      "cju6xa0qmvzun0818xjukgncj",
      "cju6xmqd9w0250817l5kxfnsk",
      "cju6yxyt0wh080871sqpepu47",
      "cju6z2616wqbk07555bvnuyr1",
      "cju6z600qwh4z081700qimgl9",
      "cju6z9a9kwsl007552s49rx6i",
      "cju76erapykj30871x5eaxh4q",
      "cju76l27oyrw907551ri2a7fl",
      "cju76lsehyia10987u54vn8rb",
      "cju76o55nymqd0871h31sph9w",
      "cju772304yw5t0818vbw8kkjf",
      "cju773hsyyosz0817pk1e7sjq",
      "cju7787c5yy3l080159mwqsnj",
      "cju77afzlz3kp07550x5nafzs",
      "cju77b3wyz4160755qis4ljsb",
      "cju77bvg0yv4r0987yh60xmjo",
      "cju77g99iyxc00817zqi2ppor",
      "cju77idwfz36d0871tzfzz51i",
      "cju77j66ez52p08019xygi0co",
      "cju77q10sz9ug0801449wu1nu",
      "cju77re6fz5bb0817vp9redjg",
      "cju77vvcwzcm50850lzoykuva",
      "cju783tmkzkqu081803g7q5vk",
      "cju784jpdzeae0987q5ypq883",
      "cju785htizjzo08017tvlhtg4",
      "cju787jnjzjuj0871p94nck9g",
      "cju7ae7bq1f820987toc8si1d",
      "cju7aez2x1jtj0871ztezs3oi",
      "cju7agj961l2r0818z29iq8yn",
      "cju7ahtkb1jr90801jck4kbds",
      "cju7aifyo1p3n07552nxjx51f",
      "cju7alcgr1lsr0871riqk84z7",
      "cju7ap09p1kz10850ldccjebj",
      "cju7apr0c1qqm0755s7msqot4",
      "cju7aqkue1i2k09879uzcpt8r",
      "cju7arvfe1ldu0850erdmphgj",
      "cju7atnm31if40817pqclnjer",
      "cju7avvi51iox0817ym55y6tt",
      "cju7azuu31mia0801pf9ib5ed",
      "cju7b10ce1mnm08011c5bwyr4",
      "cju7b1ygu1msd0801hywhy0mc",
      "cju7b2l561oas0871decgslaf",
      "cju7b3f5h1sm40755i572jden",
      "cju7b9vcs1luz0987ta60j1dy",
      "cju7bc95p1mdm0817yqj5jc6j",
      "cju7bduyq1rjf08719giru9ho",
      "cju7bfx651qr80801cs7epotb",
      "cju7bgnvb1sf808717qa799ir",
      "cju7cl8zm1xcu0817ado0jpas",
      "cju7cp6dw244p0818gncdol4m",
      "cju7cq6su27qv075574dir0r3",
      "cju7crgxa28550755wbsgqkel",
      "cju7csvlb22fr0850lvm45n3x",
      "cju7ctvqn25dy08186g442m1r",
      "cju7cue9b232j0801qdzk1ykj",
      "cju7d1tvt25bu08019dvw3uff",
      "cju7d2q1k27nf08715zshsckt",
      "cju7d3oc82cho0755dajlwldz",
      "cju7d5m0p23kn09871rk7pu3v",
      "cju7d7tly27h408016fyp5nr7",
      "cju7d8m3b2e210755l8fj1yph",
      "cju7da88w2eod0755wejzynvt",
      "cju7db7lp2f400755tntd1ohf",
      "cju7dda8w2br20818zhsuz8s7",
      "cju7ddtz729960801uazp1knc",
      "cju7deifq2fzn0755lc8idyh8",
      "cju7dglf226g50987ohbthl19",
      "cju7dizi82h2i0755doucgnt3",
      "cju7dlk532dsh0871zvr6qz0r",
      "cju7dmlgf2ebw0871ieqas5fh",
      "cju7dn24o296i09871qfxb8s2",
      "cju7dp3dw2k4n0755zhe003ad",
      "cju7dqcwi2dz00850gcmr2ert",
      "cju7dsrtb2f8i085064kwugfk",
      "cju7dtb1e2j0t0818deq51ib3",
      "cju7dvl5m2n4t0755hlnnjjet",
      "cju7dwe282dc309876rco45ts",
      "cju7dxffn2eam0817qxosfwch",
      "cju7dymur2od30755eg8yv2ht",
      "cju7ea4om2l910801bohqjccy",
      "cju7ebe962hr409872ovibahw",
      "cju7ecl9i2i060987xawjp4l0",
      "cju7efffp2ivf0817etg3jehl",
      "cju7ehljc2or70871261br8ai",
      "cju7ejm2l2ncl0801wq6y84nw",
      "cju7ekbo32pft0871fv7kzwb9",
      "cju7el6xv2k520817qxx9wdr5",
      "cju7emdni2py40871ivhxjtut",
      "cju7eotqi2qea0871y8yc7tqh",
      "cju7epwj82koz098713apjnzo",
      "cju7er4kc2opa0801anuxc0eb",
      "cju7es23b2vcp0755gpbm9s7v",
      "cju7et17a2vjk0755e743npl1",
      "cju7etr3y2p4t0801cdzjj8ab",
      "cju7eueum2oqn0850rodmx8zo",
      "cju7evxt12m730987rxivne3x",
      "cju7ey10f2rvf0871bwbi9x82",
      "cju7ez7r22qbc08015xfoz2wb",
      "cju7ezs7g2mxm098787atbran",
      "cju7f4sc62xqj075597xpmuoy",
      "cju7f5ghb2r5s0801chwkxxh9",
      "cju7fbndk2sl608015ravktum",
      "cju7fd6yt2p740987wkr8exo1",
      "cju7ff97z2ow40817u2r83my5",
      "cju7fmvpk2q170987v6i3ola8",
      "cju7fob3x301u0755x985pmmq",
      "cju7fpfzq2wyf0818xxd1oziv",
      "cju7fq7mm2pw508176uk5ugtx",
      "cju7frtqu2xa20818wq8r9fzf",
      "cju7g7ba42z310987bqzbi2bq",
      "cju83h9ysjwe808716nt35oah",
      "cju83ipu3jwpx0801z5pvguf8",
      "cju83k8fyjsxr0817d6nxs6r4",
      "cju83kxitjv340987z09m0ezy",
      "cju83mki1jv5w0817kubxm31r",
      "cju83nwu1jxte0987h1krpfmv",
      "cju83qd0yjyht0817ktkfl268",
      "cju83rcnzkbsj0755x5anfrcg",
      "cju83syhdk6gs0801rf1rekdl",
      "cju83wwn1k55e0850kw6i2d81",
      "cju83yddek68q0850d2x7zfkm",
      "cju8418jhkf7d0818ga2v0xq0",
      "cju843yjskhq30818qre4rwm2",
      "cju847pxykriq0755268ktrk2",
      "cju849c23kgnk0817cgv2hw1e",
      "cju84dsvaklpx098750hp83x4",
      "cju84ffdzkrjn08183jh1fxmb",
      "cju84gpefknwm098714oq8q61",
      "cju84jdl9kv0i0871eog9b3i9",
      "cju8567gdlcbq0801dwwyo2jt",
      "cju85a8h8llwm07559wxg4t5w",
      "cju85bf1algsq0871y9gtlq97",
      "cju85citjlnfm0755i4rk5tqj",
      "cju85dx63lic408017f0l0400",
      "cju85ia1slh220987y7c20sm2",
      "cju85je7vlht70817c9jcjwi4",
      "cju85mpuglq8k0818d2it6hzb",
      "cju85nr8elly209872w9n5m0s",
      "cju85omszllp30850b6rm9mi3",
      "cju85qefyln6v0850szeb9byi",
      "cju85rkbnlo1c08503uxcpax1",
      "cju87kbcen2av0987usezo8kn",
      "cju87li0zn3yb0817kbwgjiz8",
      "cju87nkyrnb970801q84m47yt",
      "cju87q6yoneim0871dl4phvkd",
      "cju87r56lnkyp0755hz30leew",
      "cju87tyddnnad0755bj0wxahe",
      "cju87vqa0ndwg0850onjdz7ol",
      "cju87xn2snfmv0987sc3d9xnq",
      "cju87z6o6nh73085045bzsx6o",
      "cju87zv8lni0o0850hbbecbq6",
      "cju8828oxnool0801qno9luhr",
      "cju884985nlmx0817vzpax3y4",
      "cju885ikhnmkn09878s2lqtuh",
      "cju886ryxnsl50801r93jai7q",
      "cju887ftknop008177nnjt46y",
      "cju888fr7nveu0818r9uwtiit",
      "cju88fpm4o0tl0871w1i6a4ds",
      "cju88gx09o2vk0818610zody3",
      "cju88itqbny720987hxizbj5y",
      "cju88l66no10s0850rsda7ej1",
      "cju88msmoo3470817m441j4sg",
      "cju88oh0po9gq0801nge4tgr1",
      "cju88t4fvokxf07558ymyh281",
      "cju88trl3ogi208716qvti51b",
      "cju88v2f9oi8w0871hx9auh01",
      "cju88y1mwoln50871emyfny1g",
      "cju88z8bson4h0871nnd7fdxo",
      "cju890guyoiti098753yg6cdu",
      "cju8914beokbf0850isxpocrk",
      "cju892fesoq2g0801n0e0jyia",
      "cju893jmdompz0817xn3g1w4h",
      "cju89y9h0puti0818i5yw29e6",
      "cju89z6pqpqfx0817mfv8ixjc",
      "cju8a56vxpy780850r45yu4wk",
      "cju8a84g0q76m0818hwiggkod",
      "cju8adb60qbiu080188mxpf8d",
      "cju8aj01yqeqm0850lhdz3xdw",
      "cju8ando2qqdo0818ck7i1be1",
      "cju8apjewqrk00801k5d71gky",
      "cju8arof2qpf20850ifr1bnqj",
      "cju8ashhnquqr0801rwduzt7d",
      "cju8at3s1qqqx0850hcq8nmnq",
      "cju8aw9n1qyg10801jkjlmors",
      "cju8axq24r4an0755yhv9d4ly",
      "cju8ayeq7r1fb0818z1junacy",
      "cju8azmhcr66e0755t61atz72",
      "cju8b0jr0r2oi0801jiquetd5",
      "cju8b1v3br45u087189kku66u",
      "cju8b2rmgr52s0801p54eyflx",
      "cju8b542nr81x0871uxnkm9ih",
      "cju8b5p40r2c60987ofa0mu03",
      "cju8b6rp0r5st0850184f79xt",
      "cju8b7aqtr4a00987coba14b7",
      "cju8b8yair65w09878pyqtr96",
      "cju8bafgqrf4x0818twisk3ea",
      "cju8bbznkrf5g0871jncffynk",
      "cju8bff9nrfi10850fmfzbf8v",
      "cju8bh8surexp0987o5pzklk1",
      "cju8bi8q7rlmn0871abc5ch8k",
      "cju8bj2ssrmlm0871gc2ug2rs",
      "cju8bk8oirjhw0817hgkua2w8",
      "cju8bljw9rqk20801kr54akrl",
      "cju8bm24yrrdp081829mbo8ic",
      "cju8bop5jrsid08716i24fqda",
      "cju8bpctzrqkr0850zeldv9kt",
      "cju8bqxxurs6i0850mu7mtef9",
      "cju8bssulrrcy0987h1vq5060",
      "cju8buos5rz9b08715lfr0f4f",
      "cju8bw697rwg308177tg8huas",
      "cju8bzzy2s66m08016z6mouqt",
      "cju8c1a0ws7o208181c6lbsom",
      "cju8c2rqzs5t80850d0zky5dy",
      "cju8c3xs7sauj0801ieyzezr5",
      "cju8c5mxls96t0850wvkvsity",
      "cju8c6hnxsdvr0801wn0vrsa6",
      "cju8c82iosagu0817l74s4m5g",
      "cju8c9akjsdjj0850s67uzlxq",
      "cju8cattbsivm0818p446wgel",
      "cju8cbsyssiqj0871gr4jedjp",
      "cju8cdeazsm8h0801jxifmzur",
      "cju8ceacrsqkr0755hdz145es",
      "cju8cgi2kspp308011nxdtjp6",
      "cju8chdlqsu620755azjty1tj",
      "cju8chxndsre008015uisl4si",
      "cju8cj10qsrau0871o2dr6ai1",
      "cju8cwy02t9eq08185qn12c02",
      "cju8czvnztbf40871b4m7t78w",
      "cju8d2q30tfhs0801n7lx77xl",
      "cju8dic9mtppa0987swn23wbc",
      "cju8djdqztu6408506pzhlo18",
      "cju8dm2cau2km0818jsv9eeq2",
      "cju8doa16u5gh0818w1ywda3q",
      "cju8dpa89u6l80818dj6lldh9",
      "cju8dqkrqu83i0818ev74qpxq",
      "cjyzjzssvd8pq0838f4nolj5l",
      "cjyzk8qieoboa0848ogj51wwm",
      "cjyzkmjy8evns070165gf9dmq",
      "cjyzl833ndne80838pzuq6ila",
      "cjyztzaqtrv430848l8xgcerw",
      "cjyzu3reghjya0794w7pwoi50",
      "cjyzu9th0qt4r0a46pyl4zik0",
      "cjyzufihqquiw0a46jatrbwln",
      "cjyzul1qggwwj07216mhiv5sy",
      "cjyzurzvohqnr0794es1itzek",
      "ck2395w2mb4vu07480otsu6tw",
      "ck2bxiswtxuw80838qkisqjwz",
      "ck2bxlujamu330725szlc2jdu",
      "ck2bxpfgxu2mk0748gsh7xelu",
      "ck2bxqz3evvg20794iiyv5v2m",
      "ck2bxskgxxzfv08386xkqtqdy",
      "ck2bxw18mmz1k0725litqq2mc",
      "ck2da7fwcjfis07218r1rvm95"
    ],
    "val": [
      "cju0qoxqj9q6s0835b43399p4",
      "cju0s690hkp960855tjuaqvv0",
      "cju0u82z3cuma0835wlxrnrjv",
      "cju13fwthn9mq0835gacxgy01",
      "cju13hp5rnbjx0835bf0jowgx",
      "cju14g8o4xui30878gkgbrvqj",
      "cju14pxbaoksp0835qzorx6g6",
      "cju15wdt3zla10801odjiw7sy",
      "cju16b6ynq8e40988m8vx0xnj",
      "cju17bz250pgd0799u1hqkj5u",
      "cju17hw9hr9c5098800fu4u8e",
      "cju17r8il13910799dr2wme2e",
      "cju18gzrq18zw0878wbf4ftw6",
      "cju1bhnfitmge0835ynls0l6b",
      "cju1cdxvz48hw0801i0fjwcnk",
      "cju1cfhyg48bb0799cl5pr2jh",
      "cju1d31sp4d4k0878r3fr02ul",
      "cju1ddr6p4k5z08780uuuzit2",
      "cju1drnhbrb9409935wi7vkhg",
      "cju1erep75us208553i4ofwwe",
      "cju1f15k3w4ct0835cmde6ypo",
      "cju1f5x1164xv08555654c24r",
      "cju1fb9236a110801yvg0fwju",
      "cju1fmsyf6gxb0801cimx2gle",
      "cju1g20bdwq6u0835e16xugcd",
      "cju1h5w4wxajx0835mc954kxy",
      "cju1hyolc7aqu0878rrkfn1lr",
      "cju2hlm19vjjf0801o69qnber",
      "cju2i03ptvkiu0799xbbd4det",
      "cju2i3hzclw3o0988rrgh911i",
      "cju2lcyfgkf5809932fn9gucn",
      "cju2lyynuymli0855g7fxgbhe",
      "cju2nguelpmlj0835rojdn097",
      "cju2ouil2mssu0993hvxsed6d",
      "cju2p4ddkmzxj0993p94o62av",
      "cju2qfie4rvz508357kad9z5o",
      "cju2qh5le1ock0878oahaql7d",
      "cju2qqn5ys4uo0988ewrt2ip2",
      "cju2qu37qobl50993aw7ghcfq",
      "cju2r11x7sdgx0988o8ule0wl",
      "cju2rlqdnoz9k0993cpjae3x0",
      "cju2rmd2rsw9g09888hh1efu0",
      "cju2rzpsmtb0f0835jabkbao1",
      "cju2saez63gxl08559ucjq3kt",
      "cju2srvy5440s0801y1ba9akr",
      "cju2sxf3iqbpv09937iksn8ep",
      "cju2t62nq45jl0799odpufwx6",
      "cju2t9tdwuk700835kv0ljmtl",
      "cju2tjrog4jy30878pawyazqc",
      "cju2tvrvm53ws0801a0jfjdxg",
      "cju2ulk385h170799rlklxob0",
      "cju2upu4evw7g08358guwozxv",
      "cju2wx0gh7fpz0878wwyd9ep8",
      "cju2wzu8wxtgu09880ku9x1pg",
      "cju2xa3i4y0160988i679zsqd",
      "cju2xs6na81t20878pt6nkfip",
      "cju2y5zas8m7f0801d34g5owq",
      "cju2y8s56ymqr083541ggdsml",
      "cju2yb31a8e8u0878wdashg7o",
      "cju2z2x3nvd3c099350zgty7w",
      "cju2zjcvj9qma0801dk71hhi0",
      "cju2zm0axztpe0988r8s9twjr",
      "cju2zy1e49pqk0878t6ncqn12",
      "cju300m3s04fg0988uzupuf7z",
      "cju30u1hbakn808019g15nb8b",
      "cju323ypb1fbb0988gx5rzudb",
      "cju33231uy4gi0993qc7b1jch",
      "cju3381d8bz3h07991xtl7ra0",
      "cju33qpdvc9g0087825jhf3s9",
      "cju34i3qvcyog0855qiejxx5w",
      "cju35eg0tdmjt085525sb4bua",
      "cju3tp94kfstl08181awh6z49",
      "cju3u39fog1bo0871lxjrabks",
      "cju3ua8u0g9rg0801uayhdxhu",
      "cju3x9lttikfb0818a0g104zn",
      "cju3xjqtpikx50817tppy6g84",
      "cju3y54kwj3nr0801biidlb4e",
      "cju3ya7goj6at0818v2l5ay7f",
      "cju3yht87j83m08507yk1u1fg",
      "cju40jl7skiuo0817p0smlgg8",
      "cju40m0rjkpw80871z6n6yg1u",
      "cju410dnfl0960755y8lu8d79",
      "cju42m60jlpcm08186kqppzqv",
      "cju43in5fm22c08175rxziqrk",
      "cju45rj7ln8980850a7821fov",
      "cju5bhv81abur0850ean02atv",
      "cju5bj926aiec07559rshy4wa",
      "cju5bwhapakm90987c1v4z46a",
      "cju5ddda9bkkt0850enzwatb1",
      "cju5ekty5ckzf07550c9u3ckk",
      "cju5enq1tcn1i0755hnkon787",
      "cju5es375cnzy0801nkq35ffs",
      "cju5f0dezct4q08183ydw11dx",
      "cju5fi0yxd3ei0801v7u0yudn",
      "cju5fw37edaae0801vkwvocn7",
      "cju5fydrud94708507vo6oy21",
      "cju5k3j3uf6de0817hszzfr7n",
      "cju5o4pk9h0720755lgp9jq8m",
      "cju5u4pywk81x0817vn9pe14z",
      "cju5u8gz4kj5b07552e2wpkwp",
      "cju5v8pgplg6k0755rvi2t63h",
      "cju5von04litr08718j8po40a",
      "cju5vutu7ll8w0871dfp92n9p",
      "cju5w7xn0lrkq0801f9k0htgx",
      "cju5wj0faly5008187n6530af",
      "cju5wkonqlrl409877y8zvnub",
      "cju5x7iskmad90818frchyfwd",
      "cju5xjn5mm78b09871spyqhhr",
      "cju5y84q3mdv50817eyp82xf3",
      "cju5yeqiwmkgl0801fzv2douc",
      "cju5ymyd8mmdc0801ry3by1xr",
      "cju6us80mv1b50871ebyq2wxa",
      "cju6ut4l8va6y0755tyw3vfqq",
      "cju6v1m1xv07w09870ah3njy1",
      "cju6x35ervu2808015c7eoqe4",
      "cju6xifswvwbo0987nibtdr50",
      "cju6xlygpw7bs0818n691jsq4",
      "cju6ywm40wdbo0987pbftsvtg",
      "cju77t0razbvm080106o56289",
      "cju77u1sjz77b0817ft44r3fk",
      "cju7afqon1ip40850ue2308b6",
      "cju7aklv31h4309871m29l4e7",
      "cju7amjna1ly40871ugiokehb",
      "cju7awzmu1ncs0871hziy65zx",
      "cju7b4mtw1n9n080186209f3d",
      "cju7b5afm1nfw0801xqm8bf8q",
      "cju7bd1qu1mx409877xjxibox",
      "cju7bf1lp1shi081835vs84lc",
      "cju7bmi1v1pnj0987pa52jjok",
      "cju7cufm7298k0755j09uf3of",
      "cju7d4jk723eu0817bqz2n39m",
      "cju7d6ux323ze0987xos3srkx",
      "cju7d9seq29zd0871nzl2uu5m",
      "cju7dff9529h208503w60lbil",
      "cju7do8c72dbo0801vxfzxdc4",
      "cju7eea9b2m0z0801ynqv1fqu",
      "cju7ev2b12owa08500bpfpwyw",
      "cju7f6cqy2ur20818t1saazbm",
      "cju7fazv92ywx0755xov2erga",
      "cju7fen322ou10817ziqkob4k",
      "cju83u9ftk3ni0987qnhlcinv",
      "cju83vvmik9wa08710yeh7cuk",
      "cju8402x1kcy70801t6kz6bdi",
      "cju84aoa3ktwn0755pfl4gfwd",
      "cju84ih17kp5l09876bkooocl",
      "cju84kplnl1y30755ropua1b0",
      "cju857ad0l88m0817qx4cwxnf",
      "cju858eswlepn0871pzvdrhj1",
      "cju85c2d4ln1b0755zz1z3onx",
      "cju85fc11ljr40818edpb0inh",
      "cju85hguellg50818kwu3s8d1",
      "cju87ox0kncom0801b98hqnd2",
      "cju88q6h6obpd0871ckmiabbo",
      "cju88rl5eo94l0850kf5wtrm1",
      "cju8a1jtvpt9m081712iwkca7",
      "cju8abobpqbir08189u01huru",
      "cju8aeei7q8k308173n9y4klv",
      "cju8amfdtqi4x09871tygrgqe",
      "cju8aqq8uqmoq0987hphto9gg",
      "cju8auylgqx0z0871u4o4db7o",
      "cju8bn7m2rmm70817hgxpb1uq",
      "cju8brv16rx7f0818uf5n89pv",
      "cju8c5zcbsdfz0801o5t6jag1",
      "cju8ca4geseia0850i2ru11hw",
      "cju8clorgsuwn08714toqb7v6",
      "cju8d4jgatgpj0871q2ophhkm",
      "cju8dk7eztzup08182yxko5zh",
      "cjyzkpsbjdsjq07211dfi4sru",
      "cjyzlw7f9faqr070129au64sq",
      "cjyzuio1qgh040763k56deohv"
    ]
  },
  "fold_3": {
    "train": [
      "cju0qkwl35piu0993l0dewei2",
      "cju0qoxqj9q6s0835b43399p4",
      "cju0qx73cjw570799j4n5cjze",
      "cju0roawvklrq0799vmjorwfv",
      "cju0rx1idathl0835detmsp84",
      "cju0s2a9ekvms080138tjjpxr",
      "cju0s690hkp960855tjuaqvv0",
      "cju0sr5ghl0nd08789uzf1raf",
      "cju0t4oil7vzk099370nun5h9",
      "cju0tl3uz8blh0993wxvn7ly3",
      "cju0u2g7pmnux0801vkk47ivj",
      "cju0u82z3cuma0835wlxrnrjv",
      "cju0ue769mxii08019zqgdbxn",
      "cju13fwthn9mq0835gacxgy01",
      "cju13hp5rnbjx0835bf0jowgx",
      "cju14g8o4xui30878gkgbrvqj",
      "cju14pxbaoksp0835qzorx6g6",
      "cju15czxqp3lv0835jvhgzurz",
      "cju15jr8jz8sb0855ukmkswkz",
      "cju15l5ubz9yh0855b3ivdpse",
      "cju15mhjczc8z0801kit5c6di",
      "cju15ptjtppz40988odsm9azx",
      "cju15wdt3zla10801odjiw7sy",
      "cju16b6ynq8e40988m8vx0xnj",
      "cju16fpvhzypl0799p9phnlx6",
      "cju16whaj0e7n0855q7b6cjkm",
      "cju175facms5f0993a5tjikvt",
      "cju17bz250pgd0799u1hqkj5u",
      "cju17g6ykn1cs0993dww6qdi8",
      "cju17hw9hr9c5098800fu4u8e",
      "cju17otoe119u0799nqcbl8n1",
      "cju17r8il13910799dr2wme2e",
      "cju17v6ih0u7808783zcbg1jy",
      "cju17x0j4nfc10993y31pvlgs",
      "cju1819curo000988pd5xcqme",
      "cju183od81ff608017ekzif89",
      "cju1871y11d6r0799k6cw4yze",
      "cju18gzrq18zw0878wbf4ftw6",
      "cju18kevfrojc0835bn90f1in",
      "cju1amqw6p8pw0993d9gc5crl",
      "cju1aqqv02qwz0878a5cyhr67",
      "cju1b0y2e396p08558ois175d",
      "cju1bhnfitmge0835ynls0l6b",
      "cju1c0qb4tzi308355wtsnp0y",
      "cju1c4fcu40hl07992b8gj0c8",
      "cju1c6yfz42md08550zgoz3pw",
      "cju1cbokpuiw70988j4lq1fpi",
      "cju1cdxvz48hw0801i0fjwcnk",
      "cju1cfhyg48bb0799cl5pr2jh",
      "cju1cnnziug1l0835yh4ropyg",
      "cju1cqc7n4gpy0855jt246k68",
      "cju1csmlc4ht10799b8ymmghg",
      "cju1cu1u2474n0878tt7v4tdr",
      "cju1cvkfwqrec0993wbp1jlzm",
      "cju1cyjb5qtie0993njqne9m3",
      "cju1d31sp4d4k0878r3fr02ul",
      "cju1d96gsv62d09881b3wecw2",
      "cju1ddr6p4k5z08780uuuzit2",
      "cju1dfeupuzlw0835gnxip369",
      "cju1dg44i4z3w0801nyz4p6zf",
      "cju1dia8hvc6v098827mgffnm",
      "cju1djtprvd7b0988thwwrg09",
      "cju1dq3x1vgx109889c7wyirg",
      "cju1drnhbrb9409935wi7vkhg",
      "cju1efbr0rqxz09931z0lf4vf",
      "cju1egh885m1l0855ci1lt37c",
      "cju1egx9pvz2n0988eoy8jp23",
      "cju1ejj7dvqfa0835ra184v5m",
      "cju1erep75us208553i4ofwwe",
      "cju1euant5l960878iqj5vvto",
      "cju1euuc65wm00799m4sjdnnn",
      "cju1f15k3w4ct0835cmde6ypo",
      "cju1f5x1164xv08555654c24r",
      "cju1fb9236a110801yvg0fwju",
      "cju1fj6axwfp30835ukhuzhw9",
      "cju1fjsb4sipq09931lvd8e41",
      "cju1fmsyf6gxb0801cimx2gle",
      "cju1fr4etsmrr09933u4t4aql",
      "cju1ftaji6isw0855108yqcse",
      "cju1fuoa4wmc50835qfd11sp9",
      "cju1g20bdwq6u0835e16xugcd",
      "cju1g4nsb6ngy0799l4ezm8ab",
      "cju1gghyjwxt80835vx0wgxw0",
      "cju1gi7jlwyld0835cdf6g6qz",
      "cju1gv7106qd008784gk603mg",
      "cju1h5w4wxajx0835mc954kxy",
      "cju1haab178i70799tk9z8y8x",
      "cju1hmff8tkp809931jps6fbr",
      "cju1hp9i2xu8e0988u2dazk7m",
      "cju1hs0za7jha0855vj0mdrjt",
      "cju1hyolc7aqu0878rrkfn1lr",
      "cju2hdr06v2bq0799mbm3bks1",
      "cju2hewssldzx0835ep795xu0",
      "cju2hgsptlfam0835o3b59h1o",
      "cju2hjrqcvi2j0801bx1i6gxg",
      "cju2hlm19vjjf0801o69qnber",
      "cju2hos57llxm08359g92p6jj",
      "cju2hqt33lmra0988fr5ijv8j",
      "cju2hugv9vget0799hhk7ksvg",
      "cju2hx006vidl0799igm81vmh",
      "cju2i03ptvkiu0799xbbd4det",
      "cju2i3hzclw3o0988rrgh911i",
      "cju2i6acqvo6l0799u20fift8",
      "cju2iatlki5u309930zmgkv6h",
      "cju2lcyfgkf5809932fn9gucn",
      "cju2lejzcy4pc0878c9rlonot",
      "cju2lyynuymli0855g7fxgbhe",
      "cju2lz8vqktne0993fuym6drw",
      "cju2m56cryvqd0801gtn2yp8t",
      "cju2mfjndoz700988b9lc3zeq",
      "cju2mh8t6p07008350e01tx2a",
      "cju2nd7l7z98o0799gfjvyfmw",
      "cju2nguelpmlj0835rojdn097",
      "cju2nnqrqzp580855z8mhzgd6",
      "cju2np2k9zi3v079992ypxqkn",
      "cju2nqapmzvk20801f9us40dx",
      "cju2nsmwjlzyl0993jl80chvz",
      "cju2ntxtdzlvu0799xl3j9pan",
      "cju2okvco06xc0799kxe5n1qh",
      "cju2omjpeqj5a0988pjdlb8l1",
      "cju2oq5570avm079959o20op1",
      "cju2osuru0ki00855txo0n3uu",
      "cju2ouil2mssu0993hvxsed6d",
      "cju2p0eveqtdc0835gpi3p93i",
      "cju2p4ddkmzxj0993p94o62av",
      "cju2p91qir00k08350ddfif0w",
      "cju2pag1f0s4r0878h52uq83s",
      "cju2phaksnahz0993yxogjcpv",
      "cju2pkwt3r8b90988v2ywq1px",
      "cju2pmhtr17a00855cvpelzb0",
      "cju2qfie4rvz508357kad9z5o",
      "cju2qh5le1ock0878oahaql7d",
      "cju2qqn5ys4uo0988ewrt2ip2",
      "cju2qu37qobl50993aw7ghcfq",
      "cju2qvuj1s9ok0835tp2k4ozh",
      "cju2r11x7sdgx0988o8ule0wl",
      "cju2r2obh2bjm08553kng0rh7",
      "cju2r7h21sj9608354gzks3ae",
      "cju2raxlosl630988jdbfy9b0",
      "cju2ricdv2iys0878sv1adh0u",
      "cju2rkjfwoxys0993x768l1j2",
      "cju2rlqdnoz9k0993cpjae3x0",
      "cju2rmd2rsw9g09888hh1efu0",
      "cju2rnkt22xep0801as160g9t",
      "cju2rqo702wpx0855fn7d5cxh",
      "cju2rxm8rpbaf0993o3qr2oph",
      "cju2rz4k434s70855wwx3ddtx",
      "cju2rzpsmtb0f0835jabkbao1",
      "cju2s16zp317h0799gr67jqc2",
      "cju2saez63gxl08559ucjq3kt",
      "cju2sggy13na70855tbeoqgha",
      "cju2srvy5440s0801y1ba9akr",
      "cju2suk42469908015ngmq6f2",
      "cju2sxf3iqbpv09937iksn8ep",
      "cju2syxa93yw40799x2iuwabz",
      "cju2t2ivz43i10878zeg8r1br",
      "cju2t3ibkuecr0835o7si16zv",
      "cju2t62nq45jl0799odpufwx6",
      "cju2t9tdwuk700835kv0ljmtl",
      "cju2ti1du4idn0878giuozonw",
      "cju2tjrog4jy30878pawyazqc",
      "cju2tpfa5uyx408359datxqqj",
      "cju2tqfgw4oat0799rn0g5b2z",
      "cju2trbpkv0c00988hxla5dzz",
      "cju2tvrvm53ws0801a0jfjdxg",
      "cju2tzypl4wss0799ow05oxb9",
      "cju2u4pymvc720988wsxrmi84",
      "cju2u73dj53oz0878486k8k4b",
      "cju2ulk385h170799rlklxob0",
      "cju2upu4evw7g08358guwozxv",
      "cju2uwz9f5yf1085506cfamfx",
      "cju2uy8ox62jo0801g88hh42z",
      "cju2uzabhs6er0993x3aaf87p",
      "cju2wtwj87kys0855kx6mddzw",
      "cju2wx0gh7fpz0878wwyd9ep8",
      "cju2wxv0hxs2f09884w48v8fi",
      "cju2wzu8wxtgu09880ku9x1pg",
      "cju2x7vw87mu30878hye2ca0m",
      "cju2xa3i4y0160988i679zsqd",
      "cju2xbk0080y80801eghyddi2",
      "cju2xd75m82720801q4s4ik3n",
      "cju2xf8e5y2wm08359vcgk09b",
      "cju2xjz2ju8pe0993ysv9wg17",
      "cju2xlcqxy9c60988vjacdznb",
      "cju2xs6na81t20878pt6nkfip",
      "cju2xwm1s84l50799i60mq0pu",
      "cju2xyd9vyi7m098831qcucse",
      "cju2y26c588bo07993ksd8eoz",
      "cju2y5zas8m7f0801d34g5owq",
      "cju2y8s56ymqr083541ggdsml",
      "cju2yb31a8e8u0878wdashg7o",
      "cju2ycp1u8g2r0799jslnp7cz",
      "cju2yg5ht8i4p087800js8hp4",
      "cju2yi9tz8vky0801yqip0xyl",
      "cju2ysg748ru80878sp6j0gm0",
      "cju2yw4s7z7p20988lmf2gdgd",
      "cju2yyhsp933j0855hp32e012",
      "cju2z2x3nvd3c099350zgty7w",
      "cju2z45kuzf6d0988nz2c819m",
      "cju2z6ez69g4u0801qwt088lw",
      "cju2z9vlp9j0w0801oag91sy9",
      "cju2zdhsczmn50988z64qwg2q",
      "cju2zi4l09f5807991s8do2b2",
      "cju2zjcvj9qma0801dk71hhi0",
      "cju2zm0axztpe0988r8s9twjr",
      "cju2zr3c3vwb00993jn06bbaz",
      "cju2zwg05a0oy0801yr73ig7g",
      "cju2zy1e49pqk0878t6ncqn12",
      "cju300m3s04fg0988uzupuf7z",
      "cju302fqq9spc0878rrygyzzz",
      "cju30525w04r10835ygp257sb",
      "cju30ajhw09sx0988qyahx9s8",
      "cju30bmab08bi0835mvlr6e0r",
      "cju30df2j09dd08351ayx2t6w",
      "cju30ftgja7170855xl9bkdm0",
      "cju30gxjq0djk0988jytm49rs",
      "cju30ia8da2bq0799klnehml2",
      "cju30j1rgadut0801vuyrsnt8",
      "cju30k2z40ds308353kdew70n",
      "cju30lncba3ny0878jwnous8n",
      "cju30mm25a53s0799qa5wiqe8",
      "cju30nyxe0gfb0835p256yoju",
      "cju30ov1oah920801mi8thuyg",
      "cju30qbm1ad3x0855znuhpz9u",
      "cju30u1hbakn808019g15nb8b",
      "cju30xqmh0ni00835ix3batv1",
      "cju30ywtc0oar0835bp2en7ec",
      "cju310f6val1v0855xo8tc3gu",
      "cju3128yi0rpu0988o4oo5n8n",
      "cju31rb7vb6110801p9rhacuw",
      "cju31t8xd17bk0835rnb893jk",
      "cju31y80qbawn0801twwm2l5s",
      "cju320gyvbch60801v2amdi2g",
      "cju323ypb1fbb0988gx5rzudb",
      "cju324q101fhe08350wae9cif",
      "cju326h4v1gxw08352px40p7r",
      "cju3280wv1ir009882jze27tc",
      "cju32csyfblyh080170aa3x5p",
      "cju32fhnhbds40799broyoptc",
      "cju32gzs6xo8x0993r8tedbpb",
      "cju32jcdabepz0878d0cznmfe",
      "cju32pzh9bpw10855q4vaxfhe",
      "cju32qr9tbvsj08013pkpjenq",
      "cju32zhbnc1oy0801iyv1ix6p",
      "cju330ofbc2l30801th5g3hw6",
      "cju33231uy4gi0993qc7b1jch",
      "cju336l68y7if0993wf092166",
      "cju3381d8bz3h07991xtl7ra0",
      "cju33qpdvc9g0087825jhf3s9",
      "cju33w4sdcivk0855x879zht7",
      "cju33x0f22peh0988g0ln7w5v",
      "cju34aozyyy830993bn16u32n",
      "cju34ds2531520988qjpqt6e3",
      "cju34fojcctcf0799ebolbvkn",
      "cju34i3qvcyog0855qiejxx5w",
      "cju34m7h536wq0988xz7gx79v",
      "cju34sh43d8zm08019xbwhc0o",
      "cju34uhepd3dd0799hs8782ad",
      "cju34ymm8d6700799uop0cw33",
      "cju34zivp3fq80988opxbaqyn",
      "cju3521y5d5mq0878t3ezsu4p",
      "cju353d1eda8c07992afde611",
      "cju357rxxdaz30878y2esjpjt",
      "cju35atpxdjot0855q46aqrd0",
      "cju35c4wzdhow0799h6eq4sgs",
      "cju35eg0tdmjt085525sb4bua",
      "cju35fxqyzt5p0993vusm54qz",
      "cju35ldepdtlm0801yv79y8vu",
      "cju35mdz73x890835eynq1h9v",
      "cju35oyvd3y850988km12hdz1",
      "cju3tp94kfstl08181awh6z49",
      "cju3tsh4lfsok0987w6x3a0v1",
      "cju3ttznuftyf09875t11850w",
      "cju3u39fog1bo0871lxjrabks",
      "cju3ua8u0g9rg0801uayhdxhu",
      "cju3uhb79gcgr0871orbrbi3x",
      "cju3ul8dogf1z09872y2ecowp",
      "cju3umoh1geet0817cmpef5am",
      "cju3v11mrgwwb0755u242ygye",
      "cju3v56bwgy8v0871w14pz8fx",
      "cju3v664kh0px0818y4y7wolf",
      "cju3wstckialg0871xs0vevsj",
      "cju3x2s11ibzi0817kk284k0j",
      "cju3x4blzieu30850x10uuvbm",
      "cju3x5u2tiihx0818914gzxy1",
      "cju3x9lttikfb0818a0g104zn",
      "cju3xeexgii1j0817zs68tb4g",
      "cju3xga12iixg0817dijbvjxw",
      "cju3xiic0ilzp0850lusrb42j",
      "cju3xjqtpikx50817tppy6g84",
      "cju3xl264ingx0850rcf0rshj",
      "cju3xwpgviwlx0871rwm15q7v",
      "cju3xzvnzj0hd0755xprz39nj",
      "cju3y0pjrj1c30755nxekxccj",
      "cju3y21quj0ir0818kgjagr15",
      "cju3y54kwj3nr0801biidlb4e",
      "cju3y79ofj3va0871uqfb1mzo",
      "cju3y9difj6th0801kd1rqm3w",
      "cju3ya7goj6at0818v2l5ay7f",
      "cju3yht87j83m08507yk1u1fg",
      "cju3ykamdj9u208503pygyuc8",
      "cju40jl7skiuo0817p0smlgg8",
      "cju40m0rjkpw80871z6n6yg1u",
      "cju40r6jrksyk0871wg98zgho",
      "cju40taxlkrho0987smigg0x0",
      "cju40u30gkuzc0871rq7t666d",
      "cju40wto5kxwi0755it190f2k",
      "cju410dnfl0960755y8lu8d79",
      "cju414lf2l1lt0801rl3hjllj",
      "cju418rckl3ur08012psrx1r1",
      "cju41kd7yl4nm0850gil5qqwh",
      "cju41nz76lcxu0755cya2qefx",
      "cju41p90plcsx08018cnzpndc",
      "cju41s6nbleqy0755e2mslg0b",
      "cju41z76wlgbz0801qdetlvby",
      "cju422cm8lfxn0818ojicxejb",
      "cju424hy5lckr085073fva1ok",
      "cju426tomlhll0818fc0i7nvh",
      "cju428k5fldt108177s6g6f45",
      "cju42dwedlmk60871jbgu4ehi",
      "cju42g865lorv07552ytz6xxa",
      "cju42m60jlpcm08186kqppzqv",
      "cju42nm68lpyo0818xvvqmupq",
      "cju42py9mlqyd0818u3d1d7ga",
      "cju42tauqlo5p08171l3cuo4b",
      "cju42u5bjlvi10801dc13sskp",
      "cju42wamblrqn098798r2yyok",
      "cju42xpi8lw4w0871ve317a1p",
      "cju430pm2lz0y0755jkhcc3d1",
      "cju439oazm2fu0871ma0vvrft",
      "cju43b8daly4408170e5ev06g",
      "cju43c92lm5cj0755lorsorfg",
      "cju43eigtm6ev0801mv0m96t1",
      "cju43gfosm63n08714rpih8pe",
      "cju43in5fm22c08175rxziqrk",
      "cju43jcqim2cp08172dvjvyui",
      "cju43lcnum9y10755bjs7z87f",
      "cju45jpvfn6c809873pv1i34s",
      "cju45n0oxn5vu08500yfrt9jn",
      "cju45pm27n80u08174kyow1gj",
      "cju45rj7ln8980850a7821fov",
      "cju5b9oyda4yr0850g9viziyv",
      "cju5bf6hxa6m50817rbwettgu",
      "cju5bhv81abur0850ean02atv",
      "cju5bj926aiec07559rshy4wa",
      "cju5bmhdcafs909878qfzrqzi",
      "cju5bwhapakm90987c1v4z46a",
      "cju5bycdkalkb09875f7bfrvx",
      "cju5c7oijaqmq09878qwgqv8n",
      "cju5ca9hcatkc0801jzwe7tfx",
      "cju5ccpvqash50850kb4bs22k",
      "cju5cetivauok0987ok3e5bre",
      "cju5chrxxawka0871qcj171yz",
      "cju5clr68b48r0755cmuvponm",
      "cju5cu8qkb84x08186jwo8yin",
      "cju5ddda9bkkt0850enzwatb1",
      "cju5ekty5ckzf07550c9u3ckk",
      "cju5enq1tcn1i0755hnkon787",
      "cju5es375cnzy0801nkq35ffs",
      "cju5eyfe9cpk90987laa7tsl3",
      "cju5f0dezct4q08183ydw11dx",
      "cju5f26ebcuai0818xlwh6116",
      "cju5fi0yxd3ei0801v7u0yudn",
      "cju5fs6j6d8350801vglraq4u",
      "cju5fw37edaae0801vkwvocn7",
      "cju5fydrud94708507vo6oy21",
      "cju5gucasds9d0801019axylx",
      "cju5hl8nee8a40755fm8qjj0o",
      "cju5hqz50e7o90850e0prlpa0",
      "cju5ht88gedbu0755xrcuddcx",
      "cju5huurrecm70801y680y13m",
      "cju5hyi9yegob0755ho3do8en",
      "cju5k3j3uf6de0817hszzfr7n",
      "cju5k7r0yf98c09878csbxb4d",
      "cju5kre09fhka0850h7b1898j",
      "cju5nxkujgscq0817l9gss626",
      "cju5o1vu9gz8a0818eyy92bns",
      "cju5o4pk9h0720755lgp9jq8m",
      "cju5tenjojp1j0755ms4949h2",
      "cju5thdbrjp1108715xdfx356",
      "cju5u4pywk81x0817vn9pe14z",
      "cju5u6wf0kh1t0755bg1ssixv",
      "cju5u8gz4kj5b07552e2wpkwp",
      "cju5ufn3skquf0818dhapnhba",
      "cju5uget8krjy0818kvywd0zu",
      "cju5uzmaol56l0817flxh4w9p",
      "cju5v8pgplg6k0755rvi2t63h",
      "cju5vgawslbe30987ndeepc1b",
      "cju5vi4nxlc530817uoqm2m7a",
      "cju5von04litr08718j8po40a",
      "cju5vutu7ll8w0871dfp92n9p",
      "cju5vwbr4lhqn0987a1pji0ux",
      "cju5vzjoslpj708186z2fusmz",
      "cju5w7xn0lrkq0801f9k0htgx",
      "cju5waeduln160817w0agirve",
      "cju5wj0faly5008187n6530af",
      "cju5wkonqlrl409877y8zvnub",
      "cju5woy82m07m08505dmjg7g1",
      "cju5wqonpm0e60801z88ewmy1",
      "cju5wrapcm2290818jsh26ppb",
      "cju5wtdu4m0im0871mix0yvc0",
      "cju5wuhm1lwm40987vugqn3vv",
      "cju5x7iskmad90818frchyfwd",
      "cju5xjn5mm78b09871spyqhhr",
      "cju5xkwzxmf0z0818gk4xabdm",
      "cju5xopi0md7q0871r1sjc1av",
      "cju5y4hgqmk0i08180rjhbwvp",
      "cju5y7buemcw80987p0r30g9f",
      "cju5y84q3mdv50817eyp82xf3",
      "cju5yclrymlgj0818k426ud6z",
      "cju5yeqiwmkgl0801fzv2douc",
      "cju5ygh1zmmdi0755uod5e17i",
      "cju5yhgznmkzb0801cji2vi8j",
      "cju5ymyd8mmdc0801ry3by1xr",
      "cju6us80mv1b50871ebyq2wxa",
      "cju6ut4l8va6y0755tyw3vfqq",
      "cju6uy20suzbl0987rzuhz7z9",
      "cju6v1m1xv07w09870ah3njy1",
      "cju6v3bb2v7xo085090blubyw",
      "cju6v4szov55u0871qmqz3v8n",
      "cju6v5ilsv8hk0850rb5sgh6o",
      "cju6v6g6kvdw007552x6mb0po",
      "cju6vifjlv55z0987un6y4zdo",
      "cju6vqarjv7yo0987q4b1btk1",
      "cju6vrs1ov8cr098788h8gs6j",
      "cju6vvb8svhed0801jjcquh5e",
      "cju6wjm81vgsc0987enk9n3pr",
      "cju6wuojavt740818b5qcv3iw",
      "cju6x0yqbvxqt0755dhxislgb",
      "cju6x35ervu2808015c7eoqe4",
      "cju6xa0qmvzun0818xjukgncj",
      "cju6xifswvwbo0987nibtdr50",
      "cju6xlygpw7bs0818n691jsq4",
      "cju6xmqd9w0250817l5kxfnsk",
      "cju6ywm40wdbo0987pbftsvtg",
      "cju6yxyt0wh080871sqpepu47",
      "cju6z2616wqbk07555bvnuyr1",
      "cju6z9a9kwsl007552s49rx6i",
      "cju76lsehyia10987u54vn8rb",
      "cju76o55nymqd0871h31sph9w",
      "cju772304yw5t0818vbw8kkjf",
      "cju773hsyyosz0817pk1e7sjq",
      "cju7787c5yy3l080159mwqsnj",
      "cju77afzlz3kp07550x5nafzs",
      "cju77b3wyz4160755qis4ljsb",
      "cju77bvg0yv4r0987yh60xmjo",
      "cju77g99iyxc00817zqi2ppor",
      "cju77j66ez52p08019xygi0co",
      "cju77q10sz9ug0801449wu1nu",
      "cju77re6fz5bb0817vp9redjg",
      "cju77t0razbvm080106o56289",
      "cju77u1sjz77b0817ft44r3fk",
      "cju77vvcwzcm50850lzoykuva",
      "cju7ae7bq1f820987toc8si1d",
      "cju7aez2x1jtj0871ztezs3oi",
      "cju7afqon1ip40850ue2308b6",
      "cju7ahtkb1jr90801jck4kbds",
      "cju7aifyo1p3n07552nxjx51f",
      "cju7aklv31h4309871m29l4e7",
      "cju7alcgr1lsr0871riqk84z7",
      "cju7amjna1ly40871ugiokehb",
      "cju7ap09p1kz10850ldccjebj",
      "cju7aqkue1i2k09879uzcpt8r",
      "cju7arvfe1ldu0850erdmphgj",
      "cju7atnm31if40817pqclnjer",
      "cju7avvi51iox0817ym55y6tt",
      "cju7awzmu1ncs0871hziy65zx",
      "cju7b10ce1mnm08011c5bwyr4",
      "cju7b1ygu1msd0801hywhy0mc",
      "cju7b2l561oas0871decgslaf",
      "cju7b3f5h1sm40755i572jden",
      "cju7b4mtw1n9n080186209f3d",
      "cju7b5afm1nfw0801xqm8bf8q",
      "cju7bc95p1mdm0817yqj5jc6j",
      "cju7bd1qu1mx409877xjxibox",
      "cju7bduyq1rjf08719giru9ho",
      "cju7bf1lp1shi081835vs84lc",
      "cju7bfx651qr80801cs7epotb",
      "cju7bmi1v1pnj0987pa52jjok",
      "cju7cl8zm1xcu0817ado0jpas",
      "cju7cp6dw244p0818gncdol4m",
      "cju7cq6su27qv075574dir0r3",
      "cju7crgxa28550755wbsgqkel",
      "cju7csvlb22fr0850lvm45n3x",
      "cju7ctvqn25dy08186g442m1r",
      "cju7cue9b232j0801qdzk1ykj",
      "cju7cufm7298k0755j09uf3of",
      "cju7d2q1k27nf08715zshsckt",
      "cju7d3oc82cho0755dajlwldz",
      "cju7d4jk723eu0817bqz2n39m",
      "cju7d6ux323ze0987xos3srkx",
      "cju7d7tly27h408016fyp5nr7",
      "cju7d8m3b2e210755l8fj1yph",
      "cju7d9seq29zd0871nzl2uu5m",
      "cju7da88w2eod0755wejzynvt",
      "cju7db7lp2f400755tntd1ohf",
      "cju7ddtz729960801uazp1knc",
      "cju7deifq2fzn0755lc8idyh8",
      "cju7dff9529h208503w60lbil",
      "cju7dglf226g50987ohbthl19",
      "cju7dizi82h2i0755doucgnt3",
      "cju7dlk532dsh0871zvr6qz0r",
      "cju7dmlgf2ebw0871ieqas5fh",
      "cju7do8c72dbo0801vxfzxdc4",
      "cju7dqcwi2dz00850gcmr2ert",
      "cju7dsrtb2f8i085064kwugfk",
      "cju7dxffn2eam0817qxosfwch",
      "cju7dymur2od30755eg8yv2ht",
      "cju7ea4om2l910801bohqjccy",
      "cju7ebe962hr409872ovibahw",
      "cju7ecl9i2i060987xawjp4l0",
      "cju7eea9b2m0z0801ynqv1fqu",
      "cju7efffp2ivf0817etg3jehl",
      "cju7ejm2l2ncl0801wq6y84nw",
      "cju7el6xv2k520817qxx9wdr5",
      "cju7emdni2py40871ivhxjtut",
      "cju7eotqi2qea0871y8yc7tqh",
      "cju7epwj82koz098713apjnzo",
      "cju7er4kc2opa0801anuxc0eb",
      "cju7es23b2vcp0755gpbm9s7v",
      "cju7et17a2vjk0755e743npl1",
      "cju7etr3y2p4t0801cdzjj8ab",
      "cju7eueum2oqn0850rodmx8zo",
      "cju7ev2b12owa08500bpfpwyw",
      "cju7evxt12m730987rxivne3x",
      "cju7ey10f2rvf0871bwbi9x82",
      "cju7ez7r22qbc08015xfoz2wb",
      "cju7ezs7g2mxm098787atbran",
      "cju7f4sc62xqj075597xpmuoy",
      "cju7f5ghb2r5s0801chwkxxh9",
      "cju7f6cqy2ur20818t1saazbm",
      "cju7fazv92ywx0755xov2erga",
      "cju7fbndk2sl608015ravktum",
      "cju7fd6yt2p740987wkr8exo1",
      "cju7fen322ou10817ziqkob4k",
      "cju7ff97z2ow40817u2r83my5",
      "cju7fmvpk2q170987v6i3ola8",
      "cju7fob3x301u0755x985pmmq",
      "cju7fpfzq2wyf0818xxd1oziv",
      "cju7frtqu2xa20818wq8r9fzf",
      "cju83mki1jv5w0817kubxm31r",
      "cju83nwu1jxte0987h1krpfmv",
      "cju83rcnzkbsj0755x5anfrcg",
      "cju83u9ftk3ni0987qnhlcinv",
      "cju83vvmik9wa08710yeh7cuk",
      "cju83yddek68q0850d2x7zfkm",
      "cju8402x1kcy70801t6kz6bdi",
      "cju8418jhkf7d0818ga2v0xq0",
      "cju843yjskhq30818qre4rwm2",
      "cju847pxykriq0755268ktrk2",
      "cju849c23kgnk0817cgv2hw1e",
      "cju84aoa3ktwn0755pfl4gfwd",
      "cju84dsvaklpx098750hp83x4",
      "cju84ffdzkrjn08183jh1fxmb",
      "cju84gpefknwm098714oq8q61",
      "cju84ih17kp5l09876bkooocl",
      "cju84jdl9kv0i0871eog9b3i9",
      "cju84kplnl1y30755ropua1b0",
      "cju857ad0l88m0817qx4cwxnf",
      "cju858eswlepn0871pzvdrhj1",
      "cju85a8h8llwm07559wxg4t5w",
      "cju85bf1algsq0871y9gtlq97",
      "cju85c2d4ln1b0755zz1z3onx",
      "cju85dx63lic408017f0l0400",
      "cju85fc11ljr40818edpb0inh",
      "cju85hguellg50818kwu3s8d1",
      "cju85ia1slh220987y7c20sm2",
      "cju85mpuglq8k0818d2it6hzb",
      "cju85nr8elly209872w9n5m0s",
      "cju85omszllp30850b6rm9mi3",
      "cju85qefyln6v0850szeb9byi",
      "cju87li0zn3yb0817kbwgjiz8",
      "cju87ox0kncom0801b98hqnd2",
      "cju87q6yoneim0871dl4phvkd",
      "cju87r56lnkyp0755hz30leew",
      "cju87tyddnnad0755bj0wxahe",
      "cju87vqa0ndwg0850onjdz7ol",
      "cju87xn2snfmv0987sc3d9xnq",
      "cju87z6o6nh73085045bzsx6o",
      "cju87zv8lni0o0850hbbecbq6",
      "cju8828oxnool0801qno9luhr",
      "cju884985nlmx0817vzpax3y4",
      "cju885ikhnmkn09878s2lqtuh",
      "cju886ryxnsl50801r93jai7q",
      "cju887ftknop008177nnjt46y",
      "cju888fr7nveu0818r9uwtiit",
      "cju88fpm4o0tl0871w1i6a4ds",
      "cju88gx09o2vk0818610zody3",
      "cju88itqbny720987hxizbj5y",
      "cju88l66no10s0850rsda7ej1",
      "cju88msmoo3470817m441j4sg",
      "cju88oh0po9gq0801nge4tgr1",
      "cju88q6h6obpd0871ckmiabbo",
      "cju88rl5eo94l0850kf5wtrm1",
      "cju88t4fvokxf07558ymyh281",
      "cju88trl3ogi208716qvti51b",
      "cju88v2f9oi8w0871hx9auh01",
      "cju88z8bson4h0871nnd7fdxo",
      "cju890guyoiti098753yg6cdu",
      "cju8914beokbf0850isxpocrk",
      "cju892fesoq2g0801n0e0jyia",
      "cju89y9h0puti0818i5yw29e6",
      "cju89z6pqpqfx0817mfv8ixjc",
      "cju8a1jtvpt9m081712iwkca7",
      "cju8a56vxpy780850r45yu4wk",
      "cju8a84g0q76m0818hwiggkod",
      "cju8abobpqbir08189u01huru",
      "cju8adb60qbiu080188mxpf8d",
      "cju8aeei7q8k308173n9y4klv",
      "cju8aj01yqeqm0850lhdz3xdw",
      "cju8amfdtqi4x09871tygrgqe",
      "cju8ando2qqdo0818ck7i1be1",
      "cju8aqq8uqmoq0987hphto9gg",
      "cju8arof2qpf20850ifr1bnqj",
      "cju8ashhnquqr0801rwduzt7d",
      "cju8at3s1qqqx0850hcq8nmnq",
      "cju8auylgqx0z0871u4o4db7o",
      "cju8aw9n1qyg10801jkjlmors",
      "cju8ayeq7r1fb0818z1junacy",
      "cju8b0jr0r2oi0801jiquetd5",
      "cju8b2rmgr52s0801p54eyflx",
      "cju8b542nr81x0871uxnkm9ih",
      "cju8b5p40r2c60987ofa0mu03",
      "cju8b6rp0r5st0850184f79xt",
      "cju8b7aqtr4a00987coba14b7",
      "cju8b8yair65w09878pyqtr96",
      "cju8bafgqrf4x0818twisk3ea",
      "cju8bbznkrf5g0871jncffynk",
      "cju8bh8surexp0987o5pzklk1",
      "cju8bi8q7rlmn0871abc5ch8k",
      "cju8bj2ssrmlm0871gc2ug2rs",
      "cju8bk8oirjhw0817hgkua2w8",
      "cju8bm24yrrdp081829mbo8ic",
      "cju8bn7m2rmm70817hgxpb1uq",
      "cju8bop5jrsid08716i24fqda",
      "cju8bpctzrqkr0850zeldv9kt",
      "cju8bqxxurs6i0850mu7mtef9",
      "cju8brv16rx7f0818uf5n89pv",
      "cju8bssulrrcy0987h1vq5060",
      "cju8bw697rwg308177tg8huas",
      "cju8c1a0ws7o208181c6lbsom",
      "cju8c2rqzs5t80850d0zky5dy",
      "cju8c3xs7sauj0801ieyzezr5",
      "cju8c5mxls96t0850wvkvsity",
      "cju8c5zcbsdfz0801o5t6jag1",
      "cju8c6hnxsdvr0801wn0vrsa6",
      "cju8c82iosagu0817l74s4m5g",
      "cju8c9akjsdjj0850s67uzlxq",
      "cju8ca4geseia0850i2ru11hw",
      "cju8cattbsivm0818p446wgel",
      "cju8cbsyssiqj0871gr4jedjp",
      "cju8cdeazsm8h0801jxifmzur",
      "cju8ceacrsqkr0755hdz145es",
      "cju8cgi2kspp308011nxdtjp6",
      "cju8chdlqsu620755azjty1tj",
      "cju8chxndsre008015uisl4si",
      "cju8clorgsuwn08714toqb7v6",
      "cju8cwy02t9eq08185qn12c02",
      "cju8czvnztbf40871b4m7t78w",
      "cju8d4jgatgpj0871q2ophhkm",
      "cju8dk7eztzup08182yxko5zh",
      "cju8doa16u5gh0818w1ywda3q",
      "cju8dpa89u6l80818dj6lldh9",
      "cju8dqkrqu83i0818ev74qpxq",
      "cjyzjzssvd8pq0838f4nolj5l",
      "cjyzk8qieoboa0848ogj51wwm",
      "cjyzkpsbjdsjq07211dfi4sru",
      "cjyzl833ndne80838pzuq6ila",
      "cjyzlw7f9faqr070129au64sq",
      "cjyzu3reghjya0794w7pwoi50",
      "cjyzu9th0qt4r0a46pyl4zik0",
      "cjyzufihqquiw0a46jatrbwln",
      "cjyzuio1qgh040763k56deohv",
      "cjyzul1qggwwj07216mhiv5sy",
      "cjyzurzvohqnr0794es1itzek",
      "ck2395w2mb4vu07480otsu6tw",
      "ck2bxiswtxuw80838qkisqjwz",
      "ck2bxpfgxu2mk0748gsh7xelu",
      "ck2bxw18mmz1k0725litqq2mc",
      "ck2da7fwcjfis07218r1rvm95"
    ],
    "val": [
      "cju0sxqiclckk08551ycbwhno",
      "cju0vtox5ain6099360pu62rp",
      "cju160wshltz10993i1gmqxbe",
      "cju16ach3m1da0993r1dq3sn2",
      "cju16d65tzw9d0799ouslsw25",
      "cju16jgnyzp970878melv7r25",
      "cju171py4qiha0835u8sl59ds",
      "cju18849rrsgr0988p90hkygb",
      "cju18ibp219ub08783i6o98g7",
      "cju1alwgo30z60855fm3y23sm",
      "cju1ats0y372e08011yazcsxm",
      "cju1b3zgj3d8e0801kpolea6c",
      "cju1dhfok4mhe0878jlgrag0h",
      "cju1dnz61vfp40988e78bkjga",
      "cju1expq45zst0855rjqwwj4m",
      "cju1f320ewfyu0988ndz6blh5",
      "cju1f79yhsb5w0993txub59ol",
      "cju1f8w0t65en0799m9oacq0q",
      "cju1fm3id6gl50801r3fok20c",
      "cju1gkndf6yi10801o1qnje19",
      "cju1h89h6xbnx08352k2790o9",
      "cju1hirfi7ekp0855q0vgm9qq",
      "cju2hfqnmhisa0993gpleeldd",
      "cju2i8br1vqtd08784u6vmcjk",
      "cju2ij9uiic2l09933ljiv6gm",
      "cju2nfnvxzdkd0878399axlco",
      "cju2oo0wh0bqy0878biujeyhe",
      "cju2pjb9v0ywn0878j5g5n69j",
      "cju2qozsk20cq0855ugrg3cri",
      "cju2qxxko229x08786gvxxhur",
      "cju2qz06823a40878ojcz9ccx",
      "cju2r6mt2om21099352pny5gw",
      "cju2r91dg2k090801bh0xzbxk",
      "cju2rga4psq9n09881z519xx0",
      "cju2rn0hasxri0835nfy3buay",
      "cju2ro5jqsy680988pi6qsujw",
      "cju2s2527pfyr0993l3h1149a",
      "cju2s9g11pnra0993gn4eh793",
      "cju2sszfq3uye0878sucelzk2",
      "cju2t16vuucaq0835xcpsivn2",
      "cju2top2ruxxy0988p1svx36g",
      "cju2txjfzv60w098839dcimys",
      "cju2uokeg5jm20799xwgsyz89",
      "cju2urqpwvxw70835rvndvtsi",
      "cju2y0z6g87p10878fpk5d3rq",
      "cju2y40d8ulqo0993q0adtgtb",
      "cju2yo1j1v0qz09934o0e683p",
      "cju2z1nxlzaj40835wj81s1iy",
      "cju2zblxw9848087853csbrx1",
      "cju2zdvjn9h7r08553cp4eed5",
      "cju2zo0fwzv580988qlijd2xa",
      "cju306x7w05nb0835cunv799x",
      "cju32a52lb9rc0799xi40qs00",
      "cju32l161bi1v07990vm376in",
      "cju334jzo261t0835yqudnfs1",
      "cju33belnbyhm0878yxl42233",
      "cju33o12x2jm50988944mxq0v",
      "cju34eqjpcpm508788b3lhp97",
      "cju3518w2d838079939fqztbc",
      "cju35740hzm0g0993zl5ic246",
      "cju358pwtdby20878cg7nm0np",
      "cju3tvffffx5f0818t5ov22al",
      "cju3tx1qyg0c907552fglumhc",
      "cju3u1c8tfyqx08503iedc3mx",
      "cju3uz4o6gr9z0850lhxyxvsj",
      "cju3xtufwiv9c0818djsc4cqd",
      "cju3yb47cj1xq0817zfotbni4",
      "cju40poe4kt9s0755f9cnm3h5",
      "cju42qet0lsq90871e50xbnuv",
      "cju42romflni20817etb9a0fl",
      "cju43h43am1dy08176gwfhmnt",
      "cju43kj2pm34f0850l28ahpni",
      "cju43o6n7m9nk087191ijwqq9",
      "cju45lbgznahl08180xz1h7u6",
      "cju45ofd9ne1j0801ri8dup7t",
      "cju45t5ddnbio0987qtqzx762",
      "cju45v0pungu40871acnwtmu5",
      "cju5bbtwsa8cl0987wgfsqpao",
      "cju5boicjagt20871b1fotkh4",
      "cju5buy2bal250818ipl6fqwv",
      "cju5c5xc7algd0817pb1ej5yo",
      "cju5cjh3xattc0817j2vbulzi",
      "cju5eftctcdbj08712gdp989f",
      "cju5fb86jd1jp0755b1ukbhq5",
      "cju5fu081d8gc0818l3yylujk",
      "cju5g163vd6mt0817uccuga6u",
      "cju5h57xedz5h0755mjpc8694",
      "cju5hi52odyf90817prvcwg45",
      "cju5hwonqedw10801vsd3w6kk",
      "cju5i39mreass0817au8p22zy",
      "cju5jx7jzf7c90871c2i9aiov",
      "cju5jz5fff8c50871hbe6108f",
      "cju5k503sfa5f0871lx0rpu5y",
      "cju5knbbqfipk080128cggukq",
      "cju5nyu31gv8e0871zpk74a2n",
      "cju5udcufki0s09874ll1dbr5",
      "cju5uhrdwkmsu0817ervv91l8",
      "cju5uxjnol2r509871qv2yeia",
      "cju5vbo6jldrt0871jf6f1700",
      "cju5vcmrqla7i0817x4sp4pqw",
      "cju5vxuc5loxw0818u8xgf45p",
      "cju5wrrs0m2af0818vmnajbtw",
      "cju5x00l6m5j608503k78ptee",
      "cju5x15djm7ae0755h8czf6nt",
      "cju5yjq1pmlgc0801z0t24bly",
      "cju6uzxk0v83p0801rcwnexdu",
      "cju6vgdmivcvb08018fra5lnv",
      "cju6vucxvvlda0755j7msqnya",
      "cju6vvxsev9y30987kespucdg",
      "cju6w733bveoz0817e600tw72",
      "cju6wt9jvvn500871hjn3t3g0",
      "cju6x97w4vwua0850x0997r0a",
      "cju6z600qwh4z081700qimgl9",
      "cju76erapykj30871x5eaxh4q",
      "cju76l27oyrw907551ri2a7fl",
      "cju77idwfz36d0871tzfzz51i",
      "cju783tmkzkqu081803g7q5vk",
      "cju784jpdzeae0987q5ypq883",
      "cju785htizjzo08017tvlhtg4",
      "cju787jnjzjuj0871p94nck9g",
      "cju7agj961l2r0818z29iq8yn",
      "cju7apr0c1qqm0755s7msqot4",
      "cju7azuu31mia0801pf9ib5ed",
      "cju7b9vcs1luz0987ta60j1dy",
      "cju7bgnvb1sf808717qa799ir",
      "cju7d1tvt25bu08019dvw3uff",
      "cju7d5m0p23kn09871rk7pu3v",
      "cju7dda8w2br20818zhsuz8s7",
      "cju7dn24o296i09871qfxb8s2",
      "cju7dp3dw2k4n0755zhe003ad",
      "cju7dtb1e2j0t0818deq51ib3",
      "cju7dvl5m2n4t0755hlnnjjet",
      "cju7dwe282dc309876rco45ts",
      "cju7ehljc2or70871261br8ai",
      "cju7ekbo32pft0871fv7kzwb9",
      "cju7fq7mm2pw508176uk5ugtx",
      "cju7g7ba42z310987bqzbi2bq",
      "cju83h9ysjwe808716nt35oah",
      "cju83ipu3jwpx0801z5pvguf8",
      "cju83k8fyjsxr0817d6nxs6r4",
      "cju83kxitjv340987z09m0ezy",
      "cju83qd0yjyht0817ktkfl268",
      "cju83syhdk6gs0801rf1rekdl",
      "cju83wwn1k55e0850kw6i2d81",
      "cju8567gdlcbq0801dwwyo2jt",
      "cju85citjlnfm0755i4rk5tqj",
      "cju85je7vlht70817c9jcjwi4",
      "cju85rkbnlo1c08503uxcpax1",
      "cju87kbcen2av0987usezo8kn",
      "cju87nkyrnb970801q84m47yt",
      "cju88y1mwoln50871emyfny1g",
      "cju893jmdompz0817xn3g1w4h",
      "cju8apjewqrk00801k5d71gky",
      "cju8axq24r4an0755yhv9d4ly",
      "cju8azmhcr66e0755t61atz72",
      "cju8b1v3br45u087189kku66u",
      "cju8bff9nrfi10850fmfzbf8v",
      "cju8bljw9rqk20801kr54akrl",
      "cju8buos5rz9b08715lfr0f4f",
      "cju8bzzy2s66m08016z6mouqt",
      "cju8cj10qsrau0871o2dr6ai1",
      "cju8d2q30tfhs0801n7lx77xl",
      "cju8dic9mtppa0987swn23wbc",
      "cju8djdqztu6408506pzhlo18",
      "cju8dm2cau2km0818jsv9eeq2",
      "cjyzkmjy8evns070165gf9dmq",
      "cjyztzaqtrv430848l8xgcerw",
      "ck2bxlujamu330725szlc2jdu",
      "ck2bxqz3evvg20794iiyv5v2m",
      "ck2bxskgxxzfv08386xkqtqdy"
    ]
  },
  "fold_4": {
    "train": [
      "cju0qoxqj9q6s0835b43399p4",
      "cju0qx73cjw570799j4n5cjze",
      "cju0roawvklrq0799vmjorwfv",
      "cju0s2a9ekvms080138tjjpxr",
      "cju0s690hkp960855tjuaqvv0",
      "cju0sr5ghl0nd08789uzf1raf",
      "cju0sxqiclckk08551ycbwhno",
      "cju0t4oil7vzk099370nun5h9",
      "cju0tl3uz8blh0993wxvn7ly3",
      "cju0u2g7pmnux0801vkk47ivj",
      "cju0u82z3cuma0835wlxrnrjv",
      "cju0ue769mxii08019zqgdbxn",
      "cju0vtox5ain6099360pu62rp",
      "cju13fwthn9mq0835gacxgy01",
      "cju13hp5rnbjx0835bf0jowgx",
      "cju14g8o4xui30878gkgbrvqj",
      "cju14pxbaoksp0835qzorx6g6",
      "cju15czxqp3lv0835jvhgzurz",
      "cju15l5ubz9yh0855b3ivdpse",
      "cju15mhjczc8z0801kit5c6di",
      "cju15ptjtppz40988odsm9azx",
      "cju15wdt3zla10801odjiw7sy",
      "cju160wshltz10993i1gmqxbe",
      "cju16ach3m1da0993r1dq3sn2",
      "cju16b6ynq8e40988m8vx0xnj",
      "cju16d65tzw9d0799ouslsw25",
      "cju16fpvhzypl0799p9phnlx6",
      "cju16jgnyzp970878melv7r25",
      "cju16whaj0e7n0855q7b6cjkm",
      "cju171py4qiha0835u8sl59ds",
      "cju17bz250pgd0799u1hqkj5u",
      "cju17hw9hr9c5098800fu4u8e",
      "cju17otoe119u0799nqcbl8n1",
      "cju17r8il13910799dr2wme2e",
      "cju17v6ih0u7808783zcbg1jy",
      "cju183od81ff608017ekzif89",
      "cju18849rrsgr0988p90hkygb",
      "cju18gzrq18zw0878wbf4ftw6",
      "cju18ibp219ub08783i6o98g7",
      "cju18kevfrojc0835bn90f1in",
      "cju1alwgo30z60855fm3y23sm",
      "cju1ats0y372e08011yazcsxm",
      "cju1b3zgj3d8e0801kpolea6c",
      "cju1bhnfitmge0835ynls0l6b",
      "cju1c0qb4tzi308355wtsnp0y",
      "cju1cbokpuiw70988j4lq1fpi",
      "cju1cdxvz48hw0801i0fjwcnk",
      "cju1cfhyg48bb0799cl5pr2jh",
      "cju1cnnziug1l0835yh4ropyg",
      "cju1cqc7n4gpy0855jt246k68",
      "cju1csmlc4ht10799b8ymmghg",
      "cju1cu1u2474n0878tt7v4tdr",
      "cju1cvkfwqrec0993wbp1jlzm",
      "cju1d31sp4d4k0878r3fr02ul",
      "cju1d96gsv62d09881b3wecw2",
      "cju1ddr6p4k5z08780uuuzit2",
      "cju1dfeupuzlw0835gnxip369",
      "cju1dg44i4z3w0801nyz4p6zf",
      "cju1dhfok4mhe0878jlgrag0h",
      "cju1dia8hvc6v098827mgffnm",
      "cju1djtprvd7b0988thwwrg09",
      "cju1dnz61vfp40988e78bkjga",
      "cju1drnhbrb9409935wi7vkhg",
      "cju1efbr0rqxz09931z0lf4vf",
      "cju1egx9pvz2n0988eoy8jp23",
      "cju1erep75us208553i4ofwwe",
      "cju1euuc65wm00799m4sjdnnn",
      "cju1expq45zst0855rjqwwj4m",
      "cju1f15k3w4ct0835cmde6ypo",
      "cju1f320ewfyu0988ndz6blh5",
      "cju1f5x1164xv08555654c24r",
      "cju1f79yhsb5w0993txub59ol",
      "cju1f8w0t65en0799m9oacq0q",
      "cju1fb9236a110801yvg0fwju",
      "cju1fj6axwfp30835ukhuzhw9",
      "cju1fm3id6gl50801r3fok20c",
      "cju1fmsyf6gxb0801cimx2gle",
      "cju1fuoa4wmc50835qfd11sp9",
      "cju1g20bdwq6u0835e16xugcd",
      "cju1gi7jlwyld0835cdf6g6qz",
      "cju1gkndf6yi10801o1qnje19",
      "cju1gv7106qd008784gk603mg",
      "cju1h5w4wxajx0835mc954kxy",
      "cju1h89h6xbnx08352k2790o9",
      "cju1hirfi7ekp0855q0vgm9qq",
      "cju1hp9i2xu8e0988u2dazk7m",
      "cju1hs0za7jha0855vj0mdrjt",
      "cju1hyolc7aqu0878rrkfn1lr",
      "cju2hdr06v2bq0799mbm3bks1",
      "cju2hewssldzx0835ep795xu0",
      "cju2hfqnmhisa0993gpleeldd",
      "cju2hlm19vjjf0801o69qnber",
      "cju2hos57llxm08359g92p6jj",
      "cju2hqt33lmra0988fr5ijv8j",
      "cju2hugv9vget0799hhk7ksvg",
      "cju2hx006vidl0799igm81vmh",
      "cju2i03ptvkiu0799xbbd4det",
      "cju2i3hzclw3o0988rrgh911i",
      "cju2i6acqvo6l0799u20fift8",
      "cju2i8br1vqtd08784u6vmcjk",
      "cju2iatlki5u309930zmgkv6h",
      "cju2ij9uiic2l09933ljiv6gm",
      "cju2lcyfgkf5809932fn9gucn",
      "cju2lejzcy4pc0878c9rlonot",
      "cju2lyynuymli0855g7fxgbhe",
      "cju2lz8vqktne0993fuym6drw",
      "cju2mfjndoz700988b9lc3zeq",
      "cju2mh8t6p07008350e01tx2a",
      "cju2nd7l7z98o0799gfjvyfmw",
      "cju2nfnvxzdkd0878399axlco",
      "cju2nguelpmlj0835rojdn097",
      "cju2okvco06xc0799kxe5n1qh",
      "cju2omjpeqj5a0988pjdlb8l1",
      "cju2oo0wh0bqy0878biujeyhe",
      "cju2oq5570avm079959o20op1",
      "cju2ouil2mssu0993hvxsed6d",
      "cju2p0eveqtdc0835gpi3p93i",
      "cju2p4ddkmzxj0993p94o62av",
      "cju2p91qir00k08350ddfif0w",
      "cju2pjb9v0ywn0878j5g5n69j",
      "cju2pkwt3r8b90988v2ywq1px",
      "cju2pmhtr17a00855cvpelzb0",
      "cju2qfie4rvz508357kad9z5o",
      "cju2qh5le1ock0878oahaql7d",
      "cju2qozsk20cq0855ugrg3cri",
      "cju2qqn5ys4uo0988ewrt2ip2",
      "cju2qu37qobl50993aw7ghcfq",
      "cju2qvuj1s9ok0835tp2k4ozh",
      "cju2qxxko229x08786gvxxhur",
      "cju2qz06823a40878ojcz9ccx",
      "cju2r11x7sdgx0988o8ule0wl",
      "cju2r2obh2bjm08553kng0rh7",
      "cju2r6mt2om21099352pny5gw",
      "cju2r7h21sj9608354gzks3ae",
      "cju2r91dg2k090801bh0xzbxk",
      "cju2raxlosl630988jdbfy9b0",
      "cju2rga4psq9n09881z519xx0",
      "cju2ricdv2iys0878sv1adh0u",
      "cju2rkjfwoxys0993x768l1j2",
      "cju2rlqdnoz9k0993cpjae3x0",
      "cju2rmd2rsw9g09888hh1efu0",
      "cju2rn0hasxri0835nfy3buay",
      "cju2rnkt22xep0801as160g9t",
      "cju2ro5jqsy680988pi6qsujw",
      "cju2rqo702wpx0855fn7d5cxh",
      "cju2rxm8rpbaf0993o3qr2oph",
      "cju2rzpsmtb0f0835jabkbao1",
      "cju2s16zp317h0799gr67jqc2",
      "cju2s2527pfyr0993l3h1149a",
      "cju2s9g11pnra0993gn4eh793",
      "cju2saez63gxl08559ucjq3kt",
      "cju2sggy13na70855tbeoqgha",
      "cju2srvy5440s0801y1ba9akr",
      "cju2sszfq3uye0878sucelzk2",
      "cju2sxf3iqbpv09937iksn8ep",
      "cju2t16vuucaq0835xcpsivn2",
      "cju2t2ivz43i10878zeg8r1br",
      "cju2t3ibkuecr0835o7si16zv",
      "cju2t62nq45jl0799odpufwx6",
      "cju2t9tdwuk700835kv0ljmtl",
      "cju2ti1du4idn0878giuozonw",
      "cju2tjrog4jy30878pawyazqc",
      "cju2top2ruxxy0988p1svx36g",
      "cju2tpfa5uyx408359datxqqj",
      "cju2trbpkv0c00988hxla5dzz",
      "cju2tvrvm53ws0801a0jfjdxg",
      "cju2txjfzv60w098839dcimys",
      "cju2tzypl4wss0799ow05oxb9",
      "cju2u4pymvc720988wsxrmi84",
      "cju2u73dj53oz0878486k8k4b",
      "cju2ulk385h170799rlklxob0",
      "cju2uokeg5jm20799xwgsyz89",
      "cju2upu4evw7g08358guwozxv",
      "cju2urqpwvxw70835rvndvtsi",
      "cju2uy8ox62jo0801g88hh42z",
      "cju2wtwj87kys0855kx6mddzw",
      "cju2wx0gh7fpz0878wwyd9ep8",
      "cju2wxv0hxs2f09884w48v8fi",
      "cju2wzu8wxtgu09880ku9x1pg",
      "cju2x7vw87mu30878hye2ca0m",
      "cju2xa3i4y0160988i679zsqd",
      "cju2xbk0080y80801eghyddi2",
      "cju2xf8e5y2wm08359vcgk09b",
      "cju2xs6na81t20878pt6nkfip",
      "cju2xwm1s84l50799i60mq0pu",
      "cju2y0z6g87p10878fpk5d3rq",
      "cju2y26c588bo07993ksd8eoz",
      "cju2y40d8ulqo0993q0adtgtb",
      "cju2y5zas8m7f0801d34g5owq",
      "cju2y8s56ymqr083541ggdsml",
      "cju2yb31a8e8u0878wdashg7o",
      "cju2yg5ht8i4p087800js8hp4",
      "cju2yo1j1v0qz09934o0e683p",
      "cju2ysg748ru80878sp6j0gm0",
      "cju2yw4s7z7p20988lmf2gdgd",
      "cju2yyhsp933j0855hp32e012",
      "cju2z1nxlzaj40835wj81s1iy",
      "cju2z2x3nvd3c099350zgty7w",
      "cju2z45kuzf6d0988nz2c819m",
      "cju2z9vlp9j0w0801oag91sy9",
      "cju2zblxw9848087853csbrx1",
      "cju2zdvjn9h7r08553cp4eed5",
      "cju2zi4l09f5807991s8do2b2",
      "cju2zjcvj9qma0801dk71hhi0",
      "cju2zm0axztpe0988r8s9twjr",
      "cju2zo0fwzv580988qlijd2xa",
      "cju2zr3c3vwb00993jn06bbaz",
      "cju2zwg05a0oy0801yr73ig7g",
      "cju2zy1e49pqk0878t6ncqn12",
      "cju300m3s04fg0988uzupuf7z",
      "cju302fqq9spc0878rrygyzzz",
      "cju30525w04r10835ygp257sb",
      "cju306x7w05nb0835cunv799x",
      "cju30ajhw09sx0988qyahx9s8",
      "cju30bmab08bi0835mvlr6e0r",
      "cju30df2j09dd08351ayx2t6w",
      "cju30ftgja7170855xl9bkdm0",
      "cju30gxjq0djk0988jytm49rs",
      "cju30ia8da2bq0799klnehml2",
      "cju30j1rgadut0801vuyrsnt8",
      "cju30lncba3ny0878jwnous8n",
      "cju30qbm1ad3x0855znuhpz9u",
      "cju30u1hbakn808019g15nb8b",
      "cju30xqmh0ni00835ix3batv1",
      "cju3128yi0rpu0988o4oo5n8n",
      "cju31t8xd17bk0835rnb893jk",
      "cju31y80qbawn0801twwm2l5s",
      "cju320gyvbch60801v2amdi2g",
      "cju323ypb1fbb0988gx5rzudb",
      "cju324q101fhe08350wae9cif",
      "cju326h4v1gxw08352px40p7r",
      "cju3280wv1ir009882jze27tc",
      "cju32a52lb9rc0799xi40qs00",
      "cju32gzs6xo8x0993r8tedbpb",
      "cju32l161bi1v07990vm376in",
      "cju32pzh9bpw10855q4vaxfhe",
      "cju33231uy4gi0993qc7b1jch",
      "cju334jzo261t0835yqudnfs1",
      "cju336l68y7if0993wf092166",
      "cju3381d8bz3h07991xtl7ra0",
      "cju33belnbyhm0878yxl42233",
      "cju33o12x2jm50988944mxq0v",
      "cju33qpdvc9g0087825jhf3s9",
      "cju33w4sdcivk0855x879zht7",
      "cju33x0f22peh0988g0ln7w5v",
      "cju34aozyyy830993bn16u32n",
      "cju34ds2531520988qjpqt6e3",
      "cju34eqjpcpm508788b3lhp97",
      "cju34fojcctcf0799ebolbvkn",
      "cju34i3qvcyog0855qiejxx5w",
      "cju34ymm8d6700799uop0cw33",
      "cju34zivp3fq80988opxbaqyn",
      "cju3518w2d838079939fqztbc",
      "cju353d1eda8c07992afde611",
      "cju35740hzm0g0993zl5ic246",
      "cju357rxxdaz30878y2esjpjt",
      "cju358pwtdby20878cg7nm0np",
      "cju35c4wzdhow0799h6eq4sgs",
      "cju35eg0tdmjt085525sb4bua",
      "cju35fxqyzt5p0993vusm54qz",
      "cju35ldepdtlm0801yv79y8vu",
      "cju35mdz73x890835eynq1h9v",
      "cju3tp94kfstl08181awh6z49",
      "cju3tsh4lfsok0987w6x3a0v1",
      "cju3ttznuftyf09875t11850w",
      "cju3tvffffx5f0818t5ov22al",
      "cju3tx1qyg0c907552fglumhc",
      "cju3u1c8tfyqx08503iedc3mx",
      "cju3u39fog1bo0871lxjrabks",
      "cju3ua8u0g9rg0801uayhdxhu",
      "cju3ul8dogf1z09872y2ecowp",
      "cju3umoh1geet0817cmpef5am",
      "cju3uz4o6gr9z0850lhxyxvsj",
      "cju3v11mrgwwb0755u242ygye",
      "cju3v56bwgy8v0871w14pz8fx",
      "cju3v664kh0px0818y4y7wolf",
      "cju3wstckialg0871xs0vevsj",
      "cju3x2s11ibzi0817kk284k0j",
      "cju3x9lttikfb0818a0g104zn",
      "cju3xeexgii1j0817zs68tb4g",
      "cju3xga12iixg0817dijbvjxw",
      "cju3xiic0ilzp0850lusrb42j",
      "cju3xjqtpikx50817tppy6g84",
      "cju3xl264ingx0850rcf0rshj",
      "cju3xtufwiv9c0818djsc4cqd",
      "cju3xwpgviwlx0871rwm15q7v",
      "cju3y0pjrj1c30755nxekxccj",
      "cju3y21quj0ir0818kgjagr15",
      "cju3y54kwj3nr0801biidlb4e",
      "cju3y79ofj3va0871uqfb1mzo",
      "cju3y9difj6th0801kd1rqm3w",
      "cju3ya7goj6at0818v2l5ay7f",
      "cju3yb47cj1xq0817zfotbni4",
      "cju3yht87j83m08507yk1u1fg",
      "cju40jl7skiuo0817p0smlgg8",
      "cju40m0rjkpw80871z6n6yg1u",
      "cju40poe4kt9s0755f9cnm3h5",
      "cju40r6jrksyk0871wg98zgho",
      "cju40taxlkrho0987smigg0x0",
      "cju40u30gkuzc0871rq7t666d",
      "cju40wto5kxwi0755it190f2k",
      "cju410dnfl0960755y8lu8d79",
      "cju41p90plcsx08018cnzpndc",
      "cju41s6nbleqy0755e2mslg0b",
      "cju41z76wlgbz0801qdetlvby",
      "cju422cm8lfxn0818ojicxejb",
      "cju424hy5lckr085073fva1ok",
      "cju426tomlhll0818fc0i7nvh",
      "cju428k5fldt108177s6g6f45",
      "cju42dwedlmk60871jbgu4ehi",
      "cju42g865lorv07552ytz6xxa",
      "cju42m60jlpcm08186kqppzqv",
      "cju42py9mlqyd0818u3d1d7ga",
      "cju42qet0lsq90871e50xbnuv",
      "cju42romflni20817etb9a0fl",
      "cju42tauqlo5p08171l3cuo4b",
      "cju42xpi8lw4w0871ve317a1p",
      "cju430pm2lz0y0755jkhcc3d1",
      "cju439oazm2fu0871ma0vvrft",
      "cju43c92lm5cj0755lorsorfg",
      "cju43eigtm6ev0801mv0m96t1",
      "cju43h43am1dy08176gwfhmnt",
      "cju43in5fm22c08175rxziqrk",
      "cju43jcqim2cp08172dvjvyui",
      "cju43kj2pm34f0850l28ahpni",
      "cju43o6n7m9nk087191ijwqq9",
      "cju45lbgznahl08180xz1h7u6",
      "cju45ofd9ne1j0801ri8dup7t",
      "cju45pm27n80u08174kyow1gj",
      "cju45rj7ln8980850a7821fov",
      "cju45t5ddnbio0987qtqzx762",
      "cju45v0pungu40871acnwtmu5",
      "cju5b9oyda4yr0850g9viziyv",
      "cju5bbtwsa8cl0987wgfsqpao",
      "cju5bhv81abur0850ean02atv",
      "cju5bj926aiec07559rshy4wa",
      "cju5bmhdcafs909878qfzrqzi",
      "cju5boicjagt20871b1fotkh4",
      "cju5buy2bal250818ipl6fqwv",
      "cju5bwhapakm90987c1v4z46a",
      "cju5bycdkalkb09875f7bfrvx",
      "cju5c5xc7algd0817pb1ej5yo",
      "cju5c7oijaqmq09878qwgqv8n",
      "cju5ccpvqash50850kb4bs22k",
      "cju5chrxxawka0871qcj171yz",
      "cju5cjh3xattc0817j2vbulzi",
      "cju5clr68b48r0755cmuvponm",
      "cju5cu8qkb84x08186jwo8yin",
      "cju5ddda9bkkt0850enzwatb1",
      "cju5eftctcdbj08712gdp989f",
      "cju5ekty5ckzf07550c9u3ckk",
      "cju5enq1tcn1i0755hnkon787",
      "cju5es375cnzy0801nkq35ffs",
      "cju5eyfe9cpk90987laa7tsl3",
      "cju5f0dezct4q08183ydw11dx",
      "cju5f26ebcuai0818xlwh6116",
      "cju5fb86jd1jp0755b1ukbhq5",
      "cju5fi0yxd3ei0801v7u0yudn",
      "cju5fs6j6d8350801vglraq4u",
      "cju5fu081d8gc0818l3yylujk",
      "cju5fw37edaae0801vkwvocn7",
      "cju5fydrud94708507vo6oy21",
      "cju5g163vd6mt0817uccuga6u",
      "cju5gucasds9d0801019axylx",
      "cju5h57xedz5h0755mjpc8694",
      "cju5hi52odyf90817prvcwg45",
      "cju5huurrecm70801y680y13m",
      "cju5hwonqedw10801vsd3w6kk",
      "cju5i39mreass0817au8p22zy",
      "cju5jx7jzf7c90871c2i9aiov",
      "cju5jz5fff8c50871hbe6108f",
      "cju5k3j3uf6de0817hszzfr7n",
      "cju5k503sfa5f0871lx0rpu5y",
      "cju5k7r0yf98c09878csbxb4d",
      "cju5knbbqfipk080128cggukq",
      "cju5kre09fhka0850h7b1898j",
      "cju5nxkujgscq0817l9gss626",
      "cju5nyu31gv8e0871zpk74a2n",
      "cju5o1vu9gz8a0818eyy92bns",
      "cju5o4pk9h0720755lgp9jq8m",
      "cju5tenjojp1j0755ms4949h2",
      "cju5thdbrjp1108715xdfx356",
      "cju5u4pywk81x0817vn9pe14z",
      "cju5u6wf0kh1t0755bg1ssixv",
      "cju5u8gz4kj5b07552e2wpkwp",
      "cju5udcufki0s09874ll1dbr5",
      "cju5ufn3skquf0818dhapnhba",
      "cju5uhrdwkmsu0817ervv91l8",
      "cju5uxjnol2r509871qv2yeia",
      "cju5uzmaol56l0817flxh4w9p",
      "cju5v8pgplg6k0755rvi2t63h",
      "cju5vbo6jldrt0871jf6f1700",
      "cju5vcmrqla7i0817x4sp4pqw",
      "cju5von04litr08718j8po40a",
      "cju5vutu7ll8w0871dfp92n9p",
      "cju5vwbr4lhqn0987a1pji0ux",
      "cju5vxuc5loxw0818u8xgf45p",
      "cju5vzjoslpj708186z2fusmz",
      "cju5w7xn0lrkq0801f9k0htgx",
      "cju5waeduln160817w0agirve",
      "cju5wj0faly5008187n6530af",
      "cju5wkonqlrl409877y8zvnub",
      "cju5wqonpm0e60801z88ewmy1",
      "cju5wrapcm2290818jsh26ppb",
      "cju5wrrs0m2af0818vmnajbtw",
      "cju5x00l6m5j608503k78ptee",
      "cju5x15djm7ae0755h8czf6nt",
      "cju5x7iskmad90818frchyfwd",
      "cju5xjn5mm78b09871spyqhhr",
      "cju5xkwzxmf0z0818gk4xabdm",
      "cju5xopi0md7q0871r1sjc1av",
      "cju5y84q3mdv50817eyp82xf3",
      "cju5yclrymlgj0818k426ud6z",
      "cju5yeqiwmkgl0801fzv2douc",
      "cju5yjq1pmlgc0801z0t24bly",
      "cju5ymyd8mmdc0801ry3by1xr",
      "cju6us80mv1b50871ebyq2wxa",
      "cju6ut4l8va6y0755tyw3vfqq",
      "cju6uy20suzbl0987rzuhz7z9",
      "cju6uzxk0v83p0801rcwnexdu",
      "cju6v1m1xv07w09870ah3njy1",
      "cju6v4szov55u0871qmqz3v8n",
      "cju6v5ilsv8hk0850rb5sgh6o",
      "cju6v6g6kvdw007552x6mb0po",
      "cju6vgdmivcvb08018fra5lnv",
      "cju6vifjlv55z0987un6y4zdo",
      "cju6vqarjv7yo0987q4b1btk1",
      "cju6vrs1ov8cr098788h8gs6j",
      "cju6vucxvvlda0755j7msqnya",
      "cju6vvb8svhed0801jjcquh5e",
      "cju6vvxsev9y30987kespucdg",
      "cju6w733bveoz0817e600tw72",
      "cju6wjm81vgsc0987enk9n3pr",
      "cju6wt9jvvn500871hjn3t3g0",
      "cju6wuojavt740818b5qcv3iw",
      "cju6x0yqbvxqt0755dhxislgb",
      "cju6x35ervu2808015c7eoqe4",
      "cju6x97w4vwua0850x0997r0a",
      "cju6xifswvwbo0987nibtdr50",
      "cju6xlygpw7bs0818n691jsq4",
      "cju6ywm40wdbo0987pbftsvtg",
      "cju6yxyt0wh080871sqpepu47",
      "cju6z2616wqbk07555bvnuyr1",
      "cju6z600qwh4z081700qimgl9",
      "cju6z9a9kwsl007552s49rx6i",
      "cju76erapykj30871x5eaxh4q",
      "cju76l27oyrw907551ri2a7fl",
      "cju76o55nymqd0871h31sph9w",
      "cju772304yw5t0818vbw8kkjf",
      "cju773hsyyosz0817pk1e7sjq",
      "cju7787c5yy3l080159mwqsnj",
      "cju77afzlz3kp07550x5nafzs",
      "cju77b3wyz4160755qis4ljsb",
      "cju77g99iyxc00817zqi2ppor",
      "cju77idwfz36d0871tzfzz51i",
      "cju77j66ez52p08019xygi0co",
      "cju77q10sz9ug0801449wu1nu",
      "cju77t0razbvm080106o56289",
      "cju77u1sjz77b0817ft44r3fk",
      "cju783tmkzkqu081803g7q5vk",
      "cju784jpdzeae0987q5ypq883",
      "cju785htizjzo08017tvlhtg4",
      "cju787jnjzjuj0871p94nck9g",
      "cju7ae7bq1f820987toc8si1d",
      "cju7afqon1ip40850ue2308b6",
      "cju7agj961l2r0818z29iq8yn",
      "cju7aifyo1p3n07552nxjx51f",
      "cju7aklv31h4309871m29l4e7",
      "cju7alcgr1lsr0871riqk84z7",
      "cju7amjna1ly40871ugiokehb",
      "cju7apr0c1qqm0755s7msqot4",
      "cju7aqkue1i2k09879uzcpt8r",
      "cju7arvfe1ldu0850erdmphgj",
      "cju7atnm31if40817pqclnjer",
      "cju7avvi51iox0817ym55y6tt",
      "cju7awzmu1ncs0871hziy65zx",
      "cju7azuu31mia0801pf9ib5ed",
      "cju7b10ce1mnm08011c5bwyr4",
      "cju7b1ygu1msd0801hywhy0mc",
      "cju7b2l561oas0871decgslaf",
      "cju7b4mtw1n9n080186209f3d",
      "cju7b5afm1nfw0801xqm8bf8q",
      "cju7b9vcs1luz0987ta60j1dy",
      "cju7bc95p1mdm0817yqj5jc6j",
      "cju7bd1qu1mx409877xjxibox",
      "cju7bduyq1rjf08719giru9ho",
      "cju7bf1lp1shi081835vs84lc",
      "cju7bfx651qr80801cs7epotb",
      "cju7bgnvb1sf808717qa799ir",
      "cju7bmi1v1pnj0987pa52jjok",
      "cju7cl8zm1xcu0817ado0jpas",
      "cju7cp6dw244p0818gncdol4m",
      "cju7cq6su27qv075574dir0r3",
      "cju7csvlb22fr0850lvm45n3x",
      "cju7ctvqn25dy08186g442m1r",
      "cju7cufm7298k0755j09uf3of",
      "cju7d1tvt25bu08019dvw3uff",
      "cju7d4jk723eu0817bqz2n39m",
      "cju7d5m0p23kn09871rk7pu3v",
      "cju7d6ux323ze0987xos3srkx",
      "cju7d7tly27h408016fyp5nr7",
      "cju7d8m3b2e210755l8fj1yph",
      "cju7d9seq29zd0871nzl2uu5m",
      "cju7da88w2eod0755wejzynvt",
      "cju7db7lp2f400755tntd1ohf",
      "cju7dda8w2br20818zhsuz8s7",
      "cju7ddtz729960801uazp1knc",
      "cju7deifq2fzn0755lc8idyh8",
      "cju7dff9529h208503w60lbil",
      "cju7dglf226g50987ohbthl19",
      "cju7dizi82h2i0755doucgnt3",
      "cju7dlk532dsh0871zvr6qz0r",
      "cju7dmlgf2ebw0871ieqas5fh",
      "cju7dn24o296i09871qfxb8s2",
      "cju7do8c72dbo0801vxfzxdc4",
      "cju7dp3dw2k4n0755zhe003ad",
      "cju7dsrtb2f8i085064kwugfk",
      "cju7dtb1e2j0t0818deq51ib3",
      "cju7dvl5m2n4t0755hlnnjjet",
      "cju7dwe282dc309876rco45ts",
      "cju7ebe962hr409872ovibahw",
      "cju7ecl9i2i060987xawjp4l0",
      "cju7eea9b2m0z0801ynqv1fqu",
      "cju7efffp2ivf0817etg3jehl",
      "cju7ehljc2or70871261br8ai",
      "cju7ekbo32pft0871fv7kzwb9",
      "cju7el6xv2k520817qxx9wdr5",
      "cju7eotqi2qea0871y8yc7tqh",
      "cju7epwj82koz098713apjnzo",
      "cju7er4kc2opa0801anuxc0eb",
      "cju7etr3y2p4t0801cdzjj8ab",
      "cju7ev2b12owa08500bpfpwyw",
      "cju7evxt12m730987rxivne3x",
      "cju7ey10f2rvf0871bwbi9x82",
      "cju7ezs7g2mxm098787atbran",
      "cju7f4sc62xqj075597xpmuoy",
      "cju7f5ghb2r5s0801chwkxxh9",
      "cju7f6cqy2ur20818t1saazbm",
      "cju7fazv92ywx0755xov2erga",
      "cju7fbndk2sl608015ravktum",
      "cju7fen322ou10817ziqkob4k",
      "cju7fob3x301u0755x985pmmq",
      "cju7fpfzq2wyf0818xxd1oziv",
      "cju7fq7mm2pw508176uk5ugtx",
      "cju7g7ba42z310987bqzbi2bq",
      "cju83h9ysjwe808716nt35oah",
      "cju83ipu3jwpx0801z5pvguf8",
      "cju83k8fyjsxr0817d6nxs6r4",
      "cju83kxitjv340987z09m0ezy",
      "cju83nwu1jxte0987h1krpfmv",
      "cju83qd0yjyht0817ktkfl268",
      "cju83rcnzkbsj0755x5anfrcg",
      "cju83syhdk6gs0801rf1rekdl",
      "cju83u9ftk3ni0987qnhlcinv",
      "cju83vvmik9wa08710yeh7cuk",
      "cju83wwn1k55e0850kw6i2d81",
      "cju83yddek68q0850d2x7zfkm",
      "cju8402x1kcy70801t6kz6bdi",
      "cju8418jhkf7d0818ga2v0xq0",
      "cju843yjskhq30818qre4rwm2",
      "cju84aoa3ktwn0755pfl4gfwd",
      "cju84dsvaklpx098750hp83x4",
      "cju84ffdzkrjn08183jh1fxmb",
      "cju84gpefknwm098714oq8q61",
      "cju84ih17kp5l09876bkooocl",
      "cju84jdl9kv0i0871eog9b3i9",
      "cju84kplnl1y30755ropua1b0",
      "cju8567gdlcbq0801dwwyo2jt",
      "cju857ad0l88m0817qx4cwxnf",
      "cju858eswlepn0871pzvdrhj1",
      "cju85a8h8llwm07559wxg4t5w",
      "cju85c2d4ln1b0755zz1z3onx",
      "cju85citjlnfm0755i4rk5tqj",
      "cju85dx63lic408017f0l0400",
      "cju85fc11ljr40818edpb0inh",
      "cju85hguellg50818kwu3s8d1",
      "cju85ia1slh220987y7c20sm2",
      "cju85je7vlht70817c9jcjwi4",
      "cju85nr8elly209872w9n5m0s",
      "cju85omszllp30850b6rm9mi3",
      "cju85qefyln6v0850szeb9byi",
      "cju85rkbnlo1c08503uxcpax1",
      "cju87kbcen2av0987usezo8kn",
      "cju87li0zn3yb0817kbwgjiz8",
      "cju87nkyrnb970801q84m47yt",
      "cju87ox0kncom0801b98hqnd2",
      "cju87q6yoneim0871dl4phvkd",
      "cju87vqa0ndwg0850onjdz7ol",
      "cju87xn2snfmv0987sc3d9xnq",
      "cju87z6o6nh73085045bzsx6o",
      "cju87zv8lni0o0850hbbecbq6",
      "cju8828oxnool0801qno9luhr",
      "cju885ikhnmkn09878s2lqtuh",
      "cju886ryxnsl50801r93jai7q",
      "cju887ftknop008177nnjt46y",
      "cju888fr7nveu0818r9uwtiit",
      "cju88gx09o2vk0818610zody3",
      "cju88itqbny720987hxizbj5y",
      "cju88l66no10s0850rsda7ej1",
      "cju88msmoo3470817m441j4sg",
      "cju88q6h6obpd0871ckmiabbo",
      "cju88rl5eo94l0850kf5wtrm1",
      "cju88t4fvokxf07558ymyh281",
      "cju88v2f9oi8w0871hx9auh01",
      "cju88y1mwoln50871emyfny1g",
      "cju88z8bson4h0871nnd7fdxo",
      "cju8914beokbf0850isxpocrk",
      "cju892fesoq2g0801n0e0jyia",
      "cju893jmdompz0817xn3g1w4h",
      "cju89y9h0puti0818i5yw29e6",
      "cju89z6pqpqfx0817mfv8ixjc",
      "cju8a1jtvpt9m081712iwkca7",
      "cju8abobpqbir08189u01huru",
      "cju8aeei7q8k308173n9y4klv",
      "cju8aj01yqeqm0850lhdz3xdw",
      "cju8amfdtqi4x09871tygrgqe",
      "cju8ando2qqdo0818ck7i1be1",
      "cju8apjewqrk00801k5d71gky",
      "cju8aqq8uqmoq0987hphto9gg",
      "cju8arof2qpf20850ifr1bnqj",
      "cju8at3s1qqqx0850hcq8nmnq",
      "cju8auylgqx0z0871u4o4db7o",
      "cju8axq24r4an0755yhv9d4ly",
      "cju8azmhcr66e0755t61atz72",
      "cju8b1v3br45u087189kku66u",
      "cju8b2rmgr52s0801p54eyflx",
      "cju8b542nr81x0871uxnkm9ih",
      "cju8b5p40r2c60987ofa0mu03",
      "cju8b8yair65w09878pyqtr96",
      "cju8bafgqrf4x0818twisk3ea",
      "cju8bff9nrfi10850fmfzbf8v",
      "cju8bh8surexp0987o5pzklk1",
      "cju8bljw9rqk20801kr54akrl",
      "cju8bm24yrrdp081829mbo8ic",
      "cju8bn7m2rmm70817hgxpb1uq",
      "cju8bop5jrsid08716i24fqda",
      "cju8bpctzrqkr0850zeldv9kt",
      "cju8bqxxurs6i0850mu7mtef9",
      "cju8brv16rx7f0818uf5n89pv",
      "cju8buos5rz9b08715lfr0f4f",
      "cju8bw697rwg308177tg8huas",
      "cju8bzzy2s66m08016z6mouqt",
      "cju8c2rqzs5t80850d0zky5dy",
      "cju8c3xs7sauj0801ieyzezr5",
      "cju8c5mxls96t0850wvkvsity",
      "cju8c5zcbsdfz0801o5t6jag1",
      "cju8c6hnxsdvr0801wn0vrsa6",
      "cju8c9akjsdjj0850s67uzlxq",
      "cju8ca4geseia0850i2ru11hw",
      "cju8cbsyssiqj0871gr4jedjp",
      "cju8cdeazsm8h0801jxifmzur",
      "cju8ceacrsqkr0755hdz145es",
      "cju8cj10qsrau0871o2dr6ai1",
      "cju8clorgsuwn08714toqb7v6",
      "cju8cwy02t9eq08185qn12c02",
      "cju8czvnztbf40871b4m7t78w",
      "cju8d2q30tfhs0801n7lx77xl",
      "cju8d4jgatgpj0871q2ophhkm",
      "cju8dic9mtppa0987swn23wbc",
      "cju8djdqztu6408506pzhlo18",
      "cju8dk7eztzup08182yxko5zh",
      "cju8dm2cau2km0818jsv9eeq2",
      "cju8dpa89u6l80818dj6lldh9",
      "cjyzjzssvd8pq0838f4nolj5l",
      "cjyzk8qieoboa0848ogj51wwm",
      "cjyzkmjy8evns070165gf9dmq",
      "cjyzkpsbjdsjq07211dfi4sru",
      "cjyzlw7f9faqr070129au64sq",
      "cjyztzaqtrv430848l8xgcerw",
      "cjyzu3reghjya0794w7pwoi50",
      "cjyzu9th0qt4r0a46pyl4zik0",
      "cjyzufihqquiw0a46jatrbwln",
      "cjyzuio1qgh040763k56deohv",
      "cjyzul1qggwwj07216mhiv5sy",
      "ck2395w2mb4vu07480otsu6tw",
      "ck2bxiswtxuw80838qkisqjwz",
      "ck2bxlujamu330725szlc2jdu",
      "ck2bxqz3evvg20794iiyv5v2m",
      "ck2bxskgxxzfv08386xkqtqdy",
      "ck2da7fwcjfis07218r1rvm95"
    ],
    "val": [
      "cju0qkwl35piu0993l0dewei2",
      "cju0rx1idathl0835detmsp84",
      "cju15jr8jz8sb0855ukmkswkz",
      "cju175facms5f0993a5tjikvt",
      "cju17g6ykn1cs0993dww6qdi8",
      "cju17x0j4nfc10993y31pvlgs",
      "cju1819curo000988pd5xcqme",
      "cju1871y11d6r0799k6cw4yze",
      "cju1amqw6p8pw0993d9gc5crl",
      "cju1aqqv02qwz0878a5cyhr67",
      "cju1b0y2e396p08558ois175d",
      "cju1c4fcu40hl07992b8gj0c8",
      "cju1c6yfz42md08550zgoz3pw",
      "cju1cyjb5qtie0993njqne9m3",
      "cju1dq3x1vgx109889c7wyirg",
      "cju1egh885m1l0855ci1lt37c",
      "cju1ejj7dvqfa0835ra184v5m",
      "cju1euant5l960878iqj5vvto",
      "cju1fjsb4sipq09931lvd8e41",
      "cju1fr4etsmrr09933u4t4aql",
      "cju1ftaji6isw0855108yqcse",
      "cju1g4nsb6ngy0799l4ezm8ab",
      "cju1gghyjwxt80835vx0wgxw0",
      "cju1haab178i70799tk9z8y8x",
      "cju1hmff8tkp809931jps6fbr",
      "cju2hgsptlfam0835o3b59h1o",
      "cju2hjrqcvi2j0801bx1i6gxg",
      "cju2m56cryvqd0801gtn2yp8t",
      "cju2nnqrqzp580855z8mhzgd6",
      "cju2np2k9zi3v079992ypxqkn",
      "cju2nqapmzvk20801f9us40dx",
      "cju2nsmwjlzyl0993jl80chvz",
      "cju2ntxtdzlvu0799xl3j9pan",
      "cju2osuru0ki00855txo0n3uu",
      "cju2pag1f0s4r0878h52uq83s",
      "cju2phaksnahz0993yxogjcpv",
      "cju2rz4k434s70855wwx3ddtx",
      "cju2suk42469908015ngmq6f2",
      "cju2syxa93yw40799x2iuwabz",
      "cju2tqfgw4oat0799rn0g5b2z",
      "cju2uwz9f5yf1085506cfamfx",
      "cju2uzabhs6er0993x3aaf87p",
      "cju2xd75m82720801q4s4ik3n",
      "cju2xjz2ju8pe0993ysv9wg17",
      "cju2xlcqxy9c60988vjacdznb",
      "cju2xyd9vyi7m098831qcucse",
      "cju2ycp1u8g2r0799jslnp7cz",
      "cju2yi9tz8vky0801yqip0xyl",
      "cju2z6ez69g4u0801qwt088lw",
      "cju2zdhsczmn50988z64qwg2q",
      "cju30k2z40ds308353kdew70n",
      "cju30mm25a53s0799qa5wiqe8",
      "cju30nyxe0gfb0835p256yoju",
      "cju30ov1oah920801mi8thuyg",
      "cju30ywtc0oar0835bp2en7ec",
      "cju310f6val1v0855xo8tc3gu",
      "cju31rb7vb6110801p9rhacuw",
      "cju32csyfblyh080170aa3x5p",
      "cju32fhnhbds40799broyoptc",
      "cju32jcdabepz0878d0cznmfe",
      "cju32qr9tbvsj08013pkpjenq",
      "cju32zhbnc1oy0801iyv1ix6p",
      "cju330ofbc2l30801th5g3hw6",
      "cju34m7h536wq0988xz7gx79v",
      "cju34sh43d8zm08019xbwhc0o",
      "cju34uhepd3dd0799hs8782ad",
      "cju3521y5d5mq0878t3ezsu4p",
      "cju35atpxdjot0855q46aqrd0",
      "cju35oyvd3y850988km12hdz1",
      "cju3uhb79gcgr0871orbrbi3x",
      "cju3x4blzieu30850x10uuvbm",
      "cju3x5u2tiihx0818914gzxy1",
      "cju3xzvnzj0hd0755xprz39nj",
      "cju3ykamdj9u208503pygyuc8",
      "cju414lf2l1lt0801rl3hjllj",
      "cju418rckl3ur08012psrx1r1",
      "cju41kd7yl4nm0850gil5qqwh",
      "cju41nz76lcxu0755cya2qefx",
      "cju42nm68lpyo0818xvvqmupq",
      "cju42u5bjlvi10801dc13sskp",
      "cju42wamblrqn098798r2yyok",
      "cju43b8daly4408170e5ev06g",
      "cju43gfosm63n08714rpih8pe",
      "cju43lcnum9y10755bjs7z87f",
      "cju45jpvfn6c809873pv1i34s",
      "cju45n0oxn5vu08500yfrt9jn",
      "cju5bf6hxa6m50817rbwettgu",
      "cju5ca9hcatkc0801jzwe7tfx",
      "cju5cetivauok0987ok3e5bre",
      "cju5hl8nee8a40755fm8qjj0o",
      "cju5hqz50e7o90850e0prlpa0",
      "cju5ht88gedbu0755xrcuddcx",
      "cju5hyi9yegob0755ho3do8en",
      "cju5uget8krjy0818kvywd0zu",
      "cju5vgawslbe30987ndeepc1b",
      "cju5vi4nxlc530817uoqm2m7a",
      "cju5woy82m07m08505dmjg7g1",
      "cju5wtdu4m0im0871mix0yvc0",
      "cju5wuhm1lwm40987vugqn3vv",
      "cju5y4hgqmk0i08180rjhbwvp",
      "cju5y7buemcw80987p0r30g9f",
      "cju5ygh1zmmdi0755uod5e17i",
      "cju5yhgznmkzb0801cji2vi8j",
      "cju6v3bb2v7xo085090blubyw",
      "cju6xa0qmvzun0818xjukgncj",
      "cju6xmqd9w0250817l5kxfnsk",
      "cju76lsehyia10987u54vn8rb",
      "cju77bvg0yv4r0987yh60xmjo",
      "cju77re6fz5bb0817vp9redjg",
      "cju77vvcwzcm50850lzoykuva",
      "cju7aez2x1jtj0871ztezs3oi",
      "cju7ahtkb1jr90801jck4kbds",
      "cju7ap09p1kz10850ldccjebj",
      "cju7b3f5h1sm40755i572jden",
      "cju7crgxa28550755wbsgqkel",
      "cju7cue9b232j0801qdzk1ykj",
      "cju7d2q1k27nf08715zshsckt",
      "cju7d3oc82cho0755dajlwldz",
      "cju7dqcwi2dz00850gcmr2ert",
      "cju7dxffn2eam0817qxosfwch",
      "cju7dymur2od30755eg8yv2ht",
      "cju7ea4om2l910801bohqjccy",
      "cju7ejm2l2ncl0801wq6y84nw",
      "cju7emdni2py40871ivhxjtut",
      "cju7es23b2vcp0755gpbm9s7v",
      "cju7et17a2vjk0755e743npl1",
      "cju7eueum2oqn0850rodmx8zo",
      "cju7ez7r22qbc08015xfoz2wb",
      "cju7fd6yt2p740987wkr8exo1",
      "cju7ff97z2ow40817u2r83my5",
      "cju7fmvpk2q170987v6i3ola8",
      "cju7frtqu2xa20818wq8r9fzf",
      "cju83mki1jv5w0817kubxm31r",
      "cju847pxykriq0755268ktrk2",
      "cju849c23kgnk0817cgv2hw1e",
      "cju85bf1algsq0871y9gtlq97",
      "cju85mpuglq8k0818d2it6hzb",
      "cju87r56lnkyp0755hz30leew",
      "cju87tyddnnad0755bj0wxahe",
      "cju884985nlmx0817vzpax3y4",
      "cju88fpm4o0tl0871w1i6a4ds",
      "cju88oh0po9gq0801nge4tgr1",
      "cju88trl3ogi208716qvti51b",
      "cju890guyoiti098753yg6cdu",
      "cju8a56vxpy780850r45yu4wk",
      "cju8a84g0q76m0818hwiggkod",
      "cju8adb60qbiu080188mxpf8d",
      "cju8ashhnquqr0801rwduzt7d",
      "cju8aw9n1qyg10801jkjlmors",
      "cju8ayeq7r1fb0818z1junacy",
      "cju8b0jr0r2oi0801jiquetd5",
      "cju8b6rp0r5st0850184f79xt",
      "cju8b7aqtr4a00987coba14b7",
      "cju8bbznkrf5g0871jncffynk",
      "cju8bi8q7rlmn0871abc5ch8k",
      "cju8bj2ssrmlm0871gc2ug2rs",
      "cju8bk8oirjhw0817hgkua2w8",
      "cju8bssulrrcy0987h1vq5060",
      "cju8c1a0ws7o208181c6lbsom",
      "cju8c82iosagu0817l74s4m5g",
      "cju8cattbsivm0818p446wgel",
      "cju8cgi2kspp308011nxdtjp6",
      "cju8chdlqsu620755azjty1tj",
      "cju8chxndsre008015uisl4si",
      "cju8doa16u5gh0818w1ywda3q",
      "cju8dqkrqu83i0818ev74qpxq",
      "cjyzl833ndne80838pzuq6ila",
      "cjyzurzvohqnr0794es1itzek",
      "ck2bxpfgxu2mk0748gsh7xelu",
      "ck2bxw18mmz1k0725litqq2mc"
    ]
  }
}
assert "test" in REPO_SPLITS, "v2 splits must contain an independent test set"
test_stems = list(REPO_SPLITS["test"])
fold0_train = list(REPO_SPLITS["fold_0"]["train"])
fold0_val = list(REPO_SPLITS["fold_0"]["val"])
assert len(test_stems) == 150, f"expected 150 test cases, got {len(test_stems)}"
assert len(fold0_train) == 680 and len(fold0_val) == 170, "unexpected fold-0 sizes"
tset = set(test_stems)
assert len(tset & set(fold0_train)) == 0, "FATAL: test leaked into train"
assert len(tset & set(fold0_val)) == 0, "FATAL: test leaked into val"
print(f"[v2] repository splits embedded: test={len(test_stems)} | fold0 train={len(fold0_train)} val={len(fold0_val)} | disjointness verified")


In [ ]:
# Cell 4: Kvasir-SEG Ingestion (read-only use of the mounted dataset)
# NOTE: the dataset is read **in place** from /kaggle/input and never copied into
# /kaggle/working. Copying it would inflate the kernel output archive with ~2k
# redundant image files, which makes `kaggle kernels output` extremely slow to
# download (per-file latency) and needlessly consumes the working disk.
input_img_dirs = glob.glob("/kaggle/input/**/images", recursive=True)
input_mask_dirs = glob.glob("/kaggle/input/**/masks", recursive=True)

if input_img_dirs and input_mask_dirs:
    IMG_DIR, MASK_DIR = input_img_dirs[0], input_mask_dirs[0]
    print(f"[Dataset] Using mounted dataset in place: {IMG_DIR}")
else:
    print("[Dataset] No mounted dataset found -> downloading official Simula Kvasir-SEG (46.2 MB)")
    DATA_DIR = "/kaggle/working/data/kvasir_seg"
    os.makedirs(DATA_DIR, exist_ok=True)
    zip_path = "/kaggle/working/kvasir-seg.zip"
    ctx = ssl._create_unverified_context()
    with urllib.request.urlopen("https://datasets.simula.no/downloads/kvasir-seg.zip", context=ctx) as r, open(zip_path, "wb") as out:
        out.write(r.read())
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall("/kaggle/working/data")
    root = os.path.join("/kaggle/working/data", "Kvasir-SEG")
    root = root if os.path.isdir(root) else DATA_DIR
    IMG_DIR, MASK_DIR = os.path.join(root, "images"), os.path.join(root, "masks")
print(f"[Dataset] images: {len(glob.glob(os.path.join(IMG_DIR, '*.*')))} | masks: {len(glob.glob(os.path.join(MASK_DIR, '*.*')))}")


In [ ]:
# Cell 5: Contiguous Zero-Copy GPU VRAM Pre-loading
print("[VRAM] Pre-loading dataset into GPU memory...")

def records_to_tensors(records, img_size=256):
    n = len(records)
    imgs_np = np.empty((n, 3, img_size, img_size), dtype=np.float32)
    msks_np = np.empty((n, 1, img_size, img_size), dtype=np.float32)
    lbls_np = np.empty((n,), dtype=np.float32)
    for i, r in enumerate(records):
        img = cv2.imread(r["img_path"])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        msk = cv2.imread(r["mask_path"], cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
        msk = cv2.resize(msk, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
        msk_bin = (msk > 127).astype(np.float32)
        imgs_np[i] = np.transpose(img, (2, 0, 1)).astype(np.float32) / 255.0
        msks_np[i, 0] = msk_bin
        lbls_np[i] = 1.0 if msk_bin.sum() > TRIAGE_PIXEL_THRESHOLD else 0.0
    return torch.from_numpy(imgs_np), torch.from_numpy(msks_np), torch.from_numpy(lbls_np)

img_files = sorted(glob.glob(os.path.join(IMG_DIR, "*.*")))
mask_dict = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob(os.path.join(MASK_DIR, "*.*"))}
stem_map = {}
for ip in img_files:
    stem = os.path.splitext(os.path.basename(ip))[0]
    if stem in mask_dict:
        stem_map[stem] = {"stem": stem, "img_path": ip, "mask_path": mask_dict[stem]}

missing = [s for s in test_stems + fold0_train + fold0_val if s not in stem_map]
assert not missing, f"FATAL: {len(missing)} split stems missing from the dataset (e.g. {missing[:3]})"

train_records = [stem_map[s] for s in fold0_train]
val_records = [stem_map[s] for s in fold0_val]
test_records = [stem_map[s] for s in test_stems]

cpu_train = records_to_tensors(train_records)
cpu_val = records_to_tensors(val_records)
cpu_test = records_to_tensors(test_records)

gpu_datasets = {}
for dev in DEV_LIST:
    gpu_datasets[dev] = {
        "train_imgs": cpu_train[0].to(dev), "train_msks": cpu_train[1].to(dev), "train_lbls": cpu_train[2].to(dev),
        "val_imgs": cpu_val[0].to(dev), "val_msks": cpu_val[1].to(dev), "val_lbls": cpu_val[2].to(dev),
        "test_imgs": cpu_test[0].to(dev), "test_msks": cpu_test[1].to(dev), "test_lbls": cpu_test[2].to(dev),
    }
    mb = gpu_datasets[dev]["train_imgs"].numel() * gpu_datasets[dev]["train_imgs"].element_size() / 1e6
    print(f"  [{dev}] cached {len(cpu_train[0])} train / {len(cpu_val[0])} val / {len(cpu_test[0])} test ({mb:.1f} MB VRAM)")

BATCH_SIZE = 16
print(f"[VRAM] batch size {BATCH_SIZE}; triage positive threshold {TRIAGE_PIXEL_THRESHOLD} px (defined in Cell 2)")


In [ ]:
# Cell 6: Model Zoo (repository implementations)
def build_model(name):
    if name == "UNet":
        return UNet(in_channels=3, num_classes=1)
    if name == "AttentionUNet":
        return AttentionUNet(in_channels=3, num_classes=1)
    if name == "UNetPlusPlus":
        return UNetPlusPlus(in_channels=3, num_classes=1)
    if name == "FEENet":
        return FEENet(in_channels=3, num_classes=1, use_dwt=True, use_egm=True, base_channels=48)
    if name == "FEENetEarlyExit":
        return FEENetEarlyExit(in_channels=3, num_classes=1, base_channels=48)
    if name == "FEENet_noDWT":
        return FEENet(in_channels=3, num_classes=1, use_dwt=False, use_egm=True, base_channels=48)
    if name == "FEENet_noEGM":
        return FEENet(in_channels=3, num_classes=1, use_dwt=True, use_egm=False, base_channels=48)
    raise ValueError(name)

is_early_exit = lambda n: "EarlyExit" in n
for n in ["UNet", "AttentionUNet", "UNetPlusPlus", "FEENet", "FEENetEarlyExit", "FEENet_noDWT", "FEENet_noEGM"]:
    m = build_model(n)
    print(f"  {n:18s} params={sum(p.numel() for p in m.parameters())/1e6:.3f}M")
    del m
print("[Zoo] model factory ready")


In [ ]:
# Cell 7: Efficiency Profiler -- measured latency/FPS + measured triage-gate cost
ANALYTIC_GFLOPS = {  # fallback only; used when thop is unavailable
    "UNet": 13.75, "AttentionUNet": 14.03, "UNetPlusPlus": 34.62,
    "FEENet": 12.59, "FEENetEarlyExit": 12.59, "FEENet_noDWT": 12.87, "FEENet_noEGM": 11.83,
    # Screening gate (stem + layer1 + Haar DWT + SCA + triage head).
    "TriageGate": 0.94,
}

def measure_latency_ms(fn, dev, n_warmup=15, n_runs=50):
    for _ in range(n_warmup):
        fn()
    if torch.cuda.is_available():
        s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        s.record()
        for _ in range(n_runs):
            fn()
        e.record()
        torch.cuda.synchronize(dev)
        return s.elapsed_time(e) / n_runs
    t0 = time.time()
    for _ in range(n_runs):
        fn()
    return (time.time() - t0) / n_runs * 1000.0

class FullPathWrapper(nn.Module):
    # Forces the early-exit network down its full segmentation path.
    # thop/torch call `model(x)` directly, which would trigger the dynamic triage
    # gate and report the *gate* cost instead of the full-path cost.

    def __init__(self, ee):
        super().__init__()
        self.ee = ee

    def forward(self, x):
        return self.ee(x, force_full=True, return_raw_logits=True)


def measure_gflops(model, name, dev):
    if THOP_OK:
        try:
            if name == "TriageGate":
                m = model
            else:
                m = build_model(name).to(dev).eval()
                if is_early_exit(name):
                    m = FullPathWrapper(m).to(dev).eval()
            macs, _ = thop_profile(m, inputs=(torch.randn(1, 3, 256, 256, device=dev),), verbose=False)
            if name != "TriageGate":
                del m
            return round(macs / 1e9, 2), "thop"
        except Exception as ex:
            print(f"[thop] failed for {name}: {ex}")
    return ANALYTIC_GFLOPS.get(name, 12.59), "analytic-table"

def profile_model(name, dev):
    model = build_model(name).to(dev).eval()
    x = torch.randn(1, 3, 256, 256, device=dev)
    with torch.no_grad():
        if is_early_exit(name):
            lat = measure_latency_ms(lambda: model(x, force_full=True, return_raw_logits=True), dev)
        else:
            lat = measure_latency_ms(lambda: model(x), dev)
        gflops, flops_src = measure_gflops(model, name, dev)
    out = {
        "params_m": round(sum(p.numel() for p in model.parameters()) / 1e6, 3),
        "gflops": gflops,
        "flops_source": flops_src,
        "latency_ms": round(lat, 2),
        "fps": round(1000.0 / max(lat, 1e-4), 1),
    }
    del model
    torch.cuda.empty_cache()
    return out

class TriageGate(nn.Module):
    """Measurable cost of the screening gate: stem + layer1 + DWT + SCA + triage head."""
    def __init__(self, ee):
        super().__init__()
        self.stem, self.layer1, self.dwt = ee.stem, ee.layer1, ee.dwt
        self.wf1, self.wf1_attn = ee.wf1, getattr(ee, "wf1_attn", None)
        self.triage_head = ee.triage_head

    def forward(self, x):
        xs = self.stem(x)
        e1 = self.layer1(xs)
        _, lh, hl, hh = self.dwt(xs)
        cat = torch.cat([lh, hl, hh], dim=1)
        if self.wf1_attn is not None:
            cat = self.wf1_attn(cat)
        hf1 = self.wf1(cat)
        return self.triage_head(e1, hf1)[0]

def profile_triage_gate(dev):
    ee = build_model("FEENetEarlyExit").to(dev).eval()
    gate = TriageGate(ee).to(dev).eval()
    x = torch.randn(1, 3, 256, 256, device=dev)
    gflops, src = measure_gflops(gate, "TriageGate", dev)
    with torch.no_grad():
        lat = measure_latency_ms(lambda: gate(x), dev)
    res = {
        "gate_params_m": round(sum(p.numel() for p in gate.parameters()) / 1e6, 3),
        "gate_gflops": gflops, "gate_flops_source": src,
        "gate_latency_ms": round(lat, 2), "gate_fps": round(1000.0 / max(lat, 1e-4), 1),
    }
    del gate, ee
    torch.cuda.empty_cache()
    return res

print("[Profiler] ready (full-path + measured triage gate)")


In [ ]:
# Cell 8: Training Engine -- repository CompoundLoss + deep supervision + triage CE
criterion = CompoundLoss(lambda_bce=1.0, lambda_dice=1.0, lambda_boundary=0.1).to(DEV_LIST[0])
triage_criterion = nn.CrossEntropyLoss()

def compute_loss(name, out, masks, lbls):
    """Mirrors train.py: CompoundLoss(seg, edge, masks) + 0.3 * deep supervision + 0.5 * triage CE."""
    if is_early_exit(name):
        seg, edge, triage_logits, _quality, aux_preds = out
    else:
        seg, edge, aux_preds = out
        triage_logits = None

    loss, _ = criterion(seg, edge, masks)

    if aux_preds:
        aux_total = 0.0
        for aux in aux_preds:
            up = F.interpolate(aux, size=masks.shape[2:], mode="bilinear", align_corners=True)
            aux_total = aux_total + criterion(up, None, masks)[0]
        loss = loss + 0.3 * (aux_total / len(aux_preds))

    if triage_logits is not None:
        loss = loss + 0.5 * triage_criterion(triage_logits, lbls)
    return loss

def train_model(name, dev, epochs=35, lr=3e-4, lambda_bce=1.0, lambda_dice=1.0, lambda_boundary=0.1):
    print("=" * 68)
    print(f"Training {name} on [{dev}] | epochs={epochs} lr={lr} boundary_w={lambda_boundary}")
    print("=" * 68)
    model = build_model(name).to(dev)
    ds = gpu_datasets[dev]

    global criterion
    criterion = CompoundLoss(lambda_bce=lambda_bce, lambda_dice=lambda_dice, lambda_boundary=lambda_boundary).to(dev)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda") if torch.cuda.is_available() else None

    # -1.0 (as in train.py) so the first epoch always yields a checkpoint even
    # when the validation Dice is exactly 0.
    best_val_dice, best_path = -1.0, os.path.join(OUTPUT_DIR, f"best_{name.lower()}.pth")
    history = {"train_loss": [], "val_dice": [], "val_iou": []}
    n_train, n_val = len(ds["train_imgs"]), len(ds["val_imgs"])

    for epoch in range(1, epochs + 1):
        model.train()
        loss_acc, nb = torch.zeros(1, device=dev), 0
        perm = torch.randperm(n_train, device=dev)
        for i in range(0, n_train, BATCH_SIZE):
            idx = perm[i: min(i + BATCH_SIZE, n_train)]
            b_imgs, b_msks = ds["train_imgs"][idx], ds["train_msks"][idx]
            b_lbls = (ds["train_msks"][idx].sum(dim=(1, 2, 3)) > TRIAGE_PIXEL_THRESHOLD).long()
            if random.random() > 0.5:
                b_imgs, b_msks = torch.flip(b_imgs, [3]), torch.flip(b_msks, [3])
            if random.random() > 0.5:
                b_imgs, b_msks = torch.flip(b_imgs, [2]), torch.flip(b_msks, [2])

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=scaler is not None):
                out = model(b_imgs, return_aux=True) if "FEENet" in name or "EarlyExit" in name else model(b_imgs)
                if is_early_exit(name):
                    loss = compute_loss(name, out, b_msks, b_lbls)
                elif "FEENet" in name:
                    loss = compute_loss(name, out, b_msks, b_lbls)
                else:
                    loss, _ = criterion(out, None, b_msks)
            if scaler:
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            else:
                loss.backward(); optimizer.step()
            loss_acc += loss.detach(); nb += 1

        scheduler.step()
        epoch_loss = (loss_acc / max(nb, 1)).item()

        # validation with repository metrics (explicit from_logits)
        model.eval()
        tracker = MetricTracker()
        with torch.no_grad():
            for i in range(0, n_val, BATCH_SIZE):
                sl = slice(i, min(i + BATCH_SIZE, n_val))
                v_imgs, v_msks = ds["val_imgs"][sl], ds["val_msks"][sl]
                if is_early_exit(name):
                    logits = model(v_imgs, force_full=True, return_raw_logits=True)
                else:
                    logits = model(v_imgs)
                tracker.update(logits, v_msks, from_logits=True)
        res = tracker.get_results()
        history["train_loss"].append(round(epoch_loss, 4))
        history["val_dice"].append(round(res["dice"] * 100, 2))
        history["val_iou"].append(round(res["iou"] * 100, 2))
        if res["dice"] > best_val_dice:
            best_val_dice = res["dice"]
            torch.save(model.state_dict(), best_path)
        if epoch % 5 == 0 or epoch == 1 or epoch == epochs:
            print(f"Epoch [{epoch:02d}/{epochs:02d}] loss={epoch_loss:.4f} val_dice={res['dice']*100:.2f}% val_iou={res['iou']*100:.2f}% best={best_val_dice*100:.2f}%")

    print(f"Finished {name}: best val Dice = {best_val_dice*100:.2f}% -> {best_path}")
    return model, best_path, history

print("[Train] engine ready (repository loss, deep supervision, triage CE)")


In [ ]:
# Cell 9: Train All Models with Model-Level Dual-T4 Allocation
# 60 epochs (not 35): the local probe matrix showed the full compound loss still
# climbing at 35 (76.14 val Dice) and reaching 77.55 only by 60, while the plain
# baselines converge much earlier. Equal epochs for every model (see Exp-032).
EPOCHS = 60
LR = 3e-4
EXTRA_MODELS = ["FEENet_noDWT", "FEENet_noEGM"] if True else []

BASE_MODELS = ["UNet", "AttentionUNet", "UNetPlusPlus", "FEENet", "FEENetEarlyExit"]
ALL_MODELS = BASE_MODELS + EXTRA_MODELS

def assign(name):
    if NUM_GPUS > 1:
        return "cuda:0" if name in ("UNet", "AttentionUNet") else "cuda:1"
    return DEV_LIST[0]

model_assignments = {n: assign(n) for n in ALL_MODELS}
trained, histories, profiles = {}, {}, {}
gate_profile = None

for name in ALL_MODELS:
    dev = model_assignments[name]
    print(f"\n>>> Profiling {name} on {dev}")
    profiles[name] = profile_model(name, dev)
    print(f"    params={profiles[name]['params_m']}M gflops={profiles[name]['gflops']} ({profiles[name]['flops_source']}) "
          f"lat={profiles[name]['latency_ms']}ms fps={profiles[name]['fps']}")
    if name == "FEENetEarlyExit":
        gate_profile = profile_triage_gate(dev)
        print(f"    [gate] params={gate_profile['gate_params_m']}M gflops={gate_profile['gate_gflops']} "
              f"({gate_profile['gate_flops_source']}) lat={gate_profile['gate_latency_ms']}ms fps={gate_profile['gate_fps']}")
    model, ckpt, hist = train_model(name, dev, epochs=EPOCHS, lr=LR,
                                    lambda_boundary=0.0 if name == "FEENet_noBL" else 0.1)
    trained[name] = (model.to(dev), ckpt)
    histories[name] = hist

# Provenance artifact: every params/FLOPs/latency number quoted in the paper must be
# traceable to a machine-readable file shipped with the results.
with open(os.path.join(OUTPUT_DIR, "model_profiles.json"), "w", encoding="utf-8") as _f:
    json.dump({
        "full_path": {k: profiles[k] for k in profiles},
        "triage_gate": gate_profile,
        "note": ("thop MACs at 256x256 batch 1; latency/FPS by non-blocking CUDA events "
                 f"on {DEV_LIST[0]}; gate = stem+layer1+DWT+SCA+triage head"),
    }, _f, indent=2)
print("[provenance] wrote model_profiles.json")


In [ ]:
# Cell 10: Independent Test Evaluation (150 cases, repository metrics)
test_results = {}
print("=" * 78)
print("INDEPENDENT TEST SET EVALUATION -- 150 held-out cases (repository splits.json)")
print("=" * 78)

for name in ALL_MODELS:
    dev = model_assignments[name]
    model, ckpt = trained[name]
    model.load_state_dict(torch.load(ckpt, map_location=dev))
    model.eval()

    tracker = MetricTracker()
    with torch.no_grad():
        for i in range(0, len(gpu_datasets[dev]["test_imgs"]), BATCH_SIZE):
            sl = slice(i, min(i + BATCH_SIZE, len(gpu_datasets[dev]["test_imgs"])))
            t_imgs, t_msks = gpu_datasets[dev]["test_imgs"][sl], gpu_datasets[dev]["test_msks"][sl]
            # NOTE: FEENetEarlyExit.forward(force_full=True) returns *probabilities*;
            # MetricTracker(from_logits=True) would apply a second sigmoid and turn
            # every pixel positive. Always request raw logits explicitly.
            logits = (model(t_imgs, force_full=True, return_raw_logits=True)
                      if is_early_exit(name) else model(t_imgs))
            tracker.update(logits, t_msks, from_logits=True)
    r = tracker.get_results()
    test_results[name] = {
        "Test Dice (%)": round(r["dice"] * 100, 2),
        "Test mIoU (%)": round(r["iou"] * 100, 2),
        "HD95 (px)": round(r["hd95"], 2),
        "Precision (%)": round(r["precision"] * 100, 2),
        "Recall (%)": round(r["recall"] * 100, 2),
        "Params (M)": profiles[name]["params_m"],
        "FLOPs (G)": profiles[name]["gflops"],
        "Latency (ms)": profiles[name]["latency_ms"],
        "FPS": profiles[name]["fps"],
    }
    print(f"[{name:18s}] Dice {test_results[name]['Test Dice (%)']:5.2f}% | mIoU {test_results[name]['Test mIoU (%)']:5.2f}% | "
          f"HD95 {test_results[name]['HD95 (px)']:6.2f}px | {test_results[name]['Params (M)']:.3f}M | "
          f"{test_results[name]['FLOPs (G)']:.2f}G | {test_results[name]['FPS']:.1f} FPS")

# Persist immediately: do not let a later cell's failure discard these numbers
# (this is exactly how the v4 run lost its benchmark table).
import csv as _csv
with open(os.path.join(OUTPUT_DIR, "benchmark_metrics_independent_test.csv"), "w", newline="",
          encoding="utf-8") as _f:
    _w = _csv.writer(_f)
    _cols = list(next(iter(test_results.values())).keys())
    _w.writerow(["Model"] + _cols)
    for _k, _v in test_results.items():
        _w.writerow([_k] + [_v[c] for c in _cols])
print("[persist] benchmark_metrics_independent_test.csv written")

# Ablation deltas (same protocol, same seed)
if EXTRA_MODELS:
    full = test_results["FEENet"]["Test Dice (%)"]
    print("\n[Ablation] Dice deltas vs. full FEE-Net (same v2 protocol):")
    for name in EXTRA_MODELS:
        d = test_results[name]["Test Dice (%)"] - full
        print(f"  {name:16s} {test_results[name]['Test Dice (%)']:5.2f}%  ({d:+.2f})")


In [ ]:
# Cell 11: Triage Screening & Conformal Risk Control
# Protocol fix (Exp-030 audit): the threshold tau* is calibrated on the VALIDATION
# fold and only *certified* on the held-out test set. Calibrating and reporting on
# the same samples would void the exchangeability assumption that conformal risk
# control requires, so the two roles are strictly separated here.
ee_dev = model_assignments["FEENetEarlyExit"]
ee_model, ee_ckpt = trained["FEENetEarlyExit"]
ee_model.load_state_dict(torch.load(ee_ckpt, map_location=ee_dev))
ee_model.eval()


def triage_risks(split):
    imgs = gpu_datasets[ee_dev][f"{split}_imgs"]
    lbls = gpu_datasets[ee_dev][f"{split}_lbls"]
    out = []
    with torch.no_grad():
        for i in range(0, len(lbls), BATCH_SIZE):
            sl = slice(i, min(i + BATCH_SIZE, len(lbls)))
            # Repository API: risk scores are returned at index 2 when return_edge=True
            # (default forward returns only the mask, unlike the retired notebook copy).
            res = ee_model(imgs[sl], tau=0.5, return_edge=True)
            out.append(res[2].cpu().numpy())
    return np.concatenate(out), lbls.cpu().numpy()


val_risks, val_lbls = triage_risks("val")
test_risks, test_lbls = triage_risks("test")

print(f"[Cohort] positives: val {int(val_lbls.sum())}/{len(val_lbls)} | test {int(test_lbls.sum())}/{len(test_lbls)}")
print("[Cohort] Kvasir-SEG is an all-positive lesion cohort: no healthy-mucosa frames exist,")
print("        so every bypass is a false negative and the normal-bypass rate is structurally 0.")

full_gflops = profiles["FEENetEarlyExit"]["gflops"]
gate_gflops = gate_profile["gate_gflops"] if gate_profile else None


def cp_upper(k, n, conf=0.05):
    # Clopper-Pearson exact upper confidence bound on a binomial rate
    if n == 0:
        return 1.0
    if k >= n:
        return 1.0
    return float(stats.beta.ppf(1 - conf, k + 1, n - k))


# --- calibrate tau* on the validation fold only ---
n_val_pos = int(val_lbls.sum())
val_pos_risks = val_risks[val_lbls == 1]
TARGET_FNR = 0.05
tau_grid = np.linspace(0.01, 0.99, 99)
calibrated_tau, certified_upper = 0.0, 1.0
for tau in tau_grid:
    k = int((val_pos_risks < tau).sum())
    upper = cp_upper(k, n_val_pos)
    if upper <= TARGET_FNR:
        calibrated_tau, certified_upper = float(tau), upper
    else:
        break

# --- certify / report on the held-out test set (single evaluation) ---
n_test_pos = int(test_lbls.sum())
test_pos_risks = test_risks[test_lbls == 1]
test_misses = int((test_pos_risks < calibrated_tau).sum())
test_fnr = test_misses / max(n_test_pos, 1) * 100.0
test_upper = cp_upper(test_misses, n_test_pos)

# Threshold sweep on TEST for reporting only (no calibration happens here)
sweep = []
for tau in np.linspace(0.05, 0.95, 19):
    bypassed = test_risks < tau
    lesion = np.where(test_lbls == 1)[0]
    fnr = float(bypassed[lesion].sum() / max(len(lesion), 1)) * 100.0
    avg_gf = float(np.mean(np.where(bypassed, gate_gflops or 1.45, full_gflops)))
    sweep.append({"tau": round(float(tau), 2), "fnr": round(fnr, 2),
                  "bypass_rate": 0.0, "avg_gflops": round(avg_gf, 2)})

pd.DataFrame(sweep).to_csv(os.path.join(OUTPUT_DIR, "triage_screening_sweep.csv"), index=False)
pd.DataFrame({"split": ["val"] * len(val_risks) + ["test"] * len(test_risks),
              "label": list(val_lbls) + list(test_lbls),
              "risk_score": list(val_risks) + list(test_risks)}
             ).to_csv(os.path.join(OUTPUT_DIR, "triage_risk_scores.csv"), index=False)

print("=" * 68)
print("CONFORMAL RISK CONTROL CALIBRATION (calibrate on val -> certify on test)")
print(f"  target: FNR <= {TARGET_FNR*100:.1f}% @ 95% confidence")
print(f"  calibration set : fold-0 validation, n_positive = {n_val_pos}")
print(f"  calibrated tau* = {calibrated_tau:.4f} (CP upper on val = {certified_upper*100:.2f}%)")
print(f"  certification   : independent test, n_positive = {n_test_pos}")
print(f"  measured FNR    = {test_fnr:.2f}%  (misses {test_misses}/{n_test_pos})")
print(f"  CP upper on test= {test_upper*100:.2f}%  -> guarantee holds: {test_upper <= TARGET_FNR}")
if gate_profile:
    print(f"  measured gate: {gate_profile['gate_gflops']:.2f} GFLOPs @ {gate_profile['gate_fps']:.1f} FPS "
          f"({gate_profile['gate_params_m']:.3f}M params) vs full path {full_gflops:.2f} GFLOPs")
    print(f"  analytic headroom = {full_gflops / max(gate_profile['gate_gflops'], 1e-6):.1f}x "
          f"(screening mode, not a segmentation operating point)")
print("=" * 68)


In [ ]:
# Cell 12: Mixed-Cohort Conformal Screening (Kvasir-SEG is all-positive, so gate
# specificity and real compute saving are unmeasurable there). We derive a
# leakage-free mixed cohort from the HELD-OUT test frames: lesion frames plus
# same-frame mucosal crops, split 75/75 into calibration and certification.
import subprocess  # self-contained: never rely on an earlier cell's imports
KSEG_ROOT = os.path.dirname(IMG_DIR)
MIXED_DIR = "/kaggle/working/mixed_cohort"
SPLITS_LOCAL = "/kaggle/working/data/kvasir_splits.json"
os.makedirs(os.path.dirname(SPLITS_LOCAL), exist_ok=True)
with open(SPLITS_LOCAL, "w", encoding="utf-8") as f:
    json.dump(REPO_SPLITS, f, indent=2)

import subprocess
_env = dict(os.environ)
_env["PYTHONPATH"] = REPO_ROOT + os.pathsep + _env.get("PYTHONPATH", "")
_env["MIXED_COHORT_DIR"] = MIXED_DIR
_env["PROBE_RESULTS_DIR"] = OUTPUT_DIR

# Never let this optional experiment kill the run: the benchmark CSV/figures below
# matter more, and a crash here previously discarded ALL v4 results.
try:
    print("[mixed] building leakage-free mixed cohort from held-out test frames ...")
    r = subprocess.run([sys.executable, os.path.join(REPO_ROOT, "tools", "build_mixed_cohort.py"),
                        "--kseg", KSEG_ROOT, "--splits", SPLITS_LOCAL, "--out", MIXED_DIR],
                       env=_env, capture_output=True, text=True)
    print(r.stdout[-800:]); print(r.stderr[-500:])

    print("[mixed] conformal screening on the mixed cohort ...")
    r = subprocess.run([sys.executable, os.path.join(REPO_ROOT, "tools", "eval_mixed_cohort_gate.py"),
                        "--ckpt", os.path.join(OUTPUT_DIR, "best_feenetearlyexit.pth"),
                        "--device", DEV_LIST[0]],
                       env=_env, capture_output=True, text=True)
    print(r.stdout[-2500:]); print(r.stderr[-800:])
    _mixed_json = os.path.join(OUTPUT_DIR, "mixed_cohort_screening.json")
    if os.path.exists(_mixed_json):
        print("[mixed] summary:", open(_mixed_json, encoding="utf-8").read()[:900])
    else:
        print("[mixed] WARNING: screening result not produced")
except Exception as _e:  # noqa: BLE001
    import traceback
    print(f"[mixed] EXPERIMENT FAILED (non-fatal, continuing): {type(_e).__name__}: {_e}")
    traceback.print_exc()


In [ ]:
# Cell 13: Publication Figures (measured points only)
MAIN = ["UNet", "AttentionUNet", "UNetPlusPlus", "FEENet", "FEENetEarlyExit"]
names = [n for n in MAIN if n in test_results]
gflops = [test_results[n]["FLOPs (G)"] for n in names]
dices = [test_results[n]["Test Dice (%)"] for n in names]
fps_vals = [test_results[n]["FPS"] for n in names]
colors = ["#1f77b4", "#ff7f0e", "#7f7f7f", "#d62728", "#9467bd"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=300)
for i, n in enumerate(names):
    ax1.scatter(gflops[i], dices[i], s=150, c=colors[i], edgecolors="black", linewidth=1.2, zorder=5)
    ax1.annotate(f"{n}\n({gflops[i]:.2f}G, {dices[i]:.2f}%)", (gflops[i] + 0.4, dices[i] - 0.3), fontsize=9, weight="bold")
ax1.set_xlabel("Computation (GFLOPs) [Lower is Better]", fontsize=12, weight="bold")
ax1.set_ylabel("Independent Test Dice (%) [Higher is Better]", fontsize=12, weight="bold")
ax1.set_title("Pareto Frontier: Accuracy vs. Computation", fontsize=13, weight="bold")
ax1.grid(True, linestyle="--", alpha=0.5)

for i, n in enumerate(names):
    ax2.scatter(fps_vals[i], dices[i], s=150, c=colors[i], edgecolors="black", linewidth=1.2, zorder=5)
    ax2.annotate(f"{n}\n({fps_vals[i]:.1f} FPS)", (fps_vals[i] + 2, dices[i] - 0.3), fontsize=9, weight="bold")
ax2.axvline(x=30, color="#2ca02c", linestyle="--", linewidth=1.5, alpha=0.8)
ax2.text(31, min(dices) - 1.5, "Clinical Real-Time (30 FPS)", color="#2ca02c", fontsize=9, weight="bold", rotation=90)
if gate_profile:
    ax2.annotate(
        f"Triage gate (screening mode, measured):\n{gate_profile['gate_gflops']:.2f} GFLOPs @ {gate_profile['gate_fps']:.1f} FPS\n"
        f"no pixel-wise decoding - not a segmentation operating point",
        xy=(0.60, 0.06), xycoords="axes fraction",
        bbox=dict(boxstyle="round,pad=0.4", fc="#f0f8ff", ec="#4682b4", lw=1.2), fontsize=8.5)
ax2.set_xlabel("Throughput (FPS) [Higher is Better]", fontsize=12, weight="bold")
ax2.set_ylabel("Independent Test Dice (%) [Higher is Better]", fontsize=12, weight="bold")
ax2.set_title("Clinical Feasibility: Throughput vs. Accuracy", fontsize=13, weight="bold")
ax2.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figure_pareto_frontier.png"), bbox_inches="tight")
plt.close()

# Qualitative comparison
vis_n = 3
dev0 = model_assignments["FEENet"]
raw_imgs = gpu_datasets[dev0]["test_imgs"][:vis_n].permute(0, 2, 3, 1).cpu().numpy()
gt_masks = gpu_datasets[dev0]["test_msks"][:vis_n].cpu().numpy().squeeze(1)
preds = {}
for n in ["UNet", "AttentionUNet", "UNetPlusPlus", "FEENet"]:
    d = model_assignments[n]
    mdl, ck = trained[n]
    mdl.load_state_dict(torch.load(ck, map_location=d))
    mdl.eval()
    with torch.no_grad():
        p = torch.sigmoid(mdl(gpu_datasets[d]["test_imgs"][:vis_n].to(d)))
    preds[n] = (p > 0.5).cpu().numpy().squeeze(1)

fig, axes = plt.subplots(vis_n, 5, figsize=(15, 7.5), dpi=200)
for r in range(vis_n):
    axes[r, 0].imshow(raw_imgs[r]); axes[r, 0].set_ylabel(f"case {r+1}", fontsize=9)
    axes[r, 1].imshow(gt_masks[r], cmap="gray")
    for c, n in enumerate(["UNet", "AttentionUNet", "UNetPlusPlus", "FEENet"]):
        axes[r, c + 1].imshow(preds[n][r], cmap="gray")
    if r == 0:
        for c, t in enumerate(["Endoscopy", "Ground Truth", "U-Net", "Att-U-Net", "FEE-Net (Ours)"]):
            axes[r, c].set_title(t, fontsize=11, weight="bold")
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figure_qualitative_comparison.png"), bbox_inches="tight")
plt.close()
print("[Figures] pareto + qualitative saved (all plotted points measured end-to-end)")


In [ ]:
# Cell 14: Tables, Provenance Bundle & Packaging
df = pd.DataFrame.from_dict(test_results, orient="index")
for c in ["Params (M)", "FLOPs (G)"]:
    if c in df.columns:
        df[c] = df[c].astype(float)
df.to_csv(os.path.join(OUTPUT_DIR, "benchmark_metrics_independent_test.csv"))
with open(os.path.join(OUTPUT_DIR, "table_colonoscopy_benchmark.tex"), "w") as f:
    f.write(df.to_latex(caption="150-case held-out independent test set (repository splits.json).", label="tab:benchmark"))

print("=" * 78)
print("FINAL BENCHMARK SUMMARY -- 150-CASE INDEPENDENT TEST SET")
print("=" * 78)
print(df.to_markdown())
if gate_profile:
    print(f"\nTriage gate (measured): {gate_profile['gate_gflops']:.2f} GFLOPs @ {gate_profile['gate_fps']:.1f} FPS, "
          f"{gate_profile['gate_params_m']:.3f}M params | CRC tau* = {calibrated_tau:.4f}")

# Provenance: ship the exact repository sources used by this run
prov_dir = os.path.join(OUTPUT_DIR, "repo_provenance")
os.makedirs(prov_dir, exist_ok=True)
for rel, src in list(REPO_SOURCES.items()) + list(REPO_TOOL_SOURCES.items()):
    dst = os.path.join(prov_dir, rel)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    with open(dst, "w", encoding="utf-8") as f:
        f.write(src)
with open(os.path.join(prov_dir, "splits.json"), "w") as f:
    json.dump(REPO_SPLITS, f, indent=2)
with open(os.path.join(prov_dir, "PROVENANCE.txt"), "w") as f:
    f.write("Sources embedded from the repository at notebook-generation time.\n"
            "models imported: models.unet, models.attention_unet, models.unet_plus_plus,\n"
            "               models.fee_net, models.fee_net_early_exit\n"
            "losses/metrics : utils.losses.CompoundLoss, utils.metrics.MetricTracker\n"
            "splits         : data/kvasir_seg/splits.json (Exp-027 v2)\n")

zip_path = "/kaggle/working/feenet_baselines_kaggle_results.zip"
with __import__("zipfile").ZipFile(zip_path, "w", __import__("zipfile").ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUTPUT_DIR):
        for fn in files:
            full = os.path.join(root, fn)
            zf.write(full, arcname=os.path.relpath(full, OUTPUT_DIR))
print(f"[Package] {zip_path}")
print("[Done] benchmark complete")
